# FreightQuote AI — Milestone 2 — Final Deployment

Run the cells in order in a Google Colab CPU runtime. The final cell prints the application link.

**Colab Secrets:** `JWT_SECRET` (32+ characters) and `ADMIN_PASSWORD` (strong, 12–72 bytes) are required. For email OTP, configure `EMAIL_ADDRESS` and `EMAIL_PASSWORD` (Gmail app password). Optional: `ADMIN_EMAIL`, `NGROK_AUTHTOKEN`.

**Product:** A shared login and five-role workspace with Admin account management, Profile, AI Copilot, routes, quotes, carriers, weather, margin and customs. Public signup creates Customer accounts; internal roles must be assigned by an Admin. The original freight agent pages and server-side RBAC remain in place.

**Appearance:** The approved maritime landing page is retained with original copy, navigation and ship photograph. Login, registration, password recovery and dashboards have a consistent responsive theme. Logged-in users may switch light/dark appearance.

**Data and hosting:** Seeded sample records are for demonstrations, not live commercial freight pricing or authoritative customs information. Colab's `/content` filesystem is temporary. Continuous production hosting requires persistent storage, HTTPS, configured email, monitoring and deployment tests.

**Submission:** Remove outputs/secrets before committing to GitHub. Include the role matrix, README and required screenshots according to the Milestone 2 instructions.

**Demo verification:** Run the final cell; it now checks authenticated page responses before presenting the application link. For action failures, inspect the per-request log in `PERSIST_DIR/errors.log` (the UI shows the corresponding request ID).


**Startup recovery (Python 3.13 safe):** The launch cell can restore missing application files from an embedded copy after a Colab runtime reset. Setup and recovery install only missing packages into Colab’s existing Python; they do not create a virtual environment or call `ensurepip`. For the normal workflow use Runtime → Restart session and Run all.


In [ ]:
from pathlib import Path
Path('/content/app/static').mkdir(parents=True, exist_ok=True)
import sys, subprocess, importlib

# Use the Python environment Colab already provides (no virtualenv/ensurepip).
FQ_PYTHON = sys.executable
_FQ_DEPENDENCIES = (
    ('fastapi', 'fastapi>=0.110,<1'),
    ('uvicorn', 'uvicorn>=0.30,<1'),
    ('multipart', 'python-multipart>=0.0.9'),
    ('pandas', 'pandas>=2.1,<3'),
    ('numpy', 'numpy>=1.26,<3'),
    ('sklearn', 'scikit-learn>=1.4,<2'),
    ('jwt', 'PyJWT>=2.8,<3'),
    ('bcrypt', 'bcrypt>=4,<6'),
    ('pyngrok', 'pyngrok>=7,<8'),
    ('requests', 'requests>=2.31,<3'),
    ('scipy', 'scipy>=1.11,<2'),
    ('matplotlib', 'matplotlib>=3.8,<4'),
    ('reportlab', 'reportlab>=4,<5'),
)

def _fq_ensure_dependencies():
    missing = []
    for import_name, requirement in _FQ_DEPENDENCIES:
        try:
            importlib.import_module(import_name)
        except ImportError:
            missing.append(requirement)
    if missing:
        print('Installing missing packages into the existing Colab environment:', ', '.join(missing))
        installation = subprocess.run(
            [FQ_PYTHON, '-m', 'pip', 'install', '-q', '--disable-pip-version-check', *missing],
            stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True,
        )
        if installation.returncode:
            raise RuntimeError(
                'Package installation failed in the Colab environment. '
                'Check internet access and retry the cell. Pip details:\n' +
                installation.stderr[-3000:]
            )
        importlib.invalidate_caches()
        for import_name, _ in _FQ_DEPENDENCIES:
            importlib.import_module(import_name)
    return FQ_PYTHON

_fq_ensure_dependencies()
print('Environment ready (using the Colab Python runtime).')


In [ ]:
%%writefile /content/app/config.py
"""Shared configuration for FreightQuote AI.
Reads required secrets from Colab Secrets (or environment variables).
Fail closed if admin or JWT secrets are missing; never use published credentials.
"""
import os
import secrets as _s

KIND = os.environ.get("APP_KIND", "Freight")
APP_DIR = os.path.dirname(os.path.abspath(__file__))
_in_colab = os.path.isdir("/content") and os.name != "nt"
_default = "/content/app_data" if _in_colab else os.path.join(APP_DIR, "runtime_data")
DATA_DIR = os.environ.get("PERSIST_DIR") or os.environ.get("APP_DATA_DIR") or os.environ.get("FREIGHTQUOTE_DATA_DIR") or _default
os.makedirs(DATA_DIR, exist_ok=True)
_mpl_dir = os.path.join(DATA_DIR, "mpl")
os.makedirs(_mpl_dir, exist_ok=True)
os.environ["MPLCONFIGDIR"] = _mpl_dir
DB_PATH = os.path.join(DATA_DIR, "freight_database.db")

def _secret(*keys, default=""):
    for k in keys:
        try:
            from google.colab import userdata
            v = userdata.get(k)
            if v:
                return str(v).strip()
        except Exception:
            pass
        val = os.environ.get(k)
        if val:
            return str(val).strip()
    return default

JWT_SECRET = _secret("JWT_SECRET", "JWT_SECRET_KEY")
if len(JWT_SECRET) < 32:
    raise RuntimeError("Set JWT_SECRET to at least 32 characters in Colab Secrets or the environment.")
ADMIN_EMAIL = _secret("ADMIN_EMAIL", "ADMIN_EMAIL_ID", default="admin@infosys.com")
ADMIN_PASSWORD = _secret("ADMIN_PASSWORD", default="")
if not 12 <= len(ADMIN_PASSWORD.encode("utf-8")) <= 72:
    raise RuntimeError("Set a unique ADMIN_PASSWORD (12–72 UTF-8 bytes) in Colab Secrets or the environment.")

EMAIL_ADDRESS = _secret("EMAIL_ADDRESS", "EMAIL_ID", "SENDER_EMAIL", default="")
EMAIL_PASSWORD = _secret("EMAIL_PASSWORD", "EMAIL_APP_PASSWORD", default="").replace(" ", "")
NGROK_AUTHTOKEN = _secret("NGROK_AUTHTOKEN", default="")
HF_TOKEN = _secret("HF_TOKEN", "HUGGINGFACE_TOKEN", default="")


In [ ]:
%%writefile /content/app/db.py
import sqlite3, os, threading
from config import DB_PATH

# A single, long-lived connection reused across the whole app process
# instead of opening (and never explicitly closing) a brand-new SQLite
# connection + re-running PRAGMA setup on every single call. That old
# pattern was leaking file handles under load and was the single
# biggest source of slowness/lock contention across the app.
_conn = None
_conn_lock = threading.Lock()
_transaction_lock = threading.RLock()
class SerializedConnection(sqlite3.Connection):
    def __enter__(self):
        _transaction_lock.acquire()
        try:
            self.execute("BEGIN IMMEDIATE")
            return self
        except Exception:
            _transaction_lock.release()
            raise
    def __exit__(self, *args):
        try:
            return super().__exit__(*args)
        finally:
            _transaction_lock.release()


def get_conn():
    global _conn
    if _conn is None:
        with _conn_lock:
            if _conn is None:
                os.makedirs(os.path.dirname(DB_PATH), exist_ok=True)
                _conn = sqlite3.connect(DB_PATH, timeout=30, check_same_thread=False, factory=SerializedConnection)
                _conn.execute("PRAGMA journal_mode=WAL;")
                _conn.execute("PRAGMA synchronous=NORMAL;")
                _conn.execute("PRAGMA cache_size=-64000;")   # ~64MB page cache
                _conn.execute("PRAGMA temp_store=MEMORY;")
    return _conn

def save_chat_message(username, role, message):
    try:
        with get_conn() as conn:
            conn.execute("INSERT INTO chat_history (username, role, message) VALUES (?, ?, ?);", (username, role, message))
            conn.commit()
    except Exception: pass

def load_chat_history(username=None, limit=100):
    try:
        with get_conn() as conn:
            cur = conn.cursor()
            if username:
                rows = cur.execute("SELECT role, message FROM chat_history WHERE username=? ORDER BY id ASC LIMIT ?;", (username, limit)).fetchall()
            else:
                rows = cur.execute("SELECT role, message FROM chat_history ORDER BY id ASC LIMIT ?;", (limit,)).fetchall()

            res = []
            for r in rows:
                content_val = str(r[1] or "")
                res.append({
                    "role": str(r[0] or "assistant"),
                    "content": content_val,
                    "message": content_val
                })
            return res
    except Exception: return []

def clear_chat_history(username=None):
    try:
        with get_conn() as conn:
            if username:
                conn.execute("DELETE FROM chat_history WHERE username=?;", (username,))
            else:
                conn.execute("DELETE FROM chat_history;")
            conn.commit()
    except Exception: pass

def init_db():
    with get_conn() as conn:
        conn.execute("""
        CREATE TABLE IF NOT EXISTS chat_history (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            username TEXT,
            role TEXT,
            message TEXT,
            timestamp DATETIME DEFAULT CURRENT_TIMESTAMP
        );
        """)
        conn.execute("""
        CREATE TABLE IF NOT EXISTS users (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            username TEXT UNIQUE,
            email TEXT UNIQUE,
            password_hash TEXT,
            security_question TEXT,
            security_answer_hash TEXT,
            role TEXT,
            failed_attempts INTEGER DEFAULT 0,
            lock_until TIMESTAMP DEFAULT NULL,
            account_status TEXT DEFAULT 'active',
            created_at DATETIME DEFAULT CURRENT_TIMESTAMP
        );
        """)
        # Authentication columns and tables — safe migrations for existing databases.
        for _col, _ddl in [
            ("username", "ALTER TABLE users ADD COLUMN username TEXT"),
            ("security_question", "ALTER TABLE users ADD COLUMN security_question TEXT"),
            ("security_answer_hash", "ALTER TABLE users ADD COLUMN security_answer_hash TEXT"),
            ("failed_attempts", "ALTER TABLE users ADD COLUMN failed_attempts INTEGER DEFAULT 0"),
            ("lock_until", "ALTER TABLE users ADD COLUMN lock_until TIMESTAMP DEFAULT NULL"),
            ("account_status", "ALTER TABLE users ADD COLUMN account_status TEXT DEFAULT 'active'"),
            ("profile_picture", "ALTER TABLE users ADD COLUMN profile_picture TEXT DEFAULT NULL"),
            ("locked", "ALTER TABLE users ADD COLUMN locked INTEGER NOT NULL DEFAULT 0"),
            ("last_login", "ALTER TABLE users ADD COLUMN last_login TEXT DEFAULT NULL"),
            ("customer_id", "ALTER TABLE users ADD COLUMN customer_id TEXT DEFAULT NULL"),
            ("session_version", "ALTER TABLE users ADD COLUMN session_version INTEGER NOT NULL DEFAULT 0"),
            ("email_verified", "ALTER TABLE users ADD COLUMN email_verified INTEGER NOT NULL DEFAULT 1"),
        ]:
            try:
                conn.execute(_ddl)
            except Exception:
                pass

        conn.execute("""
        CREATE TABLE IF NOT EXISTS password_history (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            user_id INTEGER NOT NULL,
            password_hash TEXT NOT NULL,
            created_at TEXT NOT NULL
        );
        """)
        conn.execute("""
        CREATE TABLE IF NOT EXISTS limits (
            subject TEXT PRIMARY KEY,
            attempts INTEGER NOT NULL DEFAULT 0,
            window_start INTEGER NOT NULL DEFAULT 0,
            locked_until INTEGER NOT NULL DEFAULT 0,
            ladder_step INTEGER NOT NULL DEFAULT 0
        );
        """)
        conn.execute("""
        CREATE TABLE IF NOT EXISTS resets (
            email TEXT PRIMARY KEY COLLATE NOCASE,
            code_hash TEXT NOT NULL,
            salt TEXT NOT NULL,
            expires_at INTEGER NOT NULL,
            attempts INTEGER NOT NULL DEFAULT 0,
            sent_at INTEGER NOT NULL
        );
        """)
        conn.execute("""
        CREATE TABLE IF NOT EXISTS otp_policy (
            email TEXT NOT NULL COLLATE NOCASE,
            purpose TEXT NOT NULL,
            requests INTEGER NOT NULL DEFAULT 0,
            next_request_at INTEGER NOT NULL DEFAULT 0,
            failures INTEGER NOT NULL DEFAULT 0,
            locked_until INTEGER NOT NULL DEFAULT 0,
            ladder_step INTEGER NOT NULL DEFAULT 0,
            PRIMARY KEY(email, purpose)
        );
        """)
        conn.execute("""
        CREATE TABLE IF NOT EXISTS email_verification_codes (
            email TEXT PRIMARY KEY COLLATE NOCASE,
            code_hash TEXT NOT NULL,
            salt TEXT NOT NULL,
            expires_at INTEGER NOT NULL
        );
        """)
        conn.execute("""
        CREATE TABLE IF NOT EXISTS reset_grants (
            email TEXT PRIMARY KEY COLLATE NOCASE,
            token_hash TEXT NOT NULL,
            expires_at INTEGER NOT NULL,
            version INTEGER NOT NULL DEFAULT 0
        );
        """)
        conn.execute("""
        CREATE TABLE IF NOT EXISTS audit (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            actor TEXT NOT NULL,
            event TEXT NOT NULL,
            occurred_at TEXT NOT NULL
        );
        """)
        conn.execute("""
        CREATE TABLE IF NOT EXISTS admin_activity_log (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            actor TEXT NOT NULL,
            action TEXT NOT NULL,
            details TEXT,
            timestamp TEXT NOT NULL
        );
        """)

        # Preserve older Stage 5/Milestone 1 admin-activity tables during upgrades.
        # CREATE TABLE IF NOT EXISTS does not migrate an existing table.
        _activity_columns = {r[1] for r in conn.execute("PRAGMA table_info(admin_activity_log)")}
        for _name, _ddl in (
            ("actor", "ALTER TABLE admin_activity_log ADD COLUMN actor TEXT NOT NULL DEFAULT 'system'"),
            ("action", "ALTER TABLE admin_activity_log ADD COLUMN action TEXT NOT NULL DEFAULT ''"),
            ("details", "ALTER TABLE admin_activity_log ADD COLUMN details TEXT"),
            ("timestamp", "ALTER TABLE admin_activity_log ADD COLUMN timestamp TEXT NOT NULL DEFAULT ''"),
        ):
            if _name not in _activity_columns:
                conn.execute(_ddl)

        conn.execute("""
        CREATE TABLE IF NOT EXISTS alerts (
            alert_id INTEGER PRIMARY KEY AUTOINCREMENT,
            shipment_id TEXT,
            outlet_id TEXT,
            severity TEXT,
            category TEXT,
            message TEXT,
            date TEXT,
            resolved INT DEFAULT 0
        );
        """)
        conn.execute("""
        CREATE TABLE IF NOT EXISTS shipments (
            shipment_id TEXT PRIMARY KEY,
            origin_port TEXT,
            dest_port TEXT,
            carrier TEXT,
            status TEXT,
            eta TEXT,
            weight_kg REAL,
            hs_code TEXT,
            predicted_delay_risk REAL
        );
        """)
        conn.execute("""
        CREATE TABLE IF NOT EXISTS freight_quotes (
            quote_id TEXT PRIMARY KEY,
            shipment_id TEXT,
            customer_id TEXT,
            carrier TEXT,
            base_cost REAL,
            insurance REAL,
            customs_fee REAL,
            fuel_surcharge REAL,
            final_price REAL,
            margin_pct REAL,
            status TEXT,
            created_at TEXT
        );
        """)
        conn.execute("""
        CREATE TABLE IF NOT EXISTS ports (
            port_id TEXT PRIMARY KEY,
            port_name TEXT,
            country TEXT,
            congestion_index REAL,
            avg_dwell_days REAL,
            lat REAL,
            lon REAL,
            region TEXT
        );
        """)
        conn.execute("""
        CREATE TABLE IF NOT EXISTS carriers (
            carrier_id TEXT PRIMARY KEY,
            name TEXT,
            rating REAL,
            on_time_pct REAL,
            avg_cost_index REAL,
            risk_level TEXT
        );
        """)
        conn.execute("""
        CREATE TABLE IF NOT EXISTS customers (
            customer_id TEXT PRIMARY KEY,
            name TEXT,
            industry TEXT,
            priority_tier TEXT,
            credit_risk REAL
        );
        """)
        conn.execute("""
        CREATE TABLE IF NOT EXISTS ml_metrics (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            module TEXT,
            model_name TEXT,
            metric_name TEXT,
            metric_value REAL,
            trained_at DATETIME DEFAULT CURRENT_TIMESTAMP
        );
        """)
        conn.execute("""
        CREATE TABLE IF NOT EXISTS outlets (
            outlet_id TEXT PRIMARY KEY,
            outlet_name TEXT,
            location TEXT,
            tier TEXT,
            revenue REAL,
            operating_costs REAL,
            customer_satisfaction REAL,
            staff_headcount INT
        );
        """)
        conn.execute("""
        CREATE TABLE IF NOT EXISTS staff (
            staff_id TEXT PRIMARY KEY,
            outlet_id TEXT,
            name TEXT,
            role TEXT,
            salary REAL,
            overtime_hrs REAL,
            job_satisfaction INT,
            age INT,
            tenure_years INT,
            work_life_balance INT,
            predicted_attrition_prob REAL
        );
        """)
        conn.execute("""
        CREATE TABLE IF NOT EXISTS inventory (
            record_id TEXT PRIMARY KEY,
            outlet_id TEXT,
            sku_name TEXT,
            category TEXT,
            current_stock INT,
            reorder_threshold INT,
            weekly_demand REAL,
            lead_time_days INT,
            stockout_risk_prob REAL
        );
        """)
        conn.execute("""
        CREATE TABLE IF NOT EXISTS marketing (
            campaign_id TEXT PRIMARY KEY,
            outlet_id TEXT,
            campaign_name TEXT,
            channel TEXT,
            budget REAL,
            actual_roi REAL,
            reach INT,
            conversions INT,
            start_date TEXT,
            end_date TEXT
        );
        """)
        conn.execute("""
        CREATE TABLE IF NOT EXISTS feedback (
            feedback_id TEXT PRIMARY KEY,
            outlet_id TEXT,
            rating INT,
            comment TEXT,
            date TEXT,
            sentiment_score REAL
        );
        """)
        conn.execute("""
        CREATE TABLE IF NOT EXISTS audits (
            audit_id TEXT PRIMARY KEY,
            outlet_id TEXT,
            audit_date TEXT,
            score REAL,
            violations INT,
            category TEXT,
            status TEXT,
            notes TEXT
        );
        """)
        conn.execute("""
        CREATE TABLE IF NOT EXISTS weather_risks (
            port_name TEXT PRIMARY KEY,
            current_severity INT,
            forecast TEXT,
            wind_speed REAL,
            wave_height REAL,
            temperature REAL
        );
        """)
        conn.execute("""
        CREATE TABLE IF NOT EXISTS customs_rules (
            chapter TEXT PRIMARY KEY,
            goods TEXT,
            duty_pct REAL,
            docs TEXT,
            restricted INT
        );
        """)
        conn.execute("""
        CREATE TABLE IF NOT EXISTS vessels (
            shipment_id TEXT PRIMARY KEY,
            lat REAL,
            lon REAL,
            progress REAL,
            chain TEXT,
            route_name TEXT,
            speed_kn REAL,
            updated TEXT
        );
        """)
        conn.execute("""
        CREATE TABLE IF NOT EXISTS decisions (
            decision_id INTEGER PRIMARY KEY AUTOINCREMENT,
            ts TEXT,
            score REAL,
            risk TEXT,
            narrative TEXT
        );
        """)
        conn.commit()

In [ ]:
%%writefile /content/app/mailer.py
"""Mailer utility for FreightQuote AI.
Sends emails (OTP codes, alerts) via SMTP using Gmail app password.
"""
import ssl
import smtplib
from email.message import EmailMessage
import config

def creds():
    return config.EMAIL_ADDRESS, config.EMAIL_PASSWORD

def configured():
    em, pw = creds()
    return bool(em and pw)

def send(to, subject, body, attachments=(), html=None):
    """Send an email. Returns (ok, message). Never prints secrets or codes."""
    em, pw = creds()
    if not (em and pw):
        return False, "Email OTP is not configured, use the security question"
    try:
        m = EmailMessage()
        m["Subject"] = subject
        m["From"] = em
        m["To"] = to
        m.set_content(body)
        if html:
            m.add_alternative(html, subtype="html")
        for fn, data, mt in attachments:
            m.add_attachment(data, maintype=mt.split("/")[0], subtype=mt.split("/")[1], filename=fn)
        with smtplib.SMTP_SSL("smtp.gmail.com", 465, context=ssl.create_default_context(), timeout=20) as s:
            s.login(em, pw)
            s.send_message(m)
        return True, f"Sent to {to}"
    except Exception as e:
        return False, "Email could not be sent. Check the sender configuration and retry later."


def otp_html(username, code):
    from html import escape
    return f"""<!doctype html><html><body style="margin:0;background:#eef5f3;font-family:Arial,sans-serif;color:#173b3d"><table role="presentation" width="100%" cellpadding="24"><tr><td align="center"><table role="presentation" width="100%" style="max-width:520px;background:white;border-radius:16px" cellpadding="32"><tr><td><p style="color:#137e78;font-weight:bold">FreightQuote AI</p><h1 style="font-size:26px">Reset your password</h1><p>Hello {escape(str(username))},</p><p>Enter this code to continue your password reset.</p><p style="background:#edf7f4;padding:22px;text-align:center;font-size:32px;letter-spacing:8px;font-weight:bold">{escape(str(code))}</p><p>This code expires in <strong>5 minutes</strong>. Do not share it with anyone.</p><p style="color:#607674;font-size:13px">If you did not request a reset, you can ignore this email. Your password has not changed.</p></td></tr></table></td></tr></table></body></html>"""


In [ ]:
%%writefile /content/app/rbac.py
"""Role-Based Access Control (RBAC) for FreightQuote AI.
Defines page access and capabilities per role as required by Milestone 2 Step 5.
"""

# Pages that each role may see in navigation and open via /api/page/{pid}
ROLE_MENU = {
    "Admin": {
        "copilot", "route", "quotes", "carriers", "weather", "margin", "customs", "profile", "admin"
    },
    "Operations Manager": {
        "copilot", "route", "carriers", "weather", "customs", "profile"
    },
    "Freight Broker": {
        "copilot", "quotes", "carriers", "margin", "profile"
    },
    "Auditor": {
        "copilot", "route", "quotes", "carriers", "weather", "margin", "customs", "profile"
    },
    "Customer": {
        "copilot", "quotes", "profile"
    },
}

# Granular capabilities for action routes and page functions
ROLE_CAPS = {
    "Admin": {
        "view", "copilot", "route", "quotes", "carriers", "weather", "margin", "customs",
        "profile", "admin", "users", "run", "reroute", "storm", "quote", "quote_create", "customs_edit"
    },
    "Operations Manager": {
        "view", "copilot", "route", "carriers", "weather", "customs", "profile",
        "run", "reroute", "storm", "customs_edit"
    },
    "Freight Broker": {
        "view", "copilot", "quotes", "carriers", "margin", "profile",
        "quote", "quote_create"
    },
    "Auditor": {
        "view", "copilot", "route", "quotes", "carriers", "weather", "margin", "customs", "profile"
        # Strictly read-only: no run, reroute, storm, quote, customs_edit, or users
    },
    "Customer": {
        "view", "copilot", "quotes", "profile"
        # Scoped to own customer_id
    },
}

ALL_MENU_ITEMS = [
    ("copilot", "AI Copilot", "bot"),
    ("route", "Agent 1 · Route AI", "route"),
    ("quotes", "Agent 2 · Spot Quotes", "dollar"),
    ("carriers", "Agent 3 · Carriers", "truck"),
    ("weather", "Agent 4 · Weather Risk", "cloud"),
    ("margin", "Agent 5 · Margin", "trend"),
    ("customs", "Agent 6 · Customs", "file"),
    ("profile", "My Profile", "user"),
    ("admin", "Admin Dashboard", "shield"),
]

def menu_for_role(role):
    allowed_ids = ROLE_MENU.get(role, set())
    return [{"id": item[0], "label": item[1], "icon": item[2]} for item in ALL_MENU_ITEMS if item[0] in allowed_ids]

def check_permission(role, capability):
    if not role:
        return False
    return capability in ROLE_CAPS.get(role, set())

def can_access_page(role, page_id):
    if not role:
        return False
    return page_id in ROLE_MENU.get(role, set())

def can_access_entity(role, eid, customer_id=None):
    if not role:
        return False
    if role in ("Admin", "Auditor"):
        return True
    eid = str(eid)
    if eid.startswith("QTE"):
        if role == "Freight Broker":
            return True
        if role == "Customer":
            return bool(customer_id)
        return False
    if eid.startswith("SHP"):
        if role == "Operations Manager":
            return True
        if role == "Customer":
            return bool(customer_id)
        return False
    # Ports
    if role == "Operations Manager":
        return True
    return False


In [ ]:
%%writefile /content/app/auth_service.py
"""Authentication, JWT sessions, lockout ladders and password policies for FreightQuote AI.
Ported directly from Milestone 1 with exact behavior, limits, and wording.
"""
import re
import json
import time
import hmac
import hashlib
import secrets
import datetime as dt
import unicodedata

try:
    import bcrypt
    _HAS_BCRYPT = True
except ImportError:
    _HAS_BCRYPT = False

try:
    import jwt
    _HAS_JWT = True
except ImportError:
    _HAS_JWT = False

from db import get_conn
import config
import mailer

EMAIL_RE = re.compile(
    r"^[A-Za-z]{2,}[A-Za-z0-9._%+\-]*@[A-Za-z]{2,}[A-Za-z0-9\-]*(?:\.[A-Za-z]{2,}[A-Za-z0-9\-]*)+$"
)
QUESTIONS = [
    "What was the name of your first school?",
    "In which city were you born?",
    "What was the name of your first pet?",
    "What is the middle name of your oldest sibling?",
]
GENERIC_LOGIN_ERROR = "Invalid username/email or password."
LOCKED_MESSAGE = "Account locked. Contact the administrator to unlock it."
OTP_TTL = 300  # 5 minutes
MAX_OTP_REQUESTS = 5
MAX_OTP_FAILURES = 5

ALLOWED_SIGNUP_ROLES = ["Customer"]

def utcnow():
    return dt.datetime.now(dt.timezone.utc).isoformat()

def valid_email(value):
    return (
        isinstance(value, str)
        and len(value.strip()) <= 254
        and bool(EMAIL_RE.fullmatch(value.strip()))
    )

def password_errors(password):
    issues = []
    if len(password) < 8:
        issues.append("at least 8 characters")
    if len(password.encode("utf-8")) > 72:
        issues.append("no more than 72 UTF-8 bytes")
    if not re.search(r"[A-Z]", password):
        issues.append("one uppercase letter")
    if not re.search(r"[a-z]", password):
        issues.append("one lowercase letter")
    if not re.search(r"[0-9]", password):
        issues.append("one number")
    if not any(
        not ch.isalnum()
        and not ch.isspace()
        and not unicodedata.category(ch).startswith("C")
        for ch in password
    ):
        issues.append("one special symbol (spaces do not count)")
    if any(ch.isspace() or unicodedata.category(ch).startswith("C") for ch in password):
        issues.append("no whitespace or invisible characters")
    return issues

def clean_answer(value):
    """Ignore case and surrounding whitespace; reject invisible-only answers."""
    value = unicodedata.normalize("NFKC", value or "")
    value = "".join(ch for ch in value if unicodedata.category(ch) != "Cf")
    return value.strip().casefold()

def answer_error(value):
    cleaned = clean_answer(value)
    if not cleaned or not any(ch.isalnum() for ch in cleaned):
        return "Enter a security answer containing a letter or number."
    if len(cleaned.encode("utf-8")) > 72:
        return "Security answer is too long (maximum 72 UTF-8 bytes)."
    return None

def hashed(text):
    if _HAS_BCRYPT:
        try:
            return bcrypt.hashpw(text.encode("utf-8"), bcrypt.gensalt()).decode("ascii")
        except Exception:
            pass
    salt = secrets.token_hex(16)
    dk = hashlib.pbkdf2_hmac("sha256", text.encode("utf-8"), salt.encode("ascii"), 100000).hex()
    return f"pbkdf2:{salt}:{dk}"

def matches(text, saved):
    if not saved or not isinstance(saved, str):
        return False
    if saved.startswith("$2"):
        if _HAS_BCRYPT:
            try:
                return bcrypt.checkpw(text.encode("utf-8"), saved.encode("ascii"))
            except Exception:
                return False
        return False
    if saved.startswith("pbkdf2:"):
        try:
            parts = saved.split(":")
            if len(parts) == 3:
                salt = parts[1]
                expected_dk = parts[2]
                computed = hashlib.pbkdf2_hmac("sha256", text.encode("utf-8"), salt.encode("ascii"), 100000).hex()
                return hmac.compare_digest(computed, expected_dk)
        except Exception:
            return False
    return False

def otp_digest(code, salt, secret=None, purpose="reset"):
    sec = secret or config.JWT_SECRET
    msg = f"{purpose}:{salt}:{code}".encode("utf-8")
    return hmac.new(sec.encode("utf-8"), msg, hashlib.sha256).hexdigest()

def _b64url_encode(data: bytes) -> str:
    import base64
    return base64.urlsafe_b64encode(data).decode("ascii").rstrip("=")

def _b64url_decode(s: str) -> bytes:
    import base64
    rem = len(s) % 4
    if rem:
        s += "=" * (4 - rem)
    return base64.urlsafe_b64decode(s)

def issue_token(user, expires_in=7200):
    now = int(time.time())
    payload = {
        "sub": user["username"],
        "email": user["email"],
        "role": user["role"],
        "customer_id": user.get("customer_id"),
        "iat": now,
        "exp": now + expires_in,
        "jti": secrets.token_hex(16),
        "uid": user["id"],
        "sv": session_version(user["id"]),
    }
    if _HAS_JWT:
        try:
            return jwt.encode(payload, config.JWT_SECRET, algorithm="HS256")
        except Exception:
            pass
    h_bytes = json.dumps({"alg": "HS256", "typ": "JWT"}, separators=(",", ":")).encode("utf-8")
    p_bytes = json.dumps(payload, separators=(",", ":")).encode("utf-8")
    signing_input = f"{_b64url_encode(h_bytes)}.{_b64url_encode(p_bytes)}"
    sig = hmac.new(config.JWT_SECRET.encode("utf-8"), signing_input.encode("ascii"), hashlib.sha256).digest()
    return f"{signing_input}.{_b64url_encode(sig)}"

def decode_token(token):
    if not token or not isinstance(token, str):
        return None
    if _HAS_JWT:
        try:
            return jwt.decode(token, config.JWT_SECRET, algorithms=["HS256"])
        except Exception:
            pass
    try:
        parts = token.split(".")
        if len(parts) != 3:
            return None
        signing_input = f"{parts[0]}.{parts[1]}"
        sig = _b64url_decode(parts[2])
        expected_sig = hmac.new(config.JWT_SECRET.encode("utf-8"), signing_input.encode("ascii"), hashlib.sha256).digest()
        if not hmac.compare_digest(sig, expected_sig):
            return None
        payload = json.loads(_b64url_decode(parts[1]).decode("utf-8"))
        if payload.get("exp") and time.time() > payload["exp"]:
            return None
        return payload
    except Exception:
        return None

# ------------------------------------------------------------------ User Repository
def get_user_by_id(uid):
    with get_conn() as conn:
        c = conn.cursor()
        c.execute("SELECT id, username, email, password_hash, role, locked, created_at, last_login, security_question, security_answer_hash, customer_id FROM users WHERE id=?", (uid,))
        row = c.fetchone()
        if not row:
            return None
        return {
            "id": row[0], "username": row[1], "email": row[2], "password_hash": row[3],
            "role": row[4], "locked": row[5], "created_at": row[6], "last_login": row[7],
            "security_question": row[8], "security_answer_hash": row[9], "customer_id": row[10],
        }

def get_user(identity):
    if not identity:
        return None
    ident = identity.strip()
    with get_conn() as conn:
        c = conn.cursor()
        c.execute(
            """SELECT id, username, email, password_hash, role, locked, created_at, last_login, security_question, security_answer_hash, customer_id
               FROM users WHERE username=? COLLATE NOCASE OR email=? COLLATE NOCASE LIMIT 1""",
            (ident, ident),
        )
        row = c.fetchone()
        if not row:
            return None
        return {
            "id": row[0], "username": row[1], "email": row[2], "password_hash": row[3],
            "role": row[4], "locked": row[5], "created_at": row[6], "last_login": row[7],
            "security_question": row[8], "security_answer_hash": row[9], "customer_id": row[10],
        }

def audit(actor, event):
    try:
        with get_conn() as conn:
            conn.execute("INSERT INTO audit(actor, event, occurred_at) VALUES(?,?,?)", (str(actor)[:100], event, utcnow()))
            conn.commit()
    except Exception:
        pass

# ------------------------------------------------------------------ Progressive Lockout Ladder
# Login ladder: 3 wrong -> 120s, 4 wrong -> 300s, 5 wrong -> Admin-only unlock.
def check_login_lockout(user_or_identity):
    """Password failures are counted directly; only the administrator can unlock accounts.

    Resend cooldowns are managed separately by the OTP subsystem. Previous
    login cooldown entries must not prevent the fifth password attempt.
    """
    if isinstance(user_or_identity, dict):
        user = user_or_identity
    else:
        user = get_user(str(user_or_identity or "").strip().casefold())
    if user and user.get("locked"):
        return True, None, LOCKED_MESSAGE
    return False, 0, None


def record_login_failure(identity):
    ident = str(identity or "unknown").strip().casefold()
    user = get_user(ident) if ident != "unknown" else None
    if user and user.get("locked"):
        return True, None, LOCKED_MESSAGE

    now = int(time.time())
    subject = f"login:user:{user['id']}" if user else f"login:anon:{ident}"
    with get_conn() as conn:
        row = conn.execute("SELECT attempts FROM limits WHERE subject=?", (subject,)).fetchone()
        attempts = (int(row[0]) if row else 0) + 1
        # Retain the existing table for backward compatibility, but no cooldown
        # blocks password attempts 3 and 4. The fifth failure locks the account.
        conn.execute(
            "INSERT OR REPLACE INTO limits(subject,attempts,window_start,locked_until,ladder_step) VALUES(?,?,?,?,?)",
            (subject, attempts, now, 0, 0),
        )
        if attempts >= 5 and user:
            conn.execute("UPDATE users SET locked=1 WHERE id=?", (user["id"],))
        conn.commit()

    if attempts >= 5 and user:
        audit(user.get("username") or user.get("email") or "unknown", "account_locked_login_attempts")
        return True, None, LOCKED_MESSAGE
    # Keep identical error wording for unknown accounts and bad passwords.
    return False, 0, GENERIC_LOGIN_ERROR


def clear_login_failure(user):
    subject = f"login:user:{user['id']}"
    with get_conn() as conn:
        conn.execute("DELETE FROM limits WHERE subject=?", (subject,))
        conn.commit()

# ------------------------------------------------------------------ OTP Flow & Limits from File 1
def check_otp_lockout(email):
    now = int(time.time())
    u = get_user(email)
    if u and u.get("locked"):
        return True, None, LOCKED_MESSAGE

    with get_conn() as conn:
        c = conn.cursor()
        c.execute("SELECT requests, next_request_at, failures, locked_until, ladder_step FROM otp_policy WHERE email=? AND purpose='reset'",
                  (email.strip().casefold(),))
        row = c.fetchone()
        if not row:
            return False, 0, None
        requests, next_request_at, failures, locked_until, ladder_step = row
        if locked_until > now:
            retry_after = locked_until - now
            return True, retry_after, f"Request another code in {retry_after} seconds."
        return False, 0, None

def record_otp_failure(email):
    now = int(time.time())
    em = email.strip().casefold()
    u = get_user(em)
    with get_conn() as conn:
        c = conn.cursor()
        c.execute("SELECT requests, next_request_at, failures, locked_until, ladder_step FROM otp_policy WHERE email=? AND purpose='reset'", (em,))
        row = c.fetchone()
        if not row:
            requests, next_request_at, failures, locked_until, ladder_step = 0, 0, 1, 0, 0
        else:
            requests, next_request_at, failures, locked_until, ladder_step = row[0], row[1], row[2] + 1, row[3], row[4]

        # File 1 rule: failures >= MAX_OTP_FAILURES (5) -> account locked
        if failures >= MAX_OTP_FAILURES:
            locked_until = now + 315360000
            if u:
                c.execute("UPDATE users SET locked=1 WHERE id=?", (u["id"],))
                audit(u["username"], "account_locked_otp_failures")
        else:
            locked_until = 0

        c.execute("INSERT OR REPLACE INTO otp_policy(email, purpose, requests, next_request_at, failures, locked_until, ladder_step) VALUES(?,?,?,?,?,?,?)",
                  (em, "reset", requests, next_request_at, failures, locked_until, ladder_step))
        conn.commit()

    if failures >= MAX_OTP_FAILURES:
        return True, None, LOCKED_MESSAGE
    return False, 0, "Invalid or expired code."

# ------------------------------------------------------------------ Signup & Login Actions
def signup_user(username, email, password, confirm_password, question, answer, role="Customer"):
    username = (username or "").strip()
    email = (email or "").strip().lower()
    role = "Customer"  # Only administrators assign staff roles.

    if role not in ALLOWED_SIGNUP_ROLES:
        return False, "Invalid role. Allowed roles: Customer, Freight Broker, Operations Manager, Auditor."

    if not re.fullmatch(r"[\w.\-]{3,50}", username, re.UNICODE):
        return False, "Username must be 3–50 letters, digits, dots, dashes, or underscores."

    if valid_email(username):
        return False, "Username cannot be formatted as an email address."

    if not valid_email(email):
        return False, "Enter a valid email address."

    if password != confirm_password:
        return False, "Password and confirmation do not match."

    errs = password_errors(password)
    if errs:
        return False, "Password does not meet requirements: " + ", ".join(errs)

    if question not in QUESTIONS:
        return False, "Choose a security question from the provided list."

    a_err = answer_error(answer)
    if a_err:
        return False, a_err

    if not mailer.configured():
        return False, "Email verification is unavailable. Ask your administrator to configure EMAIL_ADDRESS and EMAIL_PASSWORD in Colab Secrets."

    # Check case-insensitive uniqueness
    with get_conn() as conn:
        c = conn.cursor()
        c.execute("SELECT id FROM users WHERE username=? COLLATE NOCASE", (username,))
        if c.fetchone():
            return False, "That username is already taken. Please choose another."
        c.execute("SELECT id FROM users WHERE email=? COLLATE NOCASE", (email,))
        if c.fetchone():
            return False, "An account with that email already exists."

        pw_hash = hashed(password)
        ans_hash = hashed(clean_answer(answer))
        now = utcnow()
        c.execute(
            """INSERT INTO users(username, email, password_hash, role, locked, created_at, last_login, security_question, security_answer_hash, customer_id, email_verified)
               VALUES(?,?,?,?,0,?,NULL,?,?,NULL,0)""",
            (username, email, pw_hash, role, now, question, ans_hash),
        )
        uid = c.lastrowid
        c.execute("INSERT INTO password_history(user_id, password_hash, created_at) VALUES(?,?,?)", (uid, pw_hash, now))
        conn.commit()

    audit(username, "signup_success")
    return True, "Account created. Verify your email before signing in."

def authenticate_user(identity, password, expected_role=None):
    ident = (identity or "").strip()
    pw = password or ""
    if not ident or not pw:
        return None, 0, GENERIC_LOGIN_ERROR

    user = get_user(ident)
    is_locked, retry_after, lock_msg = check_login_lockout(user or ident)
    if is_locked:
        return None, retry_after or 0, lock_msg

    if not user:
        is_l, ret_s, msg = record_login_failure(ident)
        audit(ident[:100], "login_failed_unknown_user")
        return None, ret_s, msg

    if user.get("locked"):
        return None, 0, LOCKED_MESSAGE

    if not matches(pw, user["password_hash"]):
        u_name = user.get("username") or user.get("email") or ident
        is_l, ret_s, msg = record_login_failure(u_name)
        audit(u_name, "login_failed_wrong_password")
        return None, ret_s, msg

    # Role selection is a sign-in confirmation, not a role assignment.
    # Verify only after the correct password to avoid leaking role metadata.
    if expected_role is not None:
        from rbac import ROLE_CAPS
        if expected_role not in ROLE_CAPS:
            return None, 0, "Select your assigned role to sign in."
        if expected_role != user["role"]:
            audit(user.get("username") or ident, "login_failed_role_mismatch")
            return None, 0, "Selected role does not match your account. Choose your assigned role."

    # Incorrect passwords are counted before disclosing verification status.
    with get_conn() as conn:
        verified = conn.execute("SELECT email_verified FROM users WHERE id=?", (user["id"],)).fetchone()
    if verified and not verified[0]:
        return None, 0, "Email verification required. Check your inbox or request a new code."

    clear_login_failure(user)
    now = utcnow()
    with get_conn() as conn:
        conn.execute("UPDATE users SET last_login=? WHERE id=?", (now, user["id"]))
        conn.commit()

    audit(user.get("username") or user.get("email") or ident, "login_success")
    token = issue_token(user)
    return {"token": token, "user": user}, 0, "Login successful."

# ------------------------------------------------------------------ Email verification (Milestone 1)
def email_is_verified(identity):
    user = get_user(identity)
    if not user:
        return False
    with get_conn() as conn:
        row = conn.execute("SELECT email_verified FROM users WHERE id=?", (user["id"],)).fetchone()
    return bool(row and row[0])

def request_email_verification(email):
    email = (email or "").strip().lower()
    if not valid_email(email):
        return False, 0, "Enter a valid email address."
    user = get_user(email)
    if not user:
        return True, 0, "If the account is pending verification, a code will be sent."
    if user.get("locked"):
        return False, 0, LOCKED_MESSAGE
    if email_is_verified(email):
        return True, 0, "Email already verified. You may sign in."
    if not mailer.configured():
        return False, 0, "Email delivery is unavailable. Configure EMAIL_ADDRESS and EMAIL_PASSWORD in Colab Secrets."
    now = int(time.time())
    with get_conn() as conn:
        row = conn.execute("SELECT requests, next_request_at FROM otp_policy WHERE email=? AND purpose='verify'", (email,)).fetchone()
        requests, next_at = row if row else (0, 0)
        if next_at > now:
            return False, next_at-now, f"Request another code in {next_at-now} seconds."
        if requests >= MAX_OTP_REQUESTS:
            # The fifth code remains usable until expiry; additional requests are blocked.
            return False, 0, "Maximum verification code requests reached. Contact your administrator."
        number = requests + 1
        delay = 60 if number <= 3 else 300
        code = f"{secrets.randbelow(1000000):06d}"
        salt = secrets.token_hex(16)
        digest = otp_digest(code, salt, config.JWT_SECRET, "verify")
        conn.execute("INSERT OR REPLACE INTO email_verification_codes(email,code_hash,salt,expires_at) VALUES(?,?,?,?)", (email,digest,salt,now+OTP_TTL))
        conn.execute("INSERT OR REPLACE INTO otp_policy(email,purpose,requests,next_request_at,failures,locked_until,ladder_step) VALUES(?,?,?,?,COALESCE((SELECT failures FROM otp_policy WHERE email=? AND purpose='verify'),0),0,0)", (email,'verify',number,now+delay,email))
        conn.commit()
    success, message = mailer.send(email, "FreightQuote AI — Verify your email", f"Your FreightQuote AI verification code is {code}. It expires in five minutes. Do not share this code.")
    if not success:
        # Failed SMTP delivery must not consume an OTP attempt or a resend slot.
        with get_conn() as conn:
            conn.execute("DELETE FROM email_verification_codes WHERE email=?", (email,))
            conn.execute("UPDATE otp_policy SET requests=?,next_request_at=? WHERE email=? AND purpose='verify'", (requests,next_at,email))
            conn.commit()
        return False, 0, message
    audit(user["username"], "email_verification_requested")
    return True, delay, "A six-digit verification code has been sent to your email."

def verify_email_code(email, code):
    email = (email or "").strip().lower()
    code = (code or "").strip()
    user = get_user(email)
    if not user or not valid_email(email):
        return False, "Invalid or expired code."
    if user.get("locked"):
        return False, LOCKED_MESSAGE
    now = int(time.time())
    with get_conn() as conn:
        row = conn.execute("SELECT code_hash,salt,expires_at FROM email_verification_codes WHERE email=?", (email,)).fetchone()
        valid = bool(row and row[2] > now and re.fullmatch(r"[0-9]{6}",code) and hmac.compare_digest(otp_digest(code,row[1],config.JWT_SECRET,'verify'),row[0]))
        if valid:
            conn.execute("UPDATE users SET email_verified=1 WHERE id=?", (user['id'],))
            conn.execute("DELETE FROM email_verification_codes WHERE email=?", (email,))
            conn.execute("DELETE FROM otp_policy WHERE email=? AND purpose='verify'", (email,))
            conn.commit()
            audit(user['username'], "email_verified")
            return True, "Email verified. You may now sign in."
        previous = conn.execute("SELECT failures FROM otp_policy WHERE email=? AND purpose='verify'", (email,)).fetchone()
        failures = (previous[0] if previous else 0) + 1
        conn.execute("INSERT OR IGNORE INTO otp_policy(email,purpose) VALUES(?,'verify')", (email,))
        conn.execute("UPDATE otp_policy SET failures=? WHERE email=? AND purpose='verify'", (failures,email))
        if failures >= MAX_OTP_FAILURES:
            conn.execute("UPDATE users SET locked=1 WHERE id=?", (user['id'],))
            conn.execute("DELETE FROM email_verification_codes WHERE email=?", (email,))
        conn.commit()
    if failures >= MAX_OTP_FAILURES:
        audit(user['username'], "account_locked_email_verification")
        return False, LOCKED_MESSAGE
    return False, "Invalid or expired code."

# ------------------------------------------------------------------ Forgot Password Flow from File 1
def request_password_reset_otp(email):
    email = (email or "").strip().lower()
    if not valid_email(email):
        return False, 0, "Enter a valid email address."

    now = int(time.time())
    user = get_user(email)

    is_locked, retry_after, lock_msg = check_otp_lockout(email)
    if is_locked:
        return False, retry_after or 0, lock_msg

    if not user:
        return False, 0, "If this address is eligible, a code will be sent."
    if user.get("locked"):
        return False, 0, LOCKED_MESSAGE

    # Cooldown & request budget from File 1:
    # 60s for requests 1-3, then 300s. Cap = MAX_OTP_REQUESTS (5) -> locks account
    with get_conn() as conn:
        c = conn.cursor()
        c.execute("SELECT requests, next_request_at, failures, locked_until, ladder_step FROM otp_policy WHERE email=? AND purpose='reset'", (email,))
        row = c.fetchone()
        if row:
            requests, next_request_at, failures, locked_until, ladder_step = row
            if next_request_at > now:
                wait_sec = next_request_at - now
                return False, wait_sec, f"Request another code in {wait_sec} seconds."
            if requests >= MAX_OTP_REQUESTS:
                c.execute("UPDATE users SET locked=1 WHERE id=?", (user["id"],))
                audit(user["username"], "account_locked_otp_budget")
                conn.commit()
                return False, 0, LOCKED_MESSAGE
        else:
            requests, next_request_at, failures, locked_until, ladder_step = 0, 0, 0, 0, 0

        number = requests + 1
        delay = 60 if number <= 3 else 300

        code_int = secrets.randbelow(1000000)
        code_str = f"{code_int:06d}"
        salt = secrets.token_hex(16)
        chash = otp_digest(code_str, salt, config.JWT_SECRET, "reset")
        expires_at = now + OTP_TTL
        sent_at = now

        c.execute("INSERT OR REPLACE INTO resets(email, code_hash, salt, expires_at, attempts, sent_at) VALUES(?,?,?,?,0,?)",
                  (email, chash, salt, expires_at, sent_at))
        c.execute("INSERT OR REPLACE INTO otp_policy(email, purpose, requests, next_request_at, failures, locked_until, ladder_step) VALUES(?,?,?,?,?,?,?)",
                  (email, "reset", number, now + delay, failures, locked_until, ladder_step))
        c.execute("DELETE FROM reset_grants WHERE email=?", (email,))
        conn.commit()

    subject = "FreightQuote AI — Your password reset code"
    body = f"Hello {user['username']},\n\nYour password reset verification code is: {code_str}\n\nThis code expires in 5 minutes. If you did not request this, please contact your administrator."
    sent, msg = mailer.send(email, subject, body, html=mailer.otp_html(user["username"], code_str))
    if not sent:
        with get_conn() as conn:
            conn.execute("DELETE FROM resets WHERE email=?", (email,))
            conn.execute("UPDATE otp_policy SET requests=?, next_request_at=? WHERE email=? AND purpose='reset'", (requests, next_request_at if row else 0, email))
            conn.commit()
        return False, 0, msg

    audit(user["username"], "otp_requested:reset")
    return True, delay, "A 6-digit verification code has been sent to your registered email."

def verify_password_reset_otp(email, code):
    email = (email or "").strip().lower()
    code = (code or "").strip()
    if not email or not code:
        return False, None, "Enter the six-digit code."

    now = int(time.time())
    user = get_user(email)
    if not user or user.get("locked"):
        return False, None, "Invalid or expired code."

    with get_conn() as conn:
        c = conn.cursor()
        c.execute("SELECT code_hash, salt, expires_at, attempts FROM resets WHERE email=?", (email,))
        row = c.fetchone()
        if not row:
            is_l, ret_s, msg = record_otp_failure(email)
            return False, None, msg

        code_hash, salt, expires_at, attempts = row
        if expires_at <= now:
            c.execute("DELETE FROM resets WHERE email=?", (email,))
            conn.commit()
            return False, None, "Invalid or expired code."

        ok = bool(re.fullmatch(r"[0-9]{6}", code)) and hmac.compare_digest(
            otp_digest(code, salt, config.JWT_SECRET, "reset"), code_hash
        )
        if not ok:
            is_l, ret_s, msg = record_otp_failure(email)
            return False, None, msg

        # Single-use code
        c.execute("DELETE FROM resets WHERE email=?", (email,))
        c.execute("UPDATE otp_policy SET failures=0 WHERE email=? AND purpose='reset'", (email,))
        grant_token = secrets.token_urlsafe(32)
        grant_hash = hashlib.sha256(grant_token.encode("utf-8")).hexdigest()
        c.execute("INSERT OR REPLACE INTO reset_grants(email, token_hash, expires_at, version) VALUES(?,?,?,0)",
                  (email, grant_hash, now + 300))
        conn.commit()

    audit(user["username"], "otp_verified:reset")
    return True, grant_token, "Code verified successfully."

def get_user_security_question(identity):
    ident = (identity or "").strip()
    if not ident:
        return None, "Enter your registered username."
    user = get_user(ident)
    if not user:
        return None, "Unable to verify that username."
    # The question may be displayed to a locked user, but answering it cannot
    # grant a password reset or unlock the account (checked below).
    if user.get("locked"):
        return user["security_question"], LOCKED_MESSAGE

    return user["security_question"], None

def verify_user_security_answer(identity, answer):
    ident = (identity or "").strip()
    answer = answer or ""
    if not ident:
        return False, None, "Enter your registered username."

    user = get_user(ident)
    if not user:
        return False, None, "Unable to verify that username."
    if user.get("locked"):
        return False, None, LOCKED_MESSAGE

    is_l, ret_s, msg = check_otp_lockout(user["email"])
    if is_l:
        return False, None, msg or LOCKED_MESSAGE

    err = answer_error(answer)
    if err:
        return False, None, err

    cleaned = clean_answer(answer)
    if not matches(cleaned, user["security_answer_hash"]):
        record_otp_failure(user["email"])
        audit(user["username"], "security_answer_failed")
        return False, None, "Incorrect answer or too many attempts."

    now = int(time.time())
    grant_token = secrets.token_urlsafe(32)
    grant_hash = hashlib.sha256(grant_token.encode("utf-8")).hexdigest()
    with get_conn() as conn:
        conn.execute("INSERT OR REPLACE INTO reset_grants(email, token_hash, expires_at, version) VALUES(?,?,?,0)",
                     (user["email"], grant_hash, now + 300))
        conn.commit()

    audit(user["username"], "security_answer_verified")
    return True, grant_token, "Answer verified successfully."

def reset_user_password(email_or_username, grant_token, new_password, confirm_password):
    ident = (email_or_username or "").strip()
    user = get_user(ident)
    if not user:
        return False, "Account not found."
    if user.get("locked"):
        return False, LOCKED_MESSAGE

    if new_password != confirm_password:
        return False, "Password and confirmation do not match."

    errs = password_errors(new_password)
    if errs:
        return False, "Password does not meet requirements: " + ", ".join(errs)

    now = int(time.time())
    with get_conn() as conn:
        c = conn.cursor()
        c.execute("SELECT token_hash, expires_at FROM reset_grants WHERE email=?", (user["email"],))
        row = c.fetchone()
        if not row or row[1] < now:
            return False, "Verification expired. Please verify again."
        expected_hash = hashlib.sha256((grant_token or "").encode("utf-8")).hexdigest()
        if not hmac.compare_digest(row[0], expected_hash):
            return False, "Invalid verification token. Please verify again."

        if matches(new_password, user["password_hash"]):
            return False, "Choose a password different from your current password."

        c.execute("SELECT password_hash FROM password_history WHERE user_id=? ORDER BY id DESC LIMIT 5", (user["id"],))
        for h_row in c.fetchall():
            if matches(new_password, h_row[0]):
                return False, "Choose a password different from your recent passwords."

        new_hash = hashed(new_password)
        now_iso = utcnow()
        c.execute("UPDATE users SET password_hash=? WHERE id=?", (new_hash, user["id"]))
        c.execute("INSERT INTO password_history(user_id, password_hash, created_at) VALUES(?,?,?)", (user["id"], new_hash, now_iso))
        c.execute("DELETE FROM reset_grants WHERE email=?", (user["email"],))
        c.execute("DELETE FROM resets WHERE email=?", (user["email"],))
        conn.execute("DELETE FROM limits WHERE subject=?", (f"login:user:{user['id']}",))
        conn.execute("DELETE FROM otp_policy WHERE email=?", (user["email"],))
        conn.commit()

    audit(user["username"], "password_changed")
    return True, "Password updated successfully. You may now sign in."

# ------------------------------------------------------------------ Admin Unblock
def unblock_user_account(admin_user, target_identity):
    if admin_user.get("role") != "Admin":
        return False, "Administrator authentication required."
    target = get_user(target_identity)
    if not target:
        return False, "Account not found."

    with get_conn() as conn:
        conn.execute("UPDATE users SET locked=0 WHERE id=?", (target["id"],))
        conn.execute("DELETE FROM limits WHERE subject=?", (f"login:user:{target['id']}",))
        conn.execute("DELETE FROM otp_policy WHERE email=?", (target["email"],))
        conn.execute("DELETE FROM resets WHERE email=?", (target["email"],))
        conn.execute("DELETE FROM reset_grants WHERE email=?", (target["email"],))
        conn.execute("INSERT INTO admin_activity_log(actor, action, details, timestamp) VALUES(?,?,?,?)",
                     (admin_user["username"], "unblock_account", f"Unblocked user {target['username']} ({target['email']})", utcnow()))
        conn.commit()

    audit(admin_user["username"], f"unblocked_user_{target['username']}")
    return True, f"Account {target['username']} has been unlocked."

# ------------------------------------------------------------------ Admin & Default Accounts Seeding
def seed_admin_account():
    # Provision once. Restarts must never change credentials or unlock accounts.
    if password_errors(config.ADMIN_PASSWORD):
        raise ValueError("ADMIN_PASSWORD requires upper/lowercase, a number and a symbol.")
    with get_conn() as conn:
        if conn.execute("SELECT 1 FROM users WHERE role='Admin'").fetchone():
            return
        if conn.execute("SELECT 1 FROM users WHERE username='admin' OR email=?", (config.ADMIN_EMAIL,)).fetchone():
            raise ValueError("Initial administrator identity already belongs to another account.")
        pw_hash = hashed(config.ADMIN_PASSWORD)
        now = utcnow()
        cur = conn.execute("INSERT INTO users(username,email,password_hash,role,locked,created_at,security_question,security_answer_hash) VALUES(?,?,?,'Admin',0,?,?,?)", ("admin",config.ADMIN_EMAIL,pw_hash,now,QUESTIONS[0],hashed(secrets.token_urlsafe(32))))
        conn.execute("INSERT INTO password_history(user_id,password_hash,created_at) VALUES(?,?,?)",(cur.lastrowid,pw_hash,now))
        conn.commit()

# ------------------------------------------------------------------ Profile Management (Stage 3)
def change_username(user_id, new_username, current_password):
    user = get_user_by_id(user_id)
    if not user:
        return False, "User not found.", None
    if not current_password or not matches(current_password, user["password_hash"]):
        return False, "Current password is incorrect.", None
    
    clean_u = (new_username or "").strip()
    if not clean_u:
        return False, "New username cannot be empty.", None
    if clean_u == user["username"]:
        return False, "New username must be different from your current username.", None
    if not re.match(r"^[a-zA-Z0-9_.]{3,30}$", clean_u):
        return False, "Username must be 3-30 characters and contain only letters, numbers, underscores, or periods.", None
    
    with get_conn() as conn:
        c = conn.cursor()
        c.execute("SELECT 1 FROM users WHERE username=? AND id != ?", (clean_u, user_id))
        if c.fetchone():
            return False, "Username is already taken. Please choose another.", None
        c.execute("UPDATE users SET username=? WHERE id=?", (clean_u, user_id))
        conn.commit()

    audit(user["username"], f"username_changed_to_{clean_u}")
    updated = get_user_by_id(user_id)
    new_token = issue_token(updated)
    sanitized = {
        "id": updated["id"],
        "username": updated["username"],
        "email": updated["email"],
        "role": updated["role"],
        "customer_id": updated["customer_id"],
        "created_at": updated["created_at"],
        "last_login": updated["last_login"],
    }
    return True, "Username updated successfully.", {"token": new_token, "user": sanitized}

def change_password(user_id, current_password, new_password, confirm_password):
    user = get_user_by_id(user_id)
    if not user:
        return False, "User not found."
    if not current_password or not matches(current_password, user["password_hash"]):
        return False, "Current password is incorrect."
    
    if new_password != confirm_password:
        return False, "Password and confirmation do not match."

    errs = password_errors(new_password)
    if errs:
        return False, "Password does not meet requirements: " + ", ".join(errs)

    if matches(new_password, user["password_hash"]):
        return False, "Choose a password different from your current password."

    with get_conn() as conn:
        c = conn.cursor()
        c.execute("SELECT password_hash FROM password_history WHERE user_id=? ORDER BY id DESC LIMIT 5", (user_id,))
        for h_row in c.fetchall():
            if matches(new_password, h_row[0]):
                return False, "Choose a password different from your recent passwords."

        new_hash = hashed(new_password)
        now = utcnow()
        c.execute("UPDATE users SET password_hash=? WHERE id=?", (new_hash, user_id))
        c.execute("INSERT INTO password_history(user_id, password_hash, created_at) VALUES(?,?,?)", (user_id, new_hash, now))
        conn.commit()

    audit(user["username"], "password_changed")
    return True, "Password updated successfully."

# ------------------------------------------------------------------ Admin Dashboard Management (Stage 4)
ADMIN_ALLOWED_ROLES = ["Admin", "Operations Manager", "Freight Broker", "Auditor", "Customer"]

def admin_list_users():
    with get_conn() as conn:
        c = conn.cursor()
        c.execute("""
            SELECT id, username, email, role, customer_id, locked, created_at, last_login
            FROM users
            ORDER BY id ASC
        """)
        rows = c.fetchall()
        return [
            {
                "id": r[0],
                "username": r[1] or "",
                "email": r[2] or "",
                "role": r[3] or "Customer",
                "customer_id": r[4],
                "locked": bool(r[5]),
                "created_at": r[6] or "—",
                "last_login": r[7] or "Never"
            }
            for r in rows
        ]

def admin_get_metrics():
    with get_conn() as conn:
        c = conn.cursor()
        c.execute("SELECT COUNT(*) FROM users")
        total = c.fetchone()[0]
        c.execute("SELECT COUNT(*) FROM users WHERE locked=1")
        locked = c.fetchone()[0]
        c.execute("SELECT role, COUNT(*) FROM users GROUP BY role")
        per_role = {r[0]: r[1] for r in c.fetchall()}
        return {
            "total_users": total,
            "locked_users": locked,
            "roles": per_role
        }

def admin_create_user(admin_username, username, email, password, role, customer_id=None):
    clean_u = (username or "").strip()
    clean_e = (email or "").strip().lower()
    pw = password or ""
    r = (role or "").strip()

    if not clean_u or not clean_e or not pw:
        return False, "Username, email, and temporary password are required."
    
    if r not in ADMIN_ALLOWED_ROLES:
        return False, f"Invalid role. Allowed roles: {', '.join(ADMIN_ALLOWED_ROLES)}."

    if not re.match(r"^[a-zA-Z0-9_.]{3,30}$", clean_u):
        return False, "Username must be 3-30 characters and contain only letters, numbers, underscores, or periods."

    if not valid_email(clean_e):
        return False, "Invalid email address format."

    errs = password_errors(pw)
    if errs:
        return False, "Password does not meet requirements: " + ", ".join(errs)

    with get_conn() as conn:
        c = conn.cursor()
        c.execute("SELECT 1 FROM users WHERE username=? COLLATE NOCASE", (clean_u,))
        if c.fetchone():
            return False, "Username is already taken."
        c.execute("SELECT 1 FROM users WHERE email=? COLLATE NOCASE", (clean_e,))
        if c.fetchone():
            return False, "Email address is already registered."

        pw_hash = hashed(pw)
        now = utcnow()
        ans_hash = hashed(secrets.token_urlsafe(32))
        c.execute(
            """INSERT INTO users(username, email, password_hash, role, locked, created_at, last_login, security_question, security_answer_hash, customer_id)
               VALUES(?, ?, ?, ?, 0, ?, NULL, ?, ?, ?)""",
            (clean_u, clean_e, pw_hash, r, now, QUESTIONS[0], ans_hash, (customer_id or None)),
        )
        uid = c.lastrowid
        c.execute("INSERT INTO password_history(user_id, password_hash, created_at) VALUES(?,?,?)", (uid, pw_hash, now))
        c.execute(
            "INSERT INTO admin_activity_log(actor, action, details, timestamp) VALUES(?,?,?,?)",
            (admin_username, "create_user", f"Created user {clean_u} ({clean_e}) with role {r}", now)
        )
        conn.commit()

    audit(admin_username, f"admin_created_user_{clean_u}")
    return True, f"User {clean_u} created successfully."

def admin_change_role(admin_username, target_id, new_role):
    r = (new_role or "").strip()
    if r not in ADMIN_ALLOWED_ROLES:
        return False, f"Invalid role. Allowed roles: {', '.join(ADMIN_ALLOWED_ROLES)}."

    target = get_user_by_id(target_id)
    if not target:
        return False, "User not found."

    if target["role"] == r:
        return True, f"User already has role {r}."

    with get_conn() as conn:
        c = conn.cursor()
        if target["role"] == "Admin" and r != "Admin":
            c.execute("SELECT COUNT(*) FROM users WHERE role='Admin' AND locked=0 AND id != ?", (target_id,))
            remaining = c.fetchone()[0]
            if remaining == 0:
                return False, "Cannot demote the last remaining active Administrator."

        c.execute("UPDATE users SET role=? WHERE id=?", (r, target_id))
        now = utcnow()
        c.execute(
            "INSERT INTO admin_activity_log(actor, action, details, timestamp) VALUES(?,?,?,?)",
            (admin_username, "change_role", f"Changed role of {target['username']} from {target['role']} to {r}", now)
        )
        conn.commit()

    audit(admin_username, f"admin_changed_role_{target['username']}_to_{r}")
    return True, f"Role for {target['username']} updated to {r}."

def admin_set_lock(admin_username, target_id, lock_bool):
    target = get_user_by_id(target_id)
    if not target:
        return False, "User not found."

    if target["username"] == admin_username and lock_bool:
        return False, "You cannot lock your own account."

    with get_conn() as conn:
        c = conn.cursor()
        now = utcnow()
        if lock_bool:
            if target["role"] == "Admin":
                c.execute("SELECT COUNT(*) FROM users WHERE role='Admin' AND locked=0 AND id != ?", (target_id,))
                remaining = c.fetchone()[0]
                if remaining == 0:
                    return False, "Cannot lock the last remaining active Administrator."

            c.execute("UPDATE users SET locked=1, account_status='Locked' WHERE id=?", (target_id,))
            c.execute(
                "INSERT INTO admin_activity_log(actor, action, details, timestamp) VALUES(?,?,?,?)",
                (admin_username, "lock_user", f"Locked account {target['username']}", now)
            )
            msg = f"Account {target['username']} has been locked."
        else:
            c.execute("UPDATE users SET locked=0, account_status='Active', failed_attempts=0, lock_until=NULL WHERE id=?", (target_id,))
            c.execute("DELETE FROM resets WHERE email=?", (target["email"],))
            c.execute("DELETE FROM reset_grants WHERE email=?", (target["email"],))
            c.execute("DELETE FROM otp_policy WHERE email=?", (target["email"],))
            c.execute("DELETE FROM limits WHERE subject=?", (f"login:user:{target['id']}",))
            c.execute(
                "INSERT INTO admin_activity_log(actor, action, details, timestamp) VALUES(?,?,?,?)",
                (admin_username, "unlock_user", f"Unlocked account {target['username']}", now)
            )
            msg = f"Account {target['username']} has been unlocked."
        conn.commit()

    audit(admin_username, f"admin_{'locked' if lock_bool else 'unlocked'}_{target['username']}")
    return True, msg

def admin_delete_user(admin_username, target_id):
    target = get_user_by_id(target_id)
    if not target:
        return False, "User not found."

    if target["username"] == admin_username:
        return False, "You cannot delete your own account."

    with get_conn() as conn:
        c = conn.cursor()
        if target["role"] == "Admin":
            c.execute("SELECT COUNT(*) FROM users WHERE role='Admin' AND locked=0 AND id != ?", (target_id,))
            remaining = c.fetchone()[0]
            if remaining == 0:
                return False, "Cannot remove the last remaining active Administrator."

        c.execute("DELETE FROM users WHERE id=?", (target_id,))
        c.execute("DELETE FROM password_history WHERE user_id=?", (target_id,))
        c.execute("DELETE FROM resets WHERE email=?", (target["email"],))
        c.execute("DELETE FROM reset_grants WHERE email=?", (target["email"],))
        now = utcnow()
        c.execute(
            "INSERT INTO admin_activity_log(actor, action, details, timestamp) VALUES(?,?,?,?)",
            (admin_username, "delete_user", f"Deleted user {target['username']} ({target['email']})", now)
        )
        conn.commit()

    audit(admin_username, f"admin_deleted_user_{target['username']}")
    return True, f"User {target['username']} has been deleted."

def admin_link_customer(admin_username, target_id, customer_id):
    target = get_user_by_id(target_id)
    if not target:
        return False, "User not found."

    clean_cid = (customer_id or "").strip() or None

    with get_conn() as conn:
        conn.execute("UPDATE users SET customer_id=? WHERE id=?", (clean_cid, target_id))
        now = utcnow()
        conn.execute(
            "INSERT INTO admin_activity_log(actor, action, details, timestamp) VALUES(?,?,?,?)",
            (admin_username, "link_customer", f"Linked {target['username']} to {clean_cid}", now)
        )
        conn.commit()

    audit(admin_username, f"admin_linked_{target['username']}_to_{clean_cid}")
    return True, f"User {target['username']} is now linked to {clean_cid or 'None'}."

def admin_get_activity_log(limit=50):
    with get_conn() as conn:
        c = conn.cursor()
        c.execute(
            "SELECT id, actor, action, details, timestamp FROM admin_activity_log ORDER BY id DESC LIMIT ?",
            (limit,)
        )
        rows = c.fetchall()
        return [
            {
                "id": r[0],
                "actor": r[1] or "system",
                "action": r[2] or "",
                "details": r[3] or "",
                "timestamp": r[4] or ""
            }
            for r in rows
        ]


def session_version(uid):
    with get_conn() as c:
        try:
            row = c.execute("SELECT session_version FROM users WHERE id=?", (uid,)).fetchone()
            return row[0] if row and row[0] is not None else 0
        except Exception:
            return 0

def init_sessions():
    with get_conn() as c:
        if "session_version" not in [x[1] for x in c.execute("PRAGMA table_info(users)")]:
            c.execute("ALTER TABLE users ADD COLUMN session_version INTEGER NOT NULL DEFAULT 0")
        c.execute("""CREATE TRIGGER IF NOT EXISTS revoke_user_sessions AFTER UPDATE OF password_hash,role,locked,username ON users BEGIN UPDATE users SET session_version=session_version+1 WHERE id=NEW.id; END""")

In [ ]:
%%writefile /content/app/blocks.py
"""Server-driven UI blocks. Pages are Python functions returning JSON blocks; the single-page front end renders them and live-updates only what changed.
Charts are ECharts option dicts built here with one consistent palette."""
import math, json, datetime
import numpy as np, pandas as pd
import live_engine as LE

TEAL, BLUE, AMBER, RED, VIOLET, GREEN, PINK, GREY = "#12a594", "#3e63dd", "#f5a524", "#e5484d", "#8e4ec6", "#30a46c", "#d6409f", "#8b8d98"
PRI = "#5b5bd6"
PAL = [PRI, TEAL, AMBER, VIOLET, RED, GREEN, PINK, BLUE, GREY]
TONE = {"Critical": RED, "High": RED, "Warning": AMBER, "Medium": AMBER, "Info": BLUE, "Low": TEAL, "Pass": TEAL, "Fail": RED}

def Q(sql, p=()):
    with LE._DB_LOCK:
        return pd.read_sql(sql, LE.conn(), params=p)

def clean(v):
    if isinstance(v, (np.integer,)): return int(v)
    if isinstance(v, (np.floating, float)): return None if (v is None or (isinstance(v, float) and (math.isnan(v) or math.isinf(v))) or (isinstance(v, np.floating) and (np.isnan(v) or np.isinf(v)))) else round(float(v), 4)
    if isinstance(v, (np.bool_,)): return bool(v)
    if isinstance(v, (pd.Timestamp, datetime.datetime, datetime.date)): return str(v)
    if isinstance(v, (list, tuple)): return [clean(x) for x in v]
    if isinstance(v, dict): return {str(k): clean(x) for k, x in v.items()}
    if v is pd.NA or v is pd.NaT: return None
    return v
def L(x): return [clean(v) for v in (x.tolist() if hasattr(x, "tolist") else list(x))]

# ------------------------------------------------------------------ charts (ECharts options)
def _base(extra=None, legend=False, grid=None):
    o = {"animation": True, "animationDuration": 350, "color": PAL, "tooltip": {"trigger": "axis", "confine": True}, "grid": grid or {"left": 8, "right": 14, "top": 28 if legend else 14, "bottom": 6, "containLabel": True}}
    if legend: o["legend"] = {"top": 0, "type": "scroll"}
    if extra: o.update(extra)
    return o
def bar(cats, series, horizontal=False, stack=False, colors=None, fmt=None, max_=None, pct=False, itemcolors=None, sort=False):
    """series: {name: values} or a list of values."""
    series = {"": series} if not isinstance(series, dict) else series; cats = L(cats)
    if sort and len(series) == 1:
        v = L(list(series.values())[0]); order = sorted(range(len(v)), key=lambda i: (v[i] is None, v[i] or 0)); cats = [cats[i] for i in order]; series = {k: [L(x)[i] for i in order] for k, x in series.items()}
    ser = []
    for i, (n, v) in enumerate(series.items()):
        d = L(v); s = {"type": "bar", "name": n, "data": d, "barMaxWidth": 22, "itemStyle": {"borderRadius": [0, 4, 4, 0] if horizontal else [4, 4, 0, 0], "color": (colors or PAL)[i % len(colors or PAL)]}, "emphasis": {"focus": "series"}}
        if itemcolors and i == 0: s["data"] = [{"value": x, "itemStyle": {"color": c}} for x, c in zip(d, itemcolors)]
        if stack: s["stack"] = "s"
        ser.append(s)
    cat = {"type": "category", "data": cats, "axisLabel": {"hideOverlap": True}}; val = {"type": "value", "splitLine": {"lineStyle": {"color": "#eceef6"}}}
    if max_: val["max"] = max_
    if pct: val["axisLabel"] = {"formatter": "{value}%"}
    o = _base(legend=len(series) > 1 and any(series)); o.update({"xAxis": val if horizontal else cat, "yAxis": cat if horizontal else val, "series": ser})
    return o
def line(x, series, area=True, band=None, colors=None, dashed=(), ymin=None, ymax=None, step=False, markers=False):
    series = {"": series} if not isinstance(series, dict) else series; ser = []
    for i, (n, v) in enumerate(series.items()):
        c = (colors or PAL)[i % len(colors or PAL)]; s = {"type": "line", "name": n, "data": L(v), "smooth": not step, "showSymbol": markers, "symbolSize": 5, "lineStyle": {"width": 2, "color": c, "type": "dashed" if n in dashed else "solid"}, "itemStyle": {"color": c}}
        if step: s["step"] = "end"
        if area and i == 0 and len(series) <= 2: s["areaStyle"] = {"color": {"type": "linear", "x": 0, "y": 0, "x2": 0, "y2": 1, "colorStops": [{"offset": 0, "color": c + "55"}, {"offset": 1, "color": c + "00"}]}}
        ser.append(s)
    if band:
        lo, hi = L(band[0]), L(band[1]); ser += [{"type": "line", "data": lo, "stack": "b", "lineStyle": {"opacity": 0}, "showSymbol": False, "silent": True, "tooltip": {"show": False}}, {"type": "line", "data": [None if (a is None or b is None) else b - a for a, b in zip(lo, hi)], "stack": "b", "lineStyle": {"opacity": 0}, "areaStyle": {"color": "rgba(251,191,36,.16)"}, "showSymbol": False, "silent": True, "tooltip": {"show": False}}]
    y = {"type": "value", "scale": True, "splitLine": {"lineStyle": {"color": "#eceef6"}}}
    if ymin is not None: y["min"] = ymin
    if ymax is not None: y["max"] = ymax
    o = _base(legend=len([s for s in series if s]) > 1); o.update({"xAxis": {"type": "category", "data": L(x), "boundaryGap": False, "axisLabel": {"hideOverlap": True}}, "yAxis": y, "series": ser}); return o
def pie(items, donut=True, colors=None):
    """items: {name: value} or list of (name, value, color?)"""
    its = list(items.items()) if isinstance(items, dict) else items; data = []
    for i, it in enumerate(its):
        d = {"name": str(it[0]), "value": clean(it[1])}
        if (colors and str(it[0]) in colors) or len(it) > 2: d["itemStyle"] = {"color": (colors or {}).get(str(it[0])) or it[2]}
        data.append(d)
    return {"animation": True, "color": PAL, "tooltip": {"trigger": "item", "confine": True}, "legend": {"bottom": 0, "type": "scroll"}, "series": [{"type": "pie", "radius": ["52%", "78%"] if donut else "75%", "center": ["50%", "44%"], "avoidLabelOverlap": True, "label": {"show": False}, "itemStyle": {"borderColor": "#ffffff", "borderWidth": 2, "borderRadius": 4}, "data": data}]}
def scatter(groups, xname="", yname="", colors=None, sized=False):
    """groups: {name: [ {x,y,label,size?} ]}. Points are [x, y, label, size]; the client resolves the {"__fn": ...} formatters."""
    ser = []
    for i, (n, pts) in enumerate(groups.items()):
        ser.append({"type": "scatter", "name": n, "symbolSize": {"__fn": "size3"} if sized else 10, "data": [[clean(p["x"]), clean(p["y"]), p.get("label", ""), clean(p.get("size", 10))] for p in pts],
                    "itemStyle": {"color": (colors or {}).get(n, PAL[i % len(PAL)]), "opacity": .85}, "emphasis": {"focus": "series"}})
    o = _base(legend=len(groups) > 1); o["tooltip"] = {"trigger": "item", "confine": True, "formatter": {"__fn": "scatterTip"}}
    ax = lambda n: {"type": "value", "name": n, "scale": True, "nameTextStyle": {"color": "#6b7080"}, "splitLine": {"lineStyle": {"color": "#eceef6"}}}
    o.update({"xAxis": ax(xname), "yAxis": ax(yname), "series": ser}); return o
def hist(values, bins=24, color=TEAL, pct=False):
    v = np.asarray([x for x in values if x is not None and not (isinstance(x, float) and math.isnan(x))], dtype=float)
    if not len(v): return bar([], [])
    h, e = np.histogram(v, bins=bins); labels = [f"{(e[i] * (100 if pct else 1)):.{0 if pct else 1}f}" + ("%" if pct else "") for i in range(len(h))]
    return bar(labels, [int(x) for x in h], colors=[color])
def waterfall(labels, deltas, total_label="Total"):
    base, run, up = [], 0.0, []
    for d in deltas: base.append(run if d >= 0 else run + d); up.append(abs(d)); run += d
    cats = list(labels) + [total_label]; base.append(0); up.append(run)
    o = _base(); o.update({"xAxis": {"type": "category", "data": cats}, "yAxis": {"type": "value", "splitLine": {"lineStyle": {"color": "#eceef6"}}},
                           "series": [{"type": "bar", "stack": "w", "data": [clean(x) for x in base], "itemStyle": {"color": "transparent"}, "tooltip": {"show": False}}, {"type": "bar", "stack": "w", "data": [{"value": clean(u), "itemStyle": {"color": TEAL if i == len(up) - 1 else (BLUE if deltas[i] >= 0 else RED)}} for i, u in enumerate(up)], "barMaxWidth": 34, "itemStyle": {"borderRadius": 4}}]}); return o
def radar(names, series, mx=100):
    o = {"animation": True, "color": PAL, "tooltip": {"confine": True}, "legend": {"bottom": 0}, "radar": {"indicator": [{"name": n, "max": mx} for n in names], "splitArea": {"show": False}, "axisName": {"color": "#6b7080"}}, "series": [{"type": "radar", "data": [{"name": k, "value": L(v), "areaStyle": {"opacity": .18}} for k, v in series.items()]}]}; return o
def gauge(value, name="", mx=100):
    c = RED if value < 60 else AMBER if value < 80 else TEAL
    return {"series": [{"type": "gauge", "startAngle": 210, "endAngle": -30, "min": 0, "max": mx, "progress": {"show": True, "width": 14, "itemStyle": {"color": c}}, "axisLine": {"lineStyle": {"width": 14, "color": [[1, "rgba(148,163,184,.15)"]]}}, "axisTick": {"show": False}, "splitLine": {"show": False}, "axisLabel": {"show": False}, "pointer": {"show": False}, "anchor": {"show": False},
                         "detail": {"valueAnimation": True, "fontSize": 30, "fontWeight": 700, "offsetCenter": [0, "8%"], "color": "#1c2024", "formatter": "{value}"}, "title": {"offsetCenter": [0, "62%"], "color": "#6b7080", "fontSize": 12}, "data": [{"value": clean(round(value, 1)), "name": name}]}]}
def graph(nodes, links, h=520):
    """nodes: [{id,name,kind,size,color,tip}] ; links: [(a,b)]"""
    cat = sorted({n.get("kind", "x") for n in nodes}); return {"animation": True, "tooltip": {"confine": True, "formatter": {"__fn": "nodeTip"}}, "legend": {"data": cat, "bottom": 0, "textStyle": {"color": "#6b7080"}}, "color": PAL,
        "series": [{"type": "graph", "layout": "force", "roam": True, "draggable": True, "force": {"repulsion": 260, "edgeLength": [40, 120], "gravity": .08}, "label": {"show": True, "position": "right", "color": "#454a58", "fontSize": 10, "formatter": {"__fn": "nodeLabel"}},
                    "categories": [{"name": c} for c in cat], "lineStyle": {"color": "#cfd3e3", "width": 1}, "emphasis": {"focus": "adjacency", "lineStyle": {"width": 2}},
                    "data": [{"id": n["id"], "name": n["id"], "category": cat.index(n.get("kind", "x")), "symbolSize": n.get("size", 12), "itemStyle": {"color": n.get("color")} if n.get("color") else {}, "tip": n.get("tip", ""), "showLabel": n.get("label", False), "open": n.get("open")} for n in nodes],
                    "links": [{"source": a, "target": b} for a, b in links]}]}

# ------------------------------------------------------------------ blocks
def kpis(*items):
    out = []
    for it in items:
        d = {"label": it[0], "value": clean(it[1])}
        if len(it) > 2 and it[2] is not None: d["sub"] = str(it[2])
        if len(it) > 3 and it[3]: d["tone"] = it[3]
        out.append(d)
    return {"t": "kpis", "items": out}
def chart(opt, title=None, h=290, open=None, sub=None, key=None): return {"t": "chart", "opt": opt, "title": title, "h": h, "open": open, "sub": sub, "key": key or title}
def table(df, cols=None, title=None, h=None, open=None, key=None, sub=None, search=False, page=None):
    """cols: list of names or dicts {k,label,type: text|num|int|pct|pill|bar|money|time|id, max}. open = {"kind":"outlet","col":"outlet_id"} makes rows clickable."""
    if isinstance(df, list): df = pd.DataFrame(df)
    cols = cols or list(df.columns); cs = [{"k": c, "label": c.replace("_", " ")} if isinstance(c, str) else c for c in cols]
    rows = [{c["k"]: clean(r.get(c["k"])) for c in cs} for r in df.to_dict("records")] if len(df) else []
    return {"t": "table", "cols": cs, "rows": rows, "title": title, "h": h, "open": open, "key": key or title, "sub": sub, "search": search}
def callout(text, tone="info", title=None): return {"t": "callout", "text": text, "tone": tone, "title": title}
def md(text): return {"t": "md", "text": text}
def row(*cols, ratio=None): return {"t": "row", "cols": list(cols), "ratio": ratio}
def col(*blocks): return {"t": "col", "blocks": list(blocks)}
def card(title, *blocks, sub=None): return {"t": "card", "title": title, "sub": sub, "blocks": list(blocks)}
def form(id, fields, submit="Submit", url=None, title=None, tone="primary", result=True, sub=None):
    """fields: {name,label,type: select|number|text|textarea|range|checks|multi|file|date, options, value, min,max,step}"""
    return {"t": "form", "id": id, "fields": fields, "submit": submit, "url": url, "title": title, "tone": tone, "sub": sub}
def button(id, label, url, tone="primary", confirm=None, payload=None): return {"t": "button", "id": id, "label": label, "url": url, "tone": tone, "confirm": confirm, "payload": payload or {}}
def map_(layers, h=470, view=None, open=None, key="map", legend=None, live=None): return {"t": "map", "layers": layers, "h": h, "view": view, "open": open, "key": key, "legend": legend, "live": live}
def tickets(items, key="tickets", title=None, sub=None): return {"t": "tickets", "items": items, "key": key, "title": title, "sub": sub}
def hero(agent, title, blurb, r, can_run=True):
    return {"t": "hero", "agent": agent, "title": title, "blurb": blurb, "can_run": can_run, "score": None if not r else clean(r["score"]), "risk": None if not r else r["risk"], "summary": None if not r else r["summary"], "ts": None if not r else r["ts"], "ms": None if not r else r["ms"]}
def tabs(items): return {"t": "tabs", "tabs": items}
def page(title, sub, blocks, tabs=None, live=8, actions=None): return {"title": title, "sub": sub, "blocks": blocks, "tabs": tabs, "live": live, "actions": actions or []}
def pill_col(k, label=None): return {"k": k, "label": label or k.replace("_", " "), "type": "pill"}
def num_col(k, label=None, fmt="num", mx=None): return {"k": k, "label": label or k.replace("_", " "), "type": fmt, "max": mx}


In [ ]:
%%writefile /content/app/views_core.py
"""Shared page builders: helpers, Command Center (tickets, agents, anomalies), Knowledge Graph."""
import json, datetime
import numpy as np, pandas as pd
import live_engine as LE, entities as EN
import blocks as B
from blocks import Q, clean, L, TEAL, BLUE, AMBER, RED, VIOLET, TONE

import rbac

def can(c, cap):
    role = c.get("role") if isinstance(c, dict) else None
    return rbac.check_permission(role, cap)
def latest(agent):
    r = LE.conn().execute("SELECT ts,score,risk,latency_ms,summary,output,cycle_id FROM agent_runs WHERE agent=? ORDER BY run_id DESC LIMIT 1", (agent,)).fetchone()
    if not r: return None
    try: out = json.loads(r[5])
    except Exception: out = {}
    return {"ts": r[0], "score": r[1], "risk": r[2], "ms": r[3], "summary": r[4], "out": out, "cycle": r[6]}
def decision():
    d = Q("SELECT decision_id, ts, score, risk, narrative, evidence FROM decisions ORDER BY decision_id DESC LIMIT 1")
    if not len(d): return None
    try: ev = json.loads(d.evidence[0])
    except Exception: ev = {}
    return {"id": int(d.decision_id[0]), "ts": d.ts[0], "score": float(d.score[0]), "risk": d.risk[0], "narrative": d.narrative[0], "ev": ev}
def scoped(df, c, col="target"):
    return df[df[col].astype(str).str.startswith(c["scope"])] if c.get("scope") and len(df) else df

# ------------------------------------------------------------------ tickets
def tk_df(c, agent=None, types=None, target=None, statuses=("open", "proposed"), limit=300):
    dom = c["dom"]; sql = "SELECT action_id id, severity, status, type, target, detail, owner, assignee, due, note, ts FROM agent_actions WHERE status IN (%s)" % ",".join("?" * len(statuses)); p = list(statuses)
    ts = types or ([t for t, a in dom.ACTION_AGENT.items() if a == agent] if agent else None)
    if ts: sql += f" AND type IN ({','.join('?' * len(ts))})"; p += list(ts)
    if target: sql += " AND target=?"; p.append(target)
    df = Q(sql + " ORDER BY CASE severity WHEN 'Critical' THEN 0 WHEN 'Warning' THEN 1 ELSE 2 END, action_id DESC LIMIT " + str(limit), p)
    return scoped(df, c)
def ticket_block(c, agent=None, types=None, title="Open tickets", target=None, key=None, limit=60):
    """Read-only list of what the agent has raised (type, target, severity, detail). There is no approval workflow in this version."""
    df = tk_df(c, agent, types, target, limit=limit)
    if not len(df): return B.callout("Nothing raised right now.", "info", title)
    return B.table(df, ["id", "ts", B.pill_col("severity"), "type", "target", "detail"], title=title, h=300, key=key or f"tk_{agent or target or 'all'}", open={"col": "target"} if False else None)
def hero(c, agent, title, blurb): return B.hero(agent, title, blurb, latest(agent), can_run=can(c, "run"))
def cross(c, targets, own, title="Flagged by other agents for these entities"):
    t = [str(x) for x in list(dict.fromkeys(targets))][:60]
    if not t: return None
    df = Q("SELECT type, target, severity, detail FROM agent_actions WHERE status IN ('open','proposed') AND target IN (%s)" % ",".join("?" * len(t)), t)
    df["agent"] = df.type.map(lambda x: c["dom"].ACTION_AGENT.get(x, "engine")); df = df[df.agent != own].head(30)
    if not len(df): return None
    return B.table(df, ["target", "agent", "type", B.pill_col("severity"), "detail"], title=title, h=240, open={"col": "target"}, key="cross_" + own)

# ------------------------------------------------------------------ Command Center

# ------------------------------------------------------------------ Approvals inbox (human in the loop)
SOURCE = {"manual_request": "Approval request", "quote_approval": "Quote approval", "ml_review": "ML second opinion", "twin_mitigation": "Digital Twin"}





# ------------------------------------------------------------------ Knowledge graph
def entity_explorer(c):
    kind, dom = c["kind"], c["dom"]; e = (c.get("params") or {}).get("e")
    ents = Q("SELECT DISTINCT target FROM agent_actions WHERE status IN ('open','proposed') LIMIT 300").target.tolist()
    base = Q("SELECT outlet_id e FROM outlets ORDER BY outlet_id").e.tolist() if kind == "Franchise" else Q("SELECT port_name e FROM ports ORDER BY congestion_index DESC LIMIT 60").e.tolist()
    opts = list(dict.fromkeys(([c["scope"]] if c.get("scope") else ents + base))); e = e or opts[0]
    d = EN.describe(kind, dom, e); nodes = [{"id": e, "kind": "focus", "size": 34, "color": "#ffffff", "tip": "selected", "label": True}]; links = []
    for n, k, tip in d["rel"]: nodes.append({"id": n, "kind": k, "size": 16, "tip": tip, "label": True, "open": n if EN.kind_of(n) != "port" or kind == "Freight" else None}); links.append((e, n))
    for r in d["tickets"][d["tickets"].status.isin(["open", "proposed"])].itertuples():
        n = f"#{r.id} {r.type}"; ag = dom.ACTION_AGENT.get(r.type, "engine"); nodes += [{"id": n, "kind": "ticket", "size": 14, "color": TONE.get(r.severity), "tip": r.detail[:120], "label": True}, {"id": ag, "kind": "agent", "size": 26, "color": TEAL, "tip": "agent", "label": True}]; links += [(e, n), (n, ag)]
    seen = set(); nodes = [n for n in nodes if not (n["id"] in seen or seen.add(n["id"]))]
    sel = {"t": "form", "id": "kg_pick", "fields": [{"name": "e", "label": "Entity", "type": "select", "options": opts, "value": e}], "submit": "Explore", "url": "page:kg:explore", "title": None, "tone": "primary", "inline": True}
    info = B.table([{"field": k, "value": v} for k, v in d["info"].items()], ["field", "value"], title="What the platform knows", h=230, key="kg_info") if d["info"] else B.callout("No master record for this entity.", "info")
    tk = d["tickets"]
    return [sel, B.row(B.card("Facts", info, B.table(tk, ["id", B.pill_col("severity"), B.pill_col("status"), "type", "owner", "detail"], title="Why is it flagged", h=260, key="kg_tk")), B.card("Relations", B.chart(B.graph(nodes, links), h=470, open={"graph": True}, key="kg_ent")), ratio=[2, 3])]


In [ ]:
%%writefile /content/app/views_freight.py
"""FreightQuote AI pages (server-driven blocks)."""
import os, re, json, datetime
import numpy as np, pandas as pd
import live_engine as LE, entities as EN
import domain_freight as dom
import blocks as B
from blocks import Q, clean, L, TEAL, BLUE, AMBER, RED, VIOLET, GREEN, TONE
from views_core import can, latest, decision, ticket_block, tk_df, cross, hero, scoped

NAME = "FreightQuote AI"
MENU = [('copilot', 'AI Copilot', 'bot'), ('route', 'Agent 1 · Route AI', 'route'), ('quotes', 'Agent 2 · Spot Quotes', 'dollar'), ('carriers', 'Agent 3 · Carriers', 'truck'), ('weather', 'Agent 4 · Weather Risk', 'cloud'), ('margin', 'Agent 5 · Margin', 'trend'), ('customs', 'Agent 6 · Customs', 'file')]
SEVC = {2: RED, 1: AMBER, 0: BLUE, -1: TEAL}
FLEET_LEGEND = [("Vessel · no ticket", TEAL), ("Vessel · info ticket", BLUE), ("Vessel · warning ticket", AMBER), ("Vessel · critical ticket", RED), ("Port · low congestion (big dot)", "#8b8d98"), ("Port · congestion 3+", "#d98e04"), ("Port · congestion 4+", "#b4262b"), ("Storm cell (circle)", "#e5484d")]
def dport(ports, key, alt=0): return next((p for p in ports if key.lower() in p.lower()), ports[min(alt, len(ports) - 1)])

# ------------------------------------------------------------------ map layers
def fleet_layers(vessels=True, ports=True, storms=True, v=None):
    layers = []
    if ports:
        p = Q("SELECT port_name, lat, lon, congestion_index ci FROM ports"); layers.append({"type": "points", "id": "ports", "pts": [{"id": r.port_name, "lat": r.lat, "lon": r.lon, "color": "#b4262b" if r.ci >= 4 else "#d98e04" if r.ci >= 3 else "#8b8d98", "r": 3 + r.ci, "tip": f"<b>{r.port_name}</b><br>congestion {r.ci:.1f}", "open": r.port_name, "z": 1} for r in p.itertuples()]})
    if vessels:
        v = Q("SELECT v.shipment_id, v.lat, v.lon, v.route_name, v.progress, s.origin_port o, s.dest_port d FROM vessels v JOIN shipments s USING(shipment_id)") if v is None else v
        tk = Q("SELECT target, MAX(CASE severity WHEN 'Critical' THEN 2 WHEN 'Warning' THEN 1 ELSE 0 END) sv FROM agent_actions WHERE status IN ('open','proposed') GROUP BY target").set_index("target").sv
        layers.append({"type": "points", "id": "vessels", "pts": [{"id": r.shipment_id, "lat": r.lat, "lon": r.lon, "color": SEVC[int(tk.get(r.shipment_id, -1))], "r": 5, "tip": f"<b>{r.shipment_id}</b><br>{r.o} → {r.d}<br>{r.route_name} · {r.progress:.0%}", "open": r.shipment_id, "z": 2} for r in v.itertuples()]})
    if storms: layers.append({"type": "circles", "items": [{"lat": s["lat"], "lon": s["lon"], "r_m": float(s.get("r", 700)) * 1852, "color": RED, "tip": f"Storm · wind {s.get('wind', '?')} m/s"} for s in dom.storms()]})
    return layers

# ------------------------------------------------------------------ Command Center: Operations
def ops(c):
    d = decision(); oa = tk_df(c, limit=500); ex = dom.kpi_extra(); fl = Q("SELECT COUNT(*) n FROM vessels").n[0]
    mp = B.map_(fleet_layers(), h=510, view={"center": [20, 60], "zoom": 2}, key="ops_map", legend=FLEET_LEGEND)
    q = Q("SELECT quote_id, ROUND(final_price) price, margin_pct margin, status FROM freight_quotes ORDER BY rowid DESC LIMIT 7"); p = Q("SELECT port_name, ROUND(congestion_index,1) ci FROM ports ORDER BY congestion_index DESC LIMIT 7")
    h = Q("SELECT substr(ts,12,8) t, score FROM decisions ORDER BY decision_id DESC LIMIT 80").iloc[::-1]; by = oa.assign(agent=oa.type.map(lambda t: dom.ACTION_AGENT.get(t, "engine"))).groupby("agent").size().sort_values() if len(oa) else pd.Series(dtype=int)
    bl = [B.kpis(("Fleet health", d["score"] if d else "…", (d or {}).get("risk", "") + " risk", "Critical" if d and d["risk"] == "High" else "Warning" if d and d["risk"] == "Medium" else "Low"), ("Vessels tracked", int(fl)), ("Open tickets", int((oa.status == "open").sum()) if len(oa) else 0), ("Critical", int(((oa.severity == "Critical") & (oa.status == "open")).sum()) if len(oa) else 0, None, "Critical"),
                  ("Awaiting approval", int((oa.status == "proposed").sum()) if len(oa) else 0, "agent proposals", "Warning"), ("Active storms", len(dom.storms()), None, "Critical" if dom.storms() else None))]
    if d: bl.append({"t": "brief", "text": d["narrative"], "id": d["id"], "can_brief": True})
    bl.append(B.row(B.card("Live fleet · click a vessel or port", mp), B.col(B.card("Latest quotes", B.table(q, ["quote_id", B.num_col("price", "price", "money"), B.num_col("margin", "margin %", "num"), B.pill_col("status")], h=230, open={"col": "quote_id"}, key="lq")), B.card("Most congested ports", B.table(p, ["port_name", B.num_col("ci", "index", "bar", 5)], h=230, open={"col": "port_name"}, key="cp"))), ratio=[3, 2]))
    bl.append(B.row(B.card("Health trend", B.chart(B.line(h.t, h.score, ymin=0, ymax=100, colors=[TEAL]), h=220, key="trend")), B.card("Open tickets by agent", B.chart(B.bar(by.index, by.values, horizontal=True, colors=[BLUE]), h=220, key="byag"))))
    if can(c, "storm"): bl.append(B.button("storm", "Inject a demo storm (tests the re-route loop)", "/api/freight/storm", "ghost", payload={}))
    return bl

# ------------------------------------------------------------------ detail drawer
def detail(c, e):
    k = EN.kind_of(e)
    if k == "shipment":
        if c.get("role") == "Customer":
            cust_id = c.get("customer_id")
            if not cust_id:
                from fastapi import HTTPException
                raise HTTPException(403, "Access denied: no customer ID associated with account.")
            owned = Q("SELECT 1 FROM freight_quotes WHERE shipment_id=? AND customer_id=?", (e, cust_id))
            if not len(owned):
                from fastapi import HTTPException
                raise HTTPException(403, "Access denied: this shipment does not belong to your account.")
        v = Q("SELECT v.lat, v.lon, v.route_name, v.progress, v.chain, s.origin_port o, s.dest_port d, s.carrier, s.status, s.eta, s.weight_kg, s.hs_code, s.predicted_delay_risk risk FROM vessels v RIGHT JOIN shipments s USING(shipment_id) WHERE s.shipment_id=?", (e,))
        if not len(v): return {"title": e, "blocks": [B.callout("Unknown shipment.", "warn")]}
        r = v.iloc[0]; bl = []
        if pd.notna(r.lat):
            w = dom.weather_at(float(r.lat), float(r.lon), allow_live=False); ch = json.loads(r.chain) if r.chain else []
            bl.append(B.kpis(("Progress", f"{r.progress * 100:.0f}%"), ("Wind m/s", round(w["wind"], 1)), ("Wave m", round(w["wave"], 1)), ("Weather", w["level"], w["src"], TONE.get(w["level"]))))
            bl.append(B.map_([{"type": "lines", "lines": [{"pts": ch, "color": TEAL, "w": 3, "dash": "6 6"}]}, {"type": "points", "id": "me", "pts": [{"id": e, "lat": r.lat, "lon": r.lon, "color": "#1f2430", "r": 8, "tip": e}]}, {"type": "circles", "items": [{"lat": s["lat"], "lon": s["lon"], "r_m": float(s.get("r", 700)) * 1852, "color": RED, "tip": "storm"} for s in dom.storms()]}], h=230, view={"center": [float(r.lat), float(r.lon)], "zoom": 3}, key="d_map", legend=[("This port", "#1f2430"), ("Planned voyage route", TEAL), ("Storm cell (red circle)", RED)]))
        else: bl.append(B.callout(f"Not at sea: status {r.status}.", "info"))
        bl.append(B.table([{"field": "Route", "value": f"{r.o} → {r.d}"}, {"field": "Carrier", "value": r.carrier}, {"field": "Status", "value": r.status}, {"field": "ETA", "value": r.eta}, {"field": "Cargo weight", "value": f"{(r.weight_kg or 0) / 1000:,.1f} t"}, {"field": "HS code", "value": r.hs_code}, {"field": "Predicted delay risk", "value": r.risk}], ["field", "value"], h=250, key="d_f"))
        bl.append(ticket_block(c, target=e, title=f"Tickets for {e}", key="d_tk"))
        try:
            import alts; po, pd_ = dom.port_xy(r.o), dom.port_xy(r.d); start = (float(r.lat), float(r.lon)) if pd.notna(r.lat) else (po[0], po[1]) if po else None
            if start and pd_:
                ao = alts.route_alternatives(dom, start, (pd_[0], pd_[1]), tonnes=float(r.weight_kg or 20000) / 1000, interp=dom.interp, dwell=(pd_[3] if pd_ else 0) * .5); bl += alts.route_blocks(ao, e, "Route alternatives" + (" from the vessel's position" if pd.notna(r.lat) else ""))
                if can(c, "approve") and pd.notna(r.lat) and ao: bl.append(B.form("rr_" + e, [{"name": "opt", "label": "Switch to", "type": "select", "options": [f"{i + 1}. {x['label']} · {x['name']}" for i, x in enumerate(ao)], "value": "1. " + ao[0]["label"] + " · " + ao[0]["name"]}, {"name": "shipment", "type": "hidden", "value": e}], "Re-route this voyage", "/api/freight/reroute", title="Apply a route", sub="Needs approve rights. The vessel is re-planned from its current position and the change is audited.", tone="ghost"))
        except Exception as ex: bl.append(B.callout(f"Route alternatives unavailable: {ex}", "info"))
        return {"title": e, "sub": f"{r.o} → {r.d}", "blocks": bl}
    if k == "quote":
        q = Q("SELECT * FROM freight_quotes WHERE quote_id=?", (e,))
        if not len(q): return {"title": e, "blocks": [B.callout("Unknown quote.", "warn")]}
        r = q.iloc[0]
        if c.get("role") == "Customer":
            cust_id = c.get("customer_id")
            if not cust_id or r.customer_id != cust_id:
                from fastapi import HTTPException
                raise HTTPException(403, "Access denied: this quote does not belong to your account.")
            bl = [
                B.kpis(("Price", f"${r.final_price:,.0f}"), ("Status", r.status)),
                B.table([
                    {"field": "Quote ID", "value": r.quote_id},
                    {"field": "Carrier", "value": r.carrier},
                    {"field": "Status", "value": r.status},
                    {"field": "Created At", "value": str(r.created_at)},
                ], ["field", "value"], h=180, key="d_qcust"),
                ticket_block(c, target=e, title="Tickets", key="d_tk")
            ]
            return {"title": e, "sub": r.status, "blocks": bl}
        cost = r.base_cost + r.fuel_surcharge + r.customs_fee + r.insurance
        return {"title": e, "sub": r.status, "blocks": [B.kpis(("Price", f"${r.final_price:,.0f}"), ("Margin", f"{r.margin_pct}%"), ("Status", r.status)), B.chart(B.waterfall(["Base (ML)", "Fuel", "Customs", "Insurance", "Margin"], [r.base_cost, r.fuel_surcharge, r.customs_fee, r.insurance, r.final_price - cost]), "How the price is built", h=260, key="d_wf"), ticket_block(c, target=e, title="Tickets", key="d_tk")]}
    p = Q("SELECT * FROM ports WHERE port_name=?", (e,))
    if not len(p): return {"title": e, "blocks": [B.callout("No detail view for this entity.", "info")]}
    r = p.iloc[0]; w = Q("SELECT current_severity sev, wind_speed, wave_height FROM weather_risks WHERE port_name=?", (e,)); sh = Q("SELECT shipment_id, carrier, status FROM shipments WHERE origin_port=? OR dest_port=? LIMIT 12", (e, e))
    bl = [B.kpis(("Congestion index", round(float(r.congestion_index), 2), None, "Critical" if r.congestion_index >= 4 else None), ("Avg dwell days", r.avg_dwell_days), ("Country", r.country)), B.table(sh, ["shipment_id", "carrier", B.pill_col("status")], title="Shipments via this port", h=250, open={"col": "shipment_id"}, key="d_sh")]
    try:
        import alts; bl += alts.port_alternatives(e)
    except Exception: pass
    if len(w): bl.insert(1, B.kpis(("Weather", w.sev[0], f"wind {w.wind_speed[0]:.0f} km/h · wave {w.wave_height[0]:.1f} m", TONE.get(w.sev[0]))))
    return {"title": e, "sub": f"{r.country} · port", "blocks": bl}

# ------------------------------------------------------------------ pages
def route(c, tab=None):
    v = Q("SELECT COUNT(*) n FROM vessels").n[0]; p = Q("SELECT port_name, country, region, congestion_index ci, avg_dwell_days dw FROM ports"); sh = Q("SELECT status, COUNT(*) n FROM shipments GROUP BY status").set_index("status").n
    ports = sorted(p.port_name); pr = c.get("params") or {}; o = pr.get("o") or dport(ports, "JNPT"); d = pr.get("d") or dport(ports, "Rotterdam", 1); cg = pr.get("cargo") or "Electronics"
    bl = [hero(c, "route_watch", "Route & congestion agent", "Compares Suez and Cape routings with a RandomForest congestion classifier and a live weather check along the lane. Watches every port."),
          B.kpis(("Vessels tracked", int(v)), ("Ports monitored", len(p)), ("Congested (≥ 4)", int((p.ci >= 4).sum()), None, "Warning"), ("Active storms", len(dom.storms())))]
    bl.append(B.card("Live fleet", B.map_(fleet_layers(), h=400, view={"center": [20, 60], "zoom": 2}, key="rt_map", legend=FLEET_LEGEND)))
    top = p.sort_values("ci", ascending=False).head(12)
    bl.append(B.row(B.card("Most congested ports (click)", B.chart(B.bar(top.port_name, top.ci.round(2), horizontal=True, itemcolors=[RED if x >= 4 else AMBER for x in top.ci], max_=5), h=330, open={"bar": "y"}, key="rt_c")), B.card("Shipments by status", B.chart(B.bar(sh.index, sh.values.astype(int), colors=[BLUE]), h=330, key="rt_s"))))
    bl.append(B.form("rt_plan", [{"name": "o", "label": "Origin", "type": "select", "options": ports, "value": o}, {"name": "d", "label": "Destination", "type": "select", "options": ports, "value": d}, {"name": "cargo", "label": "Cargo", "type": "select", "options": list(dom.CARGO), "value": cg}], "Plan route", "page:route", title="Route planner", sub="Same route, weather and congestion logic as the quote pipeline.", tone="ghost"))
    # Heavy ocean-graph work is optional for rendering the Route dashboard.
    # Keep the fleet map, KPIs and route-planner inputs visible even when
    # the navigation graph cannot be loaded or a particular lane has no path.
    try:
        po, pd_ = dom.port_xy(o), dom.port_xy(d)
        if po and pd_ and o != d:
            opts = dom.route_options(po[:2], pd_[:2]); cong = dom.predict_congestion((po[2] + pd_[2]) / 2, (po[3] + pd_[3]) / 2, cg); rows, lines = [], []
            for i, op in enumerate(opts):
                ws = [dom.weather_at(*dom.interp(op["chain"], f), allow_live=False) for f in (0, .25, .5, .75, 1)]; worst = max(ws, key=lambda w: w["wind"]); days = op["nm"] / (14 * 24) + po[3] * .5 + pd_[3] * .5
                rows.append({"route": op["name"], "distance_nm": op["nm"], "transit_days": round(days, 1), "weather_risk": worst["level"], "max_wind_ms": worst["wind"]}); lines.append({"pts": op["chain"], "color": TEAL if i == 0 else AMBER, "w": 4, "dash": None if i == 0 else "8 8", "tip": op["name"]})
            bl.append(B.row(B.card("Route options", B.table(rows, ["route", B.num_col("distance_nm", "nm", "int"), B.num_col("transit_days", "days", "num"), B.pill_col("weather_risk"), B.num_col("max_wind_ms", "max wind m/s", "num")], h=150, key="rt_o"), B.callout(f"Predicted congestion level {cong}/3 (RandomForest: port congestion index, dwell time, month, weekday, cargo).", "info")),
                            B.map_([{"type": "lines", "lines": lines}, {"type": "points", "id": "ends", "pts": [{"id": o, "lat": po[0], "lon": po[1], "color": TEAL, "r": 7, "tip": o}, {"id": d, "lat": pd_[0], "lon": pd_[1], "color": BLUE, "r": 7, "tip": d}]}], h=300, key="rt_pm", view={"bounds": [[min(p[0] for ln in lines[:1] for p in ln["pts"]) - 4, min(p[1] for ln in lines[:1] for p in ln["pts"]) - 4], [max(p[0] for ln in lines[:1] for p in ln["pts"]) + 4, max(p[1] for ln in lines[:1] for p in ln["pts"]) + 4]]}, legend=[("Recommended (sea route)", TEAL), ("Alternative", AMBER)])))
    except Exception:
        import logging, traceback
        logging.getLogger("uvicorn.error").exception("Route planner calculation failed")
        bl.append(B.callout("Route options could not be calculated for the selected ports. Try another origin or destination; check the server diagnostics if the problem continues.", "warn"))
    x = cross(c, p[p.ci >= 4].port_name.tolist(), "route_watch"); bl += [x] if x else []
    bl.append(ticket_block(c, "route_watch", title="Open tickets from this agent", key="tk_rt")); return B.page("Route AI", "Agent 1 · routing, congestion, lane planning", bl, live=10)

def _docs_all(): return sorted({d for s in Q("SELECT docs FROM customs_rules").docs for d in str(s).split(";") if d})
def quote_form(c):
    ports = Q("SELECT port_name FROM ports ORDER BY port_name").port_name.tolist(); cust = Q("SELECT customer_id FROM customers").customer_id.tolist(); alld = _docs_all(); ch, base, *_ = dom._required_docs("HS-8501")
    return B.form("quote", [{"name": "o", "label": "Origin", "type": "select", "options": ports, "value": dport(ports, "JNPT")}, {"name": "d", "label": "Destination", "type": "select", "options": ports, "value": dport(ports, "Rotterdam", 1)}, {"name": "cargo", "label": "Cargo", "type": "select", "options": list(dom.CARGO), "value": "Electronics"},
                                  {"name": "weight", "label": "Weight (tons)", "type": "number", "value": 120, "min": 1, "max": 5000}, {"name": "hs", "label": "HS code", "type": "text", "value": "HS-8501"}, {"name": "customer", "label": "Customer", "type": "select", "options": cust, "value": cust[0]},
                                  {"name": "docs", "label": "Documents you are submitting (tick all that apply; the customs agent checks them against the rules for the HS code)", "type": "checks", "options": alld, "value": base}], "Generate quote", "/api/freight/quote", title="New quote", sub="Runs the 7-agent pipeline: route → weather → (re-plan loop) → pricing (RandomForest) → customs → margin → documents → alerts.")
def quote_result(rid):
    r = Q("SELECT quote_id, result FROM quote_requests WHERE req_id=?", (rid,))
    if not len(r) or not r.result[0]: return [B.callout("The pipeline did not return a result.", "warn")]
    res = json.loads(r.result[0]); bl = [B.callout(res.get("narrative", ""), "success", "Pipeline result"), B.kpis(("Price (USD)", f"{res.get('price', 0):,.0f}"), ("Decision", res.get("decision")), ("Margin", f"{res.get('margin_pct')}%"), ("Transit", f"{res.get('days')} days"))]
    if res.get("missing"): bl.append(B.callout("Missing documents: " + ", ".join(res["missing"]) + ". Attach them on the Customs page and the quote re-runs.", "warn"))
    tr = Q("SELECT agent, risk, score, latency_ms ms, summary FROM agent_runs WHERE cycle_id LIKE ? ORDER BY run_id", (f"Q{rid}-%",)); bl.append(B.table(tr, ["agent", B.pill_col("risk"), B.num_col("score", "score", "num"), B.num_col("ms", "ms", "int"), "summary"], title="Agent trace for this quote", h=260, key="q_tr"))
    ddir = os.path.join(dom.DATA_DIR, "docs"); files = [p for p in (sorted(os.listdir(ddir)) if os.path.isdir(ddir) else []) if p.startswith(r.quote_id[0])]
    for p in files: bl.append(B.button("dl_" + p, "Download " + p, f"/api/doc/{p}", "ghost", payload={"download": True}))
    return bl
def quotes(c, tab=None):
    role = c.get("role")
    cust_id = c.get("customer_id")
    if role == "Customer":
        if not cust_id:
            return B.page("My Quotes", "Your active and past freight quotes", [
                B.callout("No customer account is currently linked to your profile. Please contact an administrator to link your customer account.", "info")
            ])
        q = Q("SELECT quote_id, carrier, final_price price, status, created_at FROM freight_quotes WHERE customer_id=? ORDER BY created_at DESC", (cust_id,))
        bl = [hero(c, "quote_watch", "My Quotes", "View spot quotes and status updates for your shipments.")]
        if len(q) == 0:
            bl.append(B.callout("You do not have any quotes in your account yet.", "info"))
        else:
            bl.append(B.kpis(("My Quotes", len(q)), ("Approved", int((q.status == "Approved").sum()), None, "Low"), ("Pending", int((q.status == "Pending Approval").sum()), None, "Warning")))
            bl.append(B.table(q, ["quote_id", "carrier", B.num_col("price", "price", "money"), B.pill_col("status"), "created_at"], title="My Quotes · click a row", h=340, open={"col": "quote_id"}, key="qt_t", search=True))
        return B.page("My Quotes", "Quotes linked to your account", bl, live=10)

    bl = [hero(c, "quote_watch", "Spot quotes agent", "Every quote goes through one pipeline: ML price (RandomForest) + 12% fuel + port fees + 15% operating cost, target margin ~20%. Quotes above $50,000 need admin approval.")]
    if can(c, "quote"): bl.append(quote_form(c))
    if can(c, "view"):
        q = Q("SELECT quote_id, customer_id, carrier, final_price price, margin_pct margin, status, created_at FROM freight_quotes ORDER BY created_at DESC, quote_id DESC LIMIT 400"); st = q.status.value_counts()
        bl.append(B.kpis(("Quotes", len(q)), ("Approved", int((q.status == "Approved").sum()), None, "Low"), ("Pending approval", int((q.status == "Pending Approval").sum()), None, "Warning"), ("On hold", int((q.status == "On Hold").sum()), None, "Critical")))
        bl.append(B.row(B.card("Quote decisions", B.chart(B.pie(st.to_dict(), colors={"Approved": TEAL, "Pending Approval": AMBER, "On Hold": RED, "Rejected": "#8b8d98"}), h=280, key="qt_p")), B.card("Margin % across quotes", B.chart(B.hist(q.margin, 24, BLUE), h=280, key="qt_h"))))
        bl.append(B.table(q.head(30), ["quote_id", "customer_id", "carrier", B.num_col("price", "price", "money"), B.num_col("margin", "margin %", "num"), B.pill_col("status"), "created_at"], title="Latest quotes · click a row", h=340, open={"col": "quote_id"}, key="qt_t", search=True))
        bl.append(ticket_block(c, "quote_watch", title="Open tickets from this agent", key="tk_qt"))
    return B.page("Spot Quotes", "Agent 2 · the 7-agent quote pipeline", bl, live=10)

def carriers(c, tab=None):
    k = Q("SELECT name, rating, on_time_pct, avg_cost_index, risk_level FROM carriers"); k["score"] = (k.rating / 5 * 40 + k.on_time_pct * .4 - k.avg_cost_index * 12 - (k.risk_level == "High") * 8).round(1)
    s = Q("SELECT carrier name, COUNT(*) shipments, SUM(status LIKE '%Delay%' OR status='Customs Hold') problems, AVG(predicted_delay_risk) delay_risk FROM shipments GROUP BY carrier"); k = k.merge(s, on="name", how="left").fillna({"shipments": 0, "problems": 0})
    top = k.sort_values("score", ascending=False).head(12); rc = {"Low": TEAL, "Medium": AMBER, "High": RED}
    sc = {t: [{"x": a.on_time_pct, "y": a.avg_cost_index, "label": a.name, "size": 6 + a.shipments ** .5} for a in k[k.risk_level == t].itertuples()] for t in ["Low", "Medium", "High"] if (k.risk_level == t).any()}
    bl = [hero(c, "route_watch", "Carrier intelligence", "Score = rating × 40% + on-time % × 0.4 − cost-index penalty − 8 points for High risk. Shipments and delays come from the live shipments table."),
          B.kpis(("Carriers", len(k)), ("High-risk carriers", int((k.risk_level == "High").sum()), None, "Warning"), ("Avg on-time", f"{k.on_time_pct.mean():.1f}%"), ("Best score", k.sort_values("score").name.iloc[-1])),
          B.row(B.card("Top carriers by agent score", B.chart(B.bar(top.name, top.score, horizontal=True, colors=[TEAL], sort=True), h=360, key="ca_t")), B.card("On-time vs cost index (size = shipments)", B.chart(B.scatter(sc, "on-time %", "cost index", colors=rc, sized=True), h=360, key="ca_s"))),
          B.table(k.sort_values("score", ascending=False), ["name", B.num_col("score", "score", "bar", 100), "rating", B.num_col("on_time_pct", "on-time %", "num"), B.num_col("avg_cost_index", "cost idx", "num"), B.pill_col("risk_level", "risk"), B.num_col("shipments", "shipments", "int"), B.num_col("problems", "problems", "int")], title="Carrier league table", h=340, key="ca_tb", search=True)]
    return B.page("Carriers", "Agent 3 · ranking and risk", bl, live=15)

METRICS = {"wind": ("Wind speed", [("Calm / light  < 8 m/s", TEAL), ("Strong  8–15 m/s", AMBER), ("Storm-force  > 15 m/s", RED)]),
           "waves": ("Wave height", [("Slight  < 2.5 m", TEAL), ("Rough  2.5–4 m", AMBER), ("Very rough  > 4 m", RED)]),
           "rain": ("Rain now", [("Dry", "#8b8d98"), ("Light rain  < 2.5 mm/h", BLUE), ("Heavy rain  ≥ 2.5 mm/h", VIOLET)]),
           "temp": ("Temperature", [("Cold  < 10 °C", BLUE), ("Mild  10–30 °C", TEAL), ("Hot  30–35 °C", AMBER), ("Very hot  > 35 °C", RED)])}
def _wxcolor(metric, r):
    if metric == "wind": return RED if r.wind_ms > 15 else AMBER if r.wind_ms >= 8 else TEAL
    if metric == "waves": w = r.wave_m if r.wave_m == r.wave_m and r.wave_m is not None else 0; return RED if w > 4 else AMBER if w >= 2.5 else TEAL
    if metric == "rain": p = r.precip_mm or 0; return VIOLET if p >= 2.5 else BLUE if p > 0 else "#8b8d98"
    t = r.temp_c if r.temp_c is not None else 25; return RED if t > 35 else AMBER if t >= 30 else TEAL if t >= 10 else BLUE
def wx_layers(metric, hi):
    import weather as WX
    t = WX.table(); layers = []
    if len(t):
        pts = []
        for r in t.itertuples():
            w = r.wave_m if r.wave_m is not None and r.wave_m == r.wave_m else None
            pts.append({"id": r.port_name, "lat": r.lat, "lon": r.lon, "color": _wxcolor(metric, r), "r": 4 + min(9, r.wind_ms / 2.2), "z": 1, "open": r.port_name,
                        "tip": f"<b>{r.port_name}</b> · {WX.WMO.get(r.wcode, '')}<br>wind {r.wind_ms:.1f} m/s ({r.wind_ms * 3.6:.0f} km/h), gusts {r.gust_ms or 0:.0f} m/s<br>waves {'-' if w is None else f'{w:.1f} m'} · rain {r.precip_mm or 0:.1f} mm/h · {r.temp_c:.0f} °C<br><i>real reading {r.ts[11:16]} UTC</i>"})
        layers.append({"type": "points", "id": "wx_ports", "pts": pts})
    else:
        w = Q("SELECT w.port_name, w.current_severity sev, w.wind_speed, w.wave_height, w.temperature, p.lat, p.lon FROM weather_risks w JOIN ports p USING(port_name)")
        layers.append({"type": "points", "id": "wx_ports", "pts": [{"id": r.port_name, "lat": r.lat, "lon": r.lon, "color": RED if r.sev == "High" else AMBER if r.sev == "Medium" else TEAL, "r": 4 + min(9, r.wind_speed / 8), "z": 1, "open": r.port_name, "tip": f"<b>{r.port_name}</b><br>wind {r.wind_speed:.0f} km/h · waves {r.wave_height:.1f} m<br><i>simulated, not a real reading</i>"} for r in w.itertuples()]})
    v = Q("SELECT v.shipment_id, v.lat, v.lon, v.route_name, s.origin_port o, s.dest_port d FROM vessels v JOIN shipments s USING(shipment_id)"); hs = set(hi)
    layers.append({"type": "points", "id": "wx_v", "pts": [{"id": r.shipment_id, "lat": r.lat, "lon": r.lon, "color": "#1f2430" if r.shipment_id in hs else "#6b7080", "r": 4.5 if r.shipment_id in hs else 3, "z": 2, "open": r.shipment_id, "tip": f"<b>{r.shipment_id}</b> {r.o} → {r.d}" + ("<br><b>in storm-force wind</b>" if r.shipment_id in hs else "")} for r in v.itertuples()]})
    layers.append({"type": "circles", "items": [{"lat": s["lat"], "lon": s["lon"], "r_m": float(s.get("r", 700)) * 1852, "color": RED, "tip": f"Storm cell · wind {s.get('wind', '?')} m/s"} for s in dom.storms()]})
    return layers
def weather(c, tab=None):
    import weather as WX
    p = c.get("params") or {}; metric = p.get("metric") if p.get("metric") in METRICS else "wind"; st = WX.status()
    r = (latest("weather_watch") or {}).get("out", {}); hi = r.get("vessels_high", []); w = Q("SELECT port_name, current_severity sev, wind_speed, wave_height, temperature, forecast FROM weather_risks")
    src = (f"Real · Open-Meteo, {st['ok']}/{st['total']} ports, updated {st['ts'][11:16]} UTC ({st['age_min']:.0f} min ago)" if st["live"] else "Simulated · Open-Meteo not reachable" + (f" ({st['err']})" if st["err"] else "") + ". Press Refresh to retry.")
    lg = list(METRICS[metric][1]) + [("Big dot = port (size = wind)", "#8b8d98"), ("Black dot = vessel in storm-force wind", "#1f2430"), ("Grey dot = other vessel", "#6b7080"), ("Red circle = storm cell", "#e5484d")]
    pick = {"t": "form", "id": "wx_pick", "fields": [{"name": "metric", "label": "Colour ports by", "type": "select", "options": list(METRICS), "value": metric}], "submit": "Show", "url": "page:weather", "title": None, "tone": "ghost", "inline": True}
    hot = w.sort_values("wind_speed", ascending=False)
    bl = [B.callout(src, "success" if st["live"] else "warning", "Weather data source"), hero(c, "weather_watch", "Weather risk agent", "Real wind, wave, rain and temperature from Open-Meteo for every port (refreshed every 20 min) plus injected storm cells. Vessels in High risk (wind > 15 m/s) get a re-route ticket; the system re-plans the voyage once it is approved."),
          B.kpis(("Vessels in High risk", len(hi), None, "Critical" if hi else None), ("Active storms", len(dom.storms())), ("Ports with weather data", len(w)), ("Ports in High risk", int((w.sev == "High").sum()), None, "Warning"), ("Data", "Real" if st["live"] else "Simulated", None, "Low" if st["live"] else "Warning")),
          pick, B.card(f"Port weather · coloured by {METRICS[metric][0].lower()} · click a port or vessel", B.map_(wx_layers(metric, hi), h=470, view={"center": [20, 60], "zoom": 2}, key="wx_map", legend=lg)),
          B.row(B.card("Strongest port winds (km/h)", B.chart(B.bar(hot.head(12).port_name, hot.head(12).wind_speed.round(0), horizontal=True, colors=[AMBER], sort=True), h=330, open={"bar": "y"}, key="wx_w")), B.card("Port weather risk levels", B.chart(B.pie({str(k): int(v) for k, v in w.sev.value_counts().items()}, colors={"High": RED, "Medium": AMBER, "Low": TEAL}), h=330, key="wx_d")))]
    if hi: bl.append(B.table(Q("SELECT shipment_id, lat, lon, route_name FROM vessels WHERE shipment_id IN (%s)" % ",".join("?" * len(hi[:50])), hi[:50]), ["shipment_id", B.num_col("lat", "lat", "num"), B.num_col("lon", "lon", "num"), "route_name"], title="Vessels in High-risk weather now · click to inspect", h=240, open={"col": "shipment_id"}, key="wx_hv"))
    bl.append(B.table(w.sort_values("wind_speed", ascending=False).head(60).rename(columns={"wind_speed": "wind km/h", "wave_height": "waves m", "temperature": "temp °C"}), ["port_name", B.pill_col("sev"), "wind km/h", "waves m", "temp °C", "forecast"], title="Port weather (real where the forecast says Open-Meteo live)", h=300, open={"col": "port_name"}, key="wx_t"))
    if can(c, "run"): bl.append(B.button("wx_ref", "Refresh real weather now", "/api/weather/refresh", "ghost", payload={}))
    bl.append(B.table(pd.DataFrame(dom.feed_status(), columns=["feed", "status"]), title="Data sources", h=190, key="wx_f"))
    if can(c, "storm"):
        PRESET_STORMS = {"Red Sea": (18.0, 40.0), "Arabian Sea": (16.0, 66.0), "Bay of Bengal": (14.0, 88.0), "South China Sea": (14.0, 114.0), "Mediterranean": (35.0, 18.0)}; bl.append(B.form("storm", [{"name": "area", "label": "Area", "type": "select", "options": list(PRESET_STORMS), "value": "Red Sea"}, {"name": "wind", "label": "Wind m/s", "type": "range", "min": 15, "max": 35, "value": 24}], "Inject storm", "/api/freight/storm", title="Inject a storm", sub="Adds a storm cell for testing the re-route loop; it does not change the real readings."))
    bl.append(ticket_block(c, "weather_watch", title="Open tickets from this agent", key="tk_wx")); return B.page("Weather Risk", "Agent 4 · real wind, waves, rain, storms", bl, live=10)

def margin(c, tab=None):
    q = Q("SELECT quote_id, customer_id, base_cost, fuel_surcharge fuel, customs_fee, insurance, final_price price, margin_pct margin, status FROM freight_quotes WHERE base_cost>0 ORDER BY created_at DESC, quote_id DESC LIMIT 600")
    bl = [hero(c, "quote_watch", "Margin & approvals agent", "Margin = (price − cost) ÷ price with tiered targets (Platinum 18%, Gold 20%, Silver/Bronze 22–24%). Quotes > $50,000 or < 10% margin wait for an approver."),
          B.kpis(("Average margin", f"{q.margin.mean():.1f}%"), ("Below 10% margin", int((q.margin < 10).sum()), None, "Warning"), ("Over $50,000", int((q.price > 50000).sum())), ("Revenue in quotes", f"${q.price.sum():,.0f}"))]
    sc = {s: [{"x": a.base_cost, "y": a.price, "label": a.quote_id} for a in q[q.status == s].head(250).itertuples()] for s in q.status.unique()}; cm = q.groupby("customer_id").margin.mean().round(1).sort_values().head(10)
    bl.append(B.row(B.card("Price vs base cost (click a dot)", B.chart(B.scatter(sc, "base cost", "price", colors={"Approved": TEAL, "Pending Approval": AMBER, "On Hold": RED, "Rejected": "#8b8d98"}), h=330, open={"scatter": True}, key="mg_s")), B.card("Lowest-margin customers", B.chart(B.bar(cm.index, cm.values, horizontal=True, colors=[AMBER]), h=330, key="mg_c"))))
    if len(q):
        sel = (c.get("params") or {}).get("q") or q.quote_id.iloc[0]; r = q[q.quote_id == sel]; r = r.iloc[0] if len(r) else q.iloc[0]; cost = r.base_cost + r.fuel + r.customs_fee + r.insurance
        bl.append({"t": "form", "id": "mg_pick", "fields": [{"name": "q", "label": "Cost build-up for quote", "type": "select", "options": q.quote_id.tolist()[:200], "value": r.quote_id}], "submit": "Show", "url": "page:margin", "title": None, "tone": "ghost", "inline": True})
        bl.append(B.card(f"{r.quote_id}: how the price is built", B.chart(B.waterfall(["Base (ML)", "Fuel 12%", "Customs", "Insurance", "Margin"], [r.base_cost, r.fuel, r.customs_fee, r.insurance, r.price - cost]), h=300, key="mg_w")))
    bl.append(ticket_block(c, "quote_watch", ["quote_approval", "margin_review"], title="Margin actions", key="tk_mg")); return B.page("Margin", "Agent 5 · margin and pricing tiers", bl, live=10)

def customs(c, tab="customs"):
    tabs = [{"id": "customs", "label": "Customs & tariff"}, {"id": "docs", "label": "Documents (OCR)"}]
    if tab == "docs":
        ddir = os.path.join(dom.DATA_DIR, "docs"); files = sorted(os.listdir(ddir), reverse=True) if os.path.isdir(ddir) else []
        bl = [B.callout("PDFs are generated by the Document agent for every quote (Bill of Lading, Packing List). Upload a document to read its fields and attach it to the matching quote.", "info"), B.kpis(("Generated documents", len(files)))]
        bl += [B.button("dl_" + f, "Download " + f, f"/api/doc/{f}", "ghost", payload={"download": True}) for f in files[:10]]
        bl.append(B.form("readdoc", [{"name": "file", "label": "PDF or TXT", "type": "file"}], "Read document", "/api/freight/readdoc", title="Read an uploaded document (OCR / text extraction)", tone="primary")); return B.page("Customs", "Agent 6 · documents", bl, tabs, live=0)
    h = Q("SELECT shipment_id, hs_code, origin_port, dest_port, carrier FROM shipments WHERE status='Customs Hold' LIMIT 200"); rules = Q("SELECT * FROM customs_rules ORDER BY chapter"); rows = []
    for r in h.head(30).itertuples():
        ch, docs, rs, duty, goods = dom._required_docs(r.hs_code); miss = [d for d in docs if d not in dom._attached(r.shipment_id)]; rows.append({"shipment": r.shipment_id, "hs": r.hs_code, "goods": goods, "duty_pct": duty, "missing": ", ".join(miss), "route": f"{r.origin_port} → {r.dest_port}"})
    bl = [hero(c, "customs_watch", "Customs & tariff agent", "Requirements come from the customs_rules table by HS chapter. A hold lifts as soon as every required document is attached: the quote re-runs and the shipment clears."),
          B.kpis(("Shipments on hold", len(h), None, "Warning" if len(h) else None), ("Quotes on hold", int(Q("SELECT COUNT(*) n FROM freight_quotes WHERE status='On Hold'").n[0])), ("Rules in force", len(rules))),
          B.table(rows, ["shipment", "hs", "goods", B.num_col("duty_pct", "duty %", "num"), "missing", "route"], title="Held shipments and what each is missing · click a row", h=300, open={"col": "shipment"}, key="cu_h"), B.table(rules, list(rules.columns), title="Customs rules", h=230, key="cu_r")]
    if can(c, "resolve") or can(c, "quote"):
        held = Q("SELECT shipment_id ref FROM shipments WHERE status='Customs Hold' UNION SELECT quote_id FROM freight_quotes WHERE status='On Hold' LIMIT 200").ref.tolist(); alld = _docs_all()
        if held and alld: bl.append(B.form("attach", [{"name": "ref", "label": "Shipment or quote on hold", "type": "select", "options": held, "value": held[0]}, {"name": "doc", "label": "Document", "type": "select", "options": alld, "value": alld[0]}], "Attach document", "/api/freight/attach", title="Attach a document", sub="The Customs agent re-checks on its next cycle; quotes re-run immediately."))
        elif held and not alld: bl.append(B.callout("Document requirements are not available yet. Reload the page after initialization.", "warning"))
    if can(c, "view"): bl.append(ticket_block(c, "customs_watch", title="Open tickets from this agent", key="tk_cu"))
    return B.page("Customs", "Agent 6 · rules, holds, documents", bl, tabs, live=12)

PAGES = {"route": route, "quotes": quotes, "carriers": carriers, "weather": weather, "margin": margin, "customs": customs}


In [ ]:
%%writefile /content/app/domain_freight.py
"""FreightQuote AI live domain (PRS v3): 7 agents, ML pricing + congestion models, vessel tracking, reroute loop, PDF documents."""
import sys, os, re, json, math, random, time, datetime, zlib
import numpy as np, pandas as pd, requests
from live_engine import AgentResult, risk_label, conn, now, _DB_LOCK, set_state, get_state, emit, run_cycle, _log_run, act, init_live_tables
from config import DB_PATH
KIND = "Freight"; NAME = "FreightQuote AI"; ENTITY_COL = "shipment_id"; SMS = False; ESCALATE_TO = "Operations Manager"
OWNER = {"quote_anomaly": "Finance Lead", "reroute": "Operations Manager", "quote_approval": "Admin", "customs_docs": "Customs Officer", "congestion_notice": "Operations Manager", "margin_review": "Finance Lead", "quote_hold": "Freight Broker"}
ACTION_AGENT = {"quote_anomaly": "anomaly_watch", "reroute": "weather_watch", "customs_docs": "customs_watch", "congestion_notice": "route_watch", "quote_approval": "quote_watch", "margin_review": "quote_watch", "quote_hold": "quote_watch"}
DATA_DIR = os.path.dirname(DB_PATH); TICK_NM = 84.0       # 1 tick = 1 sailing hour
CARGO = {"Electronics": 1.15, "Pharmaceuticals": 1.35, "Automotive Parts": 1.1, "Textiles": .95, "Heavy Machinery": 1.3, "Perishables": 1.4, "General": 1.0, "Hazardous": 1.6, "Bulk": .8}
CODES = {k: i for i, k in enumerate(CARGO)}
def Q(sql, p=()):
    with _DB_LOCK, conn() as c: return pd.read_sql(sql, c, params=p)

# ---------------------------------------------------------------- geo + routing
def nm(a, b):
    la1, lo1, la2, lo2 = map(math.radians, (a[0], a[1], b[0], b[1])); h = math.sin((la2 - la1) / 2) ** 2 + math.cos(la1) * math.cos(la2) * math.sin((lo2 - lo1) / 2) ** 2
    return 3440.065 * 2 * math.asin(min(1, math.sqrt(h)))
def chain_nm(ch): return sum(nm(ch[i], ch[i + 1]) for i in range(len(ch) - 1))
def interp(ch, frac):
    legs = [nm(ch[i], ch[i + 1]) for i in range(len(ch) - 1)]; tot = sum(legs) or 1; d = min(max(frac, 0), 1) * tot
    for i, L in enumerate(legs):
        if d <= L or i == len(legs) - 1:
            t = d / L if L else 0; return (ch[i][0] + (ch[i + 1][0] - ch[i][0]) * t, ch[i][1] + (ch[i + 1][1] - ch[i][1]) * t)
        d -= L
SUEZ_S, SUEZ_N, CAPE = (12.6, 43.4), (30.0, 32.55), (-34.4, 18.5)
_ACC = {}
def _access(xy):
    """Sea entry point for river / bay ports (Hamburg, Wuhan, Houston ...), looked up by the port's coordinates."""
    if not _ACC:
        try:
            import seanav
            for n, la, lo in Q("SELECT port_name, lat, lon FROM ports").itertuples(index=False): _ACC[(round(float(la), 2), round(float(lo), 2))] = seanav.access_for(n)
        except Exception: _ACC[None] = None
    return _ACC.get((round(float(xy[0]), 2), round(float(xy[1]), 2)))
def best_route(o, d):
    import seanav
    r = seanav.sea().route((float(o[0]), float(o[1])), (float(d[0]), float(d[1])), (), (), _access(o), _access(d))
    return {"name": "via " + " + ".join(r["via"]) if r["via"] else "direct sea route", "chain": r["pts"], "nm": round(r["nm"]), "via": r["via"]}
def route_options(o, d, avoid=()):
    """Real sea routes (water only: see seanav.py). avoid may contain 'suez', 'panama', 'malacca', 'hormuz', 'bosphorus' and/or 'storm' (steer around the active storms)."""
    import seanav
    S = seanav.sea(); blocked = tuple(a for a in avoid if a in seanav.BLOCK_GROUPS); st = tuple((s["lat"], s["lon"], s.get("r", 700)) for s in storms()) if "storm" in avoid else ()
    opts = S.options((float(o[0]), float(o[1])), (float(d[0]), float(d[1])), blocked, st, _access(o), _access(d))
    if not opts and st: opts = S.options((float(o[0]), float(o[1])), (float(d[0]), float(d[1])), blocked, (), _access(o), _access(d))
    return [{"name": x["name"], "chain": x["pts"], "nm": round(x["nm"]), "via": x["via"]} for x in opts]
def port_xy(name):
    r = Q("SELECT lat, lon, congestion_index ci, avg_dwell_days dw, country FROM ports WHERE port_name=? LIMIT 1", (name,))
    return (float(r.lat[0]), float(r.lon[0]), float(r.ci[0]), float(r.dw[0]), r.country[0]) if len(r) else None

# ---------------------------------------------------------------- weather (Open-Meteo live + cached port rows + simulated storms)
_WX = {}; _OFF = [0.0]
def storms(): return get_state("storms", []) or []
def inject_storm(lat=None, lon=None, wind=22.0):
    s = storms(); s.append({"lat": lat if lat is not None else random.uniform(-30, 45), "lon": lon if lon is not None else random.uniform(-60, 120), "r": 700, "wind": wind, "ts": now()}); set_state("storms", s[-6:])
def weather_at(lat, lon, allow_live=True):
    k = (round(lat / 2) * 2, round(lon / 2) * 2); hit = _WX.get(k); wind, wave, src = 6.0, 1.2, "cached"
    if hit and time.time() - hit[0] < 900: wind, wave, src = hit[1:]
    elif allow_live and time.time() > _OFF[0]:
        try:
            r = requests.get(f"https://api.open-meteo.com/v1/forecast?latitude={lat}&longitude={lon}&current=wind_speed_10m&wind_speed_unit=ms", timeout=3).json()["current"]
            wind, src = float(r["wind_speed_10m"]), "Open-Meteo live"
            try: wave = float(requests.get(f"https://marine-api.open-meteo.com/v1/marine?latitude={lat}&longitude={lon}&current=wave_height", timeout=3).json()["current"]["wave_height"] or 1.2)
            except Exception: wave = 1.2
            _WX[k] = (time.time(), wind, wave, src)
        except Exception: _OFF[0] = time.time() + 120; wind, src = 5 + 4 * random.random(), "simulated (offline)"
    else: wind, src = 5 + 4 * random.random(), "simulated (offline)"
    for s in storms():
        d = nm((lat, lon), (s["lat"], s["lon"]))
        if d < s["r"]: wind = max(wind, s["wind"] * (1 - .5 * d / s["r"])); src += " + storm"
    return {"wind": round(wind, 1), "wave": round(wave, 1), "src": src, "level": "High" if wind > 15 else "Medium" if wind >= 8 else "Low"}
def wscore(level): return {"High": 1.0, "Medium": .5, "Low": .12}[level]

# ---------------------------------------------------------------- ML models
def _gen_rates(n=4000):
    R = np.random.default_rng(42); dist = R.uniform(500, 12000, n); w = R.uniform(1, 600, n); cg = R.integers(0, len(CARGO), n); cong = R.integers(0, 4, n); mult = np.array(list(CARGO.values()))
    cost = (250 + .21 * dist) * (.6 + .012 * w) * (1 + .08 * cong) * mult[cg] * R.normal(1, .06, n)
    return pd.DataFrame({"distance_nm": dist, "cargo_weight": w, "cargo_type": cg, "congestion_level": cong, "freight_cost_usd": cost})
_ML = {}
def pricing_model():
    from sklearn.ensemble import RandomForestRegressor; from sklearn.model_selection import train_test_split; from sklearn.metrics import r2_score, mean_squared_error; import joblib
    path = os.path.join(DATA_DIR, "pricing_rf.joblib")
    if "rf" in _ML: return _ML
    if os.path.exists(path): _ML.update(joblib.load(path)); return _ML
    src = "bootstrap synthetic (feed a Kaggle freight-rate CSV to replace)"
    try:
        d = Q("SELECT * FROM freight_rates")
        if len(d) >= 300: src = "fed dataset (freight_rates table)"
        else: d = _gen_rates()
    except Exception: d = _gen_rates()
    X, y = d[["distance_nm", "cargo_weight", "cargo_type", "congestion_level"]], d.freight_cost_usd
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=.2, random_state=42); rf = RandomForestRegressor(100, random_state=42, n_jobs=-1).fit(Xtr, ytr)
    m = {"r2": round(r2_score(yte, rf.predict(Xte)), 3), "rmse": round(float(np.sqrt(mean_squared_error(yte, rf.predict(Xte)))), 1), "source": src, "importance": dict(zip(X.columns, rf.feature_importances_.round(3).tolist()))}
    _ML.update(rf=rf, metrics=m); joblib.dump({"rf": rf, "metrics": m}, path)
    with _DB_LOCK, conn() as c: c.execute("UPDATE ml_models SET is_active=0 WHERE model_name='rf_freight_price'"); c.execute("INSERT INTO ml_models(model_name,metrics,trained_at,is_active) VALUES(?,?,?,1)", ("rf_freight_price", json.dumps(m), now()))
    return _ML
def congestion_model():
    from sklearn.ensemble import RandomForestClassifier
    if "cg" in _ML: return _ML["cg"]
    R = np.random.default_rng(3); n = 3000; ci = R.uniform(1, 4.8, n); dw = R.uniform(1.5, 7, n); mo = R.integers(1, 13, n); dow = R.integers(0, 7, n); cg = R.integers(0, len(CARGO), n)
    lvl = np.clip(np.round((ci - 1) / 1.3 + .25 * (dw > 5) + .35 * np.isin(mo, [10, 11, 12, 1, 2]) + R.normal(0, .3, n)), 0, 3).astype(int)
    clf = RandomForestClassifier(100, random_state=42).fit(np.c_[ci, dw, mo, dow, cg], lvl); _ML["cg"] = clf; return clf
def predict_congestion(ci, dw, cargo="General"):
    t = datetime.date.today(); return int(congestion_model().predict([[ci, dw, t.month, t.weekday(), CODES.get(cargo, 6)]])[0])

# ---------------------------------------------------------------- bootstrap, simulator, vessel tracking
def bootstrap():
    from scale_data import scale_freight; scale_freight()
    with _DB_LOCK, conn() as c:
        c.executescript("""CREATE TABLE IF NOT EXISTS quote_requests(req_id INTEGER PRIMARY KEY AUTOINCREMENT, ts TEXT, origin TEXT, dest TEXT, cargo TEXT, weight_t REAL, hs_code TEXT, customer_id TEXT, language TEXT, status TEXT DEFAULT 'new', quote_id TEXT, result TEXT, docs_provided TEXT);
        CREATE TABLE IF NOT EXISTS shipment_docs(ref TEXT, doc TEXT, ts TEXT, by TEXT, UNIQUE(ref, doc));
        CREATE TABLE IF NOT EXISTS vessels(shipment_id TEXT PRIMARY KEY, lat REAL, lon REAL, progress REAL, chain TEXT, route_name TEXT, speed_kn REAL, updated TEXT);
        CREATE TABLE IF NOT EXISTS ml_models(model_id INTEGER PRIMARY KEY AUTOINCREMENT, model_name TEXT, metrics TEXT, trained_at TEXT, is_active INT);
        CREATE TABLE IF NOT EXISTS customs_rules(chapter TEXT PRIMARY KEY, goods TEXT, duty_pct REAL, docs TEXT, restricted INT);""")
        try: c.execute("ALTER TABLE quote_requests ADD COLUMN docs_provided TEXT")
        except Exception: pass
        if not c.execute("SELECT 1 FROM customs_rules").fetchone():
            base = "Commercial Invoice;Packing List;Bill of Lading"
            for ch, g, d, ex, rs in [("30", "Pharmaceuticals", 5, "Import License;Drug Control Certificate", 1), ("84", "Machinery", 7.5, "Certificate of Origin", 0), ("85", "Electronics", 10, "BIS Certificate;Certificate of Origin", 0), ("87", "Vehicles and parts", 15, "Homologation Certificate", 0),
                                     ("10", "Cereals", 12, "Phytosanitary Certificate", 0), ("62", "Apparel", 12, "Certificate of Origin", 0), ("27", "Mineral fuels", 5, "Hazmat Declaration;IMDG Certificate", 1), ("39", "Plastics", 8, "Certificate of Origin", 0), ("72", "Iron and steel", 7.5, "Mill Test Certificate", 0), ("22", "Beverages", 20, "Excise Permit", 1)]:
                c.execute("INSERT INTO customs_rules VALUES(?,?,?,?,?)", (ch, g, d, base + ";" + ex, rs))
        n = c.execute("SELECT COUNT(*) FROM vessels").fetchone()[0]
        if n == 0:
            R = random.Random(5)
            for sid, o, d in c.execute("SELECT shipment_id, origin_port, dest_port FROM shipments WHERE status IN ('In Transit','Port Congestion Delay','Anchorage Pending')").fetchall():
                po, pd_ = port_xy(o), port_xy(d)
                if not po or not pd_: continue
                opt = best_route(po[:2], pd_[:2]); p = R.uniform(.05, .9); la, lo = interp(opt["chain"], p)
                c.execute("INSERT OR REPLACE INTO vessels VALUES(?,?,?,?,?,?,?,?)", (sid, la, lo, p, json.dumps(opt["chain"]), opt["name"], R.uniform(12, 17), now()))
    pricing_model(); congestion_model()

def tick():
    reasons = []; R = random
    with _DB_LOCK, conn() as c:
        real = set()
        for sid, p, chain, spd in c.execute("SELECT shipment_id, progress, chain, speed_kn FROM vessels").fetchall():
            if sid in real: continue
            ch = json.loads(chain); tot = chain_nm(ch) or 1; p2 = p + spd * 1.0 / tot
            if p2 >= 1:
                c.execute("DELETE FROM vessels WHERE shipment_id=?", (sid,)); c.execute("UPDATE shipments SET status='Delivered' WHERE shipment_id=?", (sid,)); emit("delivered", sid); continue
            la, lo = interp(ch, p2); c.execute("UPDATE vessels SET lat=?, lon=?, progress=?, updated=? WHERE shipment_id=?", (la, lo, p2, now(), sid))
        _tickn = get_state("tickn", 0) + 1; set_state("tickn", _tickn)
        ports = c.execute("SELECT port_name, lat, lon FROM ports ORDER BY port_id").fetchall()
        try: import weather as _WXL; _fresh = _WXL.fresh_ports()
        except Exception: _fresh = set()
        for i in range(12):                                         # round-robin refresh of port weather + congestion drift
            nme, la, lo = ports[(_tickn * 12 + i) % len(ports)]; w = weather_at(la, lo, allow_live=False)
            if nme in _fresh:                                       # real Open-Meteo reading is recent: keep it, only drift congestion
                c.execute("UPDATE ports SET congestion_index=MIN(5,MAX(0.5,congestion_index+?)) WHERE port_name=?", (R.uniform(-.12, .12), nme)); continue
            c.execute("INSERT OR REPLACE INTO weather_risks(port_name,current_severity,forecast,wind_speed,wave_height,temperature) VALUES(?,?,?,?,?,?)", (nme, w["level"], w["src"], w["wind"] * 3.6, w["wave"], 24.0))
            c.execute("UPDATE ports SET congestion_index=MIN(5,MAX(0.5,congestion_index+?)) WHERE port_name=?", (R.uniform(-.12, .12), nme))
        for _ in range(max(0, min(4, 230 - c.execute("SELECT COUNT(*) FROM vessels").fetchone()[0]))):   # keep the tracked fleet topped up as ships arrive
            o, d = R.sample(ports, 2); sid = f"SHP-L{R.randint(0, 10**8):08d}"; opt = best_route((o[1], o[2]), (d[1], d[2]))
            c.execute("INSERT OR REPLACE INTO shipments(shipment_id,origin_port,dest_port,carrier,status,eta,weight_kg,hs_code,predicted_delay_risk) VALUES(?,?,?,?,?,?,?,?,?)", (sid, o[0], d[0], R.choice([r[0] for r in c.execute("SELECT name FROM carriers").fetchall()]), "In Transit", (datetime.date.today() + datetime.timedelta(days=20)).isoformat(), R.uniform(2000, 25000), f"HS-{R.choice([30,84,85,87,10,62])}{R.randint(10,99)}", .1))
            c.execute("INSERT OR REPLACE INTO vessels VALUES(?,?,?,?,?,?,?,?)", (sid, o[1], o[2], R.uniform(0, .05), json.dumps(opt["chain"]), opt["name"], R.uniform(12, 17), now()))
        if _tickn % 6 == 0 and R.random() < .6:                      # incoming customer quote request
            o, d = R.sample(ports, 2); cu = c.execute("SELECT customer_id FROM customers ORDER BY RANDOM() LIMIT 1").fetchone()[0]
            hs = f"HS-{R.choice([30,84,85,87,10,62,27])}{R.randint(10,99)}"; req_docs = _required_docs(hs)[1]; given = [x for x in req_docs if R.random() < (.97 if x in ("Commercial Invoice", "Packing List", "Bill of Lading") else .82)]
            c.execute("INSERT INTO quote_requests(ts,origin,dest,cargo,weight_t,hs_code,customer_id,language,docs_provided) VALUES(?,?,?,?,?,?,?,?,?)", (now(), o[0], d[0], R.choice(list(CARGO)), round(R.uniform(5, 450), 1), hs, cu, "English", ";".join(given))); reasons.append("quote-request")
        if _tickn % 40 == 0 and R.random() < .5: inject_storm(); reasons.append("storm")
        if _tickn % 25 == 0: set_state("storms", [s for s in storms() if (time.time() % 1e9) > 0][-3:])
    return reasons

def feed():
    with _DB_LOCK, conn() as c: ports = c.execute("SELECT port_name, lat, lon FROM ports ORDER BY port_id").fetchall()
    k = get_state("feedk", 0) or 0
    for i in range(10): _, la, lo = ports[(k * 10 + i) % len(ports)]; weather_at(la, lo, allow_live=True)
    set_state("feedk", k + 1)
DEFAULT_POLICY = {"reroute": "approval"}
def process():
    for r in Q("SELECT req_id FROM quote_requests WHERE status='new' ORDER BY req_id LIMIT 3").req_id.tolist(): run_quote(int(r))

# ---------------------------------------------------------------- the 7 pipeline agents
def route_agent(s):
    req = s["req"]; po, pd_ = port_xy(req["origin"]), port_xy(req["dest"])
    if not po or not pd_: raise ValueError("unknown port")
    opts = route_options(po[:2], pd_[:2], s.get("avoid", ())); best = opts[0]
    cong = predict_congestion((po[2] + pd_[2]) / 2, (po[3] + pd_[3]) / 2, req["cargo"]); days = best["nm"] / (14 * 24) + po[3] * .5 + pd_[3] * .5
    cars = Q("SELECT name, rating, on_time_pct, avg_cost_index, risk_level FROM carriers"); cars["score"] = cars.rating / 5 * 40 + cars.on_time_pct * .4 - cars.avg_cost_index * 12 - (cars.risk_level == "High") * 8
    top = cars.sort_values("score", ascending=False).head(3)
    return AgentResult("route", 100 - cong * 20, risk_label(100 - cong * 20), f"{best['name']}, {best['nm']:,} nm, {days:.1f} days, congestion level {cong}/3 (ML)",
        {"route": best["name"], "chain": best["chain"], "distance_nm": best["nm"], "transit_days": round(days, 1), "congestion": cong, "alternatives": [o["name"] for o in opts[1:]], "carriers": top.name.tolist(), "ports": [po[:2], pd_[:2]]})
def weather_agent(s):
    r = s["results"]["route"].findings; ch = r["chain"]; pts = [interp(ch, f) for f in (0, .25, .5, .75, 1)]; ws = [weather_at(la, lo) for la, lo in pts]; worst = max(ws, key=lambda w: w["wind"])
    sc = max(wscore(w["level"]) for w in ws); delay = round(min(.95, sc * .6 + worst["wind"] / 60), 2)
    return AgentResult("weather", 100 - sc * 100, risk_label(100 - sc * 100), f"{worst['level']} risk: wind {worst['wind']} m/s, waves {worst['wave']} m ({worst['src']}), delay probability {delay}", {"level": worst["level"], "wind": worst["wind"], "wave": worst["wave"], "risk": sc, "delay_prob": delay, "src": worst["src"]})
def pricing_agent(s):
    m = pricing_model(); rt = s["results"]["route"].findings; req = s["req"]
    ml = float(m["rf"].predict(pd.DataFrame([[rt["distance_nm"], req["weight_t"], CODES.get(req["cargo"], 6), rt["congestion"]]], columns=["distance_nm", "cargo_weight", "cargo_type", "congestion_level"]))[0])
    fuel = ml * float(get_state("fuel_pct", .12) or .12); fees = 350 + 40 * rt["congestion"] * 2
    return AgentResult("pricing", 90, "Low", f"ML base ${ml:,.0f} (R2 {m['metrics']['r2']}), fuel 12% ${fuel:,.0f}, port fees ${fees:,.0f}", {"base": round(ml, 2), "fuel": round(fuel, 2), "port_fees": fees, "model": m["metrics"]})
def _required_docs(hs):
    ch = (re.findall(r"\d+", str(hs)) or ["00"])[0][:2]; r = Q("SELECT * FROM customs_rules WHERE chapter=?", (ch,))
    base = ["Commercial Invoice", "Packing List", "Bill of Lading"]
    return ch, (r.docs[0].split(";") if len(r) else base), (int(r.restricted[0]) if len(r) else 0), (float(r.duty_pct[0]) if len(r) else 10.0), (r.goods[0] if len(r) else "General cargo")
def _attached(ref):
    try: return set(Q("SELECT doc FROM shipment_docs WHERE ref=?", (ref,)).doc)
    except Exception: return set()
def customs_agent(s):
    """Customs check against the customs_rules table: duty, restricted goods, and which required documents the customer actually provided or attached later."""
    req = s["req"]; ch, docs, restricted, duty, goods = _required_docs(req["hs_code"]); prov = req.get("docs_provided")
    have = set(str(prov).split(";")) if isinstance(prov, str) and prov else set(docs)                # legacy requests without a document list count as complete
    have |= _attached(s.get("quote_id", "")); missing = [d for d in docs if d not in have]
    risk = round(min(1, .12 + .35 * restricted + .18 * len(missing)), 2)
    return AgentResult("customs", 100 - risk * 100, risk_label(100 - risk * 100), f"Chapter {ch} ({goods}): {'restricted, ' if restricted else ''}{len(missing)} document(s) missing, duty {duty}%", {"risk": risk, "docs": docs, "missing": missing, "duty_pct": duty, "restricted": bool(restricted), "status": "Hold" if len(missing) or risk > .6 else "Ready", "goods": goods})
def margin_agent(s):
    p = s["results"]["pricing"].findings; cust = Q("SELECT priority_tier t, credit_risk cr FROM customers WHERE customer_id=?", (s["req"]["customer_id"],)); tier = cust.t[0] if len(cust) else "Silver"; cr = float(cust.cr[0]) if len(cust) else .1
    target = .20 + {"Platinum": -.02, "Gold": 0, "Silver": .02, "Bronze": .04}.get(tier, .02) + (.02 if cr > .3 else 0)
    cost = p["base"] + p["fuel"] + p["port_fees"]; op = p["base"] * .15; duty = p["base"] * s["results"]["customs"].findings["duty_pct"] / 100 * .1; total_cost = cost + op + duty
    price = total_cost / (1 - target); margin = (price - total_cost) / price * 100
    return AgentResult("margin", min(100, margin * 4), "Low" if margin >= 10 else "High", f"Target {target*100:.0f}% ({tier}), price ${price:,.0f}, margin {margin:.1f}%", {"price": round(price, 2), "margin_pct": round(margin, 1), "op_cost": round(op, 2), "target": target, "tier": tier, "cost": round(total_cost, 2)})
def document_agent(s):
    q = s["quote_id"]; out = os.path.join(DATA_DIR, "docs"); os.makedirs(out, exist_ok=True); req = s["req"]; rt = s["results"]["route"].findings; paths = []
    try:
        from reportlab.lib.pagesizes import A4; from reportlab.pdfgen import canvas
        for kind, lines in (("BillOfLading", [f"Shipper/Customer: {req['customer_id']}", f"Port of loading: {req['origin']}", f"Port of discharge: {req['dest']}", f"Cargo: {req['cargo']} ({req['hs_code']}), {req['weight_t']} t", f"Carrier: {rt['carriers'][0]}", f"Route: {rt['route']}, {rt['distance_nm']:,} nm"]),
                            ("PackingList", [f"Quote: {q}", f"Cargo type: {req['cargo']}", f"Gross weight: {req['weight_t']} t", f"Documents: {', '.join(s['results']['customs'].findings['docs'])}"])):
            p = os.path.join(out, f"{q}_{kind}.pdf"); cv = canvas.Canvas(p, pagesize=A4); cv.setFont("Helvetica-Bold", 16); cv.drawString(50, 800, f"{kind} - {q}"); cv.setFont("Helvetica", 11)
            for i, ln in enumerate(lines): cv.drawString(50, 770 - 22 * i, ln)
            cv.save(); paths.append(p)
    except Exception as e: return AgentResult("document", 60, "Medium", f"PDF generation unavailable: {e}")
    return AgentResult("document", 100, "Low", f"{len(paths)} PDFs generated", {"paths": paths})
def alert_agent(s):
    val = s["results"]["margin"].findings["price"]; w = s["results"]["weather"].findings; acts = []
    if val > 50000: acts.append({"type": "quote_approval", "target": s["quote_id"], "detail": f"Quote ${val:,.0f} exceeds $50,000: admin approval required", "severity": "Critical", "owner": "Admin"})
    elif s["results"]["margin"].findings["margin_pct"] < 10: acts.append({"type": "margin_review", "target": s["quote_id"], "detail": "Margin below 10%", "severity": "Warning"})
    if w["level"] == "High": acts.append({"type": "reroute", "target": s["quote_id"], "detail": f"High weather risk on lane (wind {w['wind']} m/s)", "severity": "Warning"})
    return AgentResult("alert", 100, "Low", f"{len(acts)} alerts raised", {}, acts)

def _compute(req, qid, cid=None, avoid=()):
    """The quote pipeline as a LangGraph graph: route -> weather -> [High risk: re-plan avoiding chokepoints, retry] -> pricing -> customs -> margin.
    cid=None runs it without logging (used by the Digital Twin and the quote preview)."""
    st = {"req": req, "results": {}, "avoid": tuple(avoid), "quote_id": qid, "tries": 0}
    def step(name, fn):
        t = time.time()
        try: res = fn(st)
        except Exception as e: res = AgentResult(name, 40, "High", f"error: {e}")
        st["results"][name] = res
        if cid: _log_run(cid, "quote", res, int((time.time() - t) * 1000))
        return res
    def replan():
        st["avoid"] = ("suez",); st["tries"] += 1
        if cid: _log_run(cid, "quote", AgentResult("re-plan", 100, "Low", f"weather High: re-routing avoiding Suez/Red Sea (try {st['tries']})"), 0)
    def need_replan(): return st["results"]["weather"].findings.get("level") == "High" and st["tries"] < 1 and st["results"]["route"].findings.get("alternatives") and "suez" not in st["avoid"]
    try:
        from langgraph.graph import StateGraph, END
        from typing import TypedDict
        class G(TypedDict, total=False): n: int
        gr = StateGraph(G)
        gr.add_node("route", lambda g: (step("route", route_agent), {})[1]); gr.add_node("weather", lambda g: (step("weather", weather_agent), {})[1]); gr.add_node("replan", lambda g: (replan(), {})[1])
        gr.add_node("rest", lambda g: ([step(n, f) for n, f in (("pricing", pricing_agent), ("customs", customs_agent), ("margin", margin_agent))], {})[1])
        gr.set_entry_point("route"); gr.add_edge("route", "weather"); gr.add_edge("replan", "route"); gr.add_conditional_edges("weather", lambda g: "replan" if need_replan() else "rest"); gr.add_edge("rest", END); gr.compile().invoke({"n": 0})
    except ImportError:
        step("route", route_agent); step("weather", weather_agent)
        if need_replan(): replan(); step("route", route_agent); step("weather", weather_agent)
        for nme, fn in (("pricing", pricing_agent), ("customs", customs_agent), ("margin", margin_agent)): step(nme, fn)
    return st, step

def _decide(st):
    res = st["results"]; req = st["req"]; wr, cr, rt = res["weather"].findings["risk"], res["customs"].findings["risk"], res["route"].findings["congestion"] / 3
    risk = round(wr * .4 + cr * .3 + rt * .3, 3); m = res["margin"].findings; val = m["price"]          # PRS v3: weather x0.40 + customs x0.30 + congestion x0.30
    decision = "On Hold" if res["customs"].findings["status"] == "Hold" or risk > .7 else "Pending Approval" if val > 50000 or m["margin_pct"] < 10 else "Approved"
    narr = f"Quote {st['quote_id']}: {req['origin']} to {req['dest']}, {res['route'].findings['route']}, {res['route'].findings['transit_days']} days. Risk score {risk} (weather {wr}, customs {cr}, congestion {rt:.2f}). Price ${val:,.0f} at {m['margin_pct']}% margin. Decision: {decision}."
    return risk, decision, val, m, narr

def preview_quote(req, avoid=()):
    """Price a lane without saving anything. The Digital Twin calls this on its database copy."""
    st, _ = _compute(dict(req), "PREVIEW", None, avoid)
    if not all(k in st["results"] and st["results"][k].findings for k in ("route", "weather", "pricing", "customs", "margin")): return None
    risk, decision, val, m, narr = _decide(st); r = st["results"]
    return {"price": val, "margin_pct": m["margin_pct"], "risk": risk, "decision": decision, "route": r["route"].findings["route"], "days": r["route"].findings["transit_days"], "weather": r["weather"].findings["level"], "narrative": narr}

def run_quote(req_id):
    from live_engine import _log_run, init_live_tables, act
    import sys; dom = sys.modules[__name__]; init_live_tables()
    r = Q("SELECT * FROM quote_requests WHERE req_id=?", (req_id,)); req = r.iloc[0].to_dict(); cid = f"Q{req_id}-" + datetime.datetime.now().strftime("%H%M%S"); qid = f"QTE-L{req_id:05d}"
    st, step = _compute(req, qid, cid); res = st["results"]
    if not all(k in res and res[k].findings for k in ("route", "weather", "pricing", "customs", "margin")):
        with _DB_LOCK, conn() as c: c.execute("UPDATE quote_requests SET status='error', result=? WHERE req_id=?", (json.dumps({k: v.summary for k, v in res.items()}), req_id)); return None
    risk, decision, val, m, narr = _decide(st); step("document", document_agent); step("alert", alert_agent); wr, cr, rt = res["weather"].findings["risk"], res["customs"].findings["risk"], res["route"].findings["congestion"] / 3
    with _DB_LOCK, conn() as c:
        sid = f"SHP-{qid[4:]}"
        c.execute("INSERT OR REPLACE INTO shipments(shipment_id,origin_port,dest_port,carrier,status,eta,weight_kg,hs_code,predicted_delay_risk) VALUES(?,?,?,?,?,?,?,?,?)", (sid, req["origin"], req["dest"], res["route"].findings["carriers"][0], "Customs Hold" if decision == "On Hold" else "Quoted", (datetime.date.today() + datetime.timedelta(days=int(res["route"].findings["transit_days"]) + 3)).isoformat(), req["weight_t"] * 1000, req["hs_code"], risk))
        p = res["pricing"].findings; c.execute("INSERT OR REPLACE INTO freight_quotes VALUES(?,?,?,?,?,?,?,?,?,?,?,?)", (qid, sid, req["customer_id"], res["route"].findings["carriers"][0], p["base"], round(p["base"] * .02, 2), round(p["base"] * res["customs"].findings["duty_pct"] / 100 * .1, 2), p["fuel"], val, m["margin_pct"], decision, now()[:10]))
        c.execute("UPDATE quote_requests SET status='done', quote_id=?, result=? WHERE req_id=?", (qid, json.dumps({"decision": decision, "risk": risk, "price": val, "narrative": narr, "missing": res["customs"].findings["missing"], "route": res["route"].findings["route"], "days": res["route"].findings["transit_days"], "weather": res["weather"].findings["level"], "margin_pct": m["margin_pct"], "parts": {"weather": wr, "customs": cr, "congestion": round(rt, 2)}}), req_id))
    dec = {"score": round((1 - risk) * 100, 1), "risk": risk_label((1 - risk) * 100), "narrative": narr, "actions": res["alert"].actions + ([{"type": "quote_hold", "target": qid, "detail": "Customs hold, missing: " + ", ".join(res["customs"].findings["missing"]), "severity": "Warning"}] if decision == "On Hold" and res["customs"].findings["missing"] else [])}
    act(dom, dec, cid, {}); return dec

def attach_doc(ref, doc, by="user"):
    """Attach a missing customs document to a quote (QTE-...) or a shipment (SHP-...). Re-runs the quote so the hold lifts when nothing is missing."""
    with _DB_LOCK, conn() as c:
        c.execute("CREATE TABLE IF NOT EXISTS shipment_docs(ref TEXT, doc TEXT, ts TEXT, by TEXT, UNIQUE(ref, doc))"); c.execute("INSERT OR IGNORE INTO shipment_docs VALUES(?,?,?,?)", (ref, doc, now(), by))
    emit("doc_attached", ref, {"doc": doc})
    if ref.startswith("QTE-L"):
        try: run_quote(int(ref[5:]))
        except Exception: pass

# ---------------------------------------------------------------- fleet monitoring cycle (the same agents, run continuously)
def route_watch(s):
    p = Q("SELECT port_name, congestion_index ci, avg_dwell_days dw FROM ports WHERE congestion_index>=4").sort_values("ci", ascending=False)
    n = Q("SELECT COUNT(*) n FROM shipments WHERE status='Port Congestion Delay'").n[0]; sc = 100 - min(60, len(p) * 3 + n * .5)
    return AgentResult("route_watch", sc, risk_label(sc), f"{len(p)} ports with congestion index >= 4, {n} shipments delayed", {"ports": p.head(8).port_name.tolist(), "n_congested": len(p), "delayed": int(n)}, [{"type": "congestion_notice", "target": r.port_name, "detail": f"Congestion index {r.ci:.1f}, dwell {r.dw:.1f} days", "severity": "Warning"} for r in p.head(6).itertuples()])
def weather_watch(s):
    v = Q("SELECT shipment_id, lat, lon, chain FROM vessels"); bad = []; acts = []
    for r in v.itertuples():
        w = weather_at(r.lat, r.lon, allow_live=False)
        if w["level"] == "High": bad.append(r.shipment_id)
    for sid in bad[:15]:
        acts.append({"type": "reroute", "target": sid, "detail": "High weather at current vessel position: alternate route proposed", "severity": "Critical"})
    sc = 100 - min(90, len(bad) / max(len(v), 1) * 400); return AgentResult("weather_watch", sc, risk_label(sc), f"{len(bad)} of {len(v)} vessels in High-risk weather ({len(storms())} active storms)", {"vessels_high": bad}, acts)
def customs_watch(s):
    """Shipments on customs hold: required documents (customs_rules by HS chapter) minus documents attached. Clears the hold when nothing is missing."""
    h = Q("SELECT shipment_id, hs_code FROM shipments WHERE status='Customs Hold'"); acts = []; cleared = 0
    for r in h.itertuples():
        ch, docs, restricted, duty, goods = _required_docs(r.hs_code); miss = [d for d in docs if d not in _attached(r.shipment_id)]
        if not miss and not restricted:
            with _DB_LOCK, conn() as c: c.execute("UPDATE shipments SET status='In Transit' WHERE shipment_id=?", (r.shipment_id,)); emit("customs_cleared", r.shipment_id); cleared += 1; continue
        acts.append({"type": "customs_docs", "target": r.shipment_id, "detail": f"Customs hold ({r.hs_code}, {goods}): missing {', '.join(miss[:4]) if miss else 'restricted goods clearance'}", "severity": "Warning"})
    n = len(h) - cleared; sc = 100 - min(70, n * 2)
    return AgentResult("customs_watch", sc, risk_label(sc), f"{n} shipments on customs hold, {cleared} cleared this cycle", {"holds": n, "cleared": cleared}, acts[:25])
def quote_watch(s):
    p = Q("SELECT q.quote_id, q.final_price, q.margin_pct, q.status, r.result FROM freight_quotes q LEFT JOIN quote_requests r ON r.quote_id=q.quote_id WHERE q.status IN ('Pending Approval','On Hold')"); acts = []
    for r in p.itertuples():
        if r.status == "Pending Approval": acts.append({"type": "quote_approval", "target": r.quote_id, "detail": f"Quote ${r.final_price:,.0f} exceeds $50,000: admin approval required" if r.final_price > 50000 else "Margin below 10%: approval required", "severity": "Critical" if r.final_price > 50000 else "Warning", "owner": "Admin"})
        else:
            try: miss = json.loads(r.result or "{}").get("missing", [])
            except Exception: miss = []
            acts.append({"type": "quote_hold", "target": r.quote_id, "detail": "Customs hold, missing: " + ", ".join(miss) if miss else "On hold: shipment risk score above 0.7", "severity": "Warning"})
    sc = 100 - min(60, len(p) * 3); return AgentResult("quote_watch", sc, risk_label(sc), f"{len(p)} quotes awaiting approval or on hold", {"pending": int((p.status == 'Pending Approval').sum()), "on_hold": int((p.status == 'On Hold').sum())}, acts[:30])

def anomaly_watch(s):
    """IsolationForest over quote economics (price/base ratio, margin, fuel share) to catch mispriced or tampered quotes."""
    from sklearn.ensemble import IsolationForest
    q = Q("SELECT quote_id, base_cost, fuel_surcharge, final_price, margin_pct FROM freight_quotes WHERE base_cost>0")
    if len(q) < 30: return AgentResult("anomaly_watch", 100, "Low", "waiting for more quotes", {"flagged": []})
    X = pd.DataFrame({"ratio": q.final_price / q.base_cost, "margin": q.margin_pct, "fuel": q.fuel_surcharge / q.base_cost}); iso = IsolationForest(contamination=.06, random_state=42).fit(X); q["flag"] = iso.predict(X) == -1
    bad = q[q.flag]; sc = 100 - len(bad) / len(q) * 250
    return AgentResult("anomaly_watch", round(max(sc, 0), 1), risk_label(sc), f"{len(bad)} of {len(q)} quotes look anomalous (IsolationForest)", {"flagged": bad.quote_id.tolist()},
        [{"type": "quote_anomaly", "target": r.quote_id, "detail": f"Price/base ratio {r.final_price / r.base_cost:.2f}, margin {r.margin_pct:.1f}%: review before sending", "severity": "Warning"} for r in bad.head(10).itertuples()])

AGENTS = {"route_watch": route_watch, "weather_watch": weather_watch, "customs_watch": customs_watch, "quote_watch": quote_watch, "anomaly_watch": anomaly_watch}
REPLAN = {"weather_watch": lambda r: ["route_watch"] if r.risk != "Low" else []}
def plan(req): return list(AGENTS)
def engine(s):
    r = s["results"]; score = float(np.mean([v.score for v in r.values()])); acts = [a for v in r.values() for a in v.actions]; order = {"Critical": 0, "Warning": 1, "Info": 2}; acts.sort(key=lambda a: order.get(a.get("severity"), 3))
    ev = {"fleet_health": round(score, 1), "vessels": int(Q("SELECT COUNT(*) n FROM vessels").n[0]), "agents": {k: {"score": v.score, "risk": v.risk, "summary": v.summary} for k, v in r.items()}, "critical": sum(a.get("severity") == "Critical" for a in acts)}
    n = f"Fleet health {ev['fleet_health']}/100 with {ev['vessels']} vessels tracked. " + " ".join(f"{k}: {v['summary']}." for k, v in ev["agents"].items())
    return {"score": round(score, 1), "risk": risk_label(score), "narrative": n, "actions": acts, "agents": ev["agents"], "evidence": ev}
def apply(a, c):
    if a["type"] == "reroute":
        sid = a["target"]; v = c.execute("SELECT lat, lon, chain FROM vessels WHERE shipment_id=?", (sid,)).fetchone()
        if v:
            ch = json.loads(v[2]); dest = ch[-1]
            if a.get("opt_idx") is not None:
                import alts; row = c.execute("SELECT s.weight_kg FROM shipments s WHERE shipment_id=?", (sid,)).fetchone(); al = alts.route_alternatives(sys.modules[__name__], (v[0], v[1]), dest, tonnes=float((row or [20000])[0] or 20000) / 1000, interp=interp); i = int(a["opt_idx"]); opt = al[i] if i < len(al) else al[0]
            else: opt = route_options((v[0], v[1]), dest, avoid=("storm",))[0]
            c.execute("UPDATE vessels SET chain=?, progress=0, route_name=? WHERE shipment_id=?", (json.dumps([(v[0], v[1])] + opt["chain"][1:]), ("rerouted: " if a.get("opt_idx") is not None else "rerouted around storm: ") + opt["name"], sid)); return f"vessel re-planned {opt['name']} ({opt['nm']:,} nm)"
    return ""
def decide(action_id, approve, by="user"):
    from live_engine import resolve_action
    with _DB_LOCK, conn() as c:
        t = c.execute("SELECT target FROM agent_actions WHERE action_id=?", (action_id,)).fetchone()
        if t and t[0].startswith("QTE"): c.execute("UPDATE freight_quotes SET status=? WHERE quote_id=?", ("Approved" if approve else "Rejected", t[0]))
    resolve_action(action_id, "approved" if approve else "rejected", f"by {by}")
def submit_request(origin, dest, cargo, weight_t, hs_code, customer_id, language="English", docs_provided=None):
    with _DB_LOCK, conn() as c: return c.execute("INSERT INTO quote_requests(ts,origin,dest,cargo,weight_t,hs_code,customer_id,language,docs_provided) VALUES(?,?,?,?,?,?,?,?,?)", (now(), origin, dest, cargo, weight_t, hs_code, customer_id, language, ";".join(docs_provided) if docs_provided is not None else None)).lastrowid
def feed_status():
    o = time.time() < _OFF[0]; return [("Open-Meteo weather and marine", "Offline: simulated winds in use" if o else "Real · live (cached 15 min)"), ("Vessel positions", "Simulated along real route waypoints (connect AIS via Data Feeder)"), ("Port congestion", "ML predicted, drifts live"), ("Freight-rate model data", "Bootstrap synthetic unless a Kaggle CSV is fed")]
def kpi_extra():
    return [("Vessels tracked", int(Q("SELECT COUNT(*) n FROM vessels").n[0])), ("Quotes awaiting approval", int(Q("SELECT COUNT(*) n FROM freight_quotes WHERE status='Pending Approval'").n[0]))]

def _prewarm():
    try:
        import seanav; seanav.sea()
    except Exception: pass
import threading as _th; _th.Thread(target=_prewarm, daemon=True).start()


# Kept from the app shell: server.py calls this at startup (the Customs page needs the table before the worker has booted).
def ensure_schema():
    with _DB_LOCK, conn() as c:
        c.executescript("CREATE TABLE IF NOT EXISTS quote_requests(req_id INTEGER PRIMARY KEY AUTOINCREMENT, ts TEXT, origin TEXT, dest TEXT, cargo TEXT, weight_t REAL, hs_code TEXT, customer_id TEXT, language TEXT, status TEXT DEFAULT 'new', quote_id TEXT, result TEXT, docs_provided TEXT);\n        CREATE TABLE IF NOT EXISTS shipment_docs(ref TEXT, doc TEXT, ts TEXT, by TEXT, UNIQUE(ref, doc));\n        CREATE TABLE IF NOT EXISTS vessels(shipment_id TEXT PRIMARY KEY, lat REAL, lon REAL, progress REAL, chain TEXT, route_name TEXT, speed_kn REAL, updated TEXT);\n        CREATE TABLE IF NOT EXISTS ml_models(model_id INTEGER PRIMARY KEY AUTOINCREMENT, model_name TEXT, metrics TEXT, trained_at TEXT, is_active INT);\n        CREATE TABLE IF NOT EXISTS customs_rules(chapter TEXT PRIMARY KEY, goods TEXT, duty_pct REAL, docs TEXT, restricted INT);")
        # On a fresh Colab session the background worker may not have started yet.
        # Populate the same professor-provided rules required by the Customs page,
        # without touching any existing user, shipment or quotation records.
        if not c.execute("SELECT 1 FROM customs_rules LIMIT 1").fetchone():
            base = "Commercial Invoice;Packing List;Bill of Lading"
            for chapter, goods, duty, extra, restricted in [
                ("30", "Pharmaceuticals", 5, "Import License;Drug Control Certificate", 1),
                ("84", "Machinery", 7.5, "Certificate of Origin", 0),
                ("85", "Electronics", 10, "BIS Certificate;Certificate of Origin", 0),
                ("87", "Vehicles and parts", 15, "Homologation Certificate", 0),
                ("10", "Cereals", 12, "Phytosanitary Certificate", 0),
                ("62", "Apparel", 12, "Certificate of Origin", 0),
                ("27", "Mineral fuels", 5, "Hazmat Declaration;IMDG Certificate", 1),
                ("39", "Plastics", 8, "Certificate of Origin", 0),
                ("72", "Iron and steel", 7.5, "Mill Test Certificate", 0),
                ("22", "Beverages", 20, "Excise Permit", 1),
            ]:
                c.execute("INSERT OR IGNORE INTO customs_rules (chapter, goods, duty_pct, docs, restricted) VALUES(?,?,?,?,?)", (chapter, goods, duty, base + ";" + extra, restricted))


In [ ]:
%%writefile /content/app/entities.py
"""One description of any entity (outlet, staff, quote, shipment, vessel, port): master data, relations, and everything flagged on it. Feeds the drawer, the knowledge graph and the Copilot."""
import json
from blocks import Q, clean

def _tickets(e, n=30):
    return Q("SELECT action_id id, type, severity, status, detail, owner, assignee, due FROM agent_actions WHERE target=? ORDER BY action_id DESC LIMIT ?", (e, n))
def latest_out(agent):
    r = Q("SELECT output FROM agent_runs WHERE agent=? ORDER BY run_id DESC LIMIT 1", (agent,))
    try: return json.loads(r.output[0]) if len(r) else {}
    except Exception: return {}
def kind_of(e):
    e = str(e)
    if e.startswith("OUT-"): return "outlet"
    if e.startswith("QTE"): return "quote"
    if e.startswith("SHP"): return "shipment"
    if e.startswith("EMP") or e.startswith("STF"): return "staff"
    return "port"

def describe(kind, dom, e):
    """-> {"kind","info":{label:value},"rel":[(name,kind,tip)], "tickets": DataFrame}"""
    k = kind_of(e); info, rel = {}, []
    if kind == "Franchise" and k == "outlet":
        o = Q("SELECT * FROM outlets WHERE outlet_id=?", (e,))
        if len(o):
            r = o.iloc[0]; info = {"Outlet": r.outlet_name, "Location": r.location, "Revenue": f"{r.revenue:,.0f}", "Customer satisfaction": round(float(r.customer_satisfaction), 2), "Headcount": int(r.staff_headcount)}
            for a in ("outlet", "inventory", "staff", "marketing", "audit", "feedback"):
                v = latest_out(a).get("per_outlet", {}).get(e)
                if v is not None: rel.append((f"{a} agent: {v:.0f}", "agent", f"{a} score for {e}"))
            for s in Q("SELECT staff_id, predicted_attrition_prob p FROM staff WHERE outlet_id=? ORDER BY p DESC LIMIT 4", (e,)).itertuples(): rel.append((s.staff_id, "staff", f"attrition {s.p:.0%}"))
            for s in Q("SELECT sku_name, stockout_risk_prob p FROM inventory WHERE outlet_id=? ORDER BY p DESC LIMIT 4", (e,)).itertuples(): rel.append((s.sku_name, "sku", f"stock-out risk {s.p:.0%}"))
            for s in Q("SELECT campaign_name, channel FROM marketing WHERE outlet_id=? AND end_date>=date('now')", (e,)).itertuples(): rel.append((s.campaign_name, "campaign", f"live via {s.channel}"))
    elif kind == "Freight" and k == "quote":
        q = Q("SELECT * FROM freight_quotes WHERE quote_id=?", (e,))
        if len(q):
            r = q.iloc[0]; info = {"Quote": e, "Price": f"${r.final_price:,.0f}", "Margin": f"{r.margin_pct}%", "Status": r.status}
            for n, tip in ((r.shipment_id, "shipment"), (r.customer_id, "customer"), (r.carrier, "carrier")): rel.append((n, tip, tip))
    elif kind == "Freight" and k == "shipment":
        s = Q("SELECT * FROM shipments WHERE shipment_id=?", (e,))
        if len(s):
            r = s.iloc[0]; info = {"Shipment": e, "Route": f"{r.origin_port} to {r.dest_port}", "Carrier": r.carrier, "Status": r.status, "Delay risk": r.predicted_delay_risk}
            for n, tip in ((r.origin_port, "port"), (r.dest_port, "port"), (r.carrier, "carrier")): rel.append((n, tip, tip))
            v = Q("SELECT lat, lon, route_name, progress FROM vessels WHERE shipment_id=?", (e,))
            if len(v): info["Vessel"] = f"{v.lat[0]:.1f}, {v.lon[0]:.1f} · {v.route_name[0]} · {v.progress[0]:.0%}"
    elif kind == "Freight":
        p = Q("SELECT * FROM ports WHERE port_name=?", (e,))
        if len(p):
            r = p.iloc[0]; info = {"Port": e, "Country": r.country, "Congestion index": round(float(r.congestion_index), 2), "Dwell days": r.avg_dwell_days}
            for s in Q("SELECT shipment_id FROM shipments WHERE origin_port=? OR dest_port=? LIMIT 6", (e, e)).itertuples(): rel.append((s.shipment_id, "shipment", "via port"))
    return {"kind": k, "info": {a: clean(b) for a, b in info.items()}, "rel": rel, "tickets": _tickets(e)}

def facts(kind, dom, text):
    import re
    out = []
    for e in dict.fromkeys(m.upper() for m in re.findall(r"\b(OUT-\d+|QTE-L\d+|SHP-[A-Z0-9-]+)\b", text, re.I)):
        d = describe(kind, dom, e); t = d["tickets"]; t = t[t.status.isin(["open", "proposed"])].head(8)
        if d["info"] or len(t): out.append(f"{e}: " + "; ".join(f"{k}={v}" for k, v in d["info"].items()) + (f". Related: {', '.join(n for n, _, _ in d['rel'][:8])}." if d["rel"] else "") + (" Open tickets: " + " | ".join(f"{r.type} ({r.severity}): {r.detail}" for r in t.itertuples()) if len(t) else " No open tickets."))
    return out


In [ ]:
%%writefile /content/app/gpu.py
"""Minimal stub (no language model in this version). Kept so hardening.py can report health."""
def status(): return {"cuda": False, "gpu": None, "models": {}, "stats": {}}


In [ ]:
%%writefile /content/app/hardening.py
"""Production hardening for the FastAPI app: request ids + JSON access log, security headers (CSP, frame, permissions), per-IP rate limits on auth routes,
a global JSON error handler that never leaks tracebacks, and Prometheus-style metrics. Install with hardening.install(app)."""
import os, time, json, uuid, threading, logging, traceback, collections
from logging.handlers import RotatingFileHandler
from fastapi import Request
from fastapi.responses import JSONResponse, PlainTextResponse
import config

_LOG = logging.getLogger("access"); _LOG.setLevel(logging.INFO); _LOG.propagate = False
try:
    _h = RotatingFileHandler(os.path.join(config.DATA_DIR, "access.log"), maxBytes=2_000_000, backupCount=3); _h.setFormatter(logging.Formatter("%(message)s")); _LOG.addHandler(_h)
except Exception: pass
_ERR = logging.getLogger("errors"); _ERR.propagate = False
try:
    _e = RotatingFileHandler(os.path.join(config.DATA_DIR, "errors.log"), maxBytes=1_000_000, backupCount=2); _e.setFormatter(logging.Formatter("%(asctime)s %(message)s")); _ERR.addHandler(_e)
except Exception: pass

M = {"requests": collections.Counter(), "status": collections.Counter(), "lat_sum": collections.Counter(), "lat_n": collections.Counter(), "errors": 0, "limited": 0, "started": time.time()}
_L = threading.Lock(); _BUCKET = {}
CSP = "default-src 'self'; img-src 'self' data: blob:; style-src 'self' 'unsafe-inline'; script-src 'self' 'unsafe-inline'; connect-src 'self'; font-src 'self' data:; frame-ancestors 'none'; base-uri 'self'; form-action 'self'"
# path prefix -> (max requests, per seconds). Everything else: 900 per minute per IP.
LIMITS = {"/api/auth/login": (12, 60), "/api/auth/signup": (5, 600), "/api/auth/forgot": (10, 600), "/api/auth/reset": (8, 600), "/api/profile/password": (8, 600), "/api/copilot": (40, 60), "/api/translate": (60, 60)}
def _ip(r):
    f = r.headers.get("x-forwarded-for"); return (f.split(",")[0].strip() if f else (r.client.host if r.client else "?"))
def _allow(ip, path):
    key, (n, per) = next(((p, v) for p, v in LIMITS.items() if path.startswith(p)), ("*", (900, 60))); k = (ip, key); now = time.time()
    with _L:
        q = _BUCKET.setdefault(k, collections.deque())
        while q and now - q[0] > per: q.popleft()
        if len(q) >= n: return False, int(per - (now - q[0])) + 1
        q.append(now)
        if len(_BUCKET) > 20000: _BUCKET.clear()
    return True, 0
def _route(path):
    parts = path.split("/")
    return "/".join(parts[:3]) if path.startswith("/api/") else "static"

def install(app, kind="app"):
    @app.middleware("http")
    async def guard(request: Request, call_next):
        rid = request.headers.get("x-request-id") or uuid.uuid4().hex[:12]; t0 = time.time(); path = request.url.path
        if os.environ.get("RATE_LIMIT_OFF") != "1" and (path.startswith("/api/") and request.method != "GET" or path.startswith("/api/login")):
            ok, wait = _allow(_ip(request), path)
            if not ok:
                M["limited"] += 1; return JSONResponse({"ok": False, "msg": f"Too many requests. Try again in {wait}s.", "request_id": rid}, status_code=429, headers={"Retry-After": str(wait), "X-Request-ID": rid})
        try: r = await call_next(request)
        except Exception as e:
            M["errors"] += 1; _ERR.error("%s %s %s\n%s", rid, request.method, path, traceback.format_exc())
            return JSONResponse({"ok": False, "msg": "Something went wrong on the server. Quote this id to the admin: " + rid, "request_id": rid}, status_code=500, headers={"X-Request-ID": rid})
        ms = (time.time() - t0) * 1000; rt = _route(path)
        with _L: M["requests"][rt] += 1; M["status"][r.status_code // 100] += 1; M["lat_sum"][rt] += ms; M["lat_n"][rt] += 1
        h = r.headers; h["X-Request-ID"] = rid; h["X-Frame-Options"] = "DENY"; h["Content-Security-Policy"] = CSP; h["Permissions-Policy"] = "microphone=(self), camera=(), geolocation=()"; h["Cross-Origin-Opener-Policy"] = "same-origin"
        if request.headers.get("x-forwarded-proto") == "https": h["Strict-Transport-Security"] = "max-age=31536000"
        if path.startswith("/api/") and path not in ("/api/stream", "/api/pulse"): _LOG.info(json.dumps({"t": round(t0), "id": rid, "ip": _ip(request), "m": request.method, "p": path, "s": r.status_code, "ms": round(ms)}))
        return r

    @app.get("/api/ready")
    def ready(): return {"ok": True, "kind": kind, "uptime_s": round(time.time() - M["started"])}

    @app.get("/api/metrics")
    def metrics(request: Request):
        """Prometheus text format. Set METRICS_TOKEN to require ?token=..., otherwise only logged-in admins (cookie) may read it."""
        tok = os.environ.get("METRICS_TOKEN")
        if tok and request.query_params.get("token") != tok: return PlainTextResponse("forbidden", status_code=403)
        if not tok:
            try:
                from server import user_of
                u = user_of(request)
                if not u or u.get("role") != "Admin": return PlainTextResponse("forbidden", status_code=403)
            except Exception: return PlainTextResponse("forbidden", status_code=403)
        L = ["# TYPE app_requests_total counter"] + [f'app_requests_total{{route="{k}"}} {v}' for k, v in M["requests"].items()] + ["# TYPE app_responses_total counter"] + [f'app_responses_total{{class="{k}xx"}} {v}' for k, v in M["status"].items()]
        L += ["# TYPE app_latency_ms_avg gauge"] + [f'app_latency_ms_avg{{route="{k}"}} {M["lat_sum"][k] / max(1, M["lat_n"][k]):.1f}' for k in M["lat_n"]] + [f"app_errors_total {M['errors']}", f"app_rate_limited_total {M['limited']}", f"app_uptime_seconds {time.time() - M['started']:.0f}"]
        try:
            import live_engine as LE; c = LE.conn(); L += [f"app_open_tickets {c.execute('SELECT COUNT(*) FROM agent_actions WHERE status=?', ('open',)).fetchone()[0]}"]
        except Exception: pass
        try:
            import gpu; g = gpu.status(); L += [f'app_model_ready{{model="{k}"}} {1 if v["state"] == "ready" else 0}' for k, v in g["models"].items()]
        except Exception: pass
        return PlainTextResponse("\n".join(L) + "\n")


In [ ]:
%%writefile /content/app/intent_router.py
import re as _re_live
import pandas as pd
import re, math
from db import get_conn

def df_to_markdown_safe(df):
    if df is None or df.empty: return ""
    try: return df.to_markdown(index=False)
    except: pass
    headers = list(df.columns)
    lines = ["| " + " | ".join([str(h) for h in headers]) + " |"]
    lines.append("| " + " | ".join(["---"] * len(headers)) + " |")
    for _, row in df.iterrows():
        vals = [str(v) if v is not None else "" for v in row.values]
        lines.append("| " + " | ".join(vals) + " |")
    return "\n".join(lines)

INTENT_MAP = {
    "ports": ["port", "ports", "harbor", "terminal", "congestion", "dwell"],
    "shipments": ["shipment", "shipments", "cargo", "carrier", "freight", "teu", "transit"],
    "quotes": ["quote", "spot", "price", "rate", "baf", "margin", "cost"],
    "weather": ["weather", "typhoon", "storm", "cyclone", "wind", "wave"],
    "customs": ["customs", "tariff", "duty", "hs code", "import", "export"],
    "alerts": ["alert", "incident", "delay", "disruption", "hold"]
}

def classify_intent(query):
    q = query.lower()
    for intent, keywords in INTENT_MAP.items():
        if any(kw in q for kw in keywords):
            return intent
    return "general"

def handle_freight_intent(query):
    q_low = query.lower()
    try:
        with get_conn() as conn:
            # 1. Ports / Terminals
            if any(k in q_low for k in ["port", "ports", "harbor", "terminal", "congestion"]):
                tot_ports = conn.execute("SELECT COUNT(*) FROM ports;").fetchone()[0]
                df_ports = pd.read_sql("SELECT port_name as 'Port Name', country as 'Country', region as 'Region', congestion_index as 'Congestion (1-5)', avg_dwell_days as 'Avg Dwell Days' FROM ports ORDER BY congestion_index DESC LIMIT 5;", conn)
                return f"CRITICAL FACT: Total Monitored Ports = {tot_ports}.\n\n[Sample Subset Preview - First 5 Records Only]:\n{df_to_markdown_safe(df_ports)}", "Ports Ledger DB"

            # 2. Shipments / Cargo
            if any(k in q_low for k in ["shipment", "shipments", "carrier", "cargo"]):
                tot_shipments = conn.execute("SELECT COUNT(*) FROM shipments;").fetchone()[0]
                df_ship = pd.read_sql("SELECT shipment_id, origin_port, dest_port, carrier, status, predicted_delay_risk FROM shipments ORDER BY predicted_delay_risk DESC LIMIT 5;", conn)
                return f"CRITICAL FACT: Total Active Shipments = {tot_shipments}.\n\n[Sample Subset Preview - First 5 Records Only]:\n{df_to_markdown_safe(df_ship)}", "Shipments Ledger DB"

            # 3. Quotes / Margins
            if any(k in q_low for k in ["quote", "quotes", "spot", "margin", "price", "rate"]):
                tot_quotes = conn.execute("SELECT COUNT(*) FROM freight_quotes;").fetchone()[0]
                df_q = pd.read_sql("SELECT quote_id, shipment_id, ROUND(base_cost, 0) AS base_cost_usd, ROUND(final_price, 0) AS final_price_usd, ROUND(margin_pct, 1) AS margin_pct FROM freight_quotes ORDER BY margin_pct DESC LIMIT 5;", conn)
                return f"CRITICAL FACT: Total Active Freight Quotes = {tot_quotes}.\n\n[Sample Subset Preview - First 5 Records Only]:\n{df_to_markdown_safe(df_q)}", "Freight Quotes DB"

            # 4. Weather
            if any(k in q_low for k in ["weather", "typhoon", "storm", "wind", "wave", "cyclone", "risk", "risks"]):
                tot_weather = conn.execute("SELECT COUNT(*) FROM weather_risks;").fetchone()[0]
                df_w = pd.read_sql("SELECT port_name AS 'Port Name', current_severity AS 'Severity (1-5)', forecast AS 'Forecast', wind_speed AS 'Wind (Knots)', wave_height AS 'Wave (m)' FROM weather_risks ORDER BY current_severity DESC LIMIT 5;", conn)
                return f"CRITICAL FACT: Total Monitored Weather Risk Hubs = {tot_weather}.\n\n[Sample Subset Preview - First 5 Records Only]:\n{df_to_markdown_safe(df_w)}", "Weather Risks DB"
    except Exception:
        pass
    return None


_AGENTIC_Q = _re_live.compile(r"(need[s]? attention|what should i do|action plan|prioriti[sz]e|overall (status|health)|run (the )?agents?|full analysis|full status|why is .{0,40}(low|down|failing|underperform|risky)|what is going wrong)")
def _agentic_query(query):
    """Open-ended 'what should I do / what needs attention' questions run the whole agent graph instead of a single table lookup."""
    if not _AGENTIC_Q.search(query.lower()): return None
    try:
        from live_engine import run_cycle
        import domain_freight as _dom
        dec = run_cycle(_dom, "copilot", query)
        top = "\n".join(f"- **{a['severity']}** · {a['type']} · {a['target']}: {a['detail']}" for a in dec["actions"][:8])
        return f"### Agent orchestration result\n\n{dec['narrative']}\n\n**Top actions**\n{top}", "Agent Orchestrator (live)"
    except Exception:
        return None

def _run_centralized_orig(query):
    freight = handle_freight_intent(query)
    if freight:
        return freight

    try:
        from rag_engine import answer_with_citation
        ctx, src = answer_with_citation(query)
        return ctx, src
    except Exception:
        return f"Retrieved enterprise freight intelligence for query: '{query}'.", "General Knowledge Index"

def run_grounded_query(query):
    return run_centralized_brain_query(query)

def text_to_sql(query):
    return run_centralized_brain_query(query)

def run_centralized_brain_query(query):
    _ag = _agentic_query(query)
    return _ag if _ag else _run_centralized_orig(query)


In [ ]:
%%writefile /content/app/live_engine.py
"""Shared live agent runtime (FranchiseOps AI + FreightQuote AI).
Domain modules (domain_franchise.py / domain_freight.py) plug in agents, a simulator and action handlers."""
import os, re, json, time, sqlite3, threading, datetime, smtplib, ssl
from dataclasses import dataclass, field
from email.message import EmailMessage
import requests
from config import DB_PATH

_DB_LOCK = threading.RLock()
_TL = threading.local()
def conn():
    """One SQLite connection per thread: nested helpers (emit/set_state) share the caller's transaction instead of deadlocking."""
    c = getattr(_TL, "c", None)
    if c is None:
        c = sqlite3.connect(DB_PATH, timeout=30); c.execute("PRAGMA journal_mode=WAL;"); c.execute("PRAGMA busy_timeout=30000;"); _TL.c = c
    return c
def now(): return datetime.datetime.now().isoformat(timespec="seconds")

@dataclass
class AgentResult:
    agent: str
    score: float = 100.0
    risk: str = "Low"
    summary: str = ""
    findings: dict = field(default_factory=dict)
    actions: list = field(default_factory=list)   # {"type","target","detail","severity","owner"?,"sla_h"?}
    confidence: float = 0.8

def risk_label(score): return "High" if score < 60 else "Medium" if score < 80 else "Low"

def init_live_tables():
    with _DB_LOCK, conn() as c:
        c.executescript("""
        CREATE TABLE IF NOT EXISTS agent_runs(run_id INTEGER PRIMARY KEY AUTOINCREMENT, ts TEXT, cycle_id TEXT, trigger_src TEXT, agent TEXT, risk TEXT, score REAL, latency_ms INT, summary TEXT, output TEXT);
        CREATE TABLE IF NOT EXISTS agent_actions(action_id INTEGER PRIMARY KEY AUTOINCREMENT, ts TEXT, type TEXT, target TEXT, detail TEXT, severity TEXT, owner TEXT, status TEXT DEFAULT 'open', due TEXT, resolved_ts TEXT, source_cycle TEXT, note TEXT);
        CREATE TABLE IF NOT EXISTS live_events(event_id INTEGER PRIMARY KEY AUTOINCREMENT, ts TEXT, etype TEXT, entity TEXT, payload TEXT);
        CREATE TABLE IF NOT EXISTS live_state(k TEXT PRIMARY KEY, v TEXT, ts TEXT);
        CREATE TABLE IF NOT EXISTS decisions(decision_id INTEGER PRIMARY KEY AUTOINCREMENT, ts TEXT, cycle_id TEXT, trigger_src TEXT, score REAL, risk TEXT, narrative TEXT, evidence TEXT);
        CREATE TABLE IF NOT EXISTS audit_log(audit_id INTEGER PRIMARY KEY AUTOINCREMENT, ts TEXT, actor TEXT, event TEXT, target TEXT, detail TEXT);
        CREATE TABLE IF NOT EXISTS ticket_comments(comment_id INTEGER PRIMARY KEY AUTOINCREMENT, action_id INT, ts TEXT, author TEXT, body TEXT);
        CREATE INDEX IF NOT EXISTS ix_act_status ON agent_actions(status, type, target);
        CREATE INDEX IF NOT EXISTS ix_runs_agent ON agent_runs(agent, run_id);
        CREATE INDEX IF NOT EXISTS ix_runs_cycle ON agent_runs(cycle_id);
        CREATE INDEX IF NOT EXISTS ix_cm_act ON ticket_comments(action_id);
        CREATE INDEX IF NOT EXISTS ix_audit_ts ON audit_log(audit_id);
        """)
        for col in ("assignee TEXT", "approval TEXT"):
            try: c.execute(f"ALTER TABLE agent_actions ADD COLUMN {col}")
            except sqlite3.OperationalError: pass
        for t, cols in (("sales", "outlet_id, ts"), ("attendance", "outlet_id, ts"), ("feedback", "outlet_id, date")):
            try: c.execute(f"CREATE INDEX IF NOT EXISTS ix_{t}_o ON {t}({cols})")
            except sqlite3.OperationalError: pass

def audit(actor, event, target="", detail=""):
    with _DB_LOCK, conn() as c:
        c.execute("INSERT INTO audit_log(ts,actor,event,target,detail) VALUES(?,?,?,?,?)", (now(), str(actor), event, str(target), str(detail)[:500]))

RETAIN = {"sales": ("ts", 6000), "attendance": ("ts", 3000), "feedback": ("date", 2000), "agent_runs": ("ts", 4000), "decisions": ("ts", 800), "live_events": ("ts", 2000), "audit_log": ("ts", 20000)}
def maintenance():
    """Keep the live DB bounded: prune to the newest N rows per table (never touches seed/master data)."""
    n = 0
    with _DB_LOCK, conn() as c:
        for t, (col, keep) in RETAIN.items():
            try:
                pk = c.execute(f"PRAGMA table_info({t})").fetchall()
                if not pk: continue
                if t in ("sales", "attendance"):
                    cnt = c.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
                    if cnt > keep * 4: n += c.execute(f"DELETE FROM {t} WHERE rowid IN (SELECT rowid FROM {t} ORDER BY rowid ASC LIMIT ?)", (cnt - keep * 2,)).rowcount
                else:
                    cnt = c.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
                    if cnt > keep * 2: n += c.execute(f"DELETE FROM {t} WHERE rowid IN (SELECT rowid FROM {t} ORDER BY rowid ASC LIMIT ?)", (cnt - keep,)).rowcount
            except sqlite3.OperationalError: pass
    return n

def set_state(k, v):
    with _DB_LOCK, conn() as c:
        c.execute("INSERT OR REPLACE INTO live_state(k,v,ts) VALUES(?,?,?)", (k, json.dumps(v, default=str), now()))
def get_state(k, default=None):
    with _DB_LOCK, conn() as c:
        r = c.execute("SELECT v FROM live_state WHERE k=?", (k,)).fetchone()
    return json.loads(r[0]) if r else default
def emit(etype, entity, payload=None):
    with _DB_LOCK, conn() as c:
        c.execute("INSERT INTO live_events(ts,etype,entity,payload) VALUES(?,?,?,?)", (now(), etype, str(entity), json.dumps(payload or {}, default=str)))

# ---------------- notifications (real channels, all optional via env) ----------------
_SENT = {}
def _nk(s): return re.sub(r"\d+(?:\.\d+)?", "#", str(s))
def notify(dom, target, severity, category, message):
    with _DB_LOCK, conn() as c:
        c.execute(f"INSERT INTO alerts ({dom.ENTITY_COL}, severity, category, message, date) VALUES (?,?,?,?,?)",
                  (target, severity, category, message, datetime.datetime.now().strftime("%Y-%m-%d %H:%M")))
    if severity not in ("Critical", "High"): return
    key = (target, category, _nk(message))
    if time.time() - _SENT.get(key, 0) < 3600: return          # no alert spam
    _SENT[key] = time.time()
    text = f"[{dom.NAME}] {severity} · {target} · {category}: {message}"
    for var in ("SLACK_WEBHOOK_URL", "TEAMS_WEBHOOK_URL"):
        if os.environ.get(var):
            try: requests.post(os.environ[var], json={"text": text}, timeout=5)
            except Exception: pass
    em, pw, to = os.environ.get("EMAIL_ID"), (os.environ.get("EMAIL_APP_PASSWORD") or os.environ.get("EMAIL_PASSWORD") or "").replace(" ", ""), os.environ.get("ALERT_EMAIL_TO")
    if em and pw and to:
        try:
            m = EmailMessage(); m["Subject"] = text[:90]; m["From"] = em; m["To"] = to; m.set_content(text)
            with smtplib.SMTP_SSL("smtp.gmail.com", 465, context=ssl.create_default_context(), timeout=15) as s:
                s.login(em, pw); s.send_message(m)
        except Exception: pass
    def _mail_subscribers():
        try:
            import mailer
            if not mailer.configured(): return
            with conn() as cc: rows = [r[0] for r in cc.execute("SELECT email FROM user_prefs WHERE email_alerts=1")]
            for e in rows: mailer.send(e, text[:90], text)
        except Exception: pass
    threading.Thread(target=_mail_subscribers, daemon=True).start()
    sid, tok, frm, tph = (os.environ.get(k) for k in ("TWILIO_SID", "TWILIO_TOKEN", "TWILIO_FROM", "ALERT_SMS_TO"))
    if sid and tok and frm and tph and getattr(dom, "SMS", False):
        try: requests.post(f"https://api.twilio.com/2010-04-01/Accounts/{sid}/Messages.json", auth=(sid, tok),
                           data={"From": frm, "To": tph, "Body": text[:300]}, timeout=8)
        except Exception: pass

# ---------------- action layer ----------------
SLA_H = {"Critical": 24, "Warning": 72, "Info": 168}
def policy(dom):
    """Autonomy policy per action type: 'auto' (agent executes) or 'approval' (human approves first). Admin-editable."""
    return {}      # no approval queue in this version: agents act on their own
def set_policy(t, mode, actor="admin"):
    p = get_state("policy", {}) or {}; p[t] = mode; set_state("policy", p); audit(actor, "policy", t, mode)
# ---------------------------------------------------------------- multi-level approval chains
# Each ticket that needs a human gets an ordered chain of steps (table approval_steps). Every step names a role; the next step opens only after the previous one is approved.
# Four-eyes rule: nobody can approve two steps of the same ticket, and nobody can approve a request they raised. Any step rejection rejects the whole ticket.
LADDER = ["Operations Manager", "Franchise Owner", "Admin"]
def _role_ok(role, need): return role == "Admin" or role == need or (need == "Operations Manager" and role == "Franchise Owner")
def chain_for(severity, typ, levels=None):
    cfg = get_state("approval_chain", {}) or {}
    if levels: return LADDER[:1] + (["Admin"] if int(levels) >= 2 else [])
    if typ in cfg and cfg[typ]: return list(cfg[typ])
    return ["Operations Manager", "Admin"] if severity == "Critical" else ["Operations Manager"]
def _steps_ddl(c): c.execute("CREATE TABLE IF NOT EXISTS approval_steps(action_id INT, step INT, role TEXT, status TEXT DEFAULT 'pending', by TEXT, ts TEXT, note TEXT, PRIMARY KEY(action_id, step))")
def _ensure_chain(c, aid, severity, typ):
    _steps_ddl(c)
    if not c.execute("SELECT 1 FROM approval_steps WHERE action_id=?", (aid,)).fetchone():
        for i, r in enumerate(chain_for(severity, typ)): c.execute("INSERT INTO approval_steps(action_id,step,role) VALUES(?,?,?)", (aid, i + 1, r))
def chains(ids):
    """{action_id: [{step, role, status, by}]} for the ticket list (creates the default chain for tickets that have none yet)."""
    out = {}
    if not ids: return out
    with _DB_LOCK, conn() as c:
        _steps_ddl(c)
        for aid, sev, typ in c.execute("SELECT action_id, severity, type FROM agent_actions WHERE action_id IN (%s) AND (status='proposed' OR type='quote_approval')" % ",".join("?" * len(ids)), list(ids)).fetchall(): _ensure_chain(c, aid, sev, typ)
        for r in c.execute("SELECT action_id, step, role, status, by FROM approval_steps WHERE action_id IN (%s) ORDER BY action_id, step" % ",".join("?" * len(ids)), list(ids)).fetchall(): out.setdefault(r[0], []).append({"step": r[1], "role": r[2], "status": r[3], "by": r[4] or ""})
    return out
def next_role(chain):
    for s in chain or []:
        if s["status"] == "pending": return s["role"]
    return None
def advance(aid, actor, role, approve=True, note=""):
    """Record one decision. Returns ('final'|'partial'|'rejected'|'denied'|'missing', message). Does not execute anything."""
    with _DB_LOCK, conn() as c:
        r = c.execute("SELECT severity, type, note FROM agent_actions WHERE action_id=? AND (status='proposed' OR type='quote_approval')", (aid,)).fetchone()
        if not r: return "missing", "nothing to approve on this ticket"
        _ensure_chain(c, aid, r[0], r[1]); steps = c.execute("SELECT step, role, status, by FROM approval_steps WHERE action_id=? ORDER BY step", (aid,)).fetchall()
        cur = next((s for s in steps if s[2] == "pending"), None)
        if not cur: return "missing", "this ticket has no pending approval step"
        if not _role_ok(role, cur[1]): return "denied", f"step {cur[0]} needs a {cur[1]} (you are {role})"
        if approve:
            if actor in [s[3] for s in steps if s[3]]: return "denied", "four-eyes rule: you already approved an earlier step of this ticket"
            if f"requested by {actor}" in (r[2] or ""): return "denied", "four-eyes rule: you cannot approve your own request"
        c.execute("UPDATE approval_steps SET status=?, by=?, ts=?, note=? WHERE action_id=? AND step=?", ("approved" if approve else "rejected", actor, now(), note, aid, cur[0]))
        left = [s for s in steps if s[2] == "pending" and s[0] != cur[0]]
        if not approve: return "rejected", f"rejected at step {cur[0]} ({cur[1]})"
        if left:
            c.execute("UPDATE agent_actions SET note=COALESCE(note,'')||? WHERE action_id=?", (f" step {cur[0]}/{len(steps)} approved by {actor};", aid)); audit(actor, "approve_step", aid, f"step {cur[0]}/{len(steps)} {note}".strip())
            return "partial", f"step {cur[0]} of {len(steps)} approved; waiting for {left[0][1]}"
    return "final", "all steps approved"
def request_approval(dom, typ, target, detail, severity, requester, levels=None):
    """Anyone with access can ask for a decision: creates a proposed ticket with its own approval chain."""
    a = {"type": "manual_request", "target": target, "detail": detail, "request": typ}
    with _DB_LOCK, conn() as c:
        cur = c.execute("INSERT INTO agent_actions(ts,type,target,detail,severity,owner,status,note,approval) VALUES(?,?,?,?,?,?,'proposed',?,?)", (now(), "manual_request", target, f"{typ}: {detail}", severity, requester, f"requested by {requester};", json.dumps(a)))
        aid = cur.lastrowid; _steps_ddl(c)
        for i, r in enumerate(chain_for(severity, "manual_request", levels)): c.execute("INSERT INTO approval_steps(action_id,step,role) VALUES(?,?,?)", (aid, i + 1, r))
    audit(requester, "request_approval", f"{typ}:{target}", detail[:120]); return aid
def approve_action(dom, aid, actor="user", approve=True, note="", role="Admin"):
    """Human-in-the-loop: runs the approval chain; the action executes only when the last step is approved. Returns (state, message)."""
    st, msg = advance(aid, actor, role, approve, note)
    if st in ("denied", "missing", "partial"): return st, msg
    with _DB_LOCK, conn() as c:
        r = c.execute("SELECT type,target,detail,severity,approval FROM agent_actions WHERE action_id=?", (aid,)).fetchone()
        if st == "final":
            a = json.loads(r[4] or "{}") or {"type": r[0], "target": r[1], "detail": r[2]}
            try: res = dom.apply(a, c) or ""
            except Exception as ex: res = f"apply failed: {ex}"
            if r[0] == "manual_request": c.execute("UPDATE agent_actions SET status='resolved', resolved_ts=?, note=COALESCE(note,'')||?, approval='approved' WHERE action_id=?", (now(), f" approved by {actor}; request granted", aid))
            else: c.execute("UPDATE agent_actions SET status='open', note=COALESCE(note,'')||?, approval=? WHERE action_id=?", (f" approved by {actor}; {res}", "approved", aid))
        else:
            c.execute("UPDATE agent_actions SET status='rejected', resolved_ts=?, note=COALESCE(note,'')||?, approval='rejected' WHERE action_id=?", (now(), f" rejected by {actor} {note}", aid))
    audit(actor, "approve" if approve else "reject", f"{r[0]}:{r[1]}", note); return st, msg
def assign_action(aid, assignee, actor="user"):
    with _DB_LOCK, conn() as c: c.execute("UPDATE agent_actions SET assignee=? WHERE action_id=?", (assignee, aid))
    audit(actor, "assign", aid, assignee)
def comment_action(aid, author, body):
    with _DB_LOCK, conn() as c: c.execute("INSERT INTO ticket_comments(action_id,ts,author,body) VALUES(?,?,?,?)", (now(), author, body) if False else (aid, now(), author, body))
    audit(author, "comment", aid, body[:80])
def act(dom, decision, cid, results):
    raised = set(); sent = 0; extra = 0; pol = policy(dom)
    for a in decision.get("actions", []):
        key = (a["type"], str(a["target"]), _nk(a["detail"])); raised.add(key)
        sev = a.get("severity", "Warning")
        due = (datetime.datetime.now() + datetime.timedelta(hours=a.get("sla_h", SLA_H.get(sev, 72)))).isoformat(timespec="seconds")
        with _DB_LOCK, conn() as c:
            hit = next((r[0] for r in c.execute("SELECT action_id, detail FROM agent_actions WHERE type=? AND target=? AND status IN ('open','proposed')", key[:2]).fetchall() if _nk(r[1].replace("OVERDUE: ", "")) == key[2]), None)
            if hit:
                c.execute("UPDATE agent_actions SET detail=CASE WHEN detail LIKE 'OVERDUE:%' THEN 'OVERDUE: '||? ELSE ? END WHERE action_id=?", (a["detail"], a["detail"], hit)); continue
            note = ""; status = "open"; appr = None
            if pol.get(a["type"]) == "approval": status = "proposed"; note = "awaiting human approval"; appr = json.dumps(a, default=str)
            else:
                try: note = dom.apply(a, c) or ""
                except Exception as ex: note = f"apply failed: {ex}"
            c.execute("INSERT INTO agent_actions(ts,type,target,detail,severity,owner,due,source_cycle,note,status,approval) VALUES(?,?,?,?,?,?,?,?,?,?,?)",
                      (now(), a["type"], str(a["target"]), a["detail"], sev, a.get("owner") or dom.OWNER.get(a["type"], "Operations"), due, cid, note, status, appr))
        if sev == "Critical":
            if sent < 8: notify(dom, a["target"], "Critical", a["type"], a["detail"]); sent += 1
            else: extra += 1
    if extra: notify(dom, "MULTIPLE", "Critical", "Summary", f"{extra} more critical actions raised this cycle; see Command Center")
    with _DB_LOCK, conn() as c:      # auto-close: the owning agent re-ran and no longer raises it
        for aid, t, tg, d in c.execute("SELECT action_id,type,target,detail FROM agent_actions WHERE status IN ('open','proposed')").fetchall():
            if dom.ACTION_AGENT.get(t) in results and (t, tg, _nk(d.replace("OVERDUE: ", ""))) not in raised:
                c.execute("UPDATE agent_actions SET status='auto-closed', resolved_ts=? WHERE action_id=?", (now(), aid))

def escalate_overdue(dom):
    n = 0
    with _DB_LOCK, conn() as c:
        rows = c.execute("SELECT action_id,target,detail,type FROM agent_actions WHERE status='open' AND due<? AND detail NOT LIKE 'OVERDUE:%'", (now(),)).fetchall()
        for aid, tg, d, t in rows:
            nd = (datetime.datetime.now() + datetime.timedelta(hours=24)).isoformat(timespec="seconds")
            c.execute("UPDATE agent_actions SET severity='Critical', owner=?, detail=?, due=? WHERE action_id=?", (dom.ESCALATE_TO, "OVERDUE: " + d, nd, aid)); n += 1
    for aid, tg, d, t in rows: notify(dom, tg, "Critical", "Escalation", "OVERDUE: " + d)
    return n

def resolve_action(aid, status="resolved", note="", actor="user"):
    audit(actor, status, aid, note)
    with _DB_LOCK, conn() as c:
        c.execute("UPDATE agent_actions SET status=?, resolved_ts=?, note=COALESCE(note,'')||? WHERE action_id=?", (status, now(), (" " + note) if note else "", aid))

# ---------------- orchestration (LangGraph when installed, plain loop otherwise) ----------------
def _log_run(cid, trig, res, ms):
    with _DB_LOCK, conn() as c:
        c.execute("INSERT INTO agent_runs(ts,cycle_id,trigger_src,agent,risk,score,latency_ms,summary,output) VALUES(?,?,?,?,?,?,?,?,?)",
                  (now(), cid, trig, res.agent, res.risk, res.score, ms, res.summary, json.dumps(res.findings, default=str)[:30000]))

def make_nodes(dom):
    def planner(s): return {"queue": dom.plan(s["request"]), "results": {}}
    def agent(s):
        name = s["queue"][0]; t = time.time()
        try: res = dom.AGENTS[name](s)
        except Exception as e: res = AgentResult(name, 50.0, "Medium", f"agent error: {e}")
        _log_run(s["cycle_id"], s["trigger"], res, int((time.time() - t) * 1000))
        new = [n for n in dom.REPLAN.get(name, lambda r: [])(res) if n not in s["results"] and n not in s["queue"]]
        if new: _log_run(s["cycle_id"], s["trigger"], AgentResult("re-plan", 100, "Low", f"{name} risk {res.risk}: adding {', '.join(new)}"), 0)
        return {"results": {**s["results"], name: res}, "queue": s["queue"][1:] + new}
    def engine(s):
        d = dom.engine(s)
        with _DB_LOCK, conn() as c:
            c.execute("INSERT INTO decisions(ts,cycle_id,trigger_src,score,risk,narrative,evidence) VALUES(?,?,?,?,?,?,?)",
                      (now(), s["cycle_id"], s["trigger"], d["score"], d["risk"], d["narrative"], json.dumps({k: v for k, v in d.items() if k not in ("narrative", "actions")}, default=str)))
        return {"decision": d}
    def act_node(s): act(dom, s["decision"], s["cycle_id"], s["results"]); return {}
    return planner, agent, engine, act_node

def _compile(dom):
    from typing import TypedDict
    from langgraph.graph import StateGraph, END
    class S(TypedDict, total=False):
        cycle_id: str; trigger: str; request: str; queue: list; results: dict; decision: dict
    planner, agent, engine, act_node = make_nodes(dom)
    g = StateGraph(S)
    for n, f in (("planner", planner), ("agent", agent), ("engine", engine), ("act", act_node)): g.add_node(n, f)
    g.set_entry_point("planner"); g.add_edge("planner", "agent")
    g.add_conditional_edges("agent", lambda s: "agent" if s["queue"] else "engine")
    g.add_edge("engine", "act"); g.add_edge("act", END)
    return g.compile()

_GRAPHS = {}
def run_cycle(dom, trigger="manual", request=""):
    init_live_tables()
    cid = datetime.datetime.now().strftime("%m%d-%H%M%S") + "-" + trigger[:4]
    s = {"cycle_id": cid, "trigger": trigger, "request": request, "queue": [], "results": {}, "decision": {}}
    try:
        if dom.NAME not in _GRAPHS: _GRAPHS[dom.NAME] = _compile(dom)
        out = _GRAPHS[dom.NAME].invoke(s)
    except ImportError:
        planner, agent, engine, act_node = make_nodes(dom)
        s.update(planner(s))
        while s["queue"]: s.update(agent(s))
        s.update(engine(s)); act_node(s); out = s
    return out["decision"]

# ---------------- background runtime ----------------
class Runtime:
    def __init__(self, dom, tick=5, cycle=60, min_gap=12):
        self.dom, self.tick, self.cycle, self.min_gap = dom, tick, cycle, min_gap
        self.stop = threading.Event(); self.ticks = 0; self.cycles = 0; self.last_cycle = 0; self.last_err = ""
        self.th = threading.Thread(target=self._loop, daemon=True); self.th.start()
        if hasattr(dom, "feed"): self.fth = threading.Thread(target=self._feed_loop, daemon=True); self.fth.start()
    def _feed_loop(self):          # network polling lives here so slow/offline APIs never stall the tick
        while not self.stop.is_set():
            try: self.dom.feed()
            except Exception as e: self.last_err = f"feed: {e}"
            self.stop.wait(int(os.environ.get("FEED_SEC", 60)))
    def _loop(self):
        init_live_tables(); set_state("boot", {"stage": "bootstrap", "ts": now()})
        try: self.dom.bootstrap()
        except Exception as e: self.last_err = f"bootstrap: {e}"
        set_state("boot", {"stage": "first_cycle", "ts": now()})
        while not self.stop.is_set():
            try:
                reasons = self.dom.tick() or []; self.ticks += 1
                if self.ticks % 120 == 0: maintenance()
                reasons = reasons + (self.dom.process() or [])
                due = time.time() - self.last_cycle
                if (reasons and due >= self.min_gap) or due >= self.cycle:
                    run_cycle(self.dom, "event" if reasons else "sched", ""); escalate_overdue(self.dom)
                    self.cycles += 1; self.last_cycle = time.time()
                    set_state("runtime", {"ticks": self.ticks, "cycles": self.cycles, "last_cycle": now(), "reasons": reasons[:3], "err": self.last_err})
                    if self.cycles == 1: set_state("boot", {"stage": "ready", "ts": now()})
                elif self.ticks % 6 == 0: set_state("runtime", {**(get_state("runtime", {}) or {}), "ticks": self.ticks})
            except Exception as e: self.last_err = str(e)
            self.stop.wait(self.tick)

_RT = None; _RT_LOCK = threading.Lock()
def start_runtime(dom):
    global _RT
    with _RT_LOCK:
        if _RT is None:
            _RT = Runtime(dom, int(os.environ.get("LIVE_TICK_SEC", 5)), int(os.environ.get("LIVE_CYCLE_SEC", 60)))
    return _RT


In [ ]:
%%writefile /content/app/alts.py
"""Alternatives shown in the detail panels: other sea routes for a shipment, nearby lower-congestion ports for a port, nearby outlets that can transfer stock to an outlet."""
import json, numpy as np, pandas as pd
import blocks as B
from blocks import Q, TEAL, BLUE, AMBER, RED, VIOLET
COLORS = [TEAL, BLUE, AMBER, VIOLET, "#d6409f", "#8b8d98"]
def hav(la1, lo1, la2, lo2, R=6371.0):
    p = np.pi / 180; a = np.sin((np.asarray(la2) - la1) * p / 2) ** 2 + np.cos(la1 * p) * np.cos(np.asarray(la2) * p) * np.sin((np.asarray(lo2) - lo1) * p / 2) ** 2; return 2 * R * np.arcsin(np.sqrt(a))
def usd(x): return "${:,.0f}".format(x)
VARIANTS = [((), "Default"), (("suez",), "Avoid Suez"), (("panama",), "Avoid Panama"), (("malacca",), "Avoid Malacca"), (("hormuz",), "Avoid Hormuz"), (("storm",), "Avoid storms")]
def route_alternatives(dom, o_xy, d_xy, tonnes=20.0, interp=None, rate=0.012, day_cost=120.0, dwell=0.0):
    """-> list of options sorted by total cost; each {label, name, nm, days, freight, delay_cost, total, wind, chain, avoid, flags}"""
    seen, out = {}, []
    for av, lab in VARIANTS:
        try: opts = dom.route_options(o_xy, d_xy, avoid=av)
        except Exception: continue
        if not opts: continue
        o = opts[0]; key = o["name"]
        if key in seen: continue
        seen[key] = 1; ch = o["chain"]; wind = 0.0
        if interp:
            try: wind = max(dom.weather_at(*interp(ch, f), allow_live=False)["wind"] for f in (.15, .35, .55, .75, .95))
            except Exception: pass
        days = o["nm"] / (14 * 24) + dwell; fr = o["nm"] * tonnes * rate; dl = days * day_cost; risk_cost = max(0, wind - 15) * 0.02 * days * day_cost * 3
        out.append({"label": lab, "name": o["name"], "nm": o["nm"], "days": round(days, 1), "freight": fr, "delay_cost": dl + risk_cost, "total": fr + dl + risk_cost, "wind": round(wind, 1), "chain": ch, "avoid": list(av)})
    if not out: return out
    out.sort(key=lambda r: r["total"]); fast = min(out, key=lambda r: r["days"]); short = min(out, key=lambda r: r["nm"])
    for r in out: r["tag"] = ", ".join(t for t, c in (("Recommended", r is out[0]), ("Fastest", r is fast), ("Shortest", r is short)) if c)
    return out
def route_blocks(opts, sid=None, title="Route alternatives", cur=None):
    if not opts: return [B.callout("No alternative sea route found for this lane.", "info")]
    lines = [{"pts": r["chain"], "color": COLORS[i % len(COLORS)], "w": 4 if i == 0 else 2.5, "dash": None if i == 0 else "6 6"} for i, r in enumerate(opts)]
    rows = [{"option": f"{i + 1}. {r['label']}", "route": r["name"], "nm": r["nm"], "days": r["days"], "freight": r["freight"], "delay": r["delay_cost"], "total": r["total"], "wind": r["wind"], "tag": r["tag"]} for i, r in enumerate(opts)]
    best, worst = opts[0], opts[-1]; pts = [p for r in opts for p in r["chain"]]; la = [p[0] for p in pts]; lo = [p[1] for p in pts]
    view = {"bounds": [[min(la) - 4, min(lo) - 6], [max(la) + 4, max(lo) + 6]]}
    ends = [{"id": "o", "lat": opts[0]["chain"][0][0], "lon": opts[0]["chain"][0][1], "color": "#1f2430", "r": 6, "tip": "start"}, {"id": "d", "lat": opts[0]["chain"][-1][0], "lon": opts[0]["chain"][-1][1], "color": RED, "r": 6, "tip": "destination"}]
    bl = [B.card(title, B.map_([{"type": "lines", "lines": lines}, {"type": "points", "id": "alt_ends", "pts": ends}], h=300, view=view, key="alt_map", legend=[(str(rows[i]["option"]) + (" (solid = recommended)" if i == 0 else " (dashed)"), COLORS[i % len(COLORS)]) for i in range(min(len(rows), len(COLORS)))] + [("Start", "#1f2430"), ("Destination", RED)]), B.table(rows, ["option", "route", B.num_col("nm", "nm", "int"), B.num_col("days", "days", "num"), B.num_col("freight", "freight cost", "money"), B.num_col("delay", "time + weather cost", "money"), B.num_col("total", "total", "money"), B.num_col("wind", "max wind m/s", "num"), "tag"], h=min(60 + 40 * len(rows), 330), key="alt_t"),
                       sub=f"Best: {best['label']} ({best['name']}), {best['days']} days, {usd(best['total'])}. Worst: {usd(worst['total'])}. Cost = $0.012 per tonne-nm + $120 per day + weather penalty above 15 m/s.")]
    return bl
def port_alternatives(port, k=6):
    p = Q("SELECT port_name, country, region, lat, lon, congestion_index ci, avg_dwell_days dw FROM ports"); me = p[p.port_name == port]
    if not len(me): return []
    me = me.iloc[0]; p["km"] = hav(me.lat, me.lon, p.lat.values, p.lon.values); c = p[(p.port_name != port) & (p.km < 1800)].copy()
    c["better"] = (me.ci - c.ci); c["score"] = c.ci * 12 + c.dw * 4 + c.km / 100; c = c[c.ci < me.ci].sort_values("score").head(k)
    rows = [{"port": r.port_name, "country": r.country, "km": r.km, "ci": r.ci, "dwell": r.dw, "days_saved": max(0, me.dw - r.dw), "haul_days": r.km / 600 / 1 if r.country == me.country else r.km / 450} for r in c.itertuples()]
    if not rows: return [B.callout(f"No less-congested port within 1,800 km of {port}.", "info")]
    return [B.card("Alternative ports nearby", B.table(rows, ["port", "country", B.num_col("km", "km away", "int"), B.num_col("ci", "congestion", "num"), B.num_col("dwell", "dwell days", "num"), B.num_col("days_saved", "dwell saved", "num"), B.num_col("haul_days", "extra haul days (approx.)", "num")], h=min(70 + 40 * len(rows), 340), open={"kind": "port", "col": "port"}, key="alt_p"),
                    sub="Cargo for the port could be diverted here; haul by road/rail is estimated at 600 km/day domestic, 450 km/day cross-border. Check customs and carrier agreements before diverting.")]
def outlet_alternatives(oid, max_km=400):
    g = Q("SELECT outlet_id, lat, lon FROM outlet_geo"); me = g[g.outlet_id == oid]
    if not len(me): return []
    me = me.iloc[0]; g["km"] = hav(me.lat, me.lon, g.lat.values, g.lon.values); near = g[(g.outlet_id != oid) & (g.km <= max_km)].sort_values("km").head(12)
    need = Q("SELECT sku_name, current_stock st, reorder_threshold thr, weekly_demand wk, lead_time_days lead FROM inventory WHERE outlet_id=? AND current_stock<=reorder_threshold*1.5 ORDER BY current_stock*1.0/MAX(weekly_demand,1) LIMIT 6", (oid,))
    if not len(need): return [B.callout("This outlet has no SKUs at or near the reorder level (within 150%), so no transfers are suggested.", "info")]
    if not len(near): return [B.callout(f"No outlet within {max_km} km to transfer stock from.", "info")]
    inv = Q(f"SELECT outlet_id, sku_name, current_stock st, reorder_threshold thr, weekly_demand wk FROM inventory WHERE outlet_id IN ({','.join('?' * len(near))}) AND sku_name IN ({','.join('?' * len(need))})", tuple(near.outlet_id) + tuple(need.sku_name))
    inv["spare"] = inv.st - (inv.thr + inv.wk); inv = inv[inv.spare > 0].merge(near[["outlet_id", "km"]], on="outlet_id"); rows = []
    for r in need.itertuples():
        want = max(1, int(r.thr + r.wk - r.st)); c = inv[inv.sku_name == r.sku_name].sort_values("km")
        if len(c):
            s = c.iloc[0]; q = int(min(want, s.spare)); days = max(.5, s.km * 1.3 / 320); rows.append({"sku": r.sku_name, "short_by": want, "from": s.outlet_id, "km": s.km, "units": q, "days": days, "vs_supplier": r.lead - days, "cost": s.km * 1.3 * 14})
        else: rows.append({"sku": r.sku_name, "short_by": want, "from": "none nearby has spare", "km": None, "units": 0, "days": None, "vs_supplier": None, "cost": None})
    d = pd.DataFrame(rows); mp = [{"type": "lines", "lines": [{"pts": [[float(me.lat), float(me.lon)], [float(g[g.outlet_id == r["from"]].lat.iloc[0]), float(g[g.outlet_id == r["from"]].lon.iloc[0])]], "color": COLORS[i % 6], "w": 3} for i, r in enumerate(rows) if r["units"]]},
                                {"type": "points", "id": "alt_o", "pts": [{"id": oid, "lat": float(me.lat), "lon": float(me.lon), "color": "#1f2430", "r": 8, "tip": oid}] + [{"id": r["from"], "lat": float(g[g.outlet_id == r["from"]].lat.iloc[0]), "lon": float(g[g.outlet_id == r["from"]].lon.iloc[0]), "color": TEAL, "r": 6, "tip": r["from"]} for r in rows if r["units"]]}]
    return [B.card("Suggested stock transfers from nearby outlets", B.map_(mp, h=240, key="alt_om", legend=[("This outlet (needs stock)", "#1f2430"), ("Outlet that can send stock", TEAL), ("Transfer line (colour per supplier outlet)", BLUE)]) if any(r["units"] for r in rows) else B.md("No nearby outlet has spare stock for these SKUs."), B.table(d, ["sku", B.num_col("short_by", "short by", "int"), "from", B.num_col("km", "km (straight line)", "int"), B.num_col("units", "transfer units", "int"), B.num_col("days", "transfer days (approx.)", "num"), B.num_col("vs_supplier", "days faster than supplier", "num"), B.num_col("cost", "est. cost ₹", "money")], h=min(80 + 40 * len(d), 330), open={"kind": "outlet", "col": "from"}, key="alt_ot"),
                    sub="Spare = stock above reorder level plus one week of demand. Distance is straight-line; road distance ≈ 1.3×. Cost ₹14 per road km. Compare with the supplier lead time before moving stock.")]


In [ ]:
%%writefile /content/app/analyst.py
"""Question understanding for the Copilot, ahead of the generic table query (qa.py). Each handler computes the answer from live data and returns
{"answer": lead text (markdown), "df": optional table, "sql": how it was computed, "extra": more markdown}. Nothing here asks the language model for a number."""
import re, json, datetime
import numpy as np, pandas as pd
from blocks import Q
import qa as QA, alts
CITY_STATE = {"Mumbai": "Maharashtra", "Pune": "Maharashtra", "Nashik": "Maharashtra", "Nagpur": "Maharashtra", "Delhi": "Delhi", "Bengaluru": "Karnataka", "Mysuru": "Karnataka", "Hyderabad": "Telangana", "Chennai": "Tamil Nadu", "Coimbatore": "Tamil Nadu", "Madurai": "Tamil Nadu",
              "Kolkata": "West Bengal", "Ahmedabad": "Gujarat", "Surat": "Gujarat", "Vadodara": "Gujarat", "Jaipur": "Rajasthan", "Lucknow": "Uttar Pradesh", "Kochi": "Kerala", "Thiruvananthapuram": "Kerala", "Chandigarh": "Chandigarh", "Bhopal": "Madhya Pradesh", "Indore": "Madhya Pradesh",
              "Patna": "Bihar", "Ranchi": "Jharkhand", "Raipur": "Chhattisgarh", "Bhubaneswar": "Odisha", "Guwahati": "Assam", "Goa": "Goa", "Visakhapatnam": "Andhra Pradesh", "Vijayawada": "Andhra Pradesh", "Kanpur": "Uttar Pradesh", "Dehradun": "Uttarakhand", "Amritsar": "Punjab", "Ludhiana": "Punjab"}
CHOKE = {"suez": ("Suez Canal", 30.5, 32.3), "panama": ("Panama Canal", 9.1, -79.7), "malacca": ("Strait of Malacca", 2.5, 101.5), "hormuz": ("Strait of Hormuz", 26.5, 56.5), "bosphorus": ("Bosphorus", 41.1, 29.0), "cape": ("Cape of Good Hope", -34.4, 18.5), "red sea": ("Red Sea", 18.0, 40.0)}
def _w(q, t): return re.search(r"(?<![a-z0-9])" + re.escape(t) + r"(?![a-z0-9])", q) is not None
def _md(df, n=10): return QA.md(df.head(n)) if df is not None and len(df) else ""
def _health(dec): return (((dec or {}).get("ev") or {}).get("health")) or {}
def _agent_scores(e):
    import entities as EN; out = {}
    for a in ("outlet", "inventory", "staff", "marketing", "audit", "feedback"):
        v = EN.latest_out(a).get("per_outlet", {}).get(e)
        if v is not None: out[a] = round(float(v), 1)
    return out
NEXT = {"staffing_shortage": "add a cover shift or move staff from a nearby outlet", "health_alert": "review the lowest agent score below and fix that first", "reorder": "place the reorder or transfer stock from a nearby outlet", "stockout": "reorder now or transfer stock",
        "corrective_action": "complete the corrective plan before the re-audit", "anomaly_review": "check the POS feed and the day's sales for this outlet", "campaign_review": "pause or re-target the campaign", "retention": "talk to the staff at risk and review overtime"}
# ------------------------------------------------------------------------------------------------ Franchise
def explain_outlet(e, dec):
    o = Q("SELECT * FROM outlets WHERE outlet_id=?", (e,))
    if not len(o): return None
    r = o.iloc[0]; h = _health(dec).get(e); sc = _agent_scores(e); tk = Q("SELECT action_id id, type, severity, status, detail FROM agent_actions WHERE target=? AND status IN ('open','proposed') ORDER BY CASE severity WHEN 'Critical' THEN 0 WHEN 'Warning' THEN 1 ELSE 2 END, action_id DESC LIMIT 8", (e,))
    avg = Q("SELECT AVG(revenue) rv, AVG(customer_satisfaction) cs FROM outlets").iloc[0]; low = Q("SELECT COUNT(*) n FROM inventory WHERE outlet_id=? AND current_stock<=reorder_threshold*1.5", (e,)).n[0]
    att = Q("SELECT AVG(predicted_attrition_prob) p, COUNT(*) n FROM staff WHERE outlet_id=?", (e,)).iloc[0]
    lines = []
    if h is not None: lines.append(f"Health score **{h:.1f}**/100 ({'Critical' if h < 60 else 'Needs attention' if h < 80 else 'Healthy'}).")
    if sc: worst = sorted(sc.items(), key=lambda kv: kv[1])[:2]; lines.append("Weakest areas: " + ", ".join(f"**{a}** {v}" for a, v in worst) + " (each agent scores 0-100, higher is better).")
    lines.append(f"Revenue {r.revenue:,.0f} ({(r.revenue / avg.rv - 1):+.0%} vs chain average), satisfaction {r.customer_satisfaction:.2f} (chain {avg.cs:.2f}), {int(att.n)} staff with average attrition risk {att.p:.0%}, {int(low)} SKUs at or near reorder level.")
    why = "\n".join(f"- {t.type} ({t.severity}): {t.detail}" + (f" → {NEXT[t.type]}" if t.type in NEXT else "") for t in tk.itertuples()) or "- no open tickets"
    return {"answer": (f"**{e}** ({r.outlet_name}, {r.location}) is flagged because:\n" if (h is None or h < 80 or len(tk)) else f"**{e}** ({r.outlet_name}, {r.location}) is not flagged right now; here is its status:\n") + "\n".join("- " + l for l in lines) + "\n\n**Open tickets:**\n" + why, "df": pd.DataFrame([{"agent": a, "score": v} for a, v in sc.items()]) if sc else None, "sql": f"decision health + agent per_outlet scores + agent_actions for {e}"}
def risk_rank(q, dec, scope, n=5):
    hl = _health(dec)
    if not hl: return None
    s = pd.Series(hl).astype(float); worst = bool(re.search(r"\b(highest risk|most at risk|worst|lowest|riskiest|critical|weakest|needs? attention|problem|bottom)\b", q)); best = bool(re.search(r"\b(best|top|highest health|strongest|healthiest)\b", q)) and not re.search(r"highest risk", q)
    both = re.search(r"best.*worst|worst.*best", q) is not None
    if scope: s = s[s.index == scope]
    o = Q("SELECT outlet_id, outlet_name, location, revenue, customer_satisfaction csat FROM outlets").set_index("outlet_id")
    def frame(ix):
        rows = []
        for e in ix:
            sc = _agent_scores(e); wk = min(sc, key=sc.get) if sc else "-"; ct = int(Q("SELECT COUNT(*) n FROM agent_actions WHERE target=? AND status IN ('open','proposed') AND severity='Critical'", (e,)).n[0])
            rows.append({"outlet": e, "name": o.name if False else (o.outlet_name.get(e, "") if e in o.index else ""), "location": o.location.get(e, ""), "health": round(float(s[e]), 1), "weakest_area": f"{wk} {sc.get(wk, '')}", "critical_tickets": ct, "revenue": o.revenue.get(e), "csat": o.csat.get(e)})
        return pd.DataFrame(rows)
    low = s.sort_values().head(n).index; hi = s.sort_values(ascending=False).head(n).index
    if both or (best and worst):
        a, b = frame(hi), frame(low); return {"answer": f"Best outlet: **{a.outlet[0]}** (health {a.health[0]}). Worst outlet: **{b.outlet[0]}** (health {b.health[0]}). Health combines outlet, inventory, staff, marketing, audit and feedback agent scores.", "df": None, "sql": "decision health per outlet", "extra": "**Top " + str(n) + " by health**\n" + QA.md(a) + "\n\n**Bottom " + str(n) + " by health**\n" + QA.md(b)}
    if best and not worst: d = frame(hi); return {"answer": f"Healthiest outlet: **{d.outlet[0]}** (health {d.health[0]}, {d.location[0]}). Top {n} below.", "df": d, "sql": "decision health per outlet, descending"}
    d = frame(low); crit = int((s < 60).sum()); return {"answer": f"Highest risk: **{d.outlet[0]}** (health {d.health[0]}, weakest area {d.weakest_area[0]}). {crit} outlets are below 60 (critical). Lowest {n} below; ask \"why is {d.outlet[0]} flagged\" for the reasons.", "df": d, "sql": "decision health per outlet, ascending"}
def inventory_status(scope):
    w = " WHERE outlet_id=?" if scope else ""; p = (scope,) if scope else ()
    d = Q("SELECT outlet_id, sku_name, category, current_stock st, reorder_threshold thr, weekly_demand wk, lead_time_days lead, stockout_risk_prob p FROM inventory" + w, p)
    if not len(d): return None
    d["cover_d"] = d.st / (d.wk / 7).replace(0, np.nan); below = d[d.st <= d.thr]; wk1 = d[d.cover_d < 7]; lt = d[d.cover_d < d.lead]
    po = Q("SELECT status, COUNT(*) n FROM purchase_orders" + w + " GROUP BY status", p); cat = d.groupby("category").agg(skus=("sku_name", "count"), below_reorder=("st", lambda s: int((d.loc[s.index, "st"] <= d.loc[s.index, "thr"]).sum())), avg_cover_days=("cover_d", "mean"), avg_risk=("p", "mean")).reset_index().sort_values("avg_risk", ascending=False)
    worst = d.sort_values("cover_d").head(8)[["outlet_id", "sku_name", "st", "cover_d", "lead", "p"]].rename(columns={"st": "stock", "cover_d": "days_of_cover", "lead": "lead_days", "p": "stockout_risk"})
    return {"answer": f"Inventory: **{len(d):,}** SKU records; **{len(below):,}** at or below reorder level, **{len(wk1):,}** have under 7 days of cover, **{len(lt):,}** will run out before a new order can arrive (cover shorter than lead time). Average stock-out risk {d.p.mean():.0%}." + (" Open purchase orders: " + ", ".join(f"{r.status} {r.n}" for r in po.itertuples()) + "." if len(po) else ""), "df": worst, "sql": "inventory: stock vs reorder threshold, cover = stock ÷ daily demand", "extra": "**By category**\n" + QA.md(cat.round(2))}
def sales_trend(scope):
    w = " AND outlet_id=?" if scope else ""; p = (scope,) if scope else ()
    d = Q(f"SELECT substr(ts,1,10) day, SUM(amount) amount, SUM(tx_count) tx FROM sales WHERE ts IS NOT NULL{w} GROUP BY day ORDER BY day DESC LIMIT 14", p)
    if len(d) < 2: return None
    d = d.iloc[::-1].reset_index(drop=True); last7, prev7 = d.tail(7).amount.sum(), d.iloc[:-7].tail(7).amount.sum() if len(d) > 7 else np.nan
    ch = f" ({(last7 / prev7 - 1):+.1%} vs the 7 days before)" if prev7 == prev7 and prev7 else ""
    return {"answer": f"Sales over the last 7 recorded days: **{last7:,.0f}**{ch}. Best day {d.loc[d.amount.idxmax(), 'day']} ({d.amount.max():,.0f}), weakest {d.loc[d.amount.idxmin(), 'day']} ({d.amount.min():,.0f}).", "df": d.tail(10), "sql": "sales grouped by day"}

def _place(ql):
    """Find a state or city named in the question. Returns ("state"|"city", name) or None."""
    flat = re.sub(r"[^a-z]", "", ql); cities = Q("SELECT DISTINCT location FROM outlets").location.dropna().tolist()
    for st in sorted(set(CITY_STATE.values()), key=len, reverse=True):
        if re.sub(r"[^a-z]", "", st.lower()) in flat: return "state", st
    for ct in sorted(cities, key=len, reverse=True):
        if re.search(r"\b" + re.escape(ct.lower()) + r"\b", ql): return "city", ct
    return None
def outlets_in_place(ql, scope=None):
    pl = _place(ql)
    if not pl: return None
    kind, name = pl; o = Q("SELECT outlet_id, outlet_name name, location, tier, revenue, customer_satisfaction csat, staff_headcount staff FROM outlets")
    o["state"] = o.location.map(CITY_STATE).fillna("Other"); d = o[o.state == name] if kind == "state" else o[o.location == name]
    if not len(d): return {"answer": f"No outlets found in {name}.", "df": None, "sql": "outlets filtered by place"}
    by = d.groupby("location").size().sort_values(ascending=False); txt = ", ".join(f"{k} {v}" for k, v in by.items())
    return {"answer": f"**{len(d)}** outlets in {name}" + (f": {txt}." if kind == "state" else "."), "df": d.drop(columns=["state"]).sort_values(["location", "outlet_id"]), "sql": f"outlets where {'state (from city)' if kind == 'state' else 'location'} = {name}"}
def staff_by_place(ql, scope=None):
    s = Q("SELECT s.staff_id, s.outlet_id, o.location FROM staff s LEFT JOIN outlets o USING(outlet_id)")
    if not len(s): return None
    s["state"] = s.location.map(CITY_STATE).fillna("Other"); pl = _place(ql)
    if pl:
        kind, name = pl; d = s[s.state == name] if kind == "state" else s[s.location == name]; g = d.groupby("location").size().sort_values(ascending=False).reset_index(name="employees")
        return {"answer": f"**{len(d):,}** employees in {name}" + (f"; most in **{g.location.iloc[0]}** ({int(g.employees.iloc[0]):,})." if len(g) else "."), "df": g, "sql": f"staff joined to outlets, place = {name}"}
    by_state = bool(re.search(r"\bstates?\b", ql)); by_out = bool(re.search(r"\boutlets?\b", ql)); col = "outlet_id" if by_out else ("state" if by_state else "location")
    g = s.groupby(col).size().sort_values(ascending=False).head(15).reset_index(name="employees"); low = bool(re.search(r"\b(fewest|least|lowest|less|low number)\b", ql)); r = (s.groupby(col).size().sort_values().reset_index(name="employees").iloc[0] if low else g.iloc[0])
    return {"answer": f"**{r[col]}** has the {'fewest' if low else 'most'} employees: **{int(r.employees):,}** (of {len(s):,} in total, by {'outlet' if by_out else 'state' if by_state else 'city'}).", "df": g, "sql": f"staff joined to outlets, grouped by {col}"}
def by_state(q):
    o = Q("SELECT outlet_id, location, revenue FROM outlets"); o["state"] = o.location.map(CITY_STATE).fillna("Other"); g = o.groupby("state").agg(outlets=("outlet_id", "count"), revenue=("revenue", "sum")).reset_index().sort_values("outlets", ascending=False)
    low = bool(re.search(r"\b(fewest|least|lowest)\b", q)); r = g.iloc[-1] if low else g.iloc[0]
    return {"answer": f"**{r.state}** has the {'fewest' if low else 'most'} outlets: **{int(r.outlets)}**. Your outlets table stores the city, so states are derived from the city name.", "df": g, "sql": "outlets grouped by state (city → state mapping)"}
def staff_status(scope):
    w = " WHERE outlet_id=?" if scope else ""; p = (scope,) if scope else ()
    s = Q("SELECT outlet_id, role, overtime_hrs ot, predicted_attrition_prob pr FROM staff" + w, p); at = Q("SELECT status, COUNT(*) n FROM attendance" + (" WHERE outlet_id=?" if scope else "") + " GROUP BY status", p)
    if not len(s): return None
    hi = s[s.pr > .6]; g = s.groupby("outlet_id").pr.mean().sort_values(ascending=False).head(8).reset_index().rename(columns={"pr": "avg_attrition_risk"})
    return {"answer": f"Workforce: **{len(s):,}** staff, **{len(hi):,}** at high attrition risk (>60%), average overtime {s.ot.mean():.1f} h" + (". Attendance: " + ", ".join(f"{r.status} {r.n:,}" for r in at.itertuples()) if len(at) else "") + ".", "df": g, "sql": "staff + attendance", }
# ------------------------------------------------------------------------------------------------ Freight
STOP = {"long", "port", "new", "fort", "saint", "south", "north", "east", "west", "city", "bay", "cape", "grand", "great", "royal", "santa", "puerto", "king", "bandar"}
def _ports_in(q):
    p = Q("SELECT port_name FROM ports").port_name.tolist(); spans = []; first = {}
    for name in p:
        core = [w for w in re.sub(r"\(.*?\)", "", name).lower().split() if w not in QA.GENERIC]
        if core and len(core[0]) >= 4 and core[0] not in STOP: first.setdefault(core[0], []).append(name)
    for name in p:
        als = set(QA._alias(name)); core = [w for w in re.sub(r"\(.*?\)", "", name).lower().split() if w not in QA.GENERIC]
        if core and core[0] not in STOP and len(first.get(core[0], [])) == 1 and len(core[0]) >= 4: als.add(core[0])
        for al in sorted(als, key=len, reverse=True):
            m = re.search(r"(?<![a-z0-9])" + re.escape(al) + r"(?![a-z0-9])", q)
            if m: spans.append((m.start(), m.end(), len(al), name)); break
    spans.sort(key=lambda x: (-x[2], x[0])); kept = []
    for s_ in spans:
        if not any(s_[0] < k[1] and k[0] < s_[1] for k in kept): kept.append(s_)
    kept.sort(); return [k[3] for k in kept], kept
def plan_lane(q, dom):
    ps, hits = _ports_in(re.sub(r"how long", "how_long", q.lower()))
    if len(ps) < 2: return None
    pre = q.lower(); o, d = ps[0], ps[1]; m = re.search(r"from\s+(.+?)\s+to\s+", pre)
    cargo = next((k for k in dom.CARGO if k.lower() in pre or k.lower().rstrip("s") in pre), "General"); wm = re.search(r"(\d+(?:\.\d+)?)\s*(t|tons?|tonnes?|mt|kg|kgs)\b", pre); wt = 20.0
    if wm: wt = float(wm.group(1)) / (1000 if wm.group(2).startswith("kg") else 1)
    po, pd_ = dom.port_xy(o), dom.port_xy(d)
    if not po or not pd_: return None
    cust = Q("SELECT customer_id FROM customers LIMIT 1").customer_id[0]; req = {"origin": o, "dest": d, "cargo": cargo, "weight_t": wt, "hs_code": "HS-8501", "customer_id": cust, "docs_provided": None}
    pq = dom.preview_quote(req); opts = alts.route_alternatives(dom, (po[0], po[1]), (pd_[0], pd_[1]), tonnes=wt, interp=dom.interp, dwell=(po[3] + pd_[3]) * .5)
    if not pq and not opts: return None
    rows = [{"option": f"{i + 1}. {x['label']}", "route": x["name"], "nm": x["nm"], "days": round(x["days"], 1), "freight_cost": round(x["freight"]), "time_cost": round(x["delay_cost"]), "total": round(x["total"]), "tag": x["tag"]} for i, x in enumerate(opts)]
    head = f"**{o} → {d}**, {wt:g} t of {cargo}: "
    if pq: head += f"recommended route **{pq['route']}**, about **{pq['days']} days** at sea and in port, quoted price **${pq['price']:,.0f}** (margin {pq['margin_pct']}%), weather risk {pq['weather']}, decision: {pq['decision']}."
    elif opts: head += f"best route **{opts[0]['name']}**, about **{opts[0]['days']} days**."
    ex = (pq["narrative"] if pq and pq.get("narrative") else "") + ("\n\nOnly one sea route exists for this lane in the routing model (it does not pass a closable canal or strait), so the alternatives are commercial: another carrier, or diverting to a nearby less-congested port (open the port in the map to see candidates)." if len(opts) == 1 else "")
    return {"answer": head, "df": pd.DataFrame(rows) if rows else None, "sql": "dom.preview_quote (route, weather, ML price, customs, margin) + sea-route alternatives; nothing was saved", "extra": ex, "map": opts}
def choke_ports(q):
    k = next((k for k in CHOKE if _w(q, k)), None)
    if not k: return None
    name, la, lo = CHOKE[k]; p = Q("SELECT port_name, country, congestion_index ci, avg_dwell_days dw, lat, lon FROM ports"); p["km"] = alts.hav(la, lo, p.lat.values, p.lon.values); near = p[p.km <= 1800].sort_values("ci", ascending=False)
    v = Q("SELECT COUNT(*) n FROM vessels WHERE route_name LIKE ?", (f"%{name.split()[0]}%",)).n[0]; sh = Q("SELECT COUNT(*) n FROM shipments s JOIN vessels v USING(shipment_id) WHERE v.route_name LIKE ? AND s.status LIKE '%Delay%'", (f"%{name.split()[0]}%",)).n[0]
    if not len(near): return None
    return {"answer": f"Ports within 1,800 km of the **{name}**: **{len(near)}**; **{int((near.ci >= 4).sum())}** are congested (index ≥ 4). Most congested: **{near.port_name.iloc[0]}** ({near.ci.iloc[0]:.1f}, dwell {near.dw.iloc[0]:.1f} days). {int(v)} tracked vessels are routed via this passage, {int(sh)} of them delayed.", "df": near[["port_name", "country", "ci", "dw", "km"]].rename(columns={"ci": "congestion", "dw": "dwell_days", "km": "km_from_passage"}), "sql": f"ports within 1,800 km of {name}"}
def freight_summary(q):
    if re.search(r"\bcustoms\b", q) and re.search(r"\b(hold|held|status|summary|how many|why|docs?|documents?)\b", q):
        d = Q("SELECT dest_port, COUNT(*) n FROM shipments WHERE status='Customs Hold' GROUP BY dest_port ORDER BY n DESC LIMIT 8"); t = int(Q("SELECT COUNT(*) n FROM shipments WHERE status='Customs Hold'").n[0]); tk = Q("SELECT detail FROM agent_actions WHERE type='customs_docs' AND status IN ('open','proposed') LIMIT 400")
        miss = pd.Series([x for dd in tk.detail for x in re.findall(r"missing ([^:]+?)(?:\.|$)", dd)[0].split(", ")] if len(tk) and all("missing" in x for x in tk.detail) else []).value_counts().head(6)
        return {"answer": f"**{t}** shipments are on customs hold. Most held at **{d.dest_port.iloc[0] if len(d) else '-'}**." + (" Most often missing: " + ", ".join(f"{k} ({v})" for k, v in miss.items()) + "." if len(miss) else ""), "df": d.rename(columns={"n": "held"}), "sql": "shipments with status Customs Hold; customs_docs tickets"}
    return None
def greeting(q, kind):
    ql = re.sub(r"[^a-z0-9 ?']", " ", q.lower()).strip()
    hi = re.match(r"^(hi+|hello+|hey+|hola|namaste|good (morning|afternoon|evening)|greetings|yo|sup|howdy)\b( there| team| copilot| buddy)?$", ql)
    thanks = re.match(r"^(thanks?|thank you|thx|ok(ay)?|cool|great|nice|got it|awesome)( a lot| so much)?$", ql)
    who = re.search(r"\b(who are you|what are you|what can you do|what do you do|help me|how can you help|your capabilities|what is this)\b", ql) or ql in ("help", "?")
    how = re.match(r"^how are you", ql)
    if not (hi or thanks or who or how): return None
    ex = (["Which stock needs immediate refill?", "Which outlets are at highest risk and why?", "Top 5 outlets by revenue", "Why is OUT-007 flagged?", "Which state has the most outlets?"] if kind == "Franchise" else
          ["Mumbai to Singapore 50 t electronics: days, cost and best route", "Which ports are most congested?", "How many shipments are on customs hold?", "Which quotes have the lowest margin?"])
    what = ("outlets, stock, staff, marketing, audits, customer feedback and anomalies" if kind == "Franchise" else "routes, ports, weather, customs, quotes and carriers")
    if thanks: return {"answer": "You're welcome. Ask me anything else about " + what + ".", "df": None, "sql": ""}
    head = "Hello! " if hi else ("I'm doing well, thank you. " if how else "")
    return {"answer": head + f"I'm your operations copilot. I answer from your live data on {what}, and I can explain *why* something is flagged, suggest actions and send them to the approval queue. I understand 37 languages.\n\nTry one of these:\n" + "\n".join(f"- {e}" for e in ex), "df": None, "sql": ""}

def refill(scope, n=10):
    w = " WHERE outlet_id=?" if scope else ""; p = (scope,) if scope else ()
    d = Q("SELECT outlet_id, sku_name sku, category, current_stock stock, reorder_threshold thr, weekly_demand wk, lead_time_days lead FROM inventory" + w, p)
    if not len(d): return None
    d["daily"] = d.wk / 7.0; d["cover"] = d.stock / d.daily.replace(0, np.nan); d["cover"] = d.cover.fillna(999)
    d["order_qty"] = np.maximum(d.thr * 3, np.ceil(d.daily * (d.lead + 7)) - d.stock).astype(int)
    d["gap"] = d.cover - d.lead
    d["urgency"] = np.where(d.cover < d.lead, "Critical: runs out before delivery", np.where(d.stock <= d.thr, "High: below reorder level", np.where(d.cover < 7, "Medium: under 7 days", "OK")))
    d["rank"] = d.urgency.str[:4].map({"Crit": 0, "High": 1, "Medi": 2}).fillna(3); need = d[d.urgency != "OK"].sort_values(["rank", "gap", "cover"])
    if not len(need): return {"answer": "No SKU needs an urgent refill right now: all have stock above the reorder level and cover longer than the supplier lead time.", "df": None, "sql": "inventory: cover = stock / daily demand vs lead time"}
    crit = need[need.urgency.str.startswith("Critical")]; po = Q("SELECT outlet_id, sku_name FROM purchase_orders WHERE status IN ('open','ordered','pending','Pending','Ordered','Open')" + (" AND outlet_id=?" if scope else ""), p) if "purchase_orders" in [r[0] for r in Q("SELECT name FROM sqlite_master WHERE type='table'").values] else None
    top = need.head(n)[["outlet_id", "sku", "stock", "thr", "cover", "lead", "order_qty", "urgency"]].rename(columns={"thr": "reorder_level", "cover": "days_of_cover", "lead": "lead_days"}); top["days_of_cover"] = top.days_of_cover.round(1)
    pool = crit if len(crit) else need[need.urgency.str.startswith("High")] if (need.urgency.str.startswith("High")).any() else need; by = pool.groupby("outlet_id").size().sort_values(ascending=False).head(5)
    outs = ", ".join(f"{k} ({v} SKUs)" for k, v in by.items()) if len(by) else "none"
    return {"answer": f"**{len(crit):,}** SKU records will run out *before a new order can arrive* (days of cover < supplier lead time) and **{len(need) - len(crit):,}** more are low. Most urgent outlets: {outs}. The table lists the {len(top)} most urgent SKUs, with the quantity to order.",
            "df": top, "sql": "inventory: days_of_cover = stock / (weekly_demand/7); urgent when cover < lead_time_days; order_qty = max(3 x reorder level, daily demand x (lead + 7 days) - stock)",
            "extra": "**How to read it:** *days_of_cover* is how long current stock lasts at current demand; *lead_days* is how long the supplier takes. When cover is shorter than lead time, stock-out is certain unless you order now (or transfer stock from a nearby outlet). The matching *reorder* tickets are raised by the agent."}

# ------------------------------------------------------------------------------------------------ router
def answer(kind, q, scope=None, dec=None, dom=None):
    r = _answer(kind, q, scope, dec, dom)
    if r: r["complete"] = True
    return r
def _answer(kind, q, scope=None, dec=None, dom=None):
    g = greeting(q, kind)
    if g: return g
    ql = q.lower().strip(); ids = re.findall(r"\b(OUT-\d+|STF-\d+|SHP-[A-Z0-9-]+|QTE-[A-Z0-9-]+)\b", q, re.I)
    try:
        if kind == "Franchise":
            if ids and ids[0].upper().startswith("OUT") and re.search(r"\b(why|reason|flag\w*|wrong|problem|issue|explain|risk)\b", ql):
                if scope and ids[0].upper() != scope: return {"answer": f"{ids[0].upper()} is outside your outlet scope ({scope}).", "df": None, "sql": ""}
                return explain_outlet(ids[0].upper(), dec)
            _pl = (bool(re.search(r"\b(where|which|list|all|show|located|locations?|states?|cities|city|how many|number)\b", ql)) or bool(_place(ql))) and not re.search(r"\b(risk\w*|health\w*|revenue|sales|csat|satisfaction|inventory|stock|attrition|salary|overtime|best|worst|weakest|critical)\b", ql)
            if _pl and re.search(r"\b(employees?|staff|workers?|people|headcount|manpower|workforce)\b", ql) and (_place(ql) or re.search(r"\b(where|which|most|highest|more|largest|big|fewest|least|lowest|number)\b", ql)): return staff_by_place(ql, scope)
            if _pl and re.search(r"\b(outlets?|stores?|branch\w*|franchises?)\b", ql) and _place(ql): return outlets_in_place(ql, scope)
            if re.search(r"\b(state|states)\b", ql) and re.search(r"\b(outlets?|stores?|branch\w*)\b", ql): return by_state(ql)
            if re.search(r"\b(outlets?|stores?|branch\w*)\b", ql) and re.search(r"\b(risk\w*|health\w*|best|worst|weakest|strongest|critical|attention|problem\w*|performing|performance)\b", ql) and not re.search(r"\b(revenue|sales|csat|satisfaction|headcount)\b", ql): return risk_rank(ql, dec, scope)
            if re.search(r"\b(inventory|stock|stocks|stocked|sku|skus|items?|products?|ingredients?)\b", ql) and re.search(r"\b(refill\w*|restock\w*|re-?order\w*|replenish\w*|immediate\w*|urgent\w*|attention|run(ning)? (out|low)|shortage\w*|low|critical|need\w*|order now)\b", ql) and not re.search(r"\b(how many|count|average|total)\b", ql): return refill(scope)
            if re.search(r"\b(inventory|stock)\b", ql) and re.search(r"\b(how|status|holding|summary|overview|situation|look|doing|week|today|health)\b", ql) and not re.search(r"\b(how many|count|average|total|top|lowest|highest|which)\b", ql): return inventory_status(scope)
            if re.search(r"\b(sales|revenue|takings)\b", ql) and re.search(r"\b(trend|this week|last week|today|yesterday|last 7|past week|recent|how (is|are|did))\b", ql): return sales_trend(scope)
            if re.search(r"\b(staff|workforce|employees?|team)\b", ql) and re.search(r"\b(how|status|summary|overview|situation|doing|look)\b", ql) and not re.search(r"\b(how many|count|average|total|top|lowest|highest|which)\b", ql): return staff_status(scope)
        else:
            if dom is not None and re.search(r"\b(days?|time|transit|cost|price|quote|route|how long|how much|eta)\b", ql):
                r = plan_lane(q, dom)
                if r: return r
            if re.search(r"\b(congest\w*|delay\w*|busy|ports?)\b", ql) and any(_w(ql, k) for k in CHOKE): return choke_ports(ql)
            r = freight_summary(ql)
            if r: return r
    except Exception as e:
        import sys; print("analyst failed:", repr(e), file=sys.stderr)
    return None


In [ ]:
%%writefile /content/app/qa.py
"""Exact-answer engine for the Copilot. Turns a plain-English question into a read-only, parameterised SQL query over the live tables
(counts, top/bottom N, averages, totals, group-by, filters, ID lookups) and returns the real rows. The LLM only explains afterwards;
it never has to guess a number. Table and column names come from a fixed whitelist, values are always bound parameters."""
import re
from blocks import Q

TABLES = {
    "Franchise": {
        "outlets": (["outlet", "store", "branch", "franchise"], "outlet_id", ["outlet_id", "outlet_name", "location", "tier", "revenue", "customer_satisfaction", "staff_headcount"], "revenue"),
        "staff": (["staff", "employee", "worker", "workforce", "people"], "staff_id", ["staff_id", "name", "role", "outlet_id", "salary", "overtime_hrs", "predicted_attrition_prob"], "predicted_attrition_prob"),
        "inventory": (["inventory", "sku", "item", "product", "stock level"], "record_id", ["outlet_id", "sku_name", "category", "current_stock", "reorder_threshold", "stockout_risk_prob"], "stockout_risk_prob"),
        "marketing": (["campaign", "marketing", "promotion", "advert"], "campaign_id", ["campaign_id", "campaign_name", "outlet_id", "channel", "budget", "actual_roi", "conversions"], "actual_roi"),
        "feedback": (["feedback", "review", "complaint", "comment"], "feedback_id", ["feedback_id", "outlet_id", "rating", "sentiment_score", "date", "comment"], "rating"),
        "audits": (["audit", "inspection", "compliance"], "audit_id", ["audit_id", "outlet_id", "audit_date", "score", "violations", "category", "status"], "score"),
        "sales": (["sale", "transaction", "takings"], "sale_id", ["outlet_id", "ts", "amount", "tx_count", "source"], "amount"),
        "purchase_orders": (["purchase order", "reorder order", "po"], "po_id", ["po_id", "outlet_id", "sku_name", "qty", "eta_ts", "status"], "qty"),
        "attendance": (["attendance", "absent", "absence", "present"], "att_id", ["staff_id", "outlet_id", "ts", "status"], None),
        "agent_actions": (["ticket", "action", "task", "approval", "alert"], "action_id", ["action_id", "type", "target", "severity", "status", "assignee", "detail"], None),
    },
    "Freight": {
        "shipments": (["shipment", "cargo", "container", "consignment"], "shipment_id", ["shipment_id", "origin_port", "dest_port", "carrier", "status", "eta", "weight_kg", "predicted_delay_risk"], "predicted_delay_risk"),
        "freight_quotes": (["quote", "quotation", "price", "pricing"], "quote_id", ["quote_id", "shipment_id", "customer_id", "carrier", "final_price", "margin_pct", "status"], "final_price"),
        "ports": (["port", "terminal", "harbour", "harbor"], "port_id", ["port_name", "country", "region", "congestion_index", "avg_dwell_days"], "congestion_index"),
        "carriers": (["carrier", "shipping line", "liner", "line"], "carrier_id", ["name", "rating", "on_time_pct", "avg_cost_index", "risk_level"], "on_time_pct"),
        "customers": (["customer", "client", "shipper"], "customer_id", ["customer_id", "name", "industry", "priority_tier", "credit_risk"], "credit_risk"),
        "weather_risks": (["weather", "storm", "wave", "wind", "forecast"], "port_name", ["port_name", "current_severity", "forecast", "wind_speed", "wave_height"], "wave_height"),
        "vessels": (["vessel", "ship ", "ais", "voyage"], "shipment_id", ["shipment_id", "route_name", "lat", "lon", "progress", "speed_kn"], "progress"),
        "quote_requests": (["request", "enquiry", "enquiries", "inquiry"], "req_id", ["req_id", "ts", "origin", "dest", "cargo", "weight_t", "status", "quote_id"], None),
        "customs_rules": (["customs", "duty", "hs code", "tariff"], "chapter", ["chapter", "goods", "duty_pct", "docs", "restricted"], "duty_pct"),
        "agent_actions": (["ticket", "action", "task", "approval", "alert"], "action_id", ["action_id", "type", "target", "severity", "status", "assignee", "detail"], None),
    },
}
CSYN = {"revenue": ["revenue", "turnover"], "customer_satisfaction": ["satisfaction", "csat"], "staff_headcount": ["headcount"], "salary": ["salary", "pay", "wage"], "overtime_hrs": ["overtime"],
        "job_satisfaction": ["job satisfaction"], "tenure_years": ["tenure"], "predicted_attrition_prob": ["attrition", "churn", "likely to leave", "quit"], "current_stock": ["current stock", "units in stock"],
        "reorder_threshold": ["reorder level", "threshold"], "weekly_demand": ["demand"], "lead_time_days": ["lead time"], "stockout_risk_prob": ["stockout", "stock-out", "stock out", "shortage", "run out"],
        "actual_roi": ["roi", "return on"], "sentiment_score": ["sentiment"], "score": ["audit score", "score"], "violations": ["violation"], "amount": ["amount", "sales value"], "qty": ["quantity", "qty"],
        "congestion_index": ["congestion", "congested", "crowded"], "avg_dwell_days": ["dwell"], "on_time_pct": ["on-time", "on time", "punctual", "punctuality", "reliab"], "avg_cost_index": ["cost index"],
        "predicted_delay_risk": ["delay risk", "delay", "delayed"], "weight_kg": ["weight"], "final_price": ["final price", "price", "quote value", "worth"], "margin_pct": ["margin"], "base_cost": ["base cost"],
        "insurance": ["insurance"], "customs_fee": ["customs fee"], "fuel_surcharge": ["fuel"], "credit_risk": ["credit risk", "credit"], "duty_pct": ["duty"], "wave_height": ["wave"], "wind_speed": ["wind"],
        "speed_kn": ["speed"], "progress": ["progress", "completed"], "operating_costs": ["operating cost", "costs"], "rating": ["rating", "rated", "stars"], "budget": ["budget", "spend"], "reach": ["reach"], "conversions": ["conversion"]}
BAD = {"predicted_attrition_prob", "stockout_risk_prob", "violations", "congestion_index", "avg_dwell_days", "predicted_delay_risk", "credit_risk", "operating_costs", "overtime_hrs", "lead_time_days", "avg_cost_index", "wave_height", "wind_speed"}
PROB = {"predicted_attrition_prob", "stockout_risk_prob", "predicted_delay_risk", "credit_risk", "progress", "sentiment_score"}
LABEL = {"outlets": "outlet_name", "staff": "name", "carriers": "name", "customers": "name", "ports": "port_name", "inventory": "sku_name", "marketing": "campaign_name"}
GENERIC = {"port", "line", "terminal", "cargo", "ocean", "shipping", "the", "of"}

def _w(q, term): return re.search(r"(?<![a-z0-9])" + re.escape(term.strip()) + r"(?:s|es)?(?![a-z0-9])", q) is not None
def _num(s, unit):
    v = float(s.replace(",", "")); u = (unit or "").lower()
    return v * {"k": 1e3, "m": 1e6, "lakh": 1e5, "lakhs": 1e5, "cr": 1e7, "crore": 1e7, "crores": 1e7}.get(u, 1)
_NUM = {}
def _cols(t):
    df = Q(f"SELECT * FROM {t} LIMIT 20"); _NUM[t] = {c for c in df.columns if str(df[c].dtype)[:3] in ("flo", "int") }
    return list(df.columns)
PL = {"inventory": "inventory records", "staff": "staff", "marketing": "campaigns", "feedback": "feedback entries", "audits": "audits", "sales": "sales records", "attendance": "attendance records", "freight_quotes": "quotes", "ports": "ports", "carriers": "carriers", "customs_rules": "customs rules", "weather_risks": "port weather entries", "agent_actions": "tickets", "purchase_orders": "purchase orders", "quote_requests": "quote requests", "outlets": "outlets", "shipments": "shipments", "customers": "customers", "vessels": "vessels"}
def pct(df, cols):
    df = df.copy()
    for c in cols:
        if c in df.columns and c in PROB: df[c] = df[c].map(lambda x: f"{x:.0%}" if isinstance(x, (int, float)) and abs(x) <= 1 else x)
    return df
def _alias(v):
    v = str(v); out = {v.lower()}; base = re.sub(r"\(.*?\)", "", v).strip().lower(); out.add(base)
    out.update(m.lower() for m in re.findall(r"\((.*?)\)", v))
    core = " ".join(w for w in base.split() if w not in GENERIC)
    if len(core) >= 3: out.add(core)
    return {a for a in out if len(a) >= 3}
def _fmt(v):
    if isinstance(v, float): return f"{v:,.2f}" if abs(v) < 1000 else f"{v:,.0f}"
    if isinstance(v, int): return f"{v:,}" if abs(v) >= 10000 else str(v)
    return str(v)
def md(df):
    if df is None or not len(df): return ""
    h = "| " + " | ".join(map(str, df.columns)) + " |\n|" + "---|" * len(df.columns) + "\n"
    return h + "\n".join("| " + " | ".join(_fmt(x).replace("|", "/").replace("\n", " ")[:90] for x in r) + " |" for r in df.itertuples(index=False))

def lookup(kind, q, scope=None):
    """ID in the question -> the exact row(s) plus its open tickets."""
    ids = list(dict.fromkeys(re.findall(r"\b(OUT-\d+|STF-\d+|EMP-\d+|SHP-[A-Z0-9-]+|QTE-[A-Z0-9-]+|PORT-\d+|CAR-\d+|CUST-\d+|CAMP-\d+|AUD-\d+|FB-\d+)\b", q, re.I)))
    ids = [i.upper() for i in ids]
    if not ids: return None
    parts, sqls = [], []
    for e in ids[:4]:
        if scope and e != scope and not e.startswith("PORT") and not e.startswith("CAR"): parts.append(f"{e} is outside your outlet scope ({scope})."); continue
        for t, (_, idc, cols, _m) in TABLES[kind].items():
            if t == "agent_actions": continue
            try: df = Q(f"SELECT * FROM {t} WHERE {idc}=?", (e,))
            except Exception: continue
            if len(df):
                r = df.iloc[0]; parts.append(f"**{e}** ({t}): " + "; ".join(f"{c}={_fmt(r[c])}" for c in df.columns if str(r[c]) not in ("None", "nan"))); sqls.append(f"SELECT * FROM {t} WHERE {idc}='{e}'"); break
        tk = Q("SELECT action_id, type, severity, status, detail FROM agent_actions WHERE target=? AND status IN ('open','proposed') ORDER BY action_id DESC LIMIT 5", (e,))
        if len(tk): parts.append(f"Open tickets on {e}:\n" + md(tk)); sqls.append(f"SELECT … FROM agent_actions WHERE target='{e}' AND status IN ('open','proposed')")
    return {"answer": "\n\n".join(parts), "df": None, "sql": "; ".join(sqls)} if parts else None

def answer(kind, q, scope=None, mask=False):
    """-> dict(answer, sql, df, table) when the question maps to the data, else None."""
    ql = q.lower().strip()
    if not ql: return None
    d = lookup(kind, q, scope)
    if d: return d
    T = TABLES[kind]; sc = {}
    for t, (syn, _, _, _) in T.items(): sc[t] = sum(3 for s in syn if _w(ql, s))
    cols_of = {}
    for t in T:
        try: cols_of[t] = _cols(t)
        except Exception: cols_of[t] = []
    hits = {c: any(_w(ql, s) for s in (CSYN.get(c, []) + [c.replace("_", " ")])) for c in {c for v in cols_of.values() for c in v}}
    for t, cs in cols_of.items(): sc[t] += sum(2 for c in cs if hits.get(c) and c not in ("status", "type", "date", "name"))
    if not sc or max(sc.values()) <= 0: return None
    order = list(T); t = max(order, key=lambda x: (sc[x], -order.index(x)))
    syn, idc, show, dmetric = T[t]; cs = cols_of[t]
    # aggregation
    agg = "list"
    if re.search(r"\b(how many|number of|count)\b", ql): agg = "count"
    elif re.search(r"\b(average|avg|mean)\b", ql): agg = "avg"
    elif re.search(r"\b(total|sum|overall|combined)\b", ql): agg = "sum"
    elif re.search(r"\b(highest|most|top|best|largest|biggest|maximum|max|greatest|riskiest|worst|lowest|least|smallest|minimum|min|bottom|weakest|cheapest|fewest|poorest)\b", ql): agg = "rank"
    # metric
    mc = [c for c in cs if hits.get(c) and c in _NUM.get(t, ()) and c not in ("lat", "lon")]
    metric = mc[0] if mc else dmetric
    if metric and metric not in cs: metric = None
    # filters
    where, params, desc = [], [], []
    def val_filter(tab, col, sub=False):
        try: vals = Q(f"SELECT DISTINCT {col} v FROM {tab} WHERE {col} IS NOT NULL LIMIT 600").v.tolist()
        except Exception: return
        found = []
        for v in vals:
            if isinstance(v, (int, float)): continue
            for a in _alias(v):
                if _w(ql, a): found.append((len(a), str(v), a)); break
        found.sort(reverse=True); used, taken = [], ql
        for _, v, a in found:
            if a in taken and not any(a in u or u in a for u in used):
                used.append(a)
                if sub: where.append(f"outlet_id IN (SELECT outlet_id FROM {tab} WHERE {col}=?)")
                else:
                    where.append(f"{col}=?")
                params.append(v); desc.append(f"{col} = {v}")
    for c in cs:
        if c in ("status", "category", "channel", "role", "tier", "severity", "type", "location", "origin_port", "dest_port", "carrier", "region", "country", "risk_level", "industry", "priority_tier", "source", "sku_name", "name", "port_name", "customer_id", "forecast", "route_name", "origin", "dest", "cargo"):
            if c in ("origin_port", "dest_port", "origin", "dest"): continue
            val_filter(t, c)
    if t in ("shipments", "quote_requests"):
        oc, dc = ("origin_port", "dest_port") if t == "shipments" else ("origin", "dest")
        try: pv = Q("SELECT DISTINCT port_name v FROM ports").v.tolist()
        except Exception: pv = []
        for pn in pv:
            for a_ in sorted(_alias(pn), key=len, reverse=True):
                mm = re.search(r"(?<![a-z0-9])" + re.escape(a_) + r"(?![a-z0-9])", ql)
                if not mm: continue
                pre = ql[max(0, mm.start() - 22):mm.start()].strip()
                frm = re.search(r"(from|origin|out of|leaving|departing|ex)$", pre); to = re.search(r"(to|towards|into|destination|dest|bound for|arriving at|arriving in|reaching)$", pre)
                if t == "quote_requests": pn_ = a_.split(" (")[0]; pat = pn_
                else: pat = pn
                if frm: where.append(f"{oc}=?"); params.append(pat); desc.append(f"{oc} = {pn}")
                elif to: where.append(f"{dc}=?"); params.append(pat); desc.append(f"{dc} = {pn}")
                else: where.append(f"({oc}=? OR {dc}=?)"); params += [pat, pat]; desc.append(f"{pn} is origin or destination")
                break
    if "outlet_id" in cs and t != "outlets":
        try: val_filter("outlets", "location", sub=True); val_filter("outlets", "tier", sub=True)
        except Exception: pass
    m = re.search(r"\b(outlet|store|branch)\s*(?:no\.?|number|#)?\s*(\d{1,3})\b", ql)
    if m and "outlet_id" in cs: where.append("outlet_id=?"); params.append(f"OUT-{int(m.group(2)):03d}"); desc.append(f"outlet_id = OUT-{int(m.group(2)):03d}")
    for pat, op in ((r"(?:below|under|less than|lower than|smaller than|<)", "<"), (r"(?:above|over|more than|greater than|higher than|exceed\w*|>)", ">"), (r"(?:at least|minimum of|>=)", ">="), (r"(?:at most|no more than|<=)", "<=")):
        mm = re.search(pat + r"\s*(?:rs\.?|₹|\$|inr|usd)?\s*([\d,]*\.?\d+)\s*(%|k|m|lakhs?|cr(?:ores?)?)?", ql)
        if mm and metric:
            v = _num(mm.group(1), mm.group(2) if mm.group(2) != "%" else "");
            if metric in PROB and v > 1: v /= 100
            where.append(f"{metric}{op}?"); params.append(v); desc.append(f"{metric} {op} {v:g}"); break
    if re.search(r"\b(open|pending|unresolved)\b", ql) and "status" in cs and not any("status" in x for x in desc) and t in ("agent_actions", "purchase_orders"):
        where.append("status IN ('open','proposed')" if t == "agent_actions" else "status NOT IN ('delivered','received','closed')"); desc.append("status = open")
    if re.search(r"\b(critical|high severity|urgent)\b", ql) and "severity" in cs and not any("severity" in x for x in desc): where.append("severity IN ('critical','high')"); desc.append("severity = critical/high")
    if re.search(r"\b(delayed|late)\b", ql) and t == "shipments" and not any("status" in x for x in desc): where.append("(status LIKE '%delay%' OR predicted_delay_risk>=0.5)"); desc.append("delayed or delay risk ≥ 0.5")
    if re.search(r"\b(low stock|running low|below reorder|need reorder|reorder)\b", ql) and t == "inventory": where.append("current_stock<=reorder_threshold"); desc.append("stock ≤ reorder threshold")
    if re.search(r"\b(high risk|at risk)\b", ql) and metric in PROB and not any(metric in x for x in desc): where.append(f"{metric}>=0.5"); desc.append(f"{metric} ≥ 0.5")
    if scope and t != "agent_actions":
        if "outlet_id" in cs: where.append("outlet_id=?"); params.append(scope); desc.append(f"outlet_id = {scope}")
        else: return {"answer": "That data is not available for your outlet role.", "sql": "", "df": None}
    elif scope: return {"answer": "Ticket lists are limited to your outlet; open the Actions page.", "sql": "", "df": None}
    # an unfiltered, unaggregated, unranked question is too vague to answer exactly
    if agg == "list" and not where and not re.search(r"\b(show|list|which|what|who|give)\b", ql): return None
    W = (" WHERE " + " AND ".join(where)) if where else ""
    nm = re.search(r"\b(?:top|bottom|first|best|worst|lowest|highest)\s+(\d{1,2})\b", ql); n = int(nm.group(1)) if nm else (5 if agg == "rank" else 10)
    tdesc = PL.get(t, syn[0] + "s"); fdesc = (" where " + ", ".join(desc)) if desc else ""
    grp = re.search(r"\b(?:by|per|for each|across|grouped by|each)\s+(location|city|tier|role|category|channel|carrier|region|country|status|type|severity|industry|outlet|port|origin|destination|sku|customer)\b", ql)
    gcol = {"city": "location", "outlet": "outlet_id", "port": "port_name" if "port_name" in cs else "origin_port", "origin": "origin_port", "destination": "dest_port", "sku": "sku_name", "customer": "customer_id"}.get(grp.group(1), grp.group(1)) if grp else None
    if gcol and gcol not in cs and "outlet_id" in cs and gcol in ("location", "tier"):
        j = f" JOIN outlets o ON o.outlet_id={t}.outlet_id"; gq = f"o.{gcol}"
    elif gcol in cs: j, gq = "", gcol
    else: j, gq = "", None
    if gq:
        fn = {"count": "COUNT(*)", "avg": f"AVG({metric})", "sum": f"SUM({metric})"}.get(agg, f"AVG({metric})" if metric else "COUNT(*)")
        if agg == "rank" and re.search(r"\b(lowest|least|smallest|minimum|min|bottom|worst|fewest|weakest|poorest|cheapest)\b", ql) and not re.search(r"\b(worst|riskiest)\b", ql) or (re.search(r"\b(worst)\b", ql) and metric not in BAD): order_dir = "ASC"
        else: order_dir = "DESC"
        if re.search(r"\bhow many|number of|count\b", ql): fn = "COUNT(*)"
        sql = f"SELECT {gq} AS grp, {fn} AS value, COUNT(*) AS rows_n FROM {t}{j}{W} GROUP BY {gq} ORDER BY value {order_dir} LIMIT 25"
        df = Q(sql, tuple(params))
        if not len(df): return {"answer": f"No {tdesc}{fdesc} found.", "sql": sql, "df": None}
        lab = {"COUNT(*)": "count"}.get(fn, fn.lower().replace("(", " of ").replace(")", ""))
        top = df.iloc[0]; df = df.rename(columns={"grp": gcol, "value": lab});
        return {"answer": f"{lab.capitalize()} of {tdesc} by {gcol}{fdesc}. Highest: **{top.iloc[0]}** ({_fmt(top.iloc[1])}); lowest: **{df.iloc[-1, 0]}** ({_fmt(df.iloc[-1, 1])}).", "sql": sql, "df": df}
    if agg == "count":
        sql = f"SELECT COUNT(*) AS n FROM {t}{W}"; n_ = int(Q(sql, tuple(params)).n[0])
        return {"answer": f"**{n_:,}** {tdesc}{fdesc}.", "sql": sql, "df": None}
    if agg in ("avg", "sum") and metric:
        fn = "AVG" if agg == "avg" else "SUM"; sql = f"SELECT {fn}({metric}) AS v, COUNT(*) AS n, MIN({metric}) AS lo, MAX({metric}) AS hi FROM {t}{W}"; r = Q(sql, tuple(params)).iloc[0]
        if not r.n: return {"answer": f"No {tdesc}{fdesc} found.", "sql": sql, "df": None}
        pv = f"{r.v:.1%}" if metric in PROB and abs(r.v) <= 1 else _fmt(float(r.v))
        return {"answer": f"{'Average' if agg == 'avg' else 'Total'} {metric.replace('_', ' ')} of {tdesc}{fdesc}: **{pv}** across {int(r.n):,} rows (min {_fmt(float(r.lo))}, max {_fmt(float(r.hi))}).", "sql": sql, "df": None}
    if mask and t == "staff" and (metric in ("salary", "age") or re.search(r"\b(salary|salaries|pay|wage|age)\b", ql)): return {"answer": "Salary and personal details are visible to Admin, Owner and Operations roles only.", "sql": "", "df": None}
    sel = list(dict.fromkeys([c for c in show if c in cs] + ([metric] if metric else [])))
    if mask: sel = [c for c in sel if not (t == "staff" and c in ("name", "salary", "age"))]
    if t == "inventory" and "outlet_id" in cs and "sku_name" not in sel: sel.append("sku_name")
    if agg == "rank" and metric:
        low = re.search(r"\b(lowest|least|smallest|minimum|min|bottom|fewest|weakest|poorest|cheapest)\b", ql) is not None; worst = re.search(r"\b(worst|riskiest)\b", ql) is not None
        asc = (low and not worst) or (worst and metric not in BAD) or (re.search(r"\b(best)\b", ql) is not None and metric in BAD)
        sql = f"SELECT {', '.join(sel)} FROM {t}{W} ORDER BY {metric} {'ASC' if asc else 'DESC'} LIMIT {n}"; df = Q(sql, tuple(params))
        if not len(df): return {"answer": f"No {tdesc}{fdesc} found.", "sql": sql, "df": None}
        r0 = df.iloc[0]; lab = r0[LABEL[t]] if t in LABEL and LABEL[t] in df.columns else r0[idc] if idc in df.columns else r0.iloc[0]; pv = f"{r0[metric]:.1%}" if metric in PROB and abs(r0[metric]) <= 1 else _fmt(r0[metric])
        return {"answer": f"{'Lowest' if asc else 'Highest'} {metric.replace('_', ' ')} among {tdesc}{fdesc}: **{lab}** at **{pv}**. Top {len(df)} below.", "sql": sql, "df": pct(df, sel)}
    sql = f"SELECT {', '.join(sel)} FROM {t}{W}" + (f" ORDER BY {metric} {'ASC' if metric in BAD and False else 'DESC'}" if metric else "") + f" LIMIT {max(n, 10) if not nm else n}"
    tot = int(Q(f"SELECT COUNT(*) n FROM {t}{W}", tuple(params)).n[0]); df = Q(sql, tuple(params))
    if not len(df): return {"answer": f"No {tdesc}{fdesc} found.", "sql": sql, "df": None}
    return {"answer": f"**{tot:,}** {tdesc}{fdesc}; showing {len(df)}" + (f", sorted by {metric.replace('_', ' ')}" if metric else "") + ".", "sql": sql, "df": pct(df, sel)}


In [ ]:
%%writefile /content/app/copilot_engine.py
"""AI Copilot Engine for FreightQuote AI.
Whitelisted SQL queries with strict RBAC enforcement, typo tolerance,
exact database answering (no LLM/RAG hallucinations), and formatted SQL citations.
"""
import re
import sqlite3
import time
from typing import Dict, Any, List, Optional, Tuple
from db import get_conn

# Role-specific question suggestions displayed when asking outside scope or greeting
ROLE_SUGGESTIONS = {
    "Customer": [
        "What are my active shipments?",
        "Show my recent quotes",
        "Where is my cargo currently located?",
    ],
    "Admin": [
        "Which port has the highest congestion?",
        "How many shipments are on customs hold?",
        "Show quotes below 10% margin",
        "Which carrier is cheapest to Rotterdam?",
        "Show storms near the Red Sea",
    ],
    "Operations Manager": [
        "Which ports are most congested?",
        "How many shipments are on customs hold?",
        "Which shipments have a high delay risk?",
        "Show storms near the Red Sea",
        "Show active vessels progress at sea",
    ],
    "Freight Broker": [
        "Which carrier is cheapest to Rotterdam?",
        "Show quotes below 10% margin",
        "Show quotes by carrier",
        "Who are our top spending customers?",
        "Show carrier reliability rankings",
    ],
    "Auditor": [
        "How many shipments are on customs hold?",
        "Show customs rules with high duty rates",
        "Show quotes below 10% margin",
        "Which carriers have the lowest on-time percentage?",
        "What are the longest port dwell days?",
    ],
}

def get_role_suggestions(role: str) -> List[str]:
    return ROLE_SUGGESTIONS.get(role, ROLE_SUGGESTIONS["Customer"])

# -----------------------------------------------------------------------------
# Normalization & Synonym Mapping
# -----------------------------------------------------------------------------
SYNONYMS = [
    (r"\bcheepest\b", "cheapest"),
    (r"\bcheap\b", "cheapest"),
    (r"\blowest price\b", "cheapest"),
    (r"\blowest rate\b", "cheapest"),
    (r"\bcarier\b", "carrier"),
    (r"\bcarriar\b", "carrier"),
    (r"\bshipping line\b", "carrier"),
    (r"\brotterdan\b", "rotterdam"),
    (r"\broterdam\b", "rotterdam"),
    (r"\brotredam\b", "rotterdam"),
    (r"\bshanghi\b", "shanghai"),
    (r"\bshang hai\b", "shanghai"),
    (r"\bsingapor\b", "singapore"),
    (r"\bnhava\b", "nhava sheva"),
    (r"\bjnpt\b", "nhava sheva"),
    (r"\bdelyed\b", "delay"),
    (r"\bdelaied\b", "delay"),
    (r"\bdelays\b", "delayed"),
    (r"\bshipmnt\b", "shipment"),
    (r"\bshipmnts\b", "shipment"),
    (r"\bshipments\b", "shipment"),
    (r"\bcargos\b", "shipment"),
    (r"\bcargo\b", "shipment"),
    (r"\bmargn\b", "margin"),
    (r"\bmargins\b", "margin"),
    (r"\bcustom\b", "customs"),
    (r"\bcustomes\b", "customs"),
    (r"\bcostom\b", "customs"),
    (r"\brelible\b", "reliable"),
    (r"\breliablity\b", "reliability"),
    (r"\bwether\b", "weather"),
    (r"\bwheather\b", "weather"),
    (r"\bstrm\b", "storm"),
    (r"\bstorms\b", "storm"),
    (r"\bcongested\b", "congestion"),
    (r"\bconjeston\b", "congestion"),
    (r"\bvessles\b", "vessel"),
    (r"\bvessels\b", "vessel"),
    (r"\bquoat\b", "quote"),
    (r"\bqoute\b", "quote"),
    (r"\bquotes\b", "quote"),
    (r"\bquotations\b", "quote"),
    (r"\bclients\b", "customer"),
    (r"\bcustomers\b", "customer"),
    (r"\bshippers\b", "customer"),
    (r"\btariff\b", "duty"),
    (r"\bdwell time\b", "dwell days"),
]

def normalize_text(text: str) -> str:
    t = text.lower().strip()
    t = t.replace("%", " percent ")
    t = re.sub(r"[^\w\s]", " ", t)
    for pat, rep in SYNONYMS:
        t = re.sub(pat, rep, t)
    t = re.sub(r"\s+", " ", t).strip()
    return t

# -----------------------------------------------------------------------------
# Markdown Table Formatter
# -----------------------------------------------------------------------------
def format_markdown_table(headers: List[str], rows: List[Tuple]) -> str:
    if not rows:
        return "_No records found matching criteria._"
    header_str = "| " + " | ".join(str(h) for h in headers) + " |"
    sep_str = "| " + " | ".join("---" for _ in headers) + " |"
    row_strs = []
    for row in rows:
        formatted_cells = []
        for cell in row:
            if cell is None:
                formatted_cells.append("-")
            elif isinstance(cell, float):
                formatted_cells.append(f"{cell:,.2f}" if abs(cell) >= 1 else f"{cell:.1%}")
            else:
                formatted_cells.append(str(cell))
        row_strs.append("| " + " | ".join(formatted_cells) + " |")
    return "\n".join([header_str, sep_str] + row_strs)

# -----------------------------------------------------------------------------
# Parameter Extractors
# -----------------------------------------------------------------------------
def extract_dest_port(norm_q: str) -> str:
    ports = ["rotterdam", "shanghai", "singapore", "tokyo", "hamburg", "dubai", "nhava sheva", "kandla", "chennai", "los angeles", "new york", "antwerp", "busan", "santos", "mundra", "colombo"]
    for p in ports:
        if p in norm_q:
            return f"%{p}%"
    m = re.search(r"to\s+([a-z\s]+)", norm_q)
    if m:
        target = m.group(1).replace("port", "").strip()
        if target:
            return f"%{target}%"
    return "%rotterdam%"

def extract_margin_threshold(norm_q: str) -> float:
    m = re.search(r"(?:below|less than|under)\s+(\d+(?:\.\d+)?)\s*(?:percent|%)?", norm_q)
    if m:
        try:
            return float(m.group(1))
        except ValueError:
            pass
    return 10.0

def extract_region(norm_q: str) -> str:
    regions = ["red sea", "arabian sea", "bay of bengal", "south china sea", "mediterranean", "gulf of aden", "baltic sea", "atlantic", "pacific"]
    for r in regions:
        if r in norm_q:
            return f"%{r}%"
    m = re.search(r"(?:near|in|around)\s+([a-z\s]+)", norm_q)
    if m:
        reg = m.group(1).strip()
        if reg:
            return f"%{reg}%"
    return "%red sea%"

# -----------------------------------------------------------------------------
# Whitelisted Query Definitions (18 Core Patterns + RBAC)
# -----------------------------------------------------------------------------
QUERIES = [
    # 1. Cheapest carrier to destination
    {
        "id": "cheapest_carrier_dest",
        "pattern": re.compile(r"cheapest\s+carrier|carrier\s+is\s+cheapest|lowest\s+(?:price|rate|cost)\s+to|best\s+price\s+to", re.I),
        "allowed_roles": ["Admin", "Operations Manager", "Freight Broker", "Auditor"],
        "sql": """SELECT COALESCE(q.carrier, s.carrier) AS carrier_name, ROUND(AVG(q.final_price), 2) AS avg_price, COUNT(*) AS quote_count
FROM freight_quotes q
LEFT JOIN shipments s ON q.shipment_id = s.shipment_id
WHERE LOWER(s.dest_port) LIKE ? OR LOWER(q.quote_id) LIKE ?
GROUP BY carrier_name
ORDER BY avg_price ASC
LIMIT 5;""",
        "params": lambda q, r, cid: (extract_dest_port(q), extract_dest_port(q)),
        "headers": ["Carrier", "Avg Price ($)", "Quote Count"],
        "lead": lambda count, p: f"Here are the carriers ranked by lowest average price for destination matching '{p[0].strip('%')}':",
    },

    # 2. Shipments on Customs Hold
    {
        "id": "customs_hold",
        "pattern": re.compile(r"customs\s+hold|held\s+(?:at|by|on)\s+customs|customs\s+clearance|shipments?\s+(?:are\s+)?on\s+customs", re.I),
        "allowed_roles": ["Admin", "Operations Manager", "Auditor"],
        "sql": """SELECT shipment_id, origin_port, dest_port, carrier, status, eta
FROM shipments
WHERE LOWER(status) LIKE '%hold%' OR LOWER(status) LIKE '%customs%'
ORDER BY shipment_id ASC
LIMIT 10;""",
        "params": lambda q, r, cid: (),
        "headers": ["Shipment ID", "Origin Port", "Destination Port", "Carrier", "Status", "ETA"],
        "lead": lambda count, p: f"There are currently {count} shipment(s) held at customs requiring documentation:",
    },

    # 3. Quotes below X% margin
    {
        "id": "low_margin_quotes",
        "pattern": re.compile(r"(?:below|under|less than)\s+\d+\s*(?:percent|%)?\s*margin|margin\s*(?:below|under|less than)|low\s+margin|unprofitable\s+quote|lowest\s+margin", re.I),
        "allowed_roles": ["Admin", "Operations Manager", "Freight Broker", "Auditor"],
        "sql": """SELECT q.quote_id, q.customer_id, q.carrier, q.final_price, q.margin_pct, q.status
FROM freight_quotes q
WHERE q.margin_pct < ?
ORDER BY q.margin_pct ASC
LIMIT 10;""",
        "params": lambda q, r, cid: (extract_margin_threshold(q),),
        "headers": ["Quote ID", "Customer ID", "Carrier", "Price ($)", "Margin (%)", "Status"],
        "lead": lambda count, p: f"Found {count} quote(s) below {p[0]:.0f}% margin threshold:",
    },

    # 4. Port congestion
    {
        "id": "port_congestion",
        "pattern": re.compile(r"port\s+congestion|congested\s+ports?|highest\s+congestion|most\s+(?:congested|congestion)", re.I),
        "allowed_roles": ["Admin", "Operations Manager", "Freight Broker", "Auditor"],
        "sql": """SELECT port_id, port_name, country, congestion_index, avg_dwell_days
FROM ports
ORDER BY congestion_index DESC
LIMIT 5;""",
        "params": lambda q, r, cid: (),
        "headers": ["Port ID", "Port Name", "Country", "Congestion Index", "Avg Dwell (Days)"],
        "lead": lambda count, p: f"Top {count} ports ranked by congestion index (scale 1.0 – 5.0):",
    },

    # 5. Storms near location / Red Sea
    {
        "id": "storms_and_cyclones",
        "pattern": re.compile(r"storms?\s+(?:near|in|around)?\s*([a-z\s]+)?|show\s+storms?|active\s+storms?|cyclones?|typhoon", re.I),
        "allowed_roles": ["Admin", "Operations Manager", "Freight Broker", "Auditor"],
        "sql": """SELECT port_name, current_severity, forecast, wind_speed, wave_height
FROM weather_risks
WHERE LOWER(port_name) LIKE ? OR LOWER(forecast) LIKE ? OR wind_speed > 35
ORDER BY wind_speed DESC
LIMIT 6;""",
        "params": lambda q, r, cid: (extract_region(q), "%storm%"),
        "headers": ["Port / Marine Zone", "Severity", "Conditions", "Wind Speed (km/h)", "Wave Height (m)"],
        "lead": lambda count, p: f"Marine storm and severe weather advisory records for '{p[0].strip('%')}':",
    },

    # 6. High delay risk shipments
    {
        "id": "high_delay_risk_shipments",
        "pattern": re.compile(r"high\s+delay\s+risk|delay\s+risk|predicted\s+delay", re.I),
        "allowed_roles": ["Admin", "Operations Manager", "Freight Broker", "Auditor"],
        "sql": """SELECT shipment_id, origin_port, dest_port, carrier, status, predicted_delay_risk
FROM shipments
WHERE predicted_delay_risk IN ('High', 'Critical') OR predicted_delay_risk >= '0.6'
ORDER BY shipment_id ASC
LIMIT 10;""",
        "params": lambda q, r, cid: (),
        "headers": ["Shipment ID", "Origin", "Destination", "Carrier", "Status", "Delay Risk"],
        "lead": lambda count, p: f"Identified {count} shipment(s) with elevated delay risk prediction:",
    },

    # 7. Delayed shipments count
    {
        "id": "delayed_shipments_count",
        "pattern": re.compile(r"how\s+many\s+shipments?\s+(?:are\s+)?delayed|delayed\s+shipments?\s+count|delayed\s+shipments?", re.I),
        "allowed_roles": ["Admin", "Operations Manager", "Freight Broker", "Auditor"],
        "sql": """SELECT status, COUNT(*) AS shipment_count
FROM shipments
WHERE LOWER(status) LIKE '%delay%' OR LOWER(status) LIKE '%hold%'
GROUP BY status;""",
        "params": lambda q, r, cid: (),
        "headers": ["Status Classification", "Shipment Count"],
        "lead": lambda count, p: f"Current shipment delay summary across the network:",
    },

    # 8. Carrier reliability & on-time performance
    {
        "id": "carrier_reliability",
        "pattern": re.compile(r"carrier\s+reliability|on\s+time\s+pct|on\s+time\s+percentage|carrier\s+performance|most\s+reliable\s+carrier|lowest\s+on\s+time", re.I),
        "allowed_roles": ["Admin", "Operations Manager", "Freight Broker", "Auditor"],
        "sql": """SELECT name, rating, on_time_pct, avg_cost_index, risk_level
FROM carriers
ORDER BY on_time_pct DESC
LIMIT 8;""",
        "params": lambda q, r, cid: (),
        "headers": ["Carrier Name", "Rating", "On-Time %", "Cost Index", "Risk Level"],
        "lead": lambda count, p: f"Carrier performance ranked by on-time delivery percentage:",
    },

    # 9. Customer active shipments (Customer-scoped)
    {
        "id": "customer_active_shipments",
        "pattern": re.compile(r"my\s+active\s+shipments?|my\s+shipments?|active\s+shipments?|where\s+is\s+my\s+cargo", re.I),
        "allowed_roles": ["Customer", "Admin", "Freight Broker"],
        "sql": """SELECT q.shipment_id, s.origin_port, s.dest_port, s.carrier, s.status, s.eta
FROM freight_quotes q
JOIN shipments s ON q.shipment_id = s.shipment_id
WHERE q.customer_id = ?
ORDER BY s.eta ASC
LIMIT 10;""",
        "params": lambda q, r, cid: (cid or "CUST-001",),
        "headers": ["Shipment ID", "Origin Port", "Destination Port", "Carrier", "Status", "ETA"],
        "lead": lambda count, p: f"Active shipment tracking for your account ({p[0]}):",
    },

    # 10. Customer recent quotes (Customer-scoped)
    {
        "id": "customer_quotes",
        "pattern": re.compile(r"my\s+recent\s+quotes?|my\s+quotes?|recent\s+quotes?|show\s+my\s+quotes?", re.I),
        "allowed_roles": ["Customer", "Admin", "Freight Broker"],
        "sql": """SELECT quote_id, carrier, final_price, margin_pct, status, created_at
FROM freight_quotes
WHERE customer_id = ?
ORDER BY created_at DESC
LIMIT 10;""",
        "params": lambda q, r, cid: (cid or "CUST-001",),
        "headers": ["Quote ID", "Carrier", "Price ($)", "Margin (%)", "Status", "Created At"],
        "lead": lambda count, p: f"Recent spot freight quotes for your account ({p[0]}):",
    },

    # 11. Top spending customers
    {
        "id": "top_spending_customers",
        "pattern": re.compile(r"top\s+(?:spending\s+)?customers?|highest\s+spending\s+customers?|top\s+clients?|biggest\s+customers?", re.I),
        "allowed_roles": ["Admin", "Freight Broker", "Auditor"],
        "sql": """SELECT c.name, c.industry, c.priority_tier, COUNT(q.quote_id) AS total_quotes, ROUND(SUM(q.final_price), 2) AS total_spend
FROM customers c
JOIN freight_quotes q ON c.customer_id = q.customer_id
GROUP BY c.customer_id
ORDER BY total_spend DESC
LIMIT 5;""",
        "params": lambda q, r, cid: (),
        "headers": ["Customer Name", "Industry", "Priority Tier", "Total Quotes", "Total Spend ($)"],
        "lead": lambda count, p: f"Top customer accounts by aggregate quote volume:",
    },

    # 12. Customs rules & duty rates
    {
        "id": "high_duty_customs",
        "pattern": re.compile(r"customs\s+rules|high\s+duty|highest\s+duty|tariff\s+rates?|duty\s+rates?", re.I),
        "allowed_roles": ["Admin", "Operations Manager", "Auditor", "Freight Broker"],
        "sql": """SELECT chapter, goods, duty_pct, docs
FROM customs_rules
ORDER BY duty_pct DESC
LIMIT 6;""",
        "params": lambda q, r, cid: (),
        "headers": ["HS Chapter", "Commodity Goods", "Standard Duty (%)", "Required Documents"],
        "lead": lambda count, p: f"Customs tariff schedule ranked by standard duty percentage:",
    },

    # 13. Active vessels progress at sea
    {
        "id": "active_vessels_progress",
        "pattern": re.compile(r"vessels?\s+progress|fleet\s+progress|vessels?\s+at\s+sea|active\s+vessels?", re.I),
        "allowed_roles": ["Admin", "Operations Manager", "Auditor"],
        "sql": """SELECT shipment_id, route_name, ROUND(progress * 100, 1) AS progress_pct, lat, lon
FROM vessels
ORDER BY progress DESC
LIMIT 8;""",
        "params": lambda q, r, cid: (),
        "headers": ["Shipment ID", "Voyage Route", "Progress (%)", "Latitude", "Longitude"],
        "lead": lambda count, p: f"Current positions and voyage completion for active vessels:",
    },

    # 14. Quotes by carrier
    {
        "id": "quotes_by_carrier",
        "pattern": re.compile(r"quotes?\s+by\s+carrier|quotes?\s+for\s+carrier|carrier\s+quotes?", re.I),
        "allowed_roles": ["Admin", "Freight Broker", "Operations Manager", "Auditor"],
        "sql": """SELECT carrier, COUNT(*) AS quote_count, ROUND(AVG(final_price), 2) AS avg_price, ROUND(AVG(margin_pct), 1) AS avg_margin
FROM freight_quotes
GROUP BY carrier
ORDER BY quote_count DESC;""",
        "params": lambda q, r, cid: (),
        "headers": ["Carrier", "Quote Count", "Avg Price ($)", "Avg Margin (%)"],
        "lead": lambda count, p: f"Spot quote breakdown and pricing performance by carrier:",
    },

    # 15. Shipments by origin port
    {
        "id": "shipments_by_origin",
        "pattern": re.compile(r"shipments?\s+(?:from|origin|by\s+origin)|origin\s+port\s+shipments?|cargo\s+from", re.I),
        "allowed_roles": ["Admin", "Operations Manager", "Auditor"],
        "sql": """SELECT origin_port, COUNT(*) AS total_shipments, carrier, status
FROM shipments
GROUP BY origin_port
ORDER BY total_shipments DESC
LIMIT 6;""",
        "params": lambda q, r, cid: (),
        "headers": ["Origin Port", "Total Shipments", "Primary Carrier", "Status"],
        "lead": lambda count, p: f"Active cargo outbound volume by origin port:",
    },

    # 16. Port dwell days
    {
        "id": "port_dwell_days",
        "pattern": re.compile(r"dwell\s+days?|dwell\s+time|longest\s+dwell|port\s+dwell", re.I),
        "allowed_roles": ["Admin", "Operations Manager", "Auditor", "Freight Broker"],
        "sql": """SELECT port_name, country, avg_dwell_days, congestion_index
FROM ports
ORDER BY avg_dwell_days DESC
LIMIT 6;""",
        "params": lambda q, r, cid: (),
        "headers": ["Port Name", "Country", "Avg Dwell (Days)", "Congestion Index"],
        "lead": lambda count, p: f"Ports ranked by average container dwell turnaround time:",
    },

    # 17. Severe weather risks & winds
    {
        "id": "severe_weather_risks",
        "pattern": re.compile(r"weather\s+risks?|severe\s+weather|high\s+wind|wave\s+heights?|port\s+weather", re.I),
        "allowed_roles": ["Admin", "Operations Manager", "Auditor"],
        "sql": """SELECT port_name, current_severity, forecast, wind_speed, wave_height
FROM weather_risks
ORDER BY wind_speed DESC
LIMIT 6;""",
        "params": lambda q, r, cid: (),
        "headers": ["Port Name", "Severity", "Forecast Conditions", "Wind Speed (km/h)", "Wave Height (m)"],
        "lead": lambda count, p: f"Ports currently reporting highest wind speed and adverse weather:",
    },

    # 18. Fleet health & decision status
    {
        "id": "fleet_health_decision",
        "pattern": re.compile(r"fleet\s+health|health\s+score|system\s+health|decision\s+status|current\s+health", re.I),
        "allowed_roles": ["Admin", "Operations Manager", "Auditor", "Freight Broker"],
        "sql": """SELECT decision_id, score, risk, narrative, ts
FROM decisions
ORDER BY decision_id DESC
LIMIT 1;""",
        "params": lambda q, r, cid: (),
        "headers": ["Decision ID", "Health Score", "Risk Tier", "Operational Narrative", "Timestamp"],
        "lead": lambda count, p: f"Latest autonomous fleet health evaluation:",
    },
]

# -----------------------------------------------------------------------------
# Main Copilot Query Dispatcher
# -----------------------------------------------------------------------------
def ask_copilot(raw_query: str, user_role: str, customer_id: Optional[str] = None) -> Dict[str, Any]:
    t0 = time.time()
    q = (raw_query or "").strip()
    norm_q = normalize_text(q)

    # 1. Greetings check
    if re.search(r"^(?:hi|hello|hey|greetings|good\s+morning|good\s+afternoon|good\s+evening)\b", norm_q):
        suggs = get_role_suggestions(user_role)
        bullets = "\n".join(f"- *\"{s}\"*" for s in suggs)
        return {
            "status": "greeting",
            "text": f"Hello! I am your FreightQuote Copilot. I answer questions directly from our database.\n\nHere are some questions you can ask:\n{bullets}",
            "sql": "",
            "row_count": 0,
            "ms": round((time.time() - t0) * 1000),
        }

    # 2. Match whitelisted query pattern
    matched_entry = None
    for entry in QUERIES:
        if entry["pattern"].search(norm_q):
            matched_entry = entry
            break

    # 3. Handle unmapped / out-of-scope query
    if not matched_entry:
        suggs = get_role_suggestions(user_role)
        bullets = "\n".join(f"- *\"{s}\"*" for s in suggs)
        return {
            "status": "out_of_scope",
            "text": f"I can only answer questions based on verified database records (routes, quotes, carriers, weather, margins, customs).\n\nHere are three valid questions for your role:\n{bullets}",
            "sql": "",
            "row_count": 0,
            "ms": round((time.time() - t0) * 1000),
        }

    # 4. Strict RBAC validation
    if user_role not in matched_entry["allowed_roles"]:
        return {
            "status": "forbidden",
            "text": f"Access Restricted: Your role ({user_role}) does not have permission to query '{matched_entry['id'].replace('_', ' ')}'. Please refer to authorized modules in your navigation.",
            "sql": "",
            "row_count": 0,
            "ms": round((time.time() - t0) * 1000),
        }

    # 5. Customer scoping
    if user_role == "Customer":
        if matched_entry["id"] not in ("customer_active_shipments", "customer_quotes"):
            return {
                "status": "forbidden",
                "text": "Customer accounts may only query active shipments and spot quotes belonging to their registered customer ID.",
                "sql": "",
                "row_count": 0,
                "ms": round((time.time() - t0) * 1000),
            }

    # 6. Execute parameterized query against SQLite
    sql = matched_entry["sql"]
    params = matched_entry["params"](norm_q, user_role, customer_id)

    try:
        conn = get_conn()
        cur = conn.cursor()
        cur.execute(sql, params)
        rows = cur.fetchall()
        row_count = len(rows)

        headers = matched_entry["headers"]
        table_md = format_markdown_table(headers, rows)
        lead_text = matched_entry["lead"](row_count, params)

        # Build clean SQL citation text with substituted literal parameters for inspection
        display_sql = sql
        for p in params:
            val_repr = f"'{p}'" if isinstance(p, str) else str(p)
            display_sql = display_sql.replace("?", val_repr, 1)

        result_text = f"{lead_text}\n\n{table_md}"
        return {
            "status": "success",
            "text": result_text,
            "sql": display_sql.strip(),
            "row_count": row_count,
            "ms": round((time.time() - t0) * 1000),
        }
    except Exception as e:
        return {
            "status": "error",
            "text": f"Database query failed: {str(e)}",
            "sql": sql.strip(),
            "row_count": 0,
            "ms": round((time.time() - t0) * 1000),
        }


In [ ]:
%%writefile /content/app/scale_data.py
"""Scale both apps to production-like volume. Idempotent: only tops tables up to the target counts.
Coordinates are approximate (1-2 decimals): replace/extend through the Data Feeder with UN/LOCODE or World Port Index data."""
import random, datetime, math
from live_engine import conn, _DB_LOCK, now

PORTS = {
"China": "Dalian 38.9 121.7;Xiamen 24.5 118.1;Fuzhou 26.0 119.4;Lianyungang 34.7 119.4;Yantai 37.6 121.4;Rizhao 35.4 119.5;Nanjing 32.1 118.8;Zhanjiang 21.2 110.4;Haikou 20.0 110.3;Zhuhai 22.2 113.6;Taicang 31.5 121.2;Wuhan 30.6 114.3;Shantou 23.4 116.7;Dongguan 22.9 113.6",
"Japan": "Yokohama 35.4 139.7;Nagoya 35.1 136.9;Osaka 34.6 135.4;Kobe 34.7 135.2;Hakata 33.6 130.4;Chiba 35.6 140.1",
"South Korea": "Incheon 37.4 126.6;Gwangyang 34.9 127.7;Ulsan 35.5 129.4;Pyeongtaek 36.9 126.8",
"Taiwan": "Keelung 25.1 121.7;Taichung 24.3 120.5", "Vietnam": "Hai Phong 20.8 106.7;Da Nang 16.1 108.2;Vung Tau 10.4 107.1;Quy Nhon 13.8 109.3",
"Thailand": "Bangkok 13.7 100.6;Songkhla 7.2 100.6", "Malaysia": "Penang 5.4 100.3;Johor 1.4 103.9;Kuantan 3.97 103.4;Kuching 1.6 110.3",
"Indonesia": "Surabaya -7.2 112.7;Semarang -6.9 110.4;Belawan 3.8 98.7;Makassar -5.1 119.4;Balikpapan -1.3 116.8;Palembang -3.0 104.8;Bitung 1.4 125.2",
"Philippines": "Cebu 10.3 123.9;Davao 7.1 125.6;Subic Bay 14.8 120.3", "Myanmar": "Yangon 16.8 96.2", "Cambodia": "Sihanoukville 10.6 103.5",
"Bangladesh": "Chittagong 22.3 91.8;Mongla 22.5 89.6", "Sri Lanka": "Hambantota 6.1 81.1;Galle 6.0 80.2", "Pakistan": "Port Qasim 24.8 67.3;Gwadar 25.1 62.3", "Iran": "Chabahar 25.3 60.6",
"India": "Mormugao 15.4 73.8;Pipavav 20.9 71.5;Hazira 21.1 72.6;Ennore 13.2 80.3;Krishnapatnam 14.3 80.1;Kakinada 16.9 82.3;Gangavaram 17.6 83.2;Dahej 21.7 72.5;Mumbai Port 18.9 72.8;Navlakhi 22.9 70.5;Karaikal 10.9 79.9;Kamarajar 13.3 80.3;Vizhinjam 8.4 77.0;Okha 22.5 69.1;Porbandar 21.6 69.6",
"UAE": "Khalifa Port 24.8 54.6;Port Rashid 25.3 55.3;Fujairah 25.1 56.4;Khor Fakkan 25.3 56.4", "Oman": "Sohar 24.5 56.6;Duqm 19.7 57.7",
"Saudi Arabia": "Dammam 26.5 50.2;King Abdullah Port 22.5 39.1;Yanbu 24.1 38.1", "Qatar": "Hamad Port 25.0 51.6", "Kuwait": "Shuwaikh 29.4 47.9", "Bahrain": "Khalifa Bin Salman 26.2 50.6",
"Iraq": "Umm Qasr 30.0 47.9", "Jordan": "Aqaba 29.5 35.0", "Israel": "Haifa 32.8 35.0;Ashdod 31.8 34.6", "Turkey": "Mersin 36.8 34.6;Ambarli 41.0 28.7;Izmir 38.4 27.1",
"Yemen": "Aden 12.8 45.0", "Djibouti": "Djibouti 11.6 43.1", "Sudan": "Port Sudan 19.6 37.2",
"Germany": "Bremerhaven 53.6 8.5", "Belgium": "Zeebrugge 51.3 3.2", "Netherlands": "Amsterdam 52.4 4.9", "United Kingdom": "Southampton 50.9 -1.4;London Gateway 51.5 0.5;Liverpool 53.4 -3.0",
"Ireland": "Dublin 53.3 -6.2", "France": "Marseille Fos 43.4 4.9", "Spain": "Barcelona 41.4 2.2;Bilbao 43.3 -3.0", "Portugal": "Lisbon 38.7 -9.1;Sines 37.9 -8.9",
"Italy": "Naples 40.8 14.3;La Spezia 44.1 9.8;Livorno 43.6 10.3;Trieste 45.6 13.8;Gioia Tauro 38.4 15.9;Taranto 40.5 17.2", "Slovenia": "Koper 45.5 13.7", "Croatia": "Rijeka 45.3 14.4",
"Greece": "Thessaloniki 40.6 22.9", "Romania": "Constanta 44.2 28.7", "Bulgaria": "Varna 43.2 27.9", "Ukraine": "Odesa 46.5 30.7", "Sweden": "Gothenburg 57.7 11.9", "Denmark": "Aarhus 56.2 10.2",
"Norway": "Oslo 59.9 10.7", "Finland": "Helsinki 60.2 25.0", "Poland": "Gdynia 54.5 18.5", "Lithuania": "Klaipeda 55.7 21.1", "Latvia": "Riga 57.0 24.1", "Estonia": "Tallinn 59.4 24.8",
"Russia": "St Petersburg 59.9 30.2;Novorossiysk 44.7 37.8", "Malta": "Marsaxlokk 35.8 14.5", "Cyprus": "Limassol 34.7 33.0",
"Egypt": "Alexandria 31.2 29.9;Port Said 31.3 32.3;Damietta 31.5 31.8;Ain Sokhna 29.6 32.4", "Libya": "Tripoli 32.9 13.2", "Tunisia": "Rades 36.8 10.3", "Algeria": "Algiers 36.8 3.1",
"Morocco": "Casablanca 33.6 -7.6", "Senegal": "Dakar 14.7 -17.4", "Ivory Coast": "Abidjan 5.3 -4.0", "Ghana": "Tema 5.6 0.0", "Togo": "Lome 6.1 1.3", "Benin": "Cotonou 6.4 2.4",
"Nigeria": "Onne 4.7 7.2", "Cameroon": "Douala 4.0 9.7", "Angola": "Luanda -8.8 13.2", "Namibia": "Walvis Bay -22.9 14.5", "South Africa": "Cape Town -33.9 18.4;Ngqura -33.8 25.7",
"Mozambique": "Maputo -25.97 32.6;Beira -19.8 34.9", "Tanzania": "Dar es Salaam -6.8 39.3", "Kenya": "Mombasa -4.1 39.7", "Somaliland": "Berbera 10.4 45.0", "Mauritius": "Port Louis -20.2 57.5", "Madagascar": "Toamasina -18.1 49.4",
"USA": "Oakland 37.8 -122.3;Seattle 47.6 -122.3;Tacoma 47.3 -122.4;Houston 29.7 -95.3;New Orleans 29.9 -90.1;Miami 25.8 -80.2;Charleston 32.8 -79.9;Norfolk 36.9 -76.3;Baltimore 39.3 -76.6;Boston 42.4 -71.0",
"Canada": "Vancouver 49.3 -123.1;Prince Rupert 54.3 -130.3;Halifax 44.6 -63.6;Montreal 45.5 -73.5", "Mexico": "Manzanillo 19.1 -104.3;Veracruz 19.2 -96.1;Lazaro Cardenas 17.9 -102.2",
"Panama": "Balboa 8.95 -79.6;Colon 9.35 -79.9", "Colombia": "Cartagena 10.4 -75.5;Buenaventura 3.9 -77.1", "Peru": "Callao -12.05 -77.15", "Ecuador": "Guayaquil -2.3 -79.9",
"Chile": "San Antonio -33.6 -71.6;Valparaiso -33.0 -71.6", "Argentina": "Buenos Aires -34.6 -58.4", "Uruguay": "Montevideo -34.9 -56.2",
"Brazil": "Santos -23.98 -46.3;Paranagua -25.5 -48.5;Rio Grande -32.0 -52.1;Itajai -26.9 -48.7;Suape -8.4 -35.0;Pecem -3.5 -38.8", "Jamaica": "Kingston 17.97 -76.8", "Dominican Republic": "Caucedo 18.4 -69.6", "Bahamas": "Freeport 26.5 -78.8",
"Australia": "Sydney -33.9 151.2;Melbourne -37.8 144.9;Brisbane -27.4 153.2;Fremantle -32.05 115.7;Adelaide -34.8 138.5;Port Hedland -20.3 118.6", "New Zealand": "Auckland -36.8 174.8;Tauranga -37.65 176.2;Lyttelton -43.6 172.7",
"Fiji": "Suva -18.1 178.4", "New Caledonia": "Noumea -22.3 166.4", "Papua New Guinea": "Port Moresby -9.5 147.1"}
CARRIERS = ["Evergreen Marine","HMM","Yang Ming","ZIM Integrated","Pacific International Lines","Wan Hai Lines","OOCL","APL","Hamburg Sud","Matson","SM Line","Sinokor","KMTC","Namsung Shipping","SITC",
"X-Press Feeders","Gold Star Line","Arkas Line","Turkon Line","Grimaldi Lines","Messina Line","CCNI","Seaboard Marine","Crowley Maritime","Tropical Shipping","Swire Shipping","Hoegh Autoliners",
"Wallenius Wilhelmsen","NYK Line","Mitsui OSK Lines","K Line","Safmarine","Emirates Shipping Line","Shipping Corporation of India","Great Eastern Shipping","Samudera Shipping","Salam Pacific",
"Interasia Lines","Heung-A Line","Unifeeder","Containerships","Finnlines","DFDS","Stena Line","Atlantic Container Line","King Ocean","Bengal Tiger Line","Transworld Shipping","Sealand Maersk Asia","Pan Ocean","Cosco Hong Kong"]
def region_of(lat, lon):
    if -170 < lon < -30: return "Americas"
    if -30 <= lon < 50 and lat > 12: return "Europe/Mediterranean" if lat > 30 else "Africa"
    if -30 <= lon < 52 and lat <= 12: return "Africa"
    if 112 < lon and lat < -10: return "Oceania"
    if 25 < lon < 60 and 12 < lat < 33: return "Middle East"
    return "Asia"

def scale_freight(target_ports=220, target_carriers=110, target_shipments=400):
    R = random.Random(42)
    with _DB_LOCK, conn() as c:
        have = {r[0].lower() for r in c.execute("SELECT port_name FROM ports")}; n = c.execute("SELECT COUNT(*) FROM ports").fetchone()[0]
        for ctry, spec in PORTS.items():
            for e in spec.split(";"):
                if n >= target_ports + 40: break
                nm, la, lo = e.rsplit(" ", 2); name = f"{nm} Port"
                if name.lower() in have: continue
                n += 1; have.add(name.lower())
                c.execute("INSERT OR REPLACE INTO ports(port_id,port_name,country,congestion_index,avg_dwell_days,lat,lon,region) VALUES(?,?,?,?,?,?,?,?)",
                          (f"PORT-{n:03d}", name, ctry, round(R.uniform(1.0, 4.6), 1), round(R.uniform(1.5, 7.0), 1), float(la), float(lo), region_of(float(la), float(lo))))
        k = c.execute("SELECT COUNT(*) FROM carriers").fetchone()[0]; names = {r[0] for r in c.execute("SELECT name FROM carriers")}
        pool = [x for x in CARRIERS if x not in names] + [f"Regional Feeder Line {i:02d}" for i in range(1, 80)]
        for nm in pool:
            if k >= target_carriers: break
            k += 1
            c.execute("INSERT OR REPLACE INTO carriers(carrier_id,name,rating,on_time_pct,avg_cost_index,risk_level) VALUES(?,?,?,?,?,?)",
                      (f"CAR-{k:03d}", nm, round(R.uniform(3.2, 4.9), 2), round(R.uniform(68, 97), 1), round(R.uniform(.8, 1.25), 2), R.choice(["Low", "Low", "Medium", "High"])))
        ports = c.execute("SELECT port_name FROM ports").fetchall(); cars = [r[0] for r in c.execute("SELECT name FROM carriers")]
        s = c.execute("SELECT COUNT(*) FROM shipments").fetchone()[0]
        while s < target_shipments:
            s += 1; a, b = R.sample(ports, 2)
            c.execute("INSERT OR REPLACE INTO shipments(shipment_id,origin_port,dest_port,carrier,status,eta,weight_kg,hs_code,predicted_delay_risk) VALUES(?,?,?,?,?,?,?,?,?)",
                      (f"SHP-{s:04d}", a[0], b[0], R.choice(cars), R.choices(["In Transit", "Delivered", "Customs Hold", "Port Congestion Delay", "Anchorage Pending"], [55, 20, 8, 10, 7])[0],
                       (datetime.date.today() + datetime.timedelta(days=R.randint(2, 40))).isoformat(), round(R.uniform(800, 28000), 1), f"HS-{R.choice([30,84,85,87,10,62,27,39,72,22])}{R.randint(10,99)}", round(R.uniform(.03, .6), 2)))

CITIES = {"Mumbai": (19.08, 72.88), "Delhi": (28.61, 77.21), "Bengaluru": (12.97, 77.59), "Chennai": (13.08, 80.27), "Hyderabad": (17.39, 78.49), "Pune": (18.52, 73.86), "Ahmedabad": (23.03, 72.57),
"Jaipur": (26.91, 75.79), "Kolkata": (22.57, 88.36), "Surat": (21.17, 72.83), "Kochi": (9.93, 76.27), "Coimbatore": (11.02, 76.96), "Lucknow": (26.85, 80.95), "Nagpur": (21.15, 79.09), "Indore": (22.72, 75.86),
"Bhopal": (23.26, 77.41), "Patna": (25.59, 85.14), "Chandigarh": (30.73, 76.78), "Visakhapatnam": (17.69, 83.22), "Vadodara": (22.31, 73.18), "Thiruvananthapuram": (8.52, 76.94), "Mysuru": (12.30, 76.64),
"Guwahati": (26.14, 91.74), "Bhubaneswar": (20.30, 85.82), "Ranchi": (23.34, 85.31), "Raipur": (21.25, 81.63), "Dehradun": (30.32, 78.03), "Goa": (15.50, 73.83), "Madurai": (9.93, 78.12), "Nashik": (19.99, 73.79)}
SKUS = [("Buns", "Food"), ("Cooking Oil", "Food"), ("Sauce", "Food"), ("Frozen Fries", "Food"), ("Chicken", "Food"), ("Cheese", "Food"), ("Lettuce", "Food"), ("Packaging", "Supplies"), ("Beverage Syrup", "Beverage"), ("Potatoes", "Food"), ("Flour", "Food"), ("Spice Mix", "Food")]
def scale_franchise(outlets=120, staff=1250, per_outlet_sku=12):
    R = random.Random(42)
    with _DB_LOCK, conn() as c:
        c.executescript("""CREATE TABLE IF NOT EXISTS outlet_geo(outlet_id TEXT PRIMARY KEY, lat REAL, lon REAL);
        CREATE TABLE IF NOT EXISTS customers_fr(customer_id TEXT PRIMARY KEY, outlet_id TEXT, total_spend_90d REAL, visit_freq REAL, avg_order_value REAL, days_since_last_visit INT);""")
        n = c.execute("SELECT COUNT(*) FROM outlets").fetchone()[0]; tiers = ["Metro Flagship", "Urban", "Suburban", "Highway"]; cities = list(CITIES)
        while n < outlets:
            n += 1; oid = f"OUT-{n:03d}"; city = R.choice(cities); rev = R.uniform(900000, 5200000)
            c.execute("INSERT OR REPLACE INTO outlets VALUES(?,?,?,?,?,?,?,?)", (oid, f"Franchise Outlet {n:03d}", city, R.choice(tiers), rev, rev * R.uniform(.55, .82), round(R.uniform(3.0, 4.9), 2), R.randint(12, 30)))
        for oid, city in c.execute("SELECT outlet_id, location FROM outlets").fetchall():
            if not c.execute("SELECT 1 FROM outlet_geo WHERE outlet_id=?", (oid,)).fetchone():
                la, lo = CITIES.get(city, CITIES["Mumbai"]); c.execute("INSERT INTO outlet_geo VALUES(?,?,?)", (oid, la + R.uniform(-.06, .06), lo + R.uniform(-.06, .06)))
        s = c.execute("SELECT COUNT(*) FROM staff").fetchone()[0]; oids = [r[0] for r in c.execute("SELECT outlet_id FROM outlets")]
        while s < staff:
            s += 1; sal = R.uniform(18000, 80000); sat = R.randint(1, 5); ot = R.uniform(0, 35)
            p = max(.05, min(.95, .9 - sal / 100000 - sat / 10 + ot / 100))
            c.execute("INSERT OR REPLACE INTO staff VALUES(?,?,?,?,?,?,?,?,?,?,?)", (f"STF-{s:04d}", R.choice(oids), f"Employee {s:04d}", R.choice(["Crew", "Crew", "Cashier", "Cook", "Shift Lead", "Store Manager"]), sal, round(ot, 1), sat, R.randint(19, 58), R.randint(0, 12), R.randint(1, 5), round(p, 2)))
        for oid in oids:
            have = {r[0] for r in c.execute("SELECT sku_name FROM inventory WHERE outlet_id=?", (oid,))}
            for sku, cat in SKUS[:per_outlet_sku]:
                if sku in have or sku in {"Oil"} and "Cooking Oil" == sku: continue
                wk = R.uniform(80, 700); lead = R.randint(2, 9)
                c.execute("INSERT OR REPLACE INTO inventory VALUES(?,?,?,?,?,?,?,?,?)", (f"INV-{oid[4:]}-{sku[:3].upper()}", oid, sku, cat, int(wk / 7 * (lead + R.uniform(2, 12))), int(wk / 7 * lead), round(wk, 1), lead, .1))
            if c.execute("SELECT COUNT(*) FROM marketing WHERE outlet_id=?", (oid,)).fetchone()[0] < 2:
                for j in range(2):
                    c.execute("INSERT OR REPLACE INTO marketing VALUES(?,?,?,?,?,?,?,?,?,?)", (f"CMP-{oid[4:]}-{j}", oid, f"Campaign {oid[4:]}-{j}", R.choice(["Social", "Email", "Influencer", "Offline"]), R.uniform(20000, 120000), round(R.uniform(1.2, 5.2), 2), R.randint(8000, 60000), R.randint(200, 2500), "2024-01-01", "2024-12-31"))
            if c.execute("SELECT COUNT(*) FROM audits WHERE outlet_id=?", (oid,)).fetchone()[0] < 2:
                for j in range(2):
                    sc = max(40, min(100, R.gauss(84, 10)))
                    c.execute("INSERT OR REPLACE INTO audits VALUES(?,?,?,?,?,?,?,?)", (f"AUD-{oid[4:]}-{j}", oid, (datetime.date.today() - datetime.timedelta(days=30 * (j + 1))).isoformat(), sc, int((100 - sc) / 6), R.choice(["Food Safety", "Hygiene", "SOP Compliance"]), "Pass" if sc >= 80 else "Warning" if sc >= 60 else "Fail", "Monthly audit"))
            if c.execute("SELECT COUNT(*) FROM feedback WHERE outlet_id=?", (oid,)).fetchone()[0] < 8:
                for j in range(8):
                    r = R.choices([1, 2, 3, 4, 5], [5, 8, 15, 35, 37])[0]
                    c.execute("INSERT OR REPLACE INTO feedback VALUES(?,?,?,?,?,?)", (f"FB-{oid[4:]}-{j}", oid, r, R.choice(["Great taste", "Slow service", "Clean ambience", "Staff was rude", "Cold food", "Good value", "Long wait time", "Dirty tables"]), datetime.date.today().isoformat(), (r - 3) / 2))
            if c.execute("SELECT COUNT(*) FROM customers_fr WHERE outlet_id=?", (oid,)).fetchone()[0] < 30:
                for j in range(30):
                    seg = R.choices(["hv", "freq", "churn"], [.2, .4, .4])[0]
                    f = {"hv": (6, 12), "freq": (3, 8), "churn": (.2, 1.5)}[seg]; aov = R.uniform(250, 900) if seg == "hv" else R.uniform(150, 450)
                    fr = R.uniform(*f); c.execute("INSERT OR REPLACE INTO customers_fr VALUES(?,?,?,?,?,?)", (f"C-{oid[4:]}-{j}", oid, round(fr * aov * 3, 0), round(fr, 1), round(aov, 0), R.randint(60, 120) if seg == "churn" else R.randint(1, 20)))


In [ ]:
%%writefile /content/app/seanav.py
"""Sea navigation: an ocean mask built from the world coastline (Natural Earth 110m land, bundled with the app), real canal/strait corridors
(Suez, Panama, Malacca, Gibraltar, Dover, Danish straits, Bosphorus, Hormuz, Sunda, Lombok...) and Dijkstra routing over a 1 degree water grid.
Routes are then straightened with a line-of-sight check on a finer 0.25 degree mask, so no leg crosses land. Everything is pure numpy/scipy."""
import os, json, math, heapq, threading
import numpy as np
from matplotlib.path import Path as MPath
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import dijkstra, connected_components

HERE = os.path.dirname(os.path.abspath(__file__))
TOPO = [os.path.join(HERE, "static", "vendor", "world.topo.json")]
R_NM = 3440.065
ACCESS = {   # river / bay / canal-side ports: where a ship actually enters open water (lat, lon). Matched on the port name.
    "wuhan": (31.2, 122.6), "nanjing": (31.2, 122.6), "taicang": (31.3, 122.4), "shanghai": (31.1, 122.4), "guangzhou": (22.0, 114.0), "dongguan": (22.0, 114.0), "zhuhai": (21.9, 113.8),
    "tokyo": (34.9, 139.9), "yokohama": (34.9, 139.9), "chiba": (34.9, 139.9), "nagoya": (34.2, 137.0), "osaka": (34.2, 135.0), "kobe": (34.2, 135.0), "hamburg": (54.1, 8.2), "antwerp": (51.45, 3.4),
    "houston": (29.1, -94.6), "new orleans": (28.9, -89.3), "baltimore": (37.0, -75.9), "norfolk": (36.9, -75.9), "montreal": (49.3, -66.0), "palembang": (-2.0, 105.4), "yangon": (15.7, 96.3), "mongla": (21.6, 89.5), "kolkata": (21.6, 88.2),
    "ho chi minh": (10.3, 107.1), "kandla": (22.8, 69.6), "navlakhi": (22.9, 70.0), "dahej": (21.7, 72.0), "hazira": (21.0, 72.4), "ambarli": (40.9, 28.3), "izmir": (38.4, 26.6), "amsterdam": (52.5, 4.3), "london gateway": (51.5, 1.2),
    "oslo": (59.2, 10.6), "douala": (3.9, 9.4), "fuzhou": (26.0, 119.9), "lianyungang": (34.7, 119.6), "port klang": (2.9, 101.0), "belawan": (4.1, 98.6), "rotterdam": (51.95, 3.9), "buenos aires": (-35.0, -57.0), "montevideo": (-35.0, -56.2), "guayaquil": (-2.8, -80.4),
    "seattle": (47.9, -122.7), "tacoma": (47.8, -122.8), "vancouver": (49.2, -123.4), "oakland": (37.8, -122.6), "kuching": (2.0, 110.3), "bangkok": (13.3, 100.6), "haikou": (20.3, 110.3), "port said": (31.4, 32.3), "ain sokhna": (29.6, 32.6), "aqaba": (28.9, 34.9),
}
CORRIDORS = {   # name -> centre line (lat, lon); cells along it (and 0.3 degree either side) are forced to water
    "Suez Canal": [(31.9, 32.3), (31.3, 32.33), (30.6, 32.34), (30.0, 32.55), (29.4, 32.55), (28.9, 32.8), (28.3, 33.2)],
    "Panama Canal": [(9.9, -79.95), (9.35, -79.88), (9.05, -79.6), (8.6, -79.5), (8.0, -79.4)],
    "Strait of Malacca": [(1.1, 104.3), (1.25, 103.7), (1.9, 102.6), (2.7, 101.5), (3.6, 100.5), (4.6, 99.9), (5.6, 99.0), (6.3, 97.5)],
    "Singapore Strait": [(1.2, 104.0), (1.4, 104.6), (2.2, 105.5)],
    "Strait of Gibraltar": [(36.5, -7.0), (36.0, -5.8), (36.0, -5.3), (36.2, -4.5), (36.4, -3.5)],
    "Dover Strait": [(49.6, -6.0), (49.8, -3.0), (50.4, -0.5), (50.9, 1.1), (51.4, 2.1), (52.0, 3.0)],
    "Danish Straits": [(57.7, 7.5), (57.8, 10.3), (57.0, 11.6), (56.2, 12.5), (55.5, 12.8), (55.0, 13.0), (54.9, 14.0), (55.2, 16.0), (56.0, 18.0), (57.5, 19.7), (58.8, 21.0), (59.6, 23.5), (59.9, 26.0), (60.0, 29.0)],
    "Bosphorus": [(39.9, 25.9), (40.2, 26.5), (40.7, 27.7), (41.0, 28.9), (41.4, 29.15), (42.0, 29.4), (42.8, 29.8)],
    "Strait of Hormuz": [(24.5, 58.6), (25.7, 56.8), (26.5, 56.3), (26.4, 54.8), (26.6, 52.8), (27.4, 51.0), (28.8, 49.8), (29.6, 48.9)],
    "Bab-el-Mandeb": [(12.0, 43.9), (12.7, 43.3), (13.6, 42.6)],
    "Sunda Strait": [(-5.7, 105.9), (-6.1, 105.8), (-6.8, 105.3)],
    "Lombok Strait": [(-7.6, 115.7), (-8.4, 115.7), (-9.2, 115.6)],
    "Messina": [(38.4, 15.6), (38.0, 15.6), (37.5, 15.3)],
    "Seto Inland Sea": [(33.9, 135.0), (34.4, 134.9), (34.5, 135.1)],
    "Korea Strait": [(35.0, 129.2), (34.3, 129.8), (33.5, 130.3)],
    "Aegean": [(36.2, 22.8), (36.6, 24.0), (37.5, 24.2), (38.5, 24.6), (39.5, 24.6), (40.0, 25.8)],
    "Adriatic": [(39.8, 18.9), (41.0, 18.7), (42.5, 17.5), (43.8, 15.3), (45.0, 13.5), (45.6, 13.4)],
    "Red Sea": [(27.5, 34.2), (24.0, 36.2), (20.0, 38.4), (16.0, 41.2), (13.5, 42.8)],
    "Gulf of Aden": [(12.5, 43.5), (12.0, 47.0), (12.4, 51.5), (12.0, 54.5)],
    "Persian Gulf": [(24.5, 58.6), (25.5, 56.7), (26.4, 54.5), (27.0, 52.0), (28.0, 50.5), (29.3, 48.8)],
    "Gulf of Mexico": [(24.0, -84.0), (23.0, -87.0), (26.5, -90.0), (28.0, -91.0), (28.0, -95.0), (26.0, -97.0), (21.0, -97.0), (19.0, -95.0)],
    "Baltic South": [(54.2, 18.9), (54.6, 19.5)],
}
BLOCK_GROUPS = {"suez": ["Suez Canal"], "panama": ["Panama Canal"], "malacca": ["Strait of Malacca"], "hormuz": ["Strait of Hormuz"], "bosphorus": ["Bosphorus"]}
CHOKE_LABEL = {"Suez Canal": "Suez Canal", "Panama Canal": "Panama Canal", "Strait of Malacca": "Strait of Malacca", "Strait of Gibraltar": "Gibraltar", "Dover Strait": "English Channel", "Strait of Hormuz": "Hormuz", "Bab-el-Mandeb": "Bab-el-Mandeb", "Bosphorus": "Bosphorus", "Danish Straits": "Danish Straits", "Sunda Strait": "Sunda Strait", "Lombok Strait": "Lombok Strait"}

def _hav(a, b):
    la1, lo1, la2, lo2 = map(math.radians, (a[0], a[1], b[0], b[1])); h = math.sin((la2 - la1) / 2) ** 2 + math.cos(la1) * math.cos(la2) * math.sin((lo2 - lo1) / 2) ** 2
    return R_NM * 2 * math.asin(min(1.0, math.sqrt(h)))

def _land_polys():
    p = next((x for x in TOPO if os.path.exists(x)), None)
    if not p: raise FileNotFoundError("world.topo.json (bundled with the app) not found")
    t = json.load(open(p)); sx, sy = t["transform"]["scale"]; tx, ty = t["transform"]["translate"]; arcs = []
    for a in t["arcs"]:
        x = y = 0; pts = []
        for dx, dy in a: x += dx; y += dy; pts.append((x * sx + tx, y * sy + ty))
        arcs.append(pts)
    def ring(idx):
        out = []
        for i in idx:
            seg = arcs[i] if i >= 0 else arcs[~i][::-1]; out += seg if not out else seg[1:]
        return out
    polys = []
    def geoms(o):
        if o["type"] == "GeometryCollection":
            for g in o["geometries"]: yield from geoms(g)
        else: yield o
    for g in geoms(t["objects"]["land"]):
        pl = g["arcs"] if g["type"] == "MultiPolygon" else [g["arcs"]]
        for poly in pl: polys.append([_unwrap_ring(ring(r)) for r in poly])        # [outer, holes...]
    return polys
def _unwrap_ring(pts):
    """Rings that cross the antimeridian jump by ~360 degrees between two points; make the longitude continuous (it may then exceed +-180)."""
    out = [pts[0]]
    for x, y in pts[1:]:
        px = out[-1][0]
        while x - px > 180: x -= 360
        while x - px < -180: x += 360
        out.append((x, y))
    return out

def gc_arr(a, b, step_nm=60.0, include_ends=True):
    """Vectorised great-circle sampling: (lat array, lon array); longitudes continuous with a's."""
    la1, lo1, la2, lo2 = map(math.radians, (a[0], a[1], b[0], b[1]))
    v1 = np.array([math.cos(la1) * math.cos(lo1), math.cos(la1) * math.sin(lo1), math.sin(la1)]); v2 = np.array([math.cos(la2) * math.cos(lo2), math.cos(la2) * math.sin(lo2), math.sin(la2)])
    om = math.acos(max(-1.0, min(1.0, float(v1 @ v2)))); n = max(1, int(om * R_NM / step_nm)); k = np.arange(0 if include_ends else 1, n + 1 if include_ends else n); t = k / n
    if om > 1e-9: w1 = np.sin((1 - t) * om) / math.sin(om); w2 = np.sin(t * om) / math.sin(om)
    else: w1, w2 = 1 - t, t
    v = w1[:, None] * v1[None, :] + w2[:, None] * v2[None, :]; la = np.degrees(np.arcsin(np.clip(v[:, 2], -1, 1))); lo = np.degrees(np.arctan2(v[:, 1], v[:, 0]))
    lo = a[1] + (((lo - a[1]) + 180) % 360 - 180)          # continuous with a's longitude
    return la, lo
def gc_points(a, b, step_nm=60.0, include_ends=True):
    """Points along the great circle from a to b (lat, lon), spaced about step_nm apart; longitudes continuous with a's."""
    la, lo = gc_arr(a, b, step_nm, include_ends); out = list(zip(la.tolist(), lo.tolist()))
    for q in range(1, len(out)):          # keep successive longitudes continuous (no jumps > 180)
        while out[q][1] - out[q - 1][1] > 180: out[q] = (out[q][0], out[q][1] - 360)
        while out[q][1] - out[q - 1][1] < -180: out[q] = (out[q][0], out[q][1] + 360)
    return out

class Sea:
    def __init__(self, step=1.0, fine=0.25, latmin=-78.0, latmax=82.0):
        self.step, self.fine, self.latmin, self.latmax = step, fine, latmin, latmax
        self.lats = np.arange(latmin + step / 2, latmax, step); self.lons = np.arange(-180 + step / 2, 180, step); self.nlat, self.nlon = len(self.lats), len(self.lons)
        self.flats = np.arange(latmin + fine / 2, latmax, fine); self.flons = np.arange(-180 + fine / 2, 180, fine)
        polys = _land_polys(); self.land = self._raster(polys, self.lats, self.lons); self.fland = self._raster(polys, self.flats, self.flons)
        self.corr_cells = {}                                                   # corridor name -> set of coarse node ids
        for name, line in CORRIDORS.items(): self._carve(name, line)
        pol = (self.lats < -52) | (self.lats > 76); drake = (self.lons > -82) & (self.lons < -48)
        self.land[np.ix_(pol & (self.lats < 0), ~drake)] = True; self.land[np.ix_(self.lats > 76, np.ones(self.nlon, bool))] = True
        fpol = (self.flats < -52); fdrake = (self.flons > -82) & (self.flons < -48); self.fland[np.ix_(fpol, ~fdrake)] = True; self.fland[self.flats > 76, :] = True
        self.ocean = ~self.land; self.nid = -np.ones(self.land.shape, int); self.nid[self.ocean] = np.arange(int(self.ocean.sum()))
        self.nodes = np.argwhere(self.ocean); self.n = len(self.nodes); self._build_graph()
        self._cache = {}; self._variants = {}; self._lock = threading.RLock()
    def _raster(self, polys, lats, lons):
        LON, LAT = np.meshgrid(lons, lats); pts = np.c_[LON.ravel(), LAT.ravel()]; inside = np.zeros(len(pts), bool)
        for rings in polys:
            outer = np.array(rings[0]); x0, y0 = outer.min(0); x1, y1 = outer.max(0)
            for off in (0, 360, -360):
                mo = (pts[:, 0] + off >= x0) & (pts[:, 0] + off <= x1) & (pts[:, 1] >= y0) & (pts[:, 1] <= y1)
                if not mo.any(): continue
                idx = np.where(mo)[0]; q = pts[idx] + np.array([off, 0]); ins = MPath(outer).contains_points(q)
                for h in rings[1:]: ins &= ~MPath(np.array(h)).contains_points(q)
                inside[idx[ins]] = True
        return inside.reshape(len(lats), len(lons))
    def _carve(self, name, line):
        cells = set()
        for (a, b) in zip(line[:-1], line[1:]):
            n = max(2, int(_hav(a, b) / 8))
            for t in np.linspace(0, 1, n):
                la, lo = a[0] + (b[0] - a[0]) * t, a[1] + (b[1] - a[1]) * t
                i = int((la - self.latmin) / self.step); j = int((lo + 180) / self.step) % self.nlon
                if 0 <= i < self.nlat: self.land[i, j] = False; cells.add((i, j))
                fi = int((la - self.latmin) / self.fine); fj = int((lo + 180) / self.fine) % len(self.flons)
                for di in (-1, 0, 1):
                    for dj in (-1, 0, 1):
                        a_, b_ = fi + di, (fj + dj) % len(self.flons)
                        if 0 <= a_ < len(self.flats): self.fland[a_, b_] = False
        self.corr_cells[name] = cells
    def _build_graph(self):
        rows, cols, w = [], [], []
        for k, (i, j) in enumerate(self.nodes):
            for di in (-1, 0, 1):
                for dj in (-1, 0, 1):
                    if di == 0 and dj == 0: continue
                    a, b = i + di, (j + dj) % self.nlon
                    if not (0 <= a < self.nlat) or not self.ocean[a, b]: continue
                    if di != 0 and dj != 0 and not (self.ocean[i, b] or self.ocean[a, j]): continue         # no diagonal squeeze between two land cells
                    rows.append(k); cols.append(self.nid[a, b]); w.append(_hav((self.lats[i], self.lons[j]), (self.lats[a], self.lons[b])))
        self.A = csr_matrix((w, (rows, cols)), shape=(self.n, self.n))
        nc, lab = connected_components(self.A, directed=False); self.main = lab == np.bincount(lab).argmax(); self.comp = lab
    # ------------------------------------------------------------ helpers
    def is_land(self, la, lo):
        i = int((la - self.latmin) / self.fine); j = int(((lo + 180) % 360) / self.fine)
        return bool(self.fland[i, j]) if 0 <= i < len(self.flats) else True
    def dist_to_water_nm(self, la, lo, maxr=2.0):
        """Distance in nm from a point to the nearest water cell of the fine mask (0 if the point is in water)."""
        i0 = int((la - self.latmin) / self.fine); j0 = int(((lo + 180) % 360) / self.fine); r = int(maxr / self.fine); best = None
        for di in range(-r, r + 1):
            for dj in range(-r, r + 1):
                i, j = i0 + di, (j0 + dj) % len(self.flons)
                if 0 <= i < len(self.flats) and not self.fland[i, j]:
                    d = _hav((la, lo), (self.flats[i], self.flons[j])); best = d if best is None or d < best else best
        return best
    def _clear(self, a, b, ends=0.6):
        la, lo = gc_arr(a, b, 7.0)
        keep = ~(((np.abs(la - a[0]) < ends) & (np.abs(lo - a[1]) < ends)) | ((np.abs(la - b[0]) < ends) & (np.abs(lo - b[1]) < ends)))
        la, lo = la[keep], lo[keep]
        if not len(la): return True
        i = ((la - self.latmin) / self.fine).astype(int); j = (((lo + 180) % 360) / self.fine).astype(int); ok = (i >= 0) & (i < len(self.flats))
        if not ok.all(): return False
        return not bool(self.fland[i, j].any())
    def _near_nodes(self, la, lo, k=12, main_only=True):
        d = (self.lats[self.nodes[:, 0]] - la) ** 2 + ((((self.lons[self.nodes[:, 1]] - lo + 180) % 360) - 180) * math.cos(math.radians(la))) ** 2
        if main_only: d = np.where(self.main, d, 1e9)
        return np.argsort(d)[:k]
    def _snap(self, la, lo):
        near = self._near_nodes(la, lo)
        for k in near:
            p = (float(self.lats[self.nodes[k, 0]]), float(self.lons[self.nodes[k, 1]]))
            if self._clear((la, lo), p, 0.45): return int(k)
        return int(near[0])
    def _graph_for(self, blocked):
        key = tuple(sorted(blocked))
        if not key: return self.A
        with self._lock:
            if key in self._variants: return self._variants[key]
            ok = np.ones(self.n, bool)
            for g in key:
                for name in BLOCK_GROUPS.get(g, [g]):
                    for (i, j) in self.corr_cells.get(name, ()):
                        if self.nid[i, j] >= 0: ok[self.nid[i, j]] = False
            D = csr_matrix(ok.astype(float)).T if False else None
            A = self.A.multiply(ok[:, None]).multiply(ok[None, :]).tocsr(); A.eliminate_zeros(); self._variants[key] = A; return A
    # ------------------------------------------------------------ public API
    def route(self, o, d, blocked=(), storms=(), oa=None, da=None):
        """o, d = (lat, lon). Returns {"pts": [(lat, lon), ...] (longitudes unwrapped), "nm": float, "via": [names]} or None."""
        key = (round(o[0], 2), round(o[1], 2), round(d[0], 2), round(d[1], 2), oa, da, tuple(sorted(blocked)), tuple((round(s[0]), round(s[1]), round(s[2])) for s in storms))
        with self._lock:
            if key in self._cache: return self._cache[key]
        A = self._graph_for(blocked)
        if storms:
            ok = np.ones(self.n, bool)
            for (sla, slo, rn) in storms:
                dd = np.array([_hav((sla, slo), (self.lats[i], self.lons[j])) for i, j in self.nodes]) if False else None
                la_ = self.lats[self.nodes[:, 0]]; lo_ = self.lons[self.nodes[:, 1]]
                dist = R_NM * np.arccos(np.clip(np.sin(np.radians(la_)) * math.sin(math.radians(sla)) + np.cos(np.radians(la_)) * math.cos(math.radians(sla)) * np.cos(np.radians(lo_ - slo)), -1, 1)); ok &= dist > rn
            A = A.multiply(ok[:, None]).multiply(ok[None, :]).tocsr(); A.eliminate_zeros()
        s, t = self._snap(*(oa or o)), self._snap(*(da or d))
        dist, pred = dijkstra(A, directed=True, indices=s, return_predecessors=True)
        if not np.isfinite(dist[t]): res = None
        else:
            path = [t]
            while path[-1] != s: path.append(pred[path[-1]])
            path.reverse(); pts = [o] + ([oa] if oa else []) + [(float(self.lats[self.nodes[k, 0]]), float(self.lons[self.nodes[k, 1]])) for k in path] + ([da] if da else []) + [d]; pts = self._unwrap(self._straighten(pts)); pts = self._densify(pts)
            res = {"pts": [(round(a, 3), round(b, 3)) for a, b in pts], "nm": round(sum(_hav(pts[i], pts[i + 1]) for i in range(len(pts) - 1)), 1), "via": self._via(pts)}
        with self._lock: self._cache[key] = res
        return res
    def _straighten(self, pts):
        out = [pts[0]]; i = 0
        while i < len(pts) - 1:
            j = len(pts) - 1
            while j > i + 1 and not self._clear(pts[i], pts[j], 0.5): j -= 1
            out.append(pts[j]); i = j
        return out
    @staticmethod
    def _densify(pts, step=90.0):
        out = [pts[0]]
        for a, b in zip(pts[:-1], pts[1:]): out += gc_points(a, b, step)[1:]
        return out
    @staticmethod
    def _unwrap(pts):
        out = [pts[0]]
        for la, lo in pts[1:]:
            prev = out[-1][1]
            while lo - prev > 180: lo -= 360
            while lo - prev < -180: lo += 360
            out.append((la, lo))
        return out
    def _via(self, pts):
        names = []; dense = []
        for a, b in zip(pts[:-1], pts[1:]):
            for t in np.linspace(0, 1, max(2, int(_hav(a, b) / 15))): dense.append((a[0] + (b[0] - a[0]) * t, a[1] + (b[1] - a[1]) * t))
        order = []
        for la, lo in dense:
            i = int((la - self.latmin) / self.step); j = int(((lo + 180) % 360) / self.step) % self.nlon
            for name, cells in self.corr_cells.items():
                if name in CHOKE_LABEL and (i, j) in cells and CHOKE_LABEL[name] not in order: order.append(CHOKE_LABEL[name])
        if any(-47 < la < -30 and 10 < (lo % 360 if lo < 0 else lo) < 35 for la, lo in dense): order.append("Cape of Good Hope")
        if any(la < -50 and (-80 < lo < -55) for la, lo in dense): order.append("Cape Horn / Drake Passage")
        return order
    def options(self, o, d, blocked=(), storms=(), oa=None, da=None):
        """Up to three sea routes: shortest, and the alternatives that avoid Suez / Panama / Malacca when the shortest uses them."""
        base = self.route(o, d, tuple(blocked), storms, oa, da); out = []
        if not base: return out
        def label(r):
            v = r["via"]; return ("via " + " + ".join(v[:3])) if v else "direct sea route"
        out.append({"name": label(base), "pts": base["pts"], "nm": base["nm"], "via": base["via"]})
        for g, nm_ in (("suez", "Suez Canal"), ("panama", "Panama Canal"), ("malacca", "Strait of Malacca")):
            if nm_ in base["via"] and g not in blocked:
                alt = self.route(o, d, tuple(blocked) + (g,), storms, oa, da)
                if alt and abs(alt["nm"] - base["nm"]) > 50 and all(abs(alt["nm"] - x["nm"]) > 50 for x in out): out.append({"name": label(alt) + f" (without {nm_})", "pts": alt["pts"], "nm": alt["nm"], "via": alt["via"]})
        return out

def access_for(name):
    n = (name or "").lower()
    for k, v in ACCESS.items():
        if k in n: return v
    return None
_SEA = None; _SL = threading.Lock(); _VER = "v5"
def _cache_path():
    d = os.environ.get("APP_DATA_DIR") or os.path.dirname(os.path.abspath(__file__)); return os.path.join(d, f"seanav_{_VER}.pkl")
def sea():
    """Singleton. The raster + graph (about 6 s to build) is pickled next to the database, so later starts load in well under a second."""
    global _SEA
    with _SL:
        if _SEA is None:
            import pickle
            p = _cache_path()
            try:
                with open(p, "rb") as f: _SEA = pickle.load(f)
                _SEA._cache = {}; _SEA._variants = {}; _SEA._lock = threading.RLock()
            except Exception:
                _SEA = Sea()
                try:
                    st = {k: v for k, v in _SEA.__dict__.items() if k not in ("_lock", "_cache", "_variants")}
                    class _P: pass
                    o = Sea.__new__(Sea); o.__dict__.update(st)
                    with open(p + ".tmp", "wb") as f: pickle.dump(o, f, protocol=4)
                    os.replace(p + ".tmp", p)
                except Exception: pass
    return _SEA


In [ ]:
%%writefile /content/app/seed_data.py
import random, sqlite3, datetime
import pandas as pd
from db import get_conn

BASE_PORTS = [
    ("JNPT Nhava Sheva (Mumbai)", "India", 3.4, 4, 18.95, 72.95, "Asia"),
    ("Mundra Port", "India", 2.8, 3, 22.84, 69.70, "Asia"),
    ("Chennai Port", "India", 3.1, 4, 13.10, 80.30, "Asia"),
    ("Tuticorin VOC Port", "India", 2.5, 3, 8.75, 78.18, "Asia"),
    ("Cochin Port", "India", 2.2, 3, 9.96, 76.26, "Asia"),
    ("Visakhapatnam Port", "India", 2.9, 3, 17.68, 83.28, "Asia"),
    ("Kolkata Haldia Port", "India", 3.5, 5, 22.03, 88.11, "Asia"),
    ("Kandla Deendayal Port", "India", 3.0, 4, 23.01, 70.22, "Asia"),
    ("New Mangalore Port", "India", 2.3, 3, 12.92, 74.81, "Asia"),
    ("Paradip Port", "India", 3.2, 4, 20.26, 86.67, "Asia"),
    ("Shanghai Port", "China", 4.2, 5, 31.23, 121.47, "Asia"),
    ("Singapore Port", "Singapore", 1.2, 2, 1.29, 103.85, "Asia"),
    ("Busan Port", "South Korea", 1.8, 3, 35.10, 129.04, "Asia"),
    ("Tokyo Port", "Japan", 2.2, 3, 35.62, 139.77, "Asia"),
    ("Colombo Port", "Sri Lanka", 2.7, 3, 6.94, 79.84, "Asia"),
    ("Dubai Jebel Ali Port", "UAE", 1.9, 2, 25.20, 55.27, "Middle East"),
    ("Rotterdam Port", "Netherlands", 1.5, 2, 51.92, 4.47, "Europe"),
    ("Antwerp Port", "Belgium", 2.6, 3, 51.22, 4.40, "Europe"),
    ("Hamburg Port", "Germany", 2.5, 3, 53.55, 9.99, "Europe"),
    ("Los Angeles Port", "USA", 3.6, 4, 33.74, -118.27, "Americas"),
    ("Ningbo-Zhoushan Port", "China", 3.9, 5, 29.87, 121.55, "Asia"),
    ("Shenzhen Port", "China", 3.7, 4, 22.54, 114.05, "Asia"),
    ("Guangzhou Port", "China", 3.3, 4, 23.10, 113.30, "Asia"),
    ("Qingdao Port", "China", 3.4, 4, 36.07, 120.38, "Asia"),
    ("Tianjin Port", "China", 3.6, 4, 39.00, 117.72, "Asia"),
    ("Hong Kong Port", "Hong Kong", 2.1, 3, 22.29, 114.16, "Asia"),
    ("Kaohsiung Port", "Taiwan", 2.4, 3, 22.62, 120.28, "Asia"),
    ("Laem Chabang Port", "Thailand", 2.6, 3, 13.08, 100.88, "Asia"),
    ("Port Klang", "Malaysia", 2.0, 3, 3.00, 101.39, "Asia"),
    ("Tanjung Pelepas Port", "Malaysia", 1.7, 2, 1.36, 103.55, "Asia"),
    ("Jakarta Tanjung Priok", "Indonesia", 3.3, 4, -6.10, 106.88, "Asia"),
    ("Manila Port", "Philippines", 3.5, 4, 14.58, 120.95, "Asia"),
    ("Ho Chi Minh Cat Lai Port", "Vietnam", 3.0, 4, 10.76, 106.77, "Asia"),
    ("Karachi Port", "Pakistan", 3.8, 5, 24.85, 66.98, "Asia"),
    ("Bandar Abbas Port", "Iran", 3.6, 4, 27.19, 56.28, "Middle East"),
    ("Jeddah Islamic Port", "Saudi Arabia", 2.2, 3, 21.49, 39.17, "Middle East"),
    ("Salalah Port", "Oman", 1.8, 2, 17.02, 54.09, "Middle East"),
    ("Piraeus Port", "Greece", 2.3, 3, 37.94, 23.65, "Europe"),
    ("Valencia Port", "Spain", 2.1, 3, 39.44, -0.32, "Europe"),
    ("Algeciras Port", "Spain", 2.0, 2, 36.13, -5.45, "Europe"),
    ("Felixstowe Port", "UK", 2.8, 3, 51.96, 1.35, "Europe"),
    ("Le Havre Port", "France", 2.4, 3, 49.49, 0.11, "Europe"),
    ("Gdansk Port", "Poland", 2.2, 3, 54.35, 18.65, "Europe"),
    ("Genoa Port", "Italy", 2.6, 3, 44.41, 8.93, "Europe"),
    ("Durban Port", "South Africa", 3.4, 4, -29.87, 31.02, "Africa"),
    ("Tangier Med Port", "Morocco", 1.9, 2, 35.88, -5.50, "Africa"),
    ("Lagos Apapa Port", "Nigeria", 4.0, 5, 6.45, 3.37, "Africa"),
    ("New York/New Jersey Port", "USA", 2.9, 3, 40.68, -74.02, "Americas"),
    ("Savannah Port", "USA", 2.7, 3, 32.08, -81.09, "Americas"),
    ("Long Beach Port", "USA", 3.4, 4, 33.75, -118.19, "Americas"),
]


def safe_exec(conn, sql, params=()):
    try:
        conn.execute(sql, params)
    except Exception as e:
        pass

def seed_all():
    with get_conn() as conn:
        if conn.execute("SELECT 1 FROM ports LIMIT 1").fetchone():
            return
        # 1. Ports
        safe_exec(conn, "DELETE FROM ports;")
        for i, p in enumerate(BASE_PORTS, 1):
            safe_exec(conn,
                "INSERT OR REPLACE INTO ports (port_id, port_name, country, congestion_index, avg_dwell_days, lat, lon, region) VALUES (?, ?, ?, ?, ?, ?, ?, ?);",
                (f"PORT-{i:03d}", p[0], p[1], p[2], p[3], p[4], p[5], p[6])
            )

        # 2. Users

        # 3. Shipments
        safe_exec(conn, "DELETE FROM shipments;")
        carriers_list = ["Maersk Line", "MSC Cargo", "CMA CGM", "COSCO Shipping", "Hapag-Lloyd", "ONE Ocean Express"]
        statuses = ["In Transit", "Customs Hold", "Delivered", "Port Congestion Delay", "Anchorage Pending"]

        for i in range(1, 101):
            p1 = BASE_PORTS[i % len(BASE_PORTS)][0]
            p2 = BASE_PORTS[(i+3) % len(BASE_PORTS)][0]
            shp_id = f"SHP-{i:04d}"

            # Generate the exact 9 fields expected by the database
            weight = round(random.uniform(500.0, 45000.0), 1)
            eta_val = (datetime.date.today() + datetime.timedelta(days=random.randint(2, 30))).isoformat()
            sev = random.randint(1, 5)
            ch_prob = round(random.uniform(0.02, 0.45), 2)
            cong = round(random.uniform(1.0, 4.8), 1)
            d_risk = round((cong / 5.0) * 0.5 + (ch_prob) * 0.3 + (sev / 5.0) * 0.2, 2)
            hs = f"HS-{random.randint(8400, 8900)}"

            # Insert matching the 9 columns exactly
            safe_exec(conn,
                "INSERT OR REPLACE INTO shipments (shipment_id, origin_port, dest_port, carrier, status, eta, weight_kg, hs_code, predicted_delay_risk) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?);",
                (shp_id, p1, p2, random.choice(carriers_list), random.choice(statuses), eta_val, weight, hs, d_risk)
            )

        # 4. Weather Risks
        safe_exec(conn, "DELETE FROM weather_risks;")
        for i in range(1, 21):
            pname = BASE_PORTS[(i - 1) % len(BASE_PORTS)][0]
            sev = random.randint(1, 4)
            fore = "Category 3 Typhoon Warning" if sev >= 3 else "Clear Maritime Conditions"
            w_spd = round(random.uniform(12.0, 58.0), 1)
            wv_ht = round(random.uniform(0.8, 5.5), 1)
            temp = round(random.uniform(14.0, 38.0), 1)
            safe_exec(conn,
                "INSERT OR REPLACE INTO weather_risks (port_name, current_severity, forecast, wind_speed, wave_height, temperature) VALUES (?, ?, ?, ?, ?, ?);",
                (pname, sev, fore, w_spd, wv_ht, temp)
            )

        # 5. Alerts
        safe_exec(conn, "DELETE FROM alerts;")
        categories = ["Customs Hold", "Typhoon Storm", "Port Congestion", "Vessel Mechanical", "Bunker Fuel Surcharge"]
        severities = ["Critical", "High", "Medium", "Low"]
        for i in range(1, 51):
            shp_id = f"SHP-{i:04d}"
            sev = random.choice(severities)
            cat = random.choice(categories)
            msg = f"Alert #{i:03d}: Severe {cat} operational delay reported on {shp_id}."
            safe_exec(conn,
                "INSERT INTO alerts (shipment_id, severity, category, message, date, resolved) VALUES (?, ?, ?, ?, ?, ?);",
                (shp_id, sev, cat, msg, "2024-08-11", 0)
            )

        # 6. Carriers
        safe_exec(conn, "DELETE FROM carriers;")
        for idx, name in enumerate(carriers_list, 1):
            safe_exec(conn,
                "INSERT OR REPLACE INTO carriers (carrier_id, name, rating, on_time_pct, avg_cost_index, risk_level) VALUES (?, ?, ?, ?, ?, ?);",
                (f"CAR-{idx:03d}", name, round(random.uniform(3.8, 4.9), 2), round(random.uniform(78, 96), 1), round(random.uniform(0.86, 1.18), 2), "Low")
            )

        # 7. Customers
        safe_exec(conn, "DELETE FROM customers;")
        for i in range(1, 25):
            safe_exec(conn,
                "INSERT OR REPLACE INTO customers (customer_id, name, industry, priority_tier, credit_risk) VALUES (?, ?, ?, ?, ?);",
                (f"CUST-{i:03d}", f"Corporate Client {i:03d}", random.choice(["Food Service", "Retail", "QSR", "Hospitality"]), random.choice(["Platinum", "Gold", "Silver"]), round(random.uniform(0.02, 0.18), 2))
            )

        # 8. Freight Quotes
        safe_exec(conn, "DELETE FROM freight_quotes;")
        for i in range(1, 51):
            base = round(random.uniform(1200, 9000), 2)
            margin_pct = round(random.uniform(9, 24), 1)
            final = round(base * (1 + margin_pct / 100), 2)
            safe_exec(conn,
                "INSERT OR REPLACE INTO freight_quotes (quote_id, shipment_id, customer_id, base_cost, insurance, customs_fee, fuel_surcharge, final_price, margin_pct, status, created_at) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?);",
                (f"QTE-{i:04d}", f"SHP-{random.randint(1, 50):04d}", f"CUST-{random.randint(1, 20):03d}", base, round(base * 0.02, 2), round(random.uniform(100, 600), 2), round(base * 0.08, 2), final, margin_pct, random.choice(["Draft", "Accepted", "Submitted"]), datetime.date.today().isoformat())
            )

        # 9. Customs Tariffs
        safe_exec(conn, "DELETE FROM customs_tariffs;")

        # ADD THE MISSING LIST HERE:
        cargos = ["Electronics", "Pharmaceuticals", "Automotive Parts", "Textiles", "Heavy Machinery", "Perishables"]

        for i, cargo in enumerate(cargos, 1):
            safe_exec(conn,
                "INSERT OR REPLACE INTO customs_tariffs (tariff_id, hs_code, cargo_type, origin_country, destination_country, duty_rate, clearance_risk, required_docs, advisory) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?);",
                (f"TAR-{i:03d}", f"HS-{8400 + i * 31}", cargo, "India", "UAE", round(random.uniform(4, 16), 2), round(random.uniform(0.08, 0.42), 2), "Commercial invoice, packing list, bill of lading", "Validate HS code and pre-clear high-risk lanes.")
            )

        # 10. Outlets (FranchiseOps)
        outlet_cities = ["Chennai", "Bengaluru", "Hyderabad", "Mumbai", "Pune", "Delhi", "Kochi", "Coimbatore", "Ahmedabad", "Kolkata"]
        tiers = ["Metro Flagship", "Urban", "Express", "Mall"]
        safe_exec(conn, "DELETE FROM outlets;")
        for i in range(1, 51):
            revenue = round(random.uniform(850000, 6400000), 2)
            cost = round(revenue * random.uniform(0.58, 0.82), 2)
            safe_exec(conn,
                "INSERT OR REPLACE INTO outlets (outlet_id, outlet_name, location, tier, revenue, operating_costs, customer_satisfaction, staff_headcount) VALUES (?, ?, ?, ?, ?, ?, ?, ?);",
                (f"OUT-{i:03d}", f"Franchise Outlet {i:03d}", random.choice(outlet_cities), random.choice(tiers), revenue, cost, round(random.uniform(3.2, 4.9), 2), random.randint(12, 55))
            )

        # 11. Staff
        roles = ["Store Manager", "Shift Lead", "Crew", "Chef", "Cashier", "Inventory Associate"]
        safe_exec(conn, "DELETE FROM staff;")
        for i in range(1, 151):
            satisfaction = random.randint(1, 5)
            overtime = round(random.uniform(0, 42), 1)
            attrition = min(0.95, max(0.03, 0.55 - satisfaction * 0.08 + overtime * 0.009 + random.uniform(-0.08, 0.08)))
            safe_exec(conn,
                "INSERT OR REPLACE INTO staff (staff_id, outlet_id, name, role, salary, overtime_hrs, job_satisfaction, age, tenure_years, work_life_balance, predicted_attrition_prob) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?);",
                (f"STF-{i:04d}", f"OUT-{random.randint(1, 50):03d}", f"Employee {i:04d}", random.choice(roles), round(random.uniform(18000, 95000), 2), overtime, satisfaction, random.randint(19, 56), random.randint(0, 14), random.randint(1, 5), round(attrition, 2))
            )

        # 12. Inventory
        skus = ["Buns", "Cheese", "Sauce", "Chicken", "Paneer", "Coffee Beans", "Packaging", "Oil", "Frozen Fries", "Dessert Mix"]
        safe_exec(conn, "DELETE FROM inventory;")
        for i in range(1, 151):
            demand = round(random.uniform(20, 420), 1)
            threshold = random.randint(30, 180)
            stock = random.randint(5, 420)
            risk = min(0.95, max(0.02, (threshold - stock) / max(threshold, 1) + random.uniform(0.05, 0.28)))
            safe_exec(conn,
                "INSERT OR REPLACE INTO inventory (record_id, outlet_id, sku_name, category, current_stock, reorder_threshold, weekly_demand, lead_time_days, stockout_risk_prob) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?);",
                (f"INV-{i:04d}", f"OUT-{random.randint(1, 50):03d}", random.choice(skus), random.choice(["Food", "Beverage", "Packaging", "Consumable"]), stock, threshold, demand, random.randint(1, 9), round(risk, 2))
            )

        # 13. Marketing
        channels = ["Digital Ads", "Social Media", "Local Print", "Influencer Campaign", "Radio Spots"]
        safe_exec(conn, "DELETE FROM marketing;")
        for i in range(1, 51):
            budget = round(random.uniform(15000, 120000), 2)
            roi = round(random.uniform(1.8, 5.4), 2)
            safe_exec(conn,
                "INSERT OR REPLACE INTO marketing (campaign_id, outlet_id, campaign_name, channel, budget, actual_roi, reach, conversions, start_date, end_date) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?);",
                (f"CMP-{i:03d}", f"OUT-{random.randint(1, 50):03d}", f"Campaign {i:03d}", random.choice(channels), budget, roi, random.randint(5000, 80000), random.randint(200, 4500), "2024-01-01", "2024-12-31")
            )

        # 14. Feedback
        safe_exec(conn, "DELETE FROM feedback;")
        comments = ["Great food!", "Slow service", "Clean ambience", "Polite staff", "Average experience"]
        for i in range(1, 101):
            rating = random.randint(1, 5)
            sentiment = round((rating - 3) / 2.0, 2)
            safe_exec(conn,
                "INSERT OR REPLACE INTO feedback (feedback_id, outlet_id, rating, comment, date, sentiment_score) VALUES (?, ?, ?, ?, ?, ?);",
                (f"FB-{i:04d}", f"OUT-{random.randint(1, 50):03d}", rating, random.choice(comments), "2024-08-01", sentiment)
            )

        # 15. Audits
        safe_exec(conn, "DELETE FROM audits;")
        categories_audit = ["Food Safety", "Hygiene & Sanitation", "Fire & Safety", "Financial Compliance"]
        for i in range(1, 51):
            score = round(random.uniform(65, 99), 1)
            status = "Pass" if score >= 85 else ("Conditional Pass" if score >= 75 else "Action Required")
            safe_exec(conn,
                "INSERT OR REPLACE INTO audits (audit_id, outlet_id, audit_date, score, violations, category, status, notes) VALUES (?, ?, ?, ?, ?, ?, ?, ?);",
                (f"AUD-{i:03d}", f"OUT-{random.randint(1, 50):03d}", "2024-08-01", score, int((100-score)/5), random.choice(categories_audit), status, "Audit completed cleanly.")
            )

        conn.commit()


In [ ]:
%%writefile /content/app/weather.py
"""Real weather for every port: Open-Meteo forecast + marine APIs (free, no key), fetched in batches and stored in wx_live.
Falls back to the simulated values already in weather_risks when the network is closed; the page says which one it is showing."""
import time, threading, datetime, json
import live_engine as LE

try: import requests
except Exception: requests = None

FORECAST = "https://api.open-meteo.com/v1/forecast"
MARINE = "https://marine-api.open-meteo.com/v1/marine"
WMO = {0: "Clear", 1: "Mostly clear", 2: "Partly cloudy", 3: "Overcast", 45: "Fog", 48: "Rime fog", 51: "Light drizzle", 53: "Drizzle", 55: "Heavy drizzle", 56: "Freezing drizzle", 57: "Freezing drizzle", 61: "Light rain", 63: "Rain", 65: "Heavy rain", 66: "Freezing rain", 67: "Freezing rain",
       71: "Light snow", 73: "Snow", 75: "Heavy snow", 77: "Snow grains", 80: "Rain showers", 81: "Rain showers", 82: "Violent showers", 85: "Snow showers", 86: "Snow showers", 95: "Thunderstorm", 96: "Thunderstorm, hail", 99: "Thunderstorm, heavy hail"}
_get = None   # tests can replace this with a function(url, params) -> json

def ensure():
    c = LE.conn(); c.execute("CREATE TABLE IF NOT EXISTS wx_live(port_name TEXT PRIMARY KEY, ts TEXT, wind_ms REAL, gust_ms REAL, wind_dir REAL, temp_c REAL, precip_mm REAL, wcode INT, wave_m REAL, wave_dir REAL, src TEXT)"); c.commit()

def _fetch(url, params):
    if _get: return _get(url, params)
    r = requests.get(url, params=params, timeout=10); r.raise_for_status(); return r.json()

def _listify(j): return j if isinstance(j, list) else [j]

def refresh(batch=50):
    """Fetch current weather for every port. Returns (rows_updated, message)."""
    ensure(); c = LE.conn(); ports = c.execute("SELECT port_name, lat, lon FROM ports").fetchall()
    if not ports: return 0, "no ports"
    n = 0; err = ""; ts = datetime.datetime.utcnow().isoformat(timespec="seconds")
    for i in range(0, len(ports), batch):
        chunk = ports[i:i + batch]; la = ",".join(f"{p[1]:.3f}" for p in chunk); lo = ",".join(f"{p[2]:.3f}" for p in chunk)
        try:
            f = _listify(_fetch(FORECAST, {"latitude": la, "longitude": lo, "current": "temperature_2m,precipitation,weather_code,wind_speed_10m,wind_direction_10m,wind_gusts_10m", "wind_speed_unit": "ms", "timezone": "UTC"}))
        except Exception as e:
            err = f"{type(e).__name__}: {str(e)[:120]}"; break
        try: m = _listify(_fetch(MARINE, {"latitude": la, "longitude": lo, "current": "wave_height,wave_direction", "timezone": "UTC"}))
        except Exception: m = [{}] * len(chunk)
        for p, a, b in zip(chunk, f, m + [{}] * (len(chunk) - len(m))):
            cur = (a or {}).get("current") or {}
            if cur.get("wind_speed_10m") is None: continue
            mc = (b or {}).get("current") or {}; wave = mc.get("wave_height")
            c.execute("INSERT OR REPLACE INTO wx_live VALUES(?,?,?,?,?,?,?,?,?,?,?)", (p[0], ts, cur.get("wind_speed_10m"), cur.get("wind_gusts_10m"), cur.get("wind_direction_10m"), cur.get("temperature_2m"), cur.get("precipitation") or 0.0, cur.get("weather_code"), wave, mc.get("wave_direction"), "Open-Meteo"))
            wind = float(cur["wind_speed_10m"]); lvl = "High" if wind > 15 else "Medium" if wind >= 8 else "Low"
            c.execute("INSERT OR REPLACE INTO weather_risks(port_name,current_severity,forecast,wind_speed,wave_height,temperature) VALUES(?,?,?,?,?,?)", (p[0], lvl, f"Open-Meteo live {ts[11:16]} UTC · {WMO.get(cur.get('weather_code'), '')}", wind * 3.6, float(wave) if wave is not None else 1.2, cur.get("temperature_2m")))
            n += 1
    c.commit(); LE.set_state("wx_status", {"ts": ts if n else (LE.get_state("wx_status", {}) or {}).get("ts"), "ok": n, "total": len(ports), "err": err, "tried": ts})
    return n, (err or f"{n} ports updated")

def status():
    s = LE.get_state("wx_status", {}) or {}; ok = int(s.get("ok") or 0); age = None
    if s.get("ts"):
        try: age = (datetime.datetime.utcnow() - datetime.datetime.fromisoformat(s["ts"])).total_seconds() / 60
        except Exception: pass
    live = bool(ok) and age is not None and age < 120
    return {"live": live, "ts": s.get("ts"), "age_min": age, "ok": ok, "total": s.get("total"), "err": s.get("err", ""), "tried": s.get("tried")}

def fresh_ports(max_age_min=90):
    """Ports whose real weather is recent: the simulator must not overwrite them."""
    try:
        cut = (datetime.datetime.utcnow() - datetime.timedelta(minutes=max_age_min)).isoformat(timespec="seconds")
        return {r[0] for r in LE.conn().execute("SELECT port_name FROM wx_live WHERE ts>=?", (cut,)).fetchall()}
    except Exception: return set()

_T = {"t": None}
def start(every_min=20):
    if _T["t"]: return
    def loop():
        time.sleep(5)
        while True:
            try: refresh()
            except Exception: pass
            time.sleep(every_min * 60 if status()["live"] else 120)
    _T["t"] = threading.Thread(target=loop, daemon=True); _T["t"].start()

def table():
    import pandas as pd
    ensure()
    return pd.read_sql("SELECT w.*, p.lat, p.lon FROM wx_live w JOIN ports p USING(port_name)", LE.conn())


In [ ]:
%%writefile /content/app/weather_context.py
import requests
from seed_data import BASE_PORTS

# Auto-built from BASE_PORTS in seed_data.py — always in sync with the port list
# (currently 50 ports). Each BASE_PORTS row is:
# (name, country, congestion_index, avg_dwell_days, lat, lon, region)
PORT_COORDS = {p[0]: (p[4], p[5]) for p in BASE_PORTS}

_DEFAULT_COORDS = (1.29, 103.85)  # Singapore Port as a safe fallback

def fetch_port_weather(port_name):
    coords = PORT_COORDS.get(port_name, _DEFAULT_COORDS)
    try:
        url = f"https://api.open-meteo.com/v1/forecast?latitude={coords[0]}&longitude={coords[1]}&current_weather=true"
        r = requests.get(url, timeout=3)
        if r.status_code == 200:
            cw = r.json().get('current_weather', {})
            windspeed = cw.get('windspeed', 15.0)
            severity = 1 if windspeed < 20 else (2 if windspeed < 40 else 3)
            return {'windspeed': windspeed, 'temperature': cw.get('temperature', 22.0), 'severity': severity}
    except Exception:
        pass
    return {'windspeed': 18.5, 'temperature': 24.0, 'severity': 2}


In [ ]:
%%writefile /content/app/worker.py
"""Background worker process: the simulator, the agent cycles, escalation and the feed threads run here so the web server (and the GPU models) never share a GIL with them."""
import os, sys, time, importlib
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import config, live_engine as LE
dom = importlib.import_module("domain_franchise" if config.KIND == "Franchise" else "domain_freight")
if __name__ == "__main__":
    rt = LE.Runtime(dom, int(os.environ.get("LIVE_TICK_SEC", 5)), int(os.environ.get("LIVE_CYCLE_SEC", 60)))
    if config.KIND == "Freight":
        import weather as WX; WX.start()
    while True: time.sleep(3600)


In [ ]:
%%writefile /content/app/server.py
"""Web server: FastAPI JSON API + server-sent events + the single-page front end. Models live on the GPU in this process; the simulator and agent cycles run in worker.py."""
import sqlite3, os, sys, io, re, csv, json, time, asyncio, subprocess, threading, importlib, datetime
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import config
from config import KIND, DB_PATH
from fastapi import FastAPI, Request, Response, HTTPException, UploadFile, File, Form, Header
from fastapi.responses import JSONResponse, StreamingResponse, FileResponse, HTMLResponse, PlainTextResponse
from fastapi.staticfiles import StaticFiles
import pandas as pd
import numpy as np
import gpu, qa as QA, analyst as AN, live_engine as LE, entities as EN, blocks as B
import views_core as VC
import auth_service
import rbac
import mailer
import copilot_engine
dom = importlib.import_module("domain_franchise" if KIND == "Franchise" else "domain_freight")
V = importlib.import_module("views_franchise" if KIND == "Franchise" else "views_freight")
HERE = os.path.dirname(os.path.abspath(__file__)); STATIC = os.path.join(HERE, "static"); STATIC_DIR = STATIC
app = FastAPI(title=f"{V.NAME} Platform", docs_url=None, redoc_url=None, openapi_url=None) if os.environ.get("PUBLIC_DOCS") != "1" else FastAPI(title=f"{V.NAME} Platform"); _worker = None; _ready = {"db": False}
import hardening; hardening.install(app, KIND)

@app.middleware("http")
async def sec(request, call_next):
    r = await call_next(request); r.headers["X-Content-Type-Options"] = "nosniff"; r.headers["Referrer-Policy"] = "same-origin"; r.headers["Cache-Control"] = "no-store" if request.url.path.startswith("/api") else r.headers.get("Cache-Control", "public, max-age=300"); return r

def _boot():
    # The ocean router requires the bundled world coastline, separately from ECharts.
    # Restore either missing asset before any Route page calculation can start.
    vendor_dir = os.path.join(STATIC_DIR, "vendor")
    required_vendor = [os.path.join(vendor_dir, x) for x in ("echarts.min.js", "world.topo.json")]
    if any(not os.path.isfile(x) or os.path.getsize(x) == 0 for x in required_vendor):
        import _vendor_bundle
    missing_vendor = [x for x in required_vendor if not os.path.isfile(x) or os.path.getsize(x) == 0]
    if missing_vendor:
        raise RuntimeError("Missing required bundled assets: " + ", ".join(missing_vendor))
    from db import init_db; from seed_data import seed_all
    init_db(); auth_service.init_sessions(); seed_all(); LE.init_live_tables()
    auth_service.seed_admin_account()
    dom.ensure_schema()
    _ready["db"] = True
@app.on_event("startup")
def startup():
    global _worker
    _boot()
    if os.environ.get("NO_WORKER") != "1":
        env = {**os.environ, "APP_KIND": KIND}; _worker = subprocess.Popen([sys.executable, os.path.join(HERE, "worker.py")], env=env, cwd=HERE)
@app.on_event("shutdown")
def shutdown():
    if _worker:
        try: _worker.terminate()
        except Exception: pass

# ------------------------------------------------------------------ auth + context
def user_of(request: Request):
    auth_header = request.headers.get("Authorization") or ""
    token = auth_header[7:].strip() if auth_header.startswith("Bearer ") else ""
    if not token:
        raise HTTPException(status_code=401, detail="Authentication required. Please sign in.")
    payload = auth_service.decode_token(token)
    if not payload:
        raise HTTPException(status_code=401, detail="Session expired or invalid token. Please sign in again.")
    username = payload.get("sub")
    user = auth_service.get_user(username)
    if not user:
        raise HTTPException(status_code=401, detail="User account not found.")
    if user.get("locked"):
        raise HTTPException(status_code=401, detail="Account locked. Contact administrator.")
    if payload.get("uid") != user["id"] or payload.get("sv") != auth_service.session_version(user["id"]):
        raise HTTPException(401, "Session expired. Sign in again.")
    return user

def ctx(u, params=None):
    return {
        "dom": dom,
        "kind": KIND,
        "role": u["role"],
        "email": u["email"],
        "user": u,
        "customer_id": u.get("customer_id"),
        "scope": None,
        "params": params or {},
    }

def need(c, cap):
    role = c.get("role")
    if not rbac.check_permission(role, cap):
        raise HTTPException(status_code=403, detail=f"Permission denied: role '{role}' lacks '{cap}' privilege.")
    return None

def menu_for(role):
    return rbac.menu_for_role(role)

def ok(msg="Done", **kw): return {"ok": True, "msg": msg, **kw}
def bad(msg, **kw): return JSONResponse({"ok": False, "msg": msg, **kw}, status_code=400)

# ------------------------------------------------------------------ auth routes
@app.post("/api/auth/signup")
def auth_signup(d: dict):
    success, msg = auth_service.signup_user(
        d.get("username"),
        d.get("email"),
        d.get("password"),
        d.get("confirm_password"),
        d.get("question"),
        d.get("answer"),
        d.get("role", "Customer"),
    )
    if not success:
        return bad(msg)
    sent, cooldown, notice = auth_service.request_email_verification(d.get("email"))
    return ok(notice if sent else msg + " " + notice, email_verification_pending=True,
              verification_sent=bool(sent), retry_after_seconds=cooldown)

@app.post("/api/auth/verify-email/request")
def auth_verify_email_request(d: dict):
    success, wait_sec, msg = auth_service.request_email_verification(d.get("email"))
    if not success:
        return JSONResponse({"ok": False, "msg": msg, "retry_after_seconds": wait_sec}, status_code=429 if wait_sec else 400)
    return ok(msg, retry_after_seconds=wait_sec)

@app.post("/api/auth/verify-email/confirm")
def auth_verify_email_confirm(d: dict):
    success, msg = auth_service.verify_email_code(d.get("email"), d.get("code"))
    if not success:
        return bad(msg)
    return ok(msg)

@app.post("/api/auth/login")
def auth_login(d: dict):
    result, retry_after, msg = auth_service.authenticate_user(d.get("identity"), d.get("password"), d.get("role"))
    if not result:
        status = 423 if msg == auth_service.LOCKED_MESSAGE else (429 if retry_after else 401)
        return JSONResponse({"ok": False, "msg": msg, "retry_after_seconds": retry_after}, status_code=status)
    u = result["user"]
    return ok(
        msg,
        token=result["token"],
        user={
            "id": u["id"],
            "username": u["username"],
            "email": u["email"],
            "role": u["role"],
            "customer_id": u.get("customer_id"),
        },
    )

@app.post("/api/auth/forgot")
@app.post("/api/auth/forgot/request_otp")
def auth_forgot_otp(d: dict):
    success, wait_sec, msg = auth_service.request_password_reset_otp(d.get("email"))
    if not success:
        return JSONResponse({"ok": False, "msg": msg, "retry_after_seconds": wait_sec}, status_code=400 if wait_sec == 0 else 429)
    return ok(msg, retry_after_seconds=wait_sec)

@app.post("/api/auth/forgot/verify_otp")
def auth_forgot_verify_otp(d: dict):
    success, grant, msg = auth_service.verify_password_reset_otp(d.get("email"), d.get("code"))
    if not success:
        return bad(msg)
    return ok(msg, grant=grant)

@app.post("/api/auth/forgot/question")
def auth_forgot_question(d: dict):
    q, err = auth_service.get_user_security_question(d.get("identity"))
    if not q:
        return bad(err or "Security question unavailable for this account.")
    return ok(err or "Question found", question=q, locked=bool(err))

@app.post("/api/auth/forgot/verify_answer")
def auth_forgot_verify_answer(d: dict):
    success, grant, msg = auth_service.verify_user_security_answer(d.get("identity"), d.get("answer"))
    if not success:
        return bad(msg)
    return ok(msg, grant=grant)

@app.post("/api/auth/reset")
def auth_reset(d: dict):
    success, msg = auth_service.reset_user_password(
        d.get("identity"),
        d.get("grant"),
        d.get("new_password"),
        d.get("confirm_password"),
    )
    if not success:
        return bad(msg)
    return ok(msg)

@app.post("/api/auth/unblock")
def auth_unblock(d: dict, request: Request):
    u = user_of(request)
    if u.get("role") != "Admin":
        raise HTTPException(status_code=403, detail="Administrator authentication required")
    success, msg = auth_service.unblock_user_account(u, d.get("identity"))
    if not success:
        return bad(msg)
    return ok(msg)


# ------------------------------------------------------------------ Admin Dashboard Routes (Stage 4)
@app.get("/api/admin/users")
def api_admin_users(request: Request):
    u = user_of(request); c = ctx(u); need(c, "users")
    return {"ok": True, "users": auth_service.admin_list_users()}

@app.get("/api/admin/metrics")
def api_admin_metrics(request: Request):
    u = user_of(request); c = ctx(u); need(c, "users")
    return {"ok": True, **auth_service.admin_get_metrics()}

@app.get("/api/admin/activity")
def api_admin_activity(request: Request):
    u = user_of(request); c = ctx(u); need(c, "users")
    return {"ok": True, "activity": auth_service.admin_get_activity_log(50)}

@app.post("/api/admin/users/create")
async def api_admin_create_user(request: Request):
    u = user_of(request); c = ctx(u); need(c, "users")
    d = await request.json()
    ok_res, msg = auth_service.admin_create_user(
        u["username"], d.get("username"), d.get("email"), d.get("password"), d.get("role"), d.get("customer_id")
    )
    if not ok_res:
        return bad(msg)
    return ok(msg)

@app.post("/api/admin/users/role")
async def api_admin_change_role(request: Request):
    u = user_of(request); c = ctx(u); need(c, "users")
    d = await request.json()
    ok_res, msg = auth_service.admin_change_role(u["username"], d.get("target_id"), d.get("role"))
    if not ok_res:
        return bad(msg)
    return ok(msg)

@app.post("/api/admin/users/lock")
async def api_admin_lock_user(request: Request):
    u = user_of(request); c = ctx(u); need(c, "users")
    d = await request.json()
    ok_res, msg = auth_service.admin_set_lock(u["username"], d.get("target_id"), True)
    if not ok_res:
        return bad(msg)
    return ok(msg)

@app.post("/api/admin/users/unlock")
async def api_admin_unlock_user(request: Request):
    u = user_of(request); c = ctx(u); need(c, "users")
    d = await request.json()
    ok_res, msg = auth_service.admin_set_lock(u["username"], d.get("target_id"), False)
    if not ok_res:
        return bad(msg)
    return ok(msg)

@app.post("/api/admin/users/delete")
async def api_admin_delete_user(request: Request):
    u = user_of(request); c = ctx(u); need(c, "users")
    d = await request.json()
    ok_res, msg = auth_service.admin_delete_user(u["username"], d.get("target_id"))
    if not ok_res:
        return bad(msg)
    return ok(msg)

@app.post("/api/admin/link_customer")
async def api_admin_link_customer(request: Request):
    u = user_of(request); c = ctx(u); need(c, "users")  # Enforces admin RBAC
    d = await request.json()

    # Accepts a plain id or the "ID - Name" dropdown format
    try:
        uid = int(str(d.get("user_id", "0")).split(" - ")[0])
    except ValueError:
        return bad("Invalid user selected.")
    cid = str(d.get("customer_id", "")).split(" - ")[0]

    success, msg = auth_service.admin_link_customer(u["username"], uid, cid)
    if not success:
        return bad(msg)
    return ok(msg, refresh=True)

# ------------------------------------------------------------------ Profile routes (Stage 3)
@app.post("/api/profile/username")
async def profile_change_username(request: Request):
    u = user_of(request)
    d = await request.json()
    ok_res, msg, data = auth_service.change_username(u["id"], d.get("new_username"), d.get("current_password"))
    if not ok_res:
        return bad(msg)
    return ok(msg, token=data["token"], user=data["user"])

@app.post("/api/profile/password")
async def profile_change_password(request: Request):
    u = user_of(request)
    d = await request.json()
    ok_res, msg = auth_service.change_password(u["id"], d.get("current_password"), d.get("new_password"), d.get("confirm_password"))
    if not ok_res:
        return bad(msg)
    return ok(msg)


@app.get("/api/public")
def public():
    s = LE.get_state("boot", {}) if _ready["db"] else {}
    return {
        "kind": KIND,
        "name": V.NAME,
        "stage": (s or {}).get("stage", "starting"),
        "gpu": {"cuda": False, "name": None},
        "questions": QUESTIONS if "QUESTIONS" in globals() else auth_service.QUESTIONS,
        "email_enabled": mailer.configured(),
        "roles": auth_service.ALLOWED_SIGNUP_ROLES,
        "demo": [],
    }

@app.get("/api/me")
def me(request: Request):
    u = user_of(request)
    menu_items = menu_for(u["role"])
    caps = list(rbac.ROLE_CAPS.get(u["role"], set()))
    return {
        "ok": True,
        "user": {
            "id": u["id"],
            "username": u["username"],
            "email": u["email"],
            "role": u["role"],
            "customer_id": u.get("customer_id"),
            "created_at": u.get("created_at"),
            "last_login": u.get("last_login"),
        },
        "kind": KIND,
        "name": V.NAME,
        "menu": menu_items,
        "caps": caps,
        "scope": None,
        "langs": ["English"],
        "questions": [],
    }
# ------------------------------------------------------------------ live pulse
def pulse():
    rt = LE.get_state("runtime", {}) or {}; boot = LE.get_state("boot", {}) or {}; d = VC.decision()
    oa = B.Q("SELECT status, severity, COUNT(*) n FROM agent_actions WHERE status IN ('open','proposed') GROUP BY status, severity")
    n = lambda st, sv=None: int(oa[(oa.status == st) & ((oa.severity == sv) if sv else True)].n.sum()) if len(oa) else 0
    return {"stage": boot.get("stage", "starting"), "ticks": rt.get("ticks", 0), "cycles": rt.get("cycles", 0), "last_cycle": rt.get("last_cycle"), "health": d["score"] if d else None, "risk": d["risk"] if d else None, "dec": d["id"] if d else 0,
            "open": n("open"), "critical": n("open", "Critical"), "proposed": 0, "gpu": {"cuda": True, "name": "", "used": 0, "total": 0, "llm": "off", "llm_name": "", "llm_done": 0, "llm_total": 0}, "worker_alive": bool(rt.get("last_cycle"))}
@app.get("/api/pulse")
def pulse_http(request: Request): user_of(request); return pulse()
@app.get("/api/stream")
async def stream(request: Request):
    user_of(request)
    async def gen():
        last = None; t0 = time.time()
        while True:
            if await request.is_disconnected(): break
            try: p = await asyncio.to_thread(pulse)
            except Exception as e: p = {"error": str(e)}
            k = json.dumps(p, sort_keys=True, default=str)
            if k != last or time.time() - t0 > 10: last = k; t0 = time.time(); yield f"data: {k}\n\n"
            await asyncio.sleep(1.5)
    return StreamingResponse(gen(), media_type="text/event-stream", headers={"Cache-Control": "no-cache", "X-Accel-Buffering": "no"})

# ------------------------------------------------------------------ pages
_TABS = {}
def _tabs_of(c, f):
    if f not in _TABS: _TABS[f] = f(c, None).get("tabs")
    return _TABS[f]
def _page(c, pid, tab):
    if pid == "copilot":
        examples = copilot_engine.get_role_suggestions(c.get("role", "Customer"))
        return B.page("AI Copilot", "Ask about your data in plain English. Answers are exact database queries.", [{"t": "chat", "examples": examples, "langs": ["English"]}], [], live=0)
    if pid == "profile":
        u = c.get("user") or {}
        return {
            "title": "My Profile",
            "sub": "Manage account credentials and security settings",
            "tabs": None,
            "actions": [],
            "blocks": [
                {"t": "profile_page", "user": {
                    "username": u.get("username"),
                    "email": u.get("email"),
                    "role": u.get("role"),
                    "created_at": u.get("created_at"),
                    "last_login": u.get("last_login") or "First login session",
                    "customer_id": u.get("customer_id") or "None"
                }}
            ],
            "live": 0
        }
    if pid == "admin":
        if c.get("role") != "Admin":
            raise HTTPException(403, "Access denied: Admin role required.")
        return {
            "title": "Admin Dashboard",
            "sub": "Platform user governance, access control, and audit logs",
            "tabs": None,
            "actions": [],
            "blocks": [
                {
                    "t": "admin_page",
                    "metrics": auth_service.admin_get_metrics(),
                    "users": auth_service.admin_list_users(),
                    "activity": auth_service.admin_get_activity_log(50),
                    "customers": B.Q("SELECT customer_id, name FROM customers ORDER BY customer_id").to_dict("records"),
                    "current_admin": c.get("user", {}).get("username")
                }
            ],
            "live": 0
        }
    f = V.PAGES.get(pid)
    if not f: raise HTTPException(404, "unknown page")
    o = f(c, tab); _TABS[f] = o.get("tabs"); return o
@app.get("/api/page/{pid}")
def page(pid: str, request: Request, tab: str = "", p: str = ""):
    u = user_of(request)
    if pid not in rbac.ROLE_MENU.get(u["role"], set()):
        raise HTTPException(403, f"Access denied: '{pid}' page is not accessible for role '{u['role']}'")
    c = ctx(u, json.loads(p) if p else {})
    t = time.time(); out = _page(c, pid, tab or None); out["id"] = pid; out["ms"] = round((time.time() - t) * 1000); return JSONResponse(B.clean(out))
@app.get("/api/detail/{eid:path}")
def detail(eid: str, request: Request):
    u = user_of(request)
    if not rbac.can_access_entity(u["role"], eid, u.get("customer_id")):
        raise HTTPException(403, f"Access denied: role '{u['role']}' is not authorized to view entity '{eid}'.")
    c = ctx(u)
    return JSONResponse(B.clean(V.detail(c, eid)))

# ------------------------------------------------------------------ tickets, cycles, policy
@app.post("/api/run_cycle")
def run_cycle(request: Request):
    u = user_of(request); c = ctx(u); need(c, "run"); need(c, "users"); t = time.time(); LE.run_cycle(dom, "manual", ""); LE.audit(u["email"], "manual_cycle", "all"); return ok(f"Agents finished in {time.time() - t:.1f}s")
@app.post("/api/run_agent")
async def run_agent(request: Request):
    u = user_of(request); c = ctx(u); need(c, "run"); a = (await request.json()).get("agent")
    if u["role"] == "Operations Manager" and a not in {"route_watch", "weather_watch", "customs_watch"}:
        raise HTTPException(403, "This agent is outside your role permissions.")
    if a not in dom.AGENTS: raise HTTPException(404, "unknown agent")
    await asyncio.to_thread(_run_one, a); LE.audit(u["email"], "manual_agent", a); return ok(f"{a} re-ran")
def _run_one(agent):
    cid = time.strftime("%m%d-%H%M%S") + "-page"; t = time.time(); s = {"cycle_id": cid, "trigger": "page", "request": "", "queue": [], "results": {}}
    if agent == "digest":
        for a in ("outlet", "inventory", "staff", "marketing", "audit"): s["results"][a] = dom.AGENTS[a](s)
    res = dom.AGENTS[agent](s); LE._log_run(cid, "page", res, int((time.time() - t) * 1000)); s["results"][agent] = res; LE.act(dom, {"actions": res.actions}, cid, {agent: res})

# ------------------------------------------------------------------ admin users

# ------------------------------------------------------------------ digital twin

# ------------------------------------------------------------------ data feeds

# ------------------------------------------------------------------ Franchise actions
@app.post("/api/franchise/review")
async def fr_review(request: Request):
    import uuid; u = user_of(request); c = ctx(u); d = await request.json(); tx = str(d.get("comment", "")).strip()
    if not tx: return bad("Write a comment first")
    src = "eng_Latn"; en = re.sub(r"[\w.+-]+@[\w-]+\.[\w.]+|\+?\d[\d -]{8,}\d", "[removed]", tx)
    try: from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer; sc = SentimentIntensityAnalyzer().polarity_scores(en)["compound"]
    except Exception: sc = (int(d["rating"]) - 3) / 2
    rt = int(d["rating"])
    with LE._DB_LOCK, LE.conn() as cn: cn.execute("INSERT INTO feedback(feedback_id,outlet_id,rating,comment,date,sentiment_score) VALUES(?,?,?,?,?,?)", ("FB-U" + uuid.uuid4().hex[:10], d["outlet"], rt, en, datetime.date.today().isoformat(), .6 * (rt - 3) / 2 + .4 * sc))
    LE.emit("review", d["outlet"], {"rating": rt}); return ok(f"Stored for {d['outlet']} (sentiment {sc:+.2f}{', translated from ' + src if src != 'eng_Latn' else ''}). The Sentiment agent includes it in its next cycle.", refresh=True)
@app.post("/api/franchise/audit")
async def fr_audit(request: Request):
    import uuid; u = user_of(request); c = ctx(u); need(c, "resolve"); d = await request.json(); sc = int(d["score"])
    with LE._DB_LOCK, LE.conn() as cn: cn.execute("INSERT INTO audits(audit_id,outlet_id,audit_date,score,violations,category,status,notes) VALUES(?,?,?,?,?,?,?,?)", ("AUD-U" + uuid.uuid4().hex[:10], d["outlet"], datetime.date.today().isoformat(), sc, int((100 - sc) / 6), d["category"], "Pass" if sc >= 80 else "Warning" if sc >= 60 else "Fail", d.get("notes", "")))
    LE.emit("audit", d["outlet"], {"score": sc}); LE.audit(u["email"], "audit_logged", d["outlet"], str(sc)); await asyncio.to_thread(_run_one, "audit"); return ok("Saved and evaluated by the Audit agent.", refresh=True)

# ------------------------------------------------------------------ Freight actions
@app.post("/api/freight/quote")
async def fq_quote(request: Request):
    u = user_of(request); c = ctx(u); need(c, "quote"); d = await request.json()
    if d["o"] == d["d"]: return bad("Origin and destination must differ.")
    rid = dom.submit_request(d["o"], d["d"], d["cargo"], float(d["weight"]), d["hs"], d["customer"], "English", d.get("docs") or []); await asyncio.to_thread(dom.run_quote, rid); LE.audit(u["email"], "quote_created", str(rid), f"{d['o']}->{d['d']}")
    return ok("Quote generated", blocks=B.clean(V.quote_result(rid)))
@app.post("/api/freight/storm")
async def fq_storm(request: Request):
    PRESET_STORMS = {"Red Sea": (18.0, 40.0), "Arabian Sea": (16.0, 66.0), "Bay of Bengal": (14.0, 88.0), "South China Sea": (14.0, 114.0), "Mediterranean": (35.0, 18.0)}; u = user_of(request); c = ctx(u); need(c, "storm"); d = await request.json() if request.headers.get("content-length", "0") not in ("0", "2") else {}
    la, lo = PRESET_STORMS.get(d.get("area") or "Red Sea"); dom.inject_storm(la, lo, wind=float(d.get("wind", 24))); LE.audit(u["email"], "inject_storm", d.get("area", "Red Sea")); return ok("Storm injected. The weather agent will propose re-routes on its next cycle.", refresh=True)
@app.post("/api/freight/reroute")
async def fq_reroute(request: Request):
    u = user_of(request); c = ctx(u); need(c, "reroute"); j = await request.json(); sid = j["shipment"]; opt = j.get("opt")
    idx = int(str(opt).split(".")[0]) - 1 if opt else None; act = {"type": "reroute", "target": sid}
    if idx is not None: act["opt_label"] = str(opt).split(" · ", 1)[-1]; act["opt_idx"] = idx
    with LE._DB_LOCK, LE.conn() as cn: note = dom.apply(act, cn)
    LE.audit(u["email"], "manual_reroute", sid, note); return ok(note or "no change", refresh=True)
@app.post("/api/freight/attach")
async def fq_attach(request: Request):
    u = user_of(request); c = ctx(u); need(c, "customs_edit")
    d = await request.json(); dom.attach_doc(d["ref"], d["doc"], u["email"]); LE.audit(u["email"], "doc_attached", d["ref"], d["doc"]); return ok("Attached. Quotes re-run immediately; shipments clear on the next customs cycle.", refresh=True)
@app.post("/api/freight/readdoc")
async def fq_readdoc(request: Request, file: UploadFile = File(...)):
    u = user_of(request); c = ctx(u); need(c, "customs_edit"); data = await file.read(); txt = ""
    try:
        if file.filename.lower().endswith(".pdf"):
            import pdfplumber
            with pdfplumber.open(io.BytesIO(data)) as pdf: txt = "\n".join((pg.extract_text() or "") for pg in pdf.pages)
        else: txt = data.decode("utf-8", errors="ignore")
    except Exception as e: return bad(f"Could not read the file: {e}")
    hs = re.findall(r"HS[- ]?\d{2,8}", txt); q = re.findall(r"QTE-L\d+", txt); wt = re.findall(r"(\d+(?:\.\d+)?)\s*(?:t|tons|kg)\b", txt)
    return ok("Read", blocks=[B.table([{"field": "HS code", "value": ", ".join(dict.fromkeys(hs)) or "-"}, {"field": "Quote reference", "value": ", ".join(dict.fromkeys(q)) or "-"}, {"field": "Weights", "value": ", ".join(wt[:3]) or "-"}], ["field", "value"], title="Fields found", h=170, key="ocr"), B.md("```\n" + txt[:2500] + "\n```")])

# ------------------------------------------------------------------ language, knowledge base

# ------------------------------------------------------------------ Copilot (agents first, then the GPU LLM; NLLB for every language) + executive brief
def _sse(ev, **kw): return f"data: {json.dumps({'ev': ev, **kw}, default=str)}\n\n"
SYSTEM = "You are the operations copilot of an agentic platform. Answer ONLY from the EVIDENCE. Quote numbers exactly as given, never invent any. Be concise: a one-line answer, then up to 5 short bullets with the entity ids. If the evidence does not answer, say so. When you use a numbered source, cite it as [1], [2]."
_BRIEF = {}
def _chat_db():
    c = sqlite3.connect(DB_PATH, timeout=30); c.execute("PRAGMA journal_mode=WAL;"); c.execute("CREATE TABLE IF NOT EXISTS copilot_chats(id TEXT PRIMARY KEY, email TEXT, title TEXT, ts TEXT, data TEXT)"); c.execute("CREATE INDEX IF NOT EXISTS ix_chat_email ON copilot_chats(email, ts)"); return c
@app.get("/api/chats")
async def chats_list(request: Request):
    u = user_of(request)
    with _chat_db() as c: rows = c.execute("SELECT id, title, ts FROM copilot_chats WHERE email=? ORDER BY ts DESC LIMIT 60", (u["email"],)).fetchall()
    return {"ok": True, "chats": [{"id": r[0], "title": r[1], "ts": r[2]} for r in rows]}
@app.get("/api/chats/{cid}")
async def chats_get(cid: str, request: Request):
    u = user_of(request)
    with _chat_db() as c: r = c.execute("SELECT title, data FROM copilot_chats WHERE id=? AND email=?", (cid, u["email"])).fetchone()
    if not r: raise HTTPException(404, "chat not found")
    return {"ok": True, "title": r[0], "msgs": json.loads(r[1])}
@app.post("/api/chats/save")
async def chats_save(request: Request):
    u = user_of(request); d = await request.json(); cid = re.sub(r"[^a-zA-Z0-9_-]", "", str(d.get("id") or ""))[:40] or ("c" + str(int(time.time() * 1000))); msgs = d.get("msgs") or []
    if not isinstance(msgs, list) or not msgs: return bad("nothing to save")
    msgs = [{"role": m.get("role"), "text": str(m.get("text", ""))[:20000], "en": (str(m["en"])[:20000] if m.get("en") else None), "lang": m.get("lang"), "sql": str(m.get("sql") or "")[:1500]} for m in msgs[-80:]]
    title = next((m["text"] for m in msgs if m["role"] == "user"), "Chat")[:70]; now = datetime.datetime.now().isoformat(timespec="seconds")
    with _chat_db() as c:
        o = c.execute("SELECT email FROM copilot_chats WHERE id=?", (cid,)).fetchone()
        if o and o[0] != u["email"]: raise HTTPException(403, "not your chat")
        c.execute("INSERT OR REPLACE INTO copilot_chats VALUES(?,?,?,?,?)", (cid, u["email"], title, now, json.dumps(msgs)))
    return {"ok": True, "id": cid}
@app.delete("/api/chats/{cid}")
async def chats_del(cid: str, request: Request):
    u = user_of(request)
    with _chat_db() as c: c.execute("DELETE FROM copilot_chats WHERE id=? AND email=?", (cid, u["email"]))
    return {"ok": True}
@app.post("/api/weather/refresh")
async def wx_refresh(request: Request):
    u = user_of(request); c = ctx(u); need(c, "run")
    if KIND != "Freight": raise HTTPException(404)
    import weather as WX; n, msg = await asyncio.to_thread(WX.refresh); LE.audit(u["email"], "weather_refresh", "ports", msg)
    return ok(("Real weather updated: " if n else "Could not reach Open-Meteo, simulated values stay: ") + msg, refresh=True)

# ------------------------------------------------------------------ reports, exports, profile, invitations


@app.get("/api/health")
def health(): return {"ok": True, "gpu": gpu.status()}
@app.get("/")
def index(): return FileResponse(os.path.join(STATIC, "index.html"), headers={"Cache-Control": "no-store"})
app.mount("/static", StaticFiles(directory=STATIC), name="static")


# ------------------------------------------------------------------ AI Copilot (SQL only: greetings + exact database answers)
@app.post("/api/copilot")
async def copilot(request: Request):
    u = user_of(request); c = ctx(u); need(c, "copilot")
    d = await request.json(); q = str(d.get("q", "")).strip()[:800]
    def gen():
        t0 = time.time()
        # Stage 5 Copilot Engine with strict RBAC, Customer data scoping, and parameterized queries
        res = copilot_engine.ask_copilot(q, u.get("role", "Customer"), u.get("customer_id"))
        if res.get("status") != "out_of_scope":
            if res.get("sql"):
                yield _sse("exact", sql=res["sql"], rows=res.get("row_count", 0))
            yield _sse("final", text=res.get("text", ""), sql=res.get("sql", ""), rows=res.get("row_count", 0), ms=res.get("ms", round((time.time() - t0) * 1000)))
            yield _sse("done")
            return

        yield _sse("final", text=res.get("text", ""), sql="", rows=0, ms=round((time.time()-t0)*1000))
        yield _sse("done")
    return StreamingResponse(gen(), media_type="text/event-stream", headers={"Cache-Control": "no-cache", "X-Accel-Buffering": "no"})

In [ ]:
%%writefile /content/app/_vendor_bundle.py
import base64, io, os, tarfile
B64 = (
'...'
)
_dest = '/content/app/static' if os.path.isdir('/content') and os.name != 'nt' else os.path.join(os.path.dirname(os.path.abspath(__file__)), 'static')
os.makedirs(_dest, exist_ok=True)
tarfile.open(fileobj=io.BytesIO(base64.b64decode(B64)), mode='r:gz').extractall(_dest)
print('vendor files ready')


In [ ]:
%%writefile /content/app/static/index.html
<!doctype html>
<html lang="en">
<head>
  <meta charset="utf-8">
  <meta name="viewport" content="width=device-width, initial-scale=1">
  <title>FreightQuote AI</title>
  <link rel="icon" href="data:image/svg+xml,%3Csvg xmlns='http://www.w3.org/2000/svg' viewBox='0 0 32 32'%3E%3Crect width='32' height='32' rx='8' fill='%230284c7'/%3E%3Cpath d='M8 20l5-6 4 4 7-9' stroke='%23fff' stroke-width='3' fill='none' stroke-linecap='round'/%3E%3C/svg%3E">
  <link rel="preconnect" href="https://fonts.googleapis.com">
  <link rel="preconnect" href="https://fonts.gstatic.com" crossorigin>
  <link href="https://fonts.googleapis.com/css2?family=Plus+Jakarta+Sans:wght@400;500;600;700;800&family=JetBrains+Mono:wght@400;500&display=swap" rel="stylesheet">
  <link rel="stylesheet" href="/static/vendor/leaflet.css" onerror="this.onerror=null;this.href='https://unpkg.com/leaflet@1.9.4/dist/leaflet.css'">
  <link rel="stylesheet" href="/static/style.css">
</head>
<body>
  <div id="app"></div>
  <div id="toast"></div>
  <div id="drawer-backdrop" class="drawer-backdrop" onclick="UI.closeDrawer()"></div>
  <div id="drawer">
    <div class="dh">
      <div style="display:flex;align-items:center;gap:10px;">
        <span class="drawer-badge">Inspection</span>
        <b id="dt"></b>
      </div>
      <button class="x" onclick="UI.closeDrawer()" aria-label="Close drawer">✕</button>
    </div>
    <div id="db"></div>
  </div>
  <script src="/static/vendor/echarts.min.js" onerror="this.onerror=null;this.src='https://cdn.jsdelivr.net/npm/echarts@5.4.3/dist/echarts.min.js'"></script>
  <script src="/static/vendor/leaflet.js" onerror="this.onerror=null;this.src='https://unpkg.com/leaflet@1.9.4/dist/leaflet.js'"></script>
  <script src="/static/vendor/topojson-client.min.js" onerror="this.onerror=null;this.src='https://unpkg.com/topojson-client@3.1.0/dist/topojson-client.min.js'"></script>
  <script src="/static/app.js"></script>
</body>
</html>


In [ ]:
%%writefile /content/app/static/landing_exact.html
<div class="fq-exact-page" id="top"><style>
@import url('https://fonts.googleapis.com/css2?family=Source+Sans+3:wght@400;500;600;700&family=Source+Serif+4:ital,wght@0,600;0,700;1,500;1,600&display=swap');

:root {
    --fq-teal: #0F766E;
    --fq-teal-dark: #115E59;
    --fq-teal-light: #1F9690;
    --fq-mint: #BFE6E1;
    --fq-teal-soft: #E6F4F2;
    --fq-ink: #1E293B;
    --fq-body: #475569;
    --fq-muted: #64748B;
    --fq-border: #E2E8F0;
    --fq-bg: #F4F8FB;
    --fq-navy: #071C26;
    --fq-cta-bg: #EAF5F3;
}

html { scroll-behavior: smooth; }

/* Streamlit reset: full width, no sidebar/header, no gaps */
section[data-testid="stSidebar"],
div[data-testid="stSidebarCollapsedControl"],
header[data-testid="stHeader"] { display: none !important; }
.stApp { background: var(--fq-bg) !important; }
.block-container, div[data-testid="stMainBlockContainer"] {
    max-width: 100% !important; padding: 0 !important; margin: 0 !important;
}
div[data-testid="stVerticalBlock"] { gap: 0 !important; }
div[data-testid="stMarkdownContainer"] { margin: 0 !important; }
div[data-testid="stElementContainer"] { margin: 0 !important; }
/* hide the little "link" icon Streamlit adds next to headings */
[data-testid="stHeaderActionElements"], a.stHeadingAction { display: none !important; }

.fq-landing, .fq-landing * { box-sizing: border-box; }
.fq-landing {
    font-family: 'Source Sans 3', -apple-system, 'Segoe UI', sans-serif;
    color: var(--fq-body); line-height: 1.6;
}
.fq-landing a, .fq-landing a:hover { text-decoration: none !important; }
.fq-landing section[id] { scroll-margin-top: 90px; }
.fq-container { max-width: 1180px; margin: 0 auto; padding: 0 24px; }

/* ---------- Header (white bar, one row, melts into the photo) ---------- */
div[data-testid="stHorizontalBlock"]:has(.st-key-top_btn_login) {
    background: #fff;
    border-bottom: none;
    padding: 14px max(24px, calc((100% - 1180px) / 2 + 24px)) !important;
    align-items: center !important;
    gap: 12px !important;
    position: sticky; top: 0; z-index: 999;
    box-shadow: 0 1px 0 rgba(226, 232, 240, 0.7);
}
.fq-brand { display: flex; align-items: center; gap: 12px; color: var(--fq-ink) !important; }
.fq-brand-icon {
    width: 42px; height: 42px; border-radius: 12px;
    background: linear-gradient(135deg, var(--fq-teal-dark), var(--fq-teal-light));
    display: flex; align-items: center; justify-content: center;
    color: #fff; font-size: 14px; font-weight: 700;
    box-shadow: 0 4px 12px rgba(15, 118, 110, 0.30);
}
.fq-brand-text { display: flex; flex-direction: column; line-height: 1.15; }
.fq-brand-name { font-size: 20px; font-weight: 700; letter-spacing: -0.01em; color: var(--fq-ink); }
.fq-brand-sub { font-size: 10.5px; letter-spacing: 0.16em; text-transform: uppercase; color: var(--fq-teal); font-weight: 600; }
.fq-nav-links { display: flex; justify-content: center; align-items: center; gap: 34px; height: 44px; }
.fq-nav-link { color: var(--fq-body) !important; font-size: 15px; font-weight: 500; }
.fq-nav-link:hover { color: var(--fq-teal) !important; }

/* Buttons (header, middle call to action, final call to action) */
.st-key-top_btn_login button, .st-key-mid_btn_login button, .st-key-cta_btn_login button {
    background: linear-gradient(135deg, var(--fq-teal-dark) 0%, var(--fq-teal-light) 100%) !important;
    border: none !important; border-radius: 10px !important;
    height: 44px !important; min-height: 44px !important;
    box-shadow: 0 6px 16px rgba(15, 118, 110, 0.28) !important;
}
.st-key-top_btn_login button p, .st-key-mid_btn_login button p, .st-key-cta_btn_login button p {
    color: #fff !important; font-weight: 600 !important; font-size: 15px !important;
}
.st-key-top_btn_signup button, .st-key-mid_btn_signup button, .st-key-cta_btn_signup button {
    background: #fff !important; border: 1px solid var(--fq-border) !important;
    border-radius: 10px !important; height: 44px !important; min-height: 44px !important;
}
.st-key-top_btn_signup button p, .st-key-mid_btn_signup button p, .st-key-cta_btn_signup button p {
    color: var(--fq-ink) !important; font-weight: 600 !important; font-size: 15px !important;
}
.st-key-top_btn_signup button:hover, .st-key-mid_btn_signup button:hover, .st-key-cta_btn_signup button:hover { border-color: var(--fq-teal) !important; }
.st-key-top_btn_signup button:hover p, .st-key-mid_btn_signup button:hover p, .st-key-cta_btn_signup button:hover p { color: var(--fq-teal) !important; }

/* ---------- Hero: photo fades into the header above and the stats bar below ---------- */
.fq-hero {
    position: relative; width: 100%;
    aspect-ratio: 1600 / 780;
    min-height: 560px; max-height: 820px;
    display: flex; align-items: center;
    background-color: #1B7F88;
    background-size: cover; background-position: 80% center; background-repeat: no-repeat;
    overflow: hidden;
}
.fq-hero-shade {
    position: absolute; inset: 0;
    background: linear-gradient(90deg, rgba(3, 40, 52, 0.62) 0%, rgba(3, 40, 52, 0.32) 34%, rgba(3, 40, 52, 0) 58%);
}
.fq-hero-fade-top {
    position: absolute; left: 0; right: 0; top: 0; height: 130px;
    background: linear-gradient(180deg, #fff 0%, rgba(255,255,255,0.65) 35%, rgba(255,255,255,0) 100%);
}
.fq-hero-fade-bottom {
    position: absolute; left: 0; right: 0; bottom: 0; height: 170px;
    background: linear-gradient(0deg, #fff 0%, rgba(255,255,255,0.7) 35%, rgba(255,255,255,0) 100%);
}
.fq-hero-content { position: relative; z-index: 2; width: 100%; padding-bottom: 36px; }
.fq-hero-text { max-width: 470px; }

.fq-hero-pill {
    display: inline-block; padding: 6px 14px; border-radius: 999px;
    background: rgba(255,255,255,0.18); border: 1px solid rgba(255,255,255,0.45);
    backdrop-filter: blur(6px);
    font-size: 13px; font-weight: 600; color: #fff !important; margin-bottom: 20px;
}
.fq-landing .fq-hero-title {
    font-family: 'Source Serif 4', Georgia, serif;
    font-size: clamp(36px, 3.7vw, 54px); font-weight: 600; line-height: 1.08;
    letter-spacing: -0.025em; margin: 0 0 20px 0;
    color: #ffffff !important; -webkit-text-fill-color: #ffffff !important;
    text-shadow: 0 2px 18px rgba(0, 40, 50, 0.45);
    animation: fqRise 0.9s ease-out both;
}
.fq-landing .fq-hero-title span, .fq-landing .fq-hero-title em {
    color: #ffffff !important; -webkit-text-fill-color: #ffffff !important;
}
.fq-landing .fq-hero-title em { font-style: italic; color: #CFF3EE !important; -webkit-text-fill-color: #CFF3EE !important; }
.fq-landing .fq-hero-desc {
    font-size: 19px; max-width: 440px; margin: 0;
    color: #ffffff !important; -webkit-text-fill-color: #ffffff !important;
    text-shadow: 0 1px 10px rgba(0, 40, 50, 0.5);
}
@keyframes fqRise { from { opacity: 0; transform: translateY(18px); } to { opacity: 1; transform: none; } }
@media (prefers-reduced-motion: reduce) { .fq-landing .fq-hero-title { animation: none; } }

/* Stats bar under the hero: full width, 4 equal centred cells */
.fq-facts { background: #fff; border-bottom: none; }
.fq-facts-grid { display: grid; grid-template-columns: repeat(4, 1fr); max-width: 1180px; margin: 0 auto; }
.fq-fact { padding: 24px 16px 26px; text-align: center; border-left: 1px solid var(--fq-border); }
.fq-fact:first-child { border-left: none; }
.fq-fact-num { font-size: 25px; font-weight: 700; color: var(--fq-teal); line-height: 1.15; }
.fq-fact-label { font-size: 14px; color: var(--fq-muted); margin-top: 4px; }

/* ---------- Sections: no hard lines, every section melts into the next ---------- */
.fq-section { padding: 80px 0; background: transparent; }
.fq-section-tight { padding-bottom: 36px; }
.fq-section-compact { padding: 64px 0; }
.fq-section-alt {
    background: linear-gradient(180deg, var(--fq-bg) 0%, #ffffff 22%, #ffffff 78%, var(--fq-bg) 100%);
    border: none;
}
/* the first section after the white stats bar fades from white into the page colour */
#preview { background: linear-gradient(180deg, #ffffff 0%, var(--fq-bg) 160px); }

.fq-section-header { max-width: 640px; margin: 0 0 40px 0; }
.fq-section-title {
    font-family: 'Source Serif 4', Georgia, serif;
    font-size: 38px; font-weight: 600; color: var(--fq-ink);
    letter-spacing: -0.01em; line-height: 1.2; margin: 0 0 12px 0;
}
.fq-section-subtitle { font-size: 17px; margin: 0; }
.fq-grid-3 { display: grid; grid-template-columns: repeat(3, 1fr); gap: 20px; }
.fq-card {
    background: #fff; border: 1px solid var(--fq-border);
    border-radius: 16px; padding: 28px;
    transition: border-color .2s ease, box-shadow .2s ease, transform .2s ease;
}
.fq-card:hover { border-color: var(--fq-teal-light); box-shadow: 0 10px 28px rgba(15, 118, 110, 0.10); transform: translateY(-2px); }
.fq-icon {
    width: 46px; height: 46px; border-radius: 12px;
    background: var(--fq-teal-soft); color: var(--fq-teal);
    display: flex; align-items: center; justify-content: center; margin-bottom: 18px;
}
.fq-card-title { font-size: 18px; font-weight: 700; color: var(--fq-ink); margin: 0 0 6px 0; }
.fq-card-desc { font-size: 15px; margin: 0; }
.fq-steps { display: grid; grid-template-columns: repeat(4, 1fr); gap: 24px; }
.fq-step { border-top: 3px solid var(--fq-teal); padding-top: 18px; }
.fq-step-num { font-weight: 700; color: var(--fq-teal); font-size: 14px; margin-bottom: 6px; }

/* ---------- Product preview ---------- */
.fq-preview-grid { display: grid; grid-template-columns: 1fr 1fr; gap: 56px; align-items: center; }
.fq-preview-grid .fq-section-header { margin-bottom: 24px; }
.fq-bullets { display: flex; flex-direction: column; gap: 14px; }
.fq-bullet { display: flex; gap: 12px; align-items: flex-start; font-size: 16px; color: var(--fq-body); }
.fq-bullet-tick {
    flex: 0 0 24px; width: 24px; height: 24px; border-radius: 50%;
    background: var(--fq-teal-soft); color: var(--fq-teal);
    display: flex; align-items: center; justify-content: center; margin-top: 1px;
}
.fq-quote {
    background: #fff; border: 1px solid var(--fq-border); border-radius: 18px;
    box-shadow: 0 24px 60px rgba(15, 60, 70, 0.13);
    padding: 22px 24px 18px; width: 100%; max-width: 500px; margin-left: auto;
}
.fq-quote-head { display: flex; justify-content: space-between; align-items: center; margin-bottom: 14px; }
.fq-quote-tag {
    font-size: 12px; font-weight: 700; letter-spacing: 0.08em; text-transform: uppercase;
    color: var(--fq-teal); background: var(--fq-teal-soft); padding: 4px 10px; border-radius: 999px;
}
.fq-quote-status { font-size: 12.5px; color: var(--fq-muted); }
.fq-quote-route { display: flex; align-items: center; gap: 14px; padding: 6px 0 12px; }
.fq-quote-city small { display: block; font-size: 12px; color: var(--fq-muted); }
.fq-quote-city b { font-size: 17px; color: var(--fq-ink); font-weight: 700; }
.fq-quote-city.right { text-align: right; }
.fq-quote-line {
    flex: 1; height: 2px;
    background: repeating-linear-gradient(90deg, var(--fq-teal-light) 0 6px, transparent 6px 12px);
}
.fq-chips { display: flex; flex-wrap: wrap; gap: 8px; margin-bottom: 16px; }
.fq-chip { font-size: 12.5px; background: #F1F5F9; color: var(--fq-body); padding: 4px 10px; border-radius: 999px; }
.fq-quote-block { font-size: 12px; font-weight: 700; letter-spacing: 0.06em; text-transform: uppercase; color: var(--fq-muted); margin: 14px 0 8px; }
.fq-carrier { display: grid; grid-template-columns: 78px 1fr 70px; gap: 10px; align-items: center; font-size: 13.5px; padding: 4px 0; color: var(--fq-ink); }
.fq-carrier span:last-child { text-align: right; color: var(--fq-muted); font-size: 12.5px; }
.fq-bar { height: 7px; background: #E8EEF3; border-radius: 99px; overflow: hidden; }
.fq-bar i { display: block; height: 100%; border-radius: 99px; background: linear-gradient(90deg, var(--fq-teal-dark), var(--fq-teal-light)); }
.fq-price { display: flex; justify-content: space-between; font-size: 14px; padding: 4px 0; color: var(--fq-body); }
.fq-price b { color: var(--fq-ink); font-weight: 600; }
.fq-price-total {
    display: flex; justify-content: space-between; align-items: baseline;
    margin-top: 10px; padding-top: 12px; border-top: 1px solid var(--fq-border);
    font-size: 15px; font-weight: 700; color: var(--fq-ink);
}
.fq-price-total b { font-size: 24px; color: var(--fq-teal); }
.fq-quote-foot { margin-top: 12px; font-size: 12px; color: var(--fq-muted); text-align: center; }

/* ---------- Middle call to action (after How it works) ---------- */
.fq-mid-cta { text-align: center; margin-top: 52px; }
.fq-landing .fq-mid-cta-title {
    font-family: 'Source Serif 4', Georgia, serif; font-size: 26px; font-weight: 600;
    color: var(--fq-ink) !important; -webkit-text-fill-color: var(--fq-ink) !important;
    text-align: center !important; margin: 0 0 6px 0 !important;
}
.fq-landing .fq-mid-cta-desc { font-size: 16px; text-align: center !important; margin: 0 auto !important; max-width: 480px; }
div[data-testid="stHorizontalBlock"]:has(.st-key-mid_btn_login) {
    background: var(--fq-bg);
    padding: 4px 24px 12px 24px !important;
    margin: 0 !important;
    gap: 16px !important;
    align-items: center !important;
    justify-content: center !important;
}

/* ---------- FAQ ---------- */
.fq-faq { max-width: 800px; }
.fq-faq-item { background: #fff; border: 1px solid var(--fq-border); border-radius: 12px; margin-bottom: 12px; overflow: hidden; }
.fq-faq-summary {
    padding: 18px 22px; font-size: 16px; font-weight: 600; color: var(--fq-ink);
    cursor: pointer; list-style: none; display: flex; justify-content: space-between; align-items: center;
}
.fq-faq-summary::-webkit-details-marker { display: none; }
.fq-faq-item[open] .fq-faq-summary svg { transform: rotate(180deg); }
.fq-faq-content { padding: 0 22px 18px 22px; font-size: 15px; }

/* ---------- Final call-to-action: faded ship photo + text + buttons in ONE band ---------- */
.fq-cta {
    position: relative; overflow: hidden;
    padding: 84px 0 36px 0; text-align: center;
    background: linear-gradient(180deg, var(--fq-bg) 0%, var(--fq-cta-bg) 100%);
}
.fq-cta::before {
    content: ""; position: absolute; inset: 0;
    background-size: cover; background-position: 72% 62%; background-repeat: no-repeat;
    opacity: 0.17; filter: saturate(0.85); pointer-events: none;
    -webkit-mask-image: linear-gradient(180deg, transparent 0%, #000 38%, #000 62%, transparent 100%);
    mask-image: linear-gradient(180deg, transparent 0%, #000 38%, #000 62%, transparent 100%);
}
.fq-cta .fq-container { position: relative; z-index: 1; }
.fq-landing .fq-cta-title {
    font-family: 'Source Serif 4', Georgia, serif;
    font-size: 38px; font-weight: 600; line-height: 1.2;
    color: var(--fq-ink) !important; -webkit-text-fill-color: var(--fq-ink) !important;
    text-align: center !important; margin: 0 0 12px 0 !important;
}
.fq-landing .fq-cta-desc {
    font-size: 17px; text-align: center !important;
    max-width: 520px; margin: 0 auto !important; color: var(--fq-body);
}
div[data-testid="stHorizontalBlock"]:has(.st-key-cta_btn_login) {
    background: var(--fq-cta-bg);
    padding: 0 24px 84px 24px !important;
    margin: 0 !important;
    gap: 16px !important;
    align-items: center !important;
    justify-content: center !important;
}
.fq-footer {
    padding: 40px 0 32px; text-align: center; font-size: 14px; color: var(--fq-muted);
    background: linear-gradient(180deg, var(--fq-cta-bg) 0%, #ffffff 100%);
    border: none;
}

/* ---------- Responsive ---------- */
@media (max-width: 1000px) {
    .fq-preview-grid { grid-template-columns: 1fr; gap: 36px; }
    .fq-quote { margin: 0 auto; }
}
@media (max-width: 900px) {
    .fq-nav-links { display: none; }
    .fq-hero { min-height: 520px; background-position: 75% center; }
    .fq-hero-text { max-width: 100%; }
    .fq-facts-grid { grid-template-columns: 1fr 1fr; }
    .fq-fact:nth-child(odd) { border-left: none; }
    .fq-fact:nth-child(-n+2) { border-bottom: 1px solid var(--fq-border); }
    .fq-grid-3 { grid-template-columns: 1fr 1fr; }
    .fq-steps { grid-template-columns: 1fr 1fr; }
}
@media (max-width: 700px) {
    /* header: logo on its own row, the two buttons side by side below it */
    div[data-testid="stHorizontalBlock"]:has(.st-key-top_btn_login) { flex-wrap: wrap !important; position: static; }
    div[data-testid="stHorizontalBlock"]:has(.st-key-top_btn_login) > div:nth-child(1) { flex: 1 1 100% !important; min-width: 100% !important; }
    div[data-testid="stHorizontalBlock"]:has(.st-key-top_btn_login) > div:nth-child(2) { display: none !important; }
    div[data-testid="stHorizontalBlock"]:has(.st-key-top_btn_login) > div:nth-child(n+3) { flex: 1 1 calc(50% - 8px) !important; min-width: calc(50% - 8px) !important; }
    /* call-to-action rows: hide the spacer columns, stack the buttons */
    div[data-testid="stHorizontalBlock"]:has(.st-key-mid_btn_login) > div:nth-child(1),
    div[data-testid="stHorizontalBlock"]:has(.st-key-mid_btn_login) > div:nth-child(4),
    div[data-testid="stHorizontalBlock"]:has(.st-key-cta_btn_login) > div:nth-child(1),
    div[data-testid="stHorizontalBlock"]:has(.st-key-cta_btn_login) > div:nth-child(4) { display: none !important; }
    div[data-testid="stHorizontalBlock"]:has(.st-key-mid_btn_login),
    div[data-testid="stHorizontalBlock"]:has(.st-key-cta_btn_login) { flex-direction: column !important; align-items: center !important; }
    div[data-testid="stHorizontalBlock"]:has(.st-key-mid_btn_login) > div,
    div[data-testid="stHorizontalBlock"]:has(.st-key-cta_btn_login) > div { width: 100% !important; max-width: 340px; flex: 0 0 auto !important; }
    .fq-hero { min-height: 480px; }
    .fq-grid-3, .fq-steps { grid-template-columns: 1fr; }
    .fq-section-title, .fq-landing .fq-cta-title { font-size: 30px; }
    .fq-section { padding: 56px 0; }
    .fq-section-compact { padding: 48px 0; }
    .fq-carrier { grid-template-columns: 70px 1fr 62px; }
}


/* FreightQuote website: restored original typography and photography. */
#app:has(.fq-exact-page){display:block!important;height:auto!important;min-height:100vh;overflow:visible;background:#fff}
.fq-exact-page{width:100%;overflow:hidden;min-height:100vh;background:#fff}
.fq-exact-page .fq-exact-header{position:sticky;top:0;z-index:300;background:#fff;box-shadow:0 1px 0 rgba(226,232,240,.75)}
.fq-exact-page .fq-header-inner{display:grid;grid-template-columns:auto minmax(270px,1fr) auto;align-items:center;gap:34px;padding-top:15px;padding-bottom:15px}
.fq-exact-page .fq-nav-links{display:flex!important;align-items:center;justify-content:center;gap:32px;height:auto!important;white-space:nowrap}
.fq-exact-page .fq-header-inner>.fq-brand{margin:0;min-width:0;display:flex}
.fq-exact-page .fq-head-actions,.fq-exact-page .fq-cta-buttons{display:flex;align-items:center;gap:12px}
.fq-exact-page .fq-head-actions button,.fq-exact-page .fq-cta-buttons button{font-family:'Source Sans 3',sans-serif;cursor:pointer;font-size:14px;font-weight:650;border-radius:10px;padding:10px 22px;min-height:44px;transition:transform .18s,box-shadow .18s}
.fq-exact-page .fq-access-login{background:linear-gradient(120deg,#0f766e,#1c9e94);border:1px solid #0f766e;color:#fff;box-shadow:0 8px 20px #0f766e24}
.fq-exact-page .fq-access-create{background:#fff;border:1px solid #e2e8f0;color:#1e293b}
.fq-exact-page .fq-access-create:hover{border-color:#0f766e;color:#0f766e}
.fq-exact-page .fq-head-actions button:hover,.fq-exact-page .fq-cta-buttons button:hover{transform:translateY(-2px)}
.fq-exact-page .fq-hero{background-image:url('/static/ship.jpg')!important;background-size:cover!important;background-position:80% center!important}
.fq-exact-page .fq-cta::before{background-image:url('/static/ship.jpg')!important;opacity:.12;filter:saturate(.8)}
.fq-exact-page .fq-cta-buttons{justify-content:center;padding:10px 20px 55px;background:#eaf5f3}
.fq-exact-page .fq-cta-buttons button{min-width:174px}
.fq-exact-page .fq-nav-link:hover{color:#0f766e!important}
@media(max-width:980px){.fq-exact-page .fq-header-inner{gap:16px;grid-template-columns:auto auto}.fq-exact-page .fq-nav-links{grid-column:1/-1;grid-row:2;justify-content:center;overflow-x:auto;gap:24px;padding-bottom:4px}.fq-exact-page .fq-head-actions{justify-self:end}.fq-exact-page .fq-hero{background-position:67% center!important}}
@media(max-width:590px){.fq-exact-page .fq-header-inner{padding:12px 14px;gap:10px}.fq-exact-page .fq-brand-name{font-size:16px}.fq-exact-page .fq-brand-sub{font-size:8px}.fq-exact-page .fq-brand-icon{width:34px;height:34px}.fq-exact-page .fq-head-actions button{padding:8px 10px;font-size:12px}.fq-exact-page .fq-nav-links{justify-content:flex-start;width:100%;font-size:13px}.fq-exact-page .fq-hero{background-position:69% center!important;min-height:550px}.fq-exact-page .fq-hero-text{max-width:90%}}
@media(prefers-reduced-motion:reduce){.fq-exact-page *, .fq-exact-page *::before{animation-duration:.01ms!important;scroll-behavior:auto!important}}
</style><header class="fq-landing fq-exact-header"><div class="fq-container fq-header-inner">
<a aria-label="FreightQuote AI home" class="fq-brand" href="#/"><div class="fq-brand-icon">FQ</div><div class="fq-brand-text"><span class="fq-brand-name">FreightQuote AI</span><span class="fq-brand-sub">Freight intelligence</span></div></a>
<nav aria-label="Main navigation" class="fq-nav-links"><a class="fq-nav-link" href="#platform">Platform</a><a class="fq-nav-link" href="#how-it-works">How it works</a><a class="fq-nav-link" href="#security">Security</a><a class="fq-nav-link" href="#faq">FAQ</a></nav>
<div class="fq-head-actions"><button class="fq-access-login" type="button">Log in</button><button class="fq-access-create" type="button">Create account</button></div>
</div></header><div class="fq-landing"><section class="fq-hero"><div class="fq-hero-shade"></div><div class="fq-hero-fade-top"></div><div class="fq-hero-fade-bottom"></div><div class="fq-container fq-hero-content"><div class="fq-hero-text"><div class="fq-hero-pill">Freight intelligence, connected</div><div aria-level="1" class="fq-hero-title" role="heading"><span>A world in motion.</span><br/><em>A smarter way forward.</em></div><div class="fq-hero-desc">Plan ocean routes, compare carriers and build accurate freight quotes in one connected workspace.</div></div></div></section><div class="fq-facts"><div class="fq-facts-grid"><div class="fq-fact"><div class="fq-fact-num">6 AI agents</div><div class="fq-fact-label">Working on every quote</div></div><div class="fq-fact"><div class="fq-fact-num">7 steps</div><div class="fq-fact-label">From request to final quote</div></div><div class="fq-fact"><div class="fq-fact-num">Plain English</div><div class="fq-fact-label">Ask the AI Copilot anything</div></div><div class="fq-fact"><div class="fq-fact-num">Secure sign-in</div><div class="fq-fact-label">Hashed passwords, signed sessions</div></div></div></div></div><div class="fq-landing"><section class="fq-section" id="preview"><div class="fq-container"><div class="fq-preview-grid"><div><div class="fq-section-header"><div aria-level="2" class="fq-section-title" role="heading">See a quote come together</div><div class="fq-section-subtitle">Route, carrier, weather and pricing agents each add their part, and you get one clear breakdown.</div></div><div class="fq-bullets"><div class="fq-bullet"><span class="fq-bullet-tick"><svg fill="none" height="14" stroke="currentColor" stroke-linecap="round" stroke-linejoin="round" stroke-width="3" viewbox="0 0 24 24" width="14"><polyline points="20 6 9 17 4 12"></polyline></svg></span><span>Route and transit time checked automatically</span></div><div class="fq-bullet"><span class="fq-bullet-tick"><svg fill="none" height="14" stroke="currentColor" stroke-linecap="round" stroke-linejoin="round" stroke-width="3" viewbox="0 0 24 24" width="14"><polyline points="20 6 9 17 4 12"></polyline></svg></span><span>Carriers ranked on reliability and on-time performance</span></div><div class="fq-bullet"><span class="fq-bullet-tick"><svg fill="none" height="14" stroke="currentColor" stroke-linecap="round" stroke-linejoin="round" stroke-width="3" viewbox="0 0 24 24" width="14"><polyline points="20 6 9 17 4 12"></polyline></svg></span><span>Fuel adjustment and margin priced in, with compliance notes</span></div></div></div><div><div class="fq-quote"><div class="fq-quote-head"><span class="fq-quote-tag">Sample quote</span><span class="fq-quote-status">Ready to review</span></div><div class="fq-quote-route"><div class="fq-quote-city"><small>From</small><b>Mundra, IN</b></div><div class="fq-quote-line"></div><div class="fq-quote-city right"><small>To</small><b>Rotterdam, NL</b></div></div><div class="fq-chips"><span class="fq-chip">1 x 40 ft container</span><span class="fq-chip">About 22 days</span><span class="fq-chip">Weather risk: low</span></div><div class="fq-quote-block">Carrier ranking</div><div class="fq-carrier"><span>Carrier A</span><div class="fq-bar"><i style="width:94%"></i></div><span>94% on time</span></div><div class="fq-carrier"><span>Carrier B</span><div class="fq-bar"><i style="width:89%"></i></div><span>89% on time</span></div><div class="fq-carrier"><span>Carrier C</span><div class="fq-bar"><i style="width:83%"></i></div><span>83% on time</span></div><div class="fq-quote-block">Price breakdown</div><div class="fq-price"><span>Ocean freight</span><b>$1,850</b></div><div class="fq-price"><span>Fuel (bunker) adjustment</span><b>$210</b></div><div class="fq-price"><span>Port and documentation</span><b>$180</b></div><div class="fq-price"><span>Margin (6%)</span><b>$134</b></div><div class="fq-price-total"><span>Total</span><b>$2,374</b></div><div class="fq-quote-foot">Illustrative figures for demonstration only.</div></div></div></div></div></section></div><div class="fq-landing"><section class="fq-section fq-section-alt" id="platform"><div class="fq-container"><div class="fq-section-header"><div aria-level="2" class="fq-section-title" role="heading">Everything you need to quote freight</div><div class="fq-section-subtitle">Specialised agents work together so every quote is fast and consistent.</div></div><div class="fq-grid-3"><div class="fq-card"><div class="fq-icon"><svg fill="none" height="22" stroke="currentColor" stroke-linecap="round" stroke-linejoin="round" stroke-width="2" viewbox="0 0 24 24" width="22"><path d="M21 15a2 2 0 0 1-2 2H7l-4 4V5a2 2 0 0 1 2-2h14a2 2 0 0 1 2 2z"></path></svg></div><div class="fq-card-title">AI Copilot</div><div class="fq-card-desc">Ask freight questions in plain language and get answers from your data.</div></div><div class="fq-card"><div class="fq-icon"><svg fill="none" height="22" stroke="currentColor" stroke-linecap="round" stroke-linejoin="round" stroke-width="2" viewbox="0 0 24 24" width="22"><circle cx="12" cy="12" r="10"></circle><polygon points="16.24 7.76 14.12 14.12 7.76 16.24 9.88 9.88 16.24 7.76"></polygon></svg></div><div class="fq-card-title">Route AI</div><div class="fq-card-desc">Plan sea routes with distance and transit time estimates.</div></div><div class="fq-card"><div class="fq-icon"><svg fill="none" height="22" stroke="currentColor" stroke-linecap="round" stroke-linejoin="round" stroke-width="2" viewbox="0 0 24 24" width="22"><line x1="12" x2="12" y1="1" y2="23"></line><path d="M17 5H9.5a3.5 3.5 0 0 0 0 7h5a3.5 3.5 0 0 1 0 7H6"></path></svg></div><div class="fq-card-title">Spot Quotes</div><div class="fq-card-desc">Track spot rates and bunker adjustments across trade lanes.</div></div><div class="fq-card"><div class="fq-icon"><svg fill="none" height="22" stroke="currentColor" stroke-linecap="round" stroke-linejoin="round" stroke-width="2" viewbox="0 0 24 24" width="22"><path d="M3 9l9-7 9 7v11a2 2 0 0 1-2 2H5a2 2 0 0 1-2-2z"></path><polyline points="9 22 9 12 15 12 15 22"></polyline></svg></div><div class="fq-card-title">Carrier Ranking</div><div class="fq-card-desc">Compare carriers on reliability and on-time performance.</div></div><div class="fq-card"><div class="fq-icon"><svg fill="none" height="22" stroke="currentColor" stroke-linecap="round" stroke-linejoin="round" stroke-width="2" viewbox="0 0 24 24" width="22"><path d="M17.5 19H9a7 7 0 1 1 6.71-9h1.79a4.5 4.5 0 1 1 0 9Z"></path></svg></div><div class="fq-card-title">Weather Risk</div><div class="fq-card-desc">See storm and seasonal risks along every route.</div></div><div class="fq-card"><div class="fq-icon"><svg fill="none" height="22" stroke="currentColor" stroke-linecap="round" stroke-linejoin="round" stroke-width="2" viewbox="0 0 24 24" width="22"><path d="M14 2H6a2 2 0 0 0-2 2v16a2 2 0 0 0 2 2h12a2 2 0 0 0 2-2V8z"></path><polyline points="14 2 14 8 20 8"></polyline><line x1="16" x2="8" y1="13" y2="13"></line><line x1="16" x2="8" y1="17" y2="17"></line></svg></div><div class="fq-card-title">Margin and Customs</div><div class="fq-card-desc">Apply margins and check customs requirements before you quote.</div></div></div></div></section></div><div class="fq-landing"><section class="fq-section fq-section-tight" id="how-it-works"><div class="fq-container"><div class="fq-section-header"><div aria-level="2" class="fq-section-title" role="heading">From request to quote in four steps</div></div><div class="fq-steps"><div class="fq-step"><div class="fq-step-num">Step 1</div><div class="fq-card-title">Submit your request</div><div class="fq-card-desc">Enter ports, container type and departure date.</div></div><div class="fq-step"><div class="fq-step-num">Step 2</div><div class="fq-card-title">Agents assess the route</div><div class="fq-card-desc">Route, carrier and weather agents review your shipment.</div></div><div class="fq-step"><div class="fq-step-num">Step 3</div><div class="fq-card-title">Pricing is calculated</div><div class="fq-card-desc">Freight, fuel adjustment and margin are priced together.</div></div><div class="fq-step"><div class="fq-step-num">Step 4</div><div class="fq-card-title">Review the quote</div><div class="fq-card-desc">See the full breakdown and compliance notes.</div></div></div><div class="fq-mid-cta"><div aria-level="3" class="fq-mid-cta-title" role="heading">Try it on your own shipment</div><div class="fq-mid-cta-desc">Create an account in a minute and build your first quote.</div></div></div></section></div><div class="fq-landing fq-cta-buttons"><button class="fq-access-login" type="button">Log in</button><button class="fq-access-create" type="button">Create account</button></div><div class="fq-landing"><section class="fq-section fq-section-alt" id="who-for"><div class="fq-container"><div class="fq-section-header"><div aria-level="2" class="fq-section-title" role="heading">Built for people who move cargo</div><div class="fq-section-subtitle">Whether you quote, ship or analyse freight, the workspace fits the way you work.</div></div><div class="fq-grid-3"><div class="fq-card"><div class="fq-icon"><svg fill="none" height="22" stroke="currentColor" stroke-linecap="round" stroke-linejoin="round" stroke-width="2" viewbox="0 0 24 24" width="22"><path d="M17 21v-2a4 4 0 0 0-4-4H5a4 4 0 0 0-4 4v2"></path><circle cx="9" cy="7" r="4"></circle><path d="M23 21v-2a4 4 0 0 0-3-3.87"></path><path d="M16 3.13a4 4 0 0 1 0 7.75"></path></svg></div><div class="fq-card-title">Freight forwarders</div><div class="fq-card-desc">Turn customer enquiries into accurate quotes faster, with carrier comparison and margin built in.</div></div><div class="fq-card"><div class="fq-icon"><svg fill="none" height="22" stroke="currentColor" stroke-linecap="round" stroke-linejoin="round" stroke-width="2" viewbox="0 0 24 24" width="22"><path d="M21 16V8a2 2 0 0 0-1-1.73l-7-4a2 2 0 0 0-2 0l-7 4A2 2 0 0 0 3 8v8a2 2 0 0 0 1 1.73l7 4a2 2 0 0 0 2 0l7-4A2 2 0 0 0 21 16z"></path><polyline points="3.27 6.96 12 12.01 20.73 6.96"></polyline><line x1="12" x2="12" y1="22.08" y2="12"></line></svg></div><div class="fq-card-title">Shippers and exporters</div><div class="fq-card-desc">Check routes, transit times and likely costs before you commit to a booking.</div></div><div class="fq-card"><div class="fq-icon"><svg fill="none" height="22" stroke="currentColor" stroke-linecap="round" stroke-linejoin="round" stroke-width="2" viewbox="0 0 24 24" width="22"><line x1="18" x2="18" y1="20" y2="10"></line><line x1="12" x2="12" y1="20" y2="4"></line><line x1="6" x2="6" y1="20" y2="14"></line></svg></div><div class="fq-card-title">Logistics analysts</div><div class="fq-card-desc">Ask the AI Copilot about rates and trade lanes in plain language and get answers from your data.</div></div></div></div></section></div><div class="fq-landing"><section class="fq-section fq-section-compact" id="security"><div class="fq-container"><div class="fq-section-header"><div aria-level="2" class="fq-section-title" role="heading">Your account stays protected</div><div class="fq-section-subtitle">Sign-in security is built in from the start.</div></div><div class="fq-grid-3"><div class="fq-card"><div class="fq-icon"><svg fill="none" height="22" stroke="currentColor" stroke-linecap="round" stroke-linejoin="round" stroke-width="2" viewbox="0 0 24 24" width="22"><rect height="11" rx="2" ry="2" width="18" x="3" y="11"></rect><path d="M7 11V7a5 5 0 0 1 10 0v4"></path></svg></div><div class="fq-card-title">Encrypted passwords</div><div class="fq-card-desc">Passwords are hashed with bcrypt and never stored in plain text.</div></div><div class="fq-card"><div class="fq-icon"><svg fill="none" height="22" stroke="currentColor" stroke-linecap="round" stroke-linejoin="round" stroke-width="2" viewbox="0 0 24 24" width="22"><path d="M12 22s8-4 8-10V5l-8-3-8 3v7c0 6 8 10 8 10z"></path></svg></div><div class="fq-card-title">Secure sessions</div><div class="fq-card-desc">Signed JWT tokens keep you signed in safely.</div></div><div class="fq-card"><div class="fq-icon"><svg fill="none" height="22" stroke="currentColor" stroke-linecap="round" stroke-linejoin="round" stroke-width="2" viewbox="0 0 24 24" width="22"><path d="M4 4h16c1.1 0 2 .9 2 2v12c0 1.1-.9 2-2 2H4c-1.1 0-2-.9-2-2V6c0-1.1.9-2 2-2z"></path><polyline points="22,6 12,13 2,6"></polyline></svg></div><div class="fq-card-title">Email verification</div><div class="fq-card-desc">Six-digit codes confirm your email and help you recover access.</div></div></div></div></section></div><div class="fq-landing"><section class="fq-section fq-section-alt" id="faq"><div class="fq-container"><div class="fq-section-header"><div aria-level="2" class="fq-section-title" role="heading">Frequently asked questions</div></div><div class="fq-faq"><details class="fq-faq-item"><summary class="fq-faq-summary"><span>What is FreightQuote AI?</span><svg fill="none" height="18" stroke="currentColor" stroke-width="2" viewbox="0 0 24 24" width="18"><polyline points="6 9 12 15 18 9"></polyline></svg></summary><div class="fq-faq-content">A freight workspace where AI agents plan routes, compare carriers and price ocean freight quotes.</div></details><details class="fq-faq-item"><summary class="fq-faq-summary"><span>Do I need to enter payment details?</span><svg fill="none" height="18" stroke="currentColor" stroke-width="2" viewbox="0 0 24 24" width="18"><polyline points="6 9 12 15 18 9"></polyline></svg></summary><div class="fq-faq-content">No. This is a demonstration project and no payment details are collected.</div></details><details class="fq-faq-item"><summary class="fq-faq-summary"><span>What is the difference between User and Admin accounts?</span><svg fill="none" height="18" stroke="currentColor" stroke-width="2" viewbox="0 0 24 24" width="18"><polyline points="6 9 12 15 18 9"></polyline></svg></summary><div class="fq-faq-content">Users work with quotes and the Copilot. Admins can also manage registered users and unlock accounts.</div></details><details class="fq-faq-item"><summary class="fq-faq-summary"><span>How do I recover my password?</span><svg fill="none" height="18" stroke="currentColor" stroke-width="2" viewbox="0 0 24 24" width="18"><polyline points="6 9 12 15 18 9"></polyline></svg></summary><div class="fq-faq-content">Answer your security question or request a six-digit code sent to your registered email.</div></details></div></div></section></div><div class="fq-landing"><section class="fq-cta"><div class="fq-container"><div aria-level="2" class="fq-cta-title" role="heading">Ready to get started?</div><div class="fq-cta-desc">Log in to your workspace or create an account in a minute.</div></div></section></div><div class="fq-landing fq-cta-buttons"><button class="fq-access-login" type="button">Log in</button><button class="fq-access-create" type="button">Create account</button></div><div class="fq-landing"><footer class="fq-footer"><div class="fq-container">FreightQuote AI · Infosys Springboard Virtual Internship 7.0, Team 02</div></footer></div></div>


In [ ]:
%%writefile /content/app/static/style.css
/* Streamlit reset: full width, no sidebar/header, no gaps */
section[data-testid="stSidebar"],
div[data-testid="stSidebarCollapsedControl"],
header[data-testid="stHeader"] { display: none !important; }
.stApp { background: var(--fq-bg) !important; }
.block-container, div[data-testid="stMainBlockContainer"] {
    max-width: 100% !important; padding: 0 !important; margin: 0 !important;
}
div[data-testid="stVerticalBlock"] { gap: 0 !important; }
div[data-testid="stMarkdownContainer"] { margin: 0 !important; }
div[data-testid="stElementContainer"] { margin: 0 !important; }
/* hide the little "link" icon Streamlit adds next to headings */
[data-testid="stHeaderActionElements"], a.stHeadingAction { display: none !important; }

.fq-landing, .fq-landing * { box-sizing: border-box; }
.fq-landing {
    font-family: 'Source Sans 3', -apple-system, 'Segoe UI', sans-serif;
    color: var(--fq-body); line-height: 1.6;
}
.fq-landing a, .fq-landing a:hover { text-decoration: none !important; }
.fq-landing section[id] { scroll-margin-top: 90px; }
.fq-container { max-width: 1180px; margin: 0 auto; padding: 0 24px; }

/* ---------- Header (white bar, one row, melts into the photo) ---------- */
div[data-testid="stHorizontalBlock"]:has(.st-key-top_btn_login) {
    background: #fff;
    border-bottom: none;
    padding: 14px max(24px, calc((100% - 1180px) / 2 + 24px)) !important;
    align-items: center !important;
    gap: 12px !important;
    position: sticky; top: 0; z-index: 999;
    box-shadow: 0 1px 0 rgba(226, 232, 240, 0.7);
}
.fq-brand { display: flex; align-items: center; gap: 12px; color: var(--fq-ink) !important; }
.fq-brand-icon {
    width: 42px; height: 42px; border-radius: 12px;
    background: linear-gradient(135deg, var(--fq-teal-dark), var(--fq-teal-light));
    display: flex; align-items: center; justify-content: center;
    color: #fff; font-size: 14px; font-weight: 700;
    box-shadow: 0 4px 12px rgba(15, 118, 110, 0.30);
}
.fq-brand-text { display: flex; flex-direction: column; line-height: 1.15; }
.fq-brand-name { font-size: 20px; font-weight: 700; letter-spacing: -0.01em; color: var(--fq-ink); }
.fq-brand-sub { font-size: 10.5px; letter-spacing: 0.16em; text-transform: uppercase; color: var(--fq-teal); font-weight: 600; }
.fq-nav-links { display: flex; justify-content: center; align-items: center; gap: 34px; height: 44px; }
.fq-nav-link { color: var(--fq-body) !important; font-size: 15px; font-weight: 500; }
.fq-nav-link:hover { color: var(--fq-teal) !important; }

/* Buttons (header, middle call to action, final call to action) */
.st-key-top_btn_login button, .st-key-mid_btn_login button, .st-key-cta_btn_login button {
    background: linear-gradient(135deg, var(--fq-teal-dark) 0%, var(--fq-teal-light) 100%) !important;
    border: none !important; border-radius: 10px !important;
    height: 44px !important; min-height: 44px !important;
    box-shadow: 0 6px 16px rgba(15, 118, 110, 0.28) !important;
}
.st-key-top_btn_login button p, .st-key-mid_btn_login button p, .st-key-cta_btn_login button p {
    color: #fff !important; font-weight: 600 !important; font-size: 15px !important;
}
.st-key-top_btn_signup button, .st-key-mid_btn_signup button, .st-key-cta_btn_signup button {
    background: #fff !important; border: 1px solid var(--fq-border) !important;
    border-radius: 10px !important; height: 44px !important; min-height: 44px !important;
}
.st-key-top_btn_signup button p, .st-key-mid_btn_signup button p, .st-key-cta_btn_signup button p {
    color: var(--fq-ink) !important; font-weight: 600 !important; font-size: 15px !important;
}
.st-key-top_btn_signup button:hover, .st-key-mid_btn_signup button:hover, .st-key-cta_btn_signup button:hover { border-color: var(--fq-teal) !important; }
.st-key-top_btn_signup button:hover p, .st-key-mid_btn_signup button:hover p, .st-key-cta_btn_signup button:hover p { color: var(--fq-teal) !important; }

/* ---------- Hero: photo fades into the header above and the stats bar below ---------- */
.fq-hero {
    position: relative; width: 100%;
    aspect-ratio: 1600 / 780;
    min-height: 560px; max-height: 820px;
    display: flex; align-items: center;
    background-color: #1B7F88;
    background-size: cover; background-position: 80% center; background-repeat: no-repeat;
    overflow: hidden;
}
.fq-hero-shade {
    position: absolute; inset: 0;
    background: linear-gradient(90deg, rgba(3, 40, 52, 0.62) 0%, rgba(3, 40, 52, 0.32) 34%, rgba(3, 40, 52, 0) 58%);
}
.fq-hero-fade-top {
    position: absolute; left: 0; right: 0; top: 0; height: 130px;
    background: linear-gradient(180deg, #fff 0%, rgba(255,255,255,0.65) 35%, rgba(255,255,255,0) 100%);
}
.fq-hero-fade-bottom {
    position: absolute; left: 0; right: 0; bottom: 0; height: 170px;
    background: linear-gradient(0deg, #fff 0%, rgba(255,255,255,0.7) 35%, rgba(255,255,255,0) 100%);
}
.fq-hero-content { position: relative; z-index: 2; width: 100%; padding-bottom: 36px; }
.fq-hero-text { max-width: 470px; }

.fq-hero-pill {
    display: inline-block; padding: 6px 14px; border-radius: 999px;
    background: rgba(255,255,255,0.18); border: 1px solid rgba(255,255,255,0.45);
    backdrop-filter: blur(6px);
    font-size: 13px; font-weight: 600; color: #fff !important; margin-bottom: 20px;
}
.fq-landing .fq-hero-title {
    font-family: 'Source Serif 4', Georgia, serif;
    font-size: clamp(36px, 3.7vw, 54px); font-weight: 600; line-height: 1.08;
    letter-spacing: -0.025em; margin: 0 0 20px 0;
    color: #ffffff !important; -webkit-text-fill-color: #ffffff !important;
    text-shadow: 0 2px 18px rgba(0, 40, 50, 0.45);
    animation: fqRise 0.9s ease-out both;
}
.fq-landing .fq-hero-title span, .fq-landing .fq-hero-title em {
    color: #ffffff !important; -webkit-text-fill-color: #ffffff !important;
}
.fq-landing .fq-hero-title em { font-style: italic; color: #CFF3EE !important; -webkit-text-fill-color: #CFF3EE !important; }
.fq-landing .fq-hero-desc {
    font-size: 19px; max-width: 440px; margin: 0;
    color: #ffffff !important; -webkit-text-fill-color: #ffffff !important;
    text-shadow: 0 1px 10px rgba(0, 40, 50, 0.5);
}
@keyframes fqRise { from { opacity: 0; transform: translateY(18px); } to { opacity: 1; transform: none; } }
@media (prefers-reduced-motion: reduce) { .fq-landing .fq-hero-title { animation: none; } }

/* Stats bar under the hero: full width, 4 equal centred cells */
.fq-facts { background: #fff; border-bottom: none; }
.fq-facts-grid { display: grid; grid-template-columns: repeat(4, 1fr); max-width: 1180px; margin: 0 auto; }
.fq-fact { padding: 24px 16px 26px; text-align: center; border-left: 1px solid var(--fq-border); }
.fq-fact:first-child { border-left: none; }
.fq-fact-num { font-size: 25px; font-weight: 700; color: var(--fq-teal); line-height: 1.15; }
.fq-fact-label { font-size: 14px; color: var(--fq-muted); margin-top: 4px; }

/* ---------- Sections: no hard lines, every section melts into the next ---------- */
.fq-section { padding: 80px 0; background: transparent; }
.fq-section-tight { padding-bottom: 36px; }
.fq-section-compact { padding: 64px 0; }
.fq-section-alt {
    background: linear-gradient(180deg, var(--fq-bg) 0%, #ffffff 22%, #ffffff 78%, var(--fq-bg) 100%);
    border: none;
}
/* the first section after the white stats bar fades from white into the page colour */
#preview { background: linear-gradient(180deg, #ffffff 0%, var(--fq-bg) 160px); }

.fq-section-header { max-width: 640px; margin: 0 0 40px 0; }
.fq-section-title {
    font-family: 'Source Serif 4', Georgia, serif;
    font-size: 38px; font-weight: 600; color: var(--fq-ink);
    letter-spacing: -0.01em; line-height: 1.2; margin: 0 0 12px 0;
}
.fq-section-subtitle { font-size: 17px; margin: 0; }
.fq-grid-3 { display: grid; grid-template-columns: repeat(3, 1fr); gap: 20px; }
.fq-card {
    background: #fff; border: 1px solid var(--fq-border);
    border-radius: 16px; padding: 28px;
    transition: border-color .2s ease, box-shadow .2s ease, transform .2s ease;
}
.fq-card:hover { border-color: var(--fq-teal-light); box-shadow: 0 10px 28px rgba(15, 118, 110, 0.10); transform: translateY(-2px); }
.fq-icon {
    width: 46px; height: 46px; border-radius: 12px;
    background: var(--fq-teal-soft); color: var(--fq-teal);
    display: flex; align-items: center; justify-content: center; margin-bottom: 18px;
}
.fq-card-title { font-size: 18px; font-weight: 700; color: var(--fq-ink); margin: 0 0 6px 0; }
.fq-card-desc { font-size: 15px; margin: 0; }
.fq-steps { display: grid; grid-template-columns: repeat(4, 1fr); gap: 24px; }
.fq-step { border-top: 3px solid var(--fq-teal); padding-top: 18px; }
.fq-step-num { font-weight: 700; color: var(--fq-teal); font-size: 14px; margin-bottom: 6px; }

/* ---------- Product preview ---------- */
.fq-preview-grid { display: grid; grid-template-columns: 1fr 1fr; gap: 56px; align-items: center; }
.fq-preview-grid .fq-section-header { margin-bottom: 24px; }
.fq-bullets { display: flex; flex-direction: column; gap: 14px; }
.fq-bullet { display: flex; gap: 12px; align-items: flex-start; font-size: 16px; color: var(--fq-body); }
.fq-bullet-tick {
    flex: 0 0 24px; width: 24px; height: 24px; border-radius: 50%;
    background: var(--fq-teal-soft); color: var(--fq-teal);
    display: flex; align-items: center; justify-content: center; margin-top: 1px;
}
.fq-quote {
    background: #fff; border: 1px solid var(--fq-border); border-radius: 18px;
    box-shadow: 0 24px 60px rgba(15, 60, 70, 0.13);
    padding: 22px 24px 18px; width: 100%; max-width: 500px; margin-left: auto;
}
.fq-quote-head { display: flex; justify-content: space-between; align-items: center; margin-bottom: 14px; }
.fq-quote-tag {
    font-size: 12px; font-weight: 700; letter-spacing: 0.08em; text-transform: uppercase;
    color: var(--fq-teal); background: var(--fq-teal-soft); padding: 4px 10px; border-radius: 999px;
}
.fq-quote-status { font-size: 12.5px; color: var(--fq-muted); }
.fq-quote-route { display: flex; align-items: center; gap: 14px; padding: 6px 0 12px; }
.fq-quote-city small { display: block; font-size: 12px; color: var(--fq-muted); }
.fq-quote-city b { font-size: 17px; color: var(--fq-ink); font-weight: 700; }
.fq-quote-city.right { text-align: right; }
.fq-quote-line {
    flex: 1; height: 2px;
    background: repeating-linear-gradient(90deg, var(--fq-teal-light) 0 6px, transparent 6px 12px);
}
.fq-chips { display: flex; flex-wrap: wrap; gap: 8px; margin-bottom: 16px; }
.fq-chip { font-size: 12.5px; background: #F1F5F9; color: var(--fq-body); padding: 4px 10px; border-radius: 999px; }
.fq-quote-block { font-size: 12px; font-weight: 700; letter-spacing: 0.06em; text-transform: uppercase; color: var(--fq-muted); margin: 14px 0 8px; }
.fq-carrier { display: grid; grid-template-columns: 78px 1fr 70px; gap: 10px; align-items: center; font-size: 13.5px; padding: 4px 0; color: var(--fq-ink); }
.fq-carrier span:last-child { text-align: right; color: var(--fq-muted); font-size: 12.5px; }
.fq-bar { height: 7px; background: #E8EEF3; border-radius: 99px; overflow: hidden; }
.fq-bar i { display: block; height: 100%; border-radius: 99px; background: linear-gradient(90deg, var(--fq-teal-dark), var(--fq-teal-light)); }
.fq-price { display: flex; justify-content: space-between; font-size: 14px; padding: 4px 0; color: var(--fq-body); }
.fq-price b { color: var(--fq-ink); font-weight: 600; }
.fq-price-total {
    display: flex; justify-content: space-between; align-items: baseline;
    margin-top: 10px; padding-top: 12px; border-top: 1px solid var(--fq-border);
    font-size: 15px; font-weight: 700; color: var(--fq-ink);
}
.fq-price-total b { font-size: 24px; color: var(--fq-teal); }
.fq-quote-foot { margin-top: 12px; font-size: 12px; color: var(--fq-muted); text-align: center; }

/* ---------- Middle call to action (after How it works) ---------- */
.fq-mid-cta { text-align: center; margin-top: 52px; }
.fq-landing .fq-mid-cta-title {
    font-family: 'Source Serif 4', Georgia, serif; font-size: 26px; font-weight: 600;
    color: var(--fq-ink) !important; -webkit-text-fill-color: var(--fq-ink) !important;
    text-align: center !important; margin: 0 0 6px 0 !important;
}
.fq-landing .fq-mid-cta-desc { font-size: 16px; text-align: center !important; margin: 0 auto !important; max-width: 480px; }
div[data-testid="stHorizontalBlock"]:has(.st-key-mid_btn_login) {
    background: var(--fq-bg);
    padding: 4px 24px 12px 24px !important;
    margin: 0 !important;
    gap: 16px !important;
    align-items: center !important;
    justify-content: center !important;
}

/* ---------- FAQ ---------- */
.fq-faq { max-width: 800px; }
.fq-faq-item { background: #fff; border: 1px solid var(--fq-border); border-radius: 12px; margin-bottom: 12px; overflow: hidden; }
.fq-faq-summary {
    padding: 18px 22px; font-size: 16px; font-weight: 600; color: var(--fq-ink);
    cursor: pointer; list-style: none; display: flex; justify-content: space-between; align-items: center;
}
.fq-faq-summary::-webkit-details-marker { display: none; }
.fq-faq-item[open] .fq-faq-summary svg { transform: rotate(180deg); }
.fq-faq-content { padding: 0 22px 18px 22px; font-size: 15px; }

/* ---------- Final call-to-action: faded ship photo + text + buttons in ONE band ---------- */
.fq-cta {
    position: relative; overflow: hidden;
    padding: 84px 0 36px 0; text-align: center;
    background: linear-gradient(180deg, var(--fq-bg) 0%, var(--fq-cta-bg) 100%);
}
.fq-cta::before {
    content: ""; position: absolute; inset: 0;
    background-size: cover; background-position: 72% 62%; background-repeat: no-repeat;
    opacity: 0.17; filter: saturate(0.85); pointer-events: none;
    -webkit-mask-image: linear-gradient(180deg, transparent 0%, #000 38%, #000 62%, transparent 100%);
    mask-image: linear-gradient(180deg, transparent 0%, #000 38%, #000 62%, transparent 100%);
}
.fq-cta .fq-container { position: relative; z-index: 1; }
.fq-landing .fq-cta-title {
    font-family: 'Source Serif 4', Georgia, serif;
    font-size: 38px; font-weight: 600; line-height: 1.2;
    color: var(--fq-ink) !important; -webkit-text-fill-color: var(--fq-ink) !important;
    text-align: center !important; margin: 0 0 12px 0 !important;
}
.fq-landing .fq-cta-desc {
    font-size: 17px; text-align: center !important;
    max-width: 520px; margin: 0 auto !important; color: var(--fq-body);
}
div[data-testid="stHorizontalBlock"]:has(.st-key-cta_btn_login) {
    background: var(--fq-cta-bg);
    padding: 0 24px 84px 24px !important;
    margin: 0 !important;
    gap: 16px !important;
    align-items: center !important;
    justify-content: center !important;
}
.fq-footer {
    padding: 40px 0 32px; text-align: center; font-size: 14px; color: var(--fq-muted);
    background: linear-gradient(180deg, var(--fq-cta-bg) 0%, #ffffff 100%);
    border: none;
}

/* ---------- Responsive ---------- */
@media (max-width: 1000px) {
    .fq-preview-grid { grid-template-columns: 1fr; gap: 36px; }
    .fq-quote { margin: 0 auto; }
}
@media (max-width: 900px) {
    .fq-nav-links { display: none; }
    .fq-hero { min-height: 520px; background-position: 75% center; }
    .fq-hero-text { max-width: 100%; }
    .fq-facts-grid { grid-template-columns: 1fr 1fr; }
    .fq-fact:nth-child(odd) { border-left: none; }
    .fq-fact:nth-child(-n+2) { border-bottom: 1px solid var(--fq-border); }
    .fq-grid-3 { grid-template-columns: 1fr 1fr; }
    .fq-steps { grid-template-columns: 1fr 1fr; }
}
@media (max-width: 700px) {
    /* header: logo on its own row, the two buttons side by side below it */
    div[data-testid="stHorizontalBlock"]:has(.st-key-top_btn_login) { flex-wrap: wrap !important; position: static; }
    div[data-testid="stHorizontalBlock"]:has(.st-key-top_btn_login) > div:nth-child(1) { flex: 1 1 100% !important; min-width: 100% !important; }
    div[data-testid="stHorizontalBlock"]:has(.st-key-top_btn_login) > div:nth-child(2) { display: none !important; }
    div[data-testid="stHorizontalBlock"]:has(.st-key-top_btn_login) > div:nth-child(n+3) { flex: 1 1 calc(50% - 8px) !important; min-width: calc(50% - 8px) !important; }
    /* call-to-action rows: hide the spacer columns, stack the buttons */
    div[data-testid="stHorizontalBlock"]:has(.st-key-mid_btn_login) > div:nth-child(1),
    div[data-testid="stHorizontalBlock"]:has(.st-key-mid_btn_login) > div:nth-child(4),
    div[data-testid="stHorizontalBlock"]:has(.st-key-cta_btn_login) > div:nth-child(1),
    div[data-testid="stHorizontalBlock"]:has(.st-key-cta_btn_login) > div:nth-child(4) { display: none !important; }
    div[data-testid="stHorizontalBlock"]:has(.st-key-mid_btn_login),
    div[data-testid="stHorizontalBlock"]:has(.st-key-cta_btn_login) { flex-direction: column !important; align-items: center !important; }
    div[data-testid="stHorizontalBlock"]:has(.st-key-mid_btn_login) > div,
    div[data-testid="stHorizontalBlock"]:has(.st-key-cta_btn_login) > div { width: 100% !important; max-width: 340px; flex: 0 0 auto !important; }
    .fq-hero { min-height: 480px; }
    .fq-grid-3, .fq-steps { grid-template-columns: 1fr; }
    .fq-section-title, .fq-landing .fq-cta-title { font-size: 30px; }
    .fq-section { padding: 56px 0; }
    .fq-section-compact { padding: 48px 0; }
    .fq-carrier { grid-template-columns: 70px 1fr 62px; }
}

/* PORTWISE presentation. All motion respects reduced-motion and Pause motion. */
html, body, [data-testid="stApp"] {
  font-family: Inter, -apple-system, BlinkMacSystemFont, "Segoe UI", sans-serif;
  color: var(--pw-ink);
}
[data-testid="stApp"] { background: var(--pw-bg); }
[data-testid="stHeader"] { background: transparent; }
.block-container { max-width: 1280px; padding: 2rem 2.6rem 1rem; }
h1,h2,h3,h4 { color: var(--pw-ink)!important; letter-spacing: -.035em; font-weight: 650; }
p, [data-testid="stCaptionContainer"] { color: var(--pw-sub); }
[data-testid="stWidgetLabel"] p { color: var(--pw-ink); font-weight: 500; }
.top-brand { display: flex; align-items: center; gap: 10px; color: var(--pw-ink); font-size: 19px; font-weight: 750; letter-spacing: .06em; padding: 5px 0; }
.brand-symbol { display: inline-grid; place-items: center; width: 34px; height: 34px; background: #6855c7; border-radius: 10px; color: white; font-size: 21px; letter-spacing: -.1em; padding-right: 3px; }
.top-brand small { margin-left: 12px; font-size: 9px; letter-spacing: .15em; color: var(--pw-sub); font-weight: 550; }
.auth-hero { position: relative; overflow: hidden; padding: 44px 38px 25px; margin: 22px 0; border-radius: 24px; min-height: 560px; background: linear-gradient(140deg,#282d63 0%,#424381 45%,#1b6269 100%); box-shadow: 0 20px 50px #1b285518; animation: pw-slide .55s ease-out both; }
.auth-hero::before { content: ''; position: absolute; width: 340px; height: 340px; border-radius: 50%; border: 1px solid #ffffff0d; top: -180px; right: -100px; box-shadow: 0 0 0 60px #ffffff04,0 0 0 120px #ffffff03; pointer-events: none; }
.hero-eyebrow { color: #d2d5f1; font-size: 10px; font-weight: 650; letter-spacing: .16em; display: flex; align-items: center; gap: 10px; }
.hero-eyebrow span { width: 7px; height: 7px; border-radius: 50%; background: #a2e9dc; }
.auth-hero h1 { color: #fff!important; font-size: clamp(36px,4vw,55px); font-weight: 650; line-height: 1.16; margin: 36px 0 20px; letter-spacing: -.055em; }
.auth-hero h1 em { font-style: normal; color: #b6e9e2; }
.auth-hero .hero-copy { font-size: 15px; line-height: 1.8; color: #dae0f2; }
.route-illustration { margin: 24px -8px 16px; }
.route-illustration svg { display: block; width: 100%; }
.route-line { animation: pw-route 24s linear infinite; }
.hero-bottom { display: flex; justify-content: space-between; gap: 16px; padding-top: 20px; border-top: 1px solid #ffffff22; color: #d5e5f1; font-size: 11px; }
[data-testid="stForm"] { background: var(--pw-card); border: 1px solid var(--pw-line); border-radius: 18px; padding: 24px; box-shadow: 0 12px 35px #1a295706; animation: pw-slide .4s ease-out both; }
[data-testid="stTextInput"] div[data-baseweb="input"], [data-testid="stTextInput"] div[data-baseweb="base-input"], [data-testid="stSelectbox"] div[data-baseweb="select"] > div { background: var(--pw-card)!important; color: var(--pw-ink)!important; border-color: var(--pw-line)!important; border-radius: 9px; }
[data-testid="stTextInput"] input { background: var(--pw-card)!important; color: var(--pw-ink)!important; -webkit-text-fill-color: var(--pw-ink)!important; min-height: 44px; }
[data-testid="stTextInput"] input::placeholder { color: var(--pw-sub)!important; -webkit-text-fill-color: var(--pw-sub)!important; opacity: .8; }
[data-testid="stTextInput"] div[data-baseweb="input"]:focus-within { border-color: var(--pw-accent)!important; box-shadow: 0 0 0 3px #8972dc1a; }
[data-testid="stButton"] button, [data-testid="stFormSubmitButton"] button { min-height: 42px; border-radius: 9px; border: 1px solid var(--pw-line); background: var(--pw-card); color: var(--pw-ink); transition: transform .18s ease, box-shadow .18s ease; }
[data-testid="stButton"] button p, [data-testid="stFormSubmitButton"] button p { color: inherit!important; font-weight: 550; font-size: 14px; }
[data-testid="stButton"] button[kind^="primary"], [data-testid="stFormSubmitButton"] button[kind^="primary"] { background: #6652c7; border-color: #6652c7; color: #fff; }
[data-testid="stButton"] button:hover:not(:disabled), [data-testid="stFormSubmitButton"] button:hover:not(:disabled) { transform: translateY(-1px); box-shadow: 0 4px 12px #54449b22; border-color: var(--pw-accent); }
button:focus-visible { outline: 3px solid var(--pw-accent)!important; outline-offset: 3px; }
button:disabled { opacity: .5; cursor: not-allowed; }
[data-testid="stSidebar"] { background: var(--pw-card); border-right: 1px solid var(--pw-line); }
[data-testid="stSidebar"] [data-testid="stSidebarContent"] { padding-top: 1.2rem; }
[data-testid="stRadio"] label p { color: var(--pw-ink)!important; }
[data-testid="stMetric"] { background: var(--pw-card); border: 1px solid var(--pw-line); border-radius: 14px; padding: 18px; }
[data-testid="stMetricValue"] { color: var(--pw-ink); font-size: clamp(21px,2.4vw,30px); }
[data-testid="stMetricLabel"] p { color: var(--pw-sub); }
.workspace-heading { display: flex; align-items: center; justify-content: space-between; gap: 15px; margin: 18px 0 14px; }
.workspace-heading h2 { font-size: 30px; margin: 4px 0 0; }
.eyebrow { font-size: 10px; letter-spacing: .15em; font-weight: 650; color: var(--pw-sub); }
.role-badge { border: 1px solid var(--pw-line); border-radius: 20px; padding: 7px 12px; background: var(--pw-card); color: var(--pw-accent); font-size: 12px; font-weight: 650; }
.welcome-panel { padding: 28px 30px; border-radius: 18px; background: linear-gradient(110deg,var(--pw-soft),var(--pw-card)); border: 1px solid var(--pw-line); margin-bottom: 18px; animation: pw-slide .45s ease-out both; }
.welcome-panel h2 { font-size: 31px; margin: 9px 0; }
.welcome-panel p { margin: 0; font-size: 14px; }
.module-grid { display: grid; grid-template-columns: repeat(2,minmax(0,1fr)); gap: 16px; margin-bottom: 20px; }
.module-card { background: var(--pw-card); border: 1px solid var(--pw-line); border-radius: 14px; padding: 22px; }
.module-label { font-size: 9px; letter-spacing: .12em; color: var(--pw-sub); }
.module-card h4 { font-size: 18px; margin: 10px 0 7px; }
.module-card p { font-size: 13px; line-height: 1.65; margin: 0; }
.footer-note { text-align: center; color: var(--pw-sub); font-size: 11px; margin: 28px 0 10px; }
@keyframes pw-slide { from { opacity: 0; transform: translateY(12px); } to { opacity: 1; transform: translateY(0); } }
@keyframes pw-route { to { stroke-dashoffset: -120; } }
@media (max-width: 1000px) { .top-brand small { display: none; } .block-container { padding: 1.3rem 1.4rem; } .auth-hero { padding: 28px 24px; min-height: 500px; } }
@media (max-width: 640px) { .block-container { padding: 1rem; } .auth-hero { min-height: 0; margin: 8px 0; padding: 24px; } .auth-hero h1 { font-size: 35px; margin: 20px 0 12px; } .route-illustration, .hero-bottom { display: none; } .hero-copy { margin-bottom: 0; } .hero-eyebrow { font-size: 8px; } .module-grid { grid-template-columns: 1fr; } .workspace-heading h2 { font-size: 25px; } .role-badge { font-size: 10px; } }
@media (prefers-reduced-motion: reduce) { *,*::before,*::after { animation: none!important; transition: none!important; scroll-behavior: auto!important; } }

[data-testid="stCaptionContainer"] p { color: var(--pw-sub)!important; opacity: 1; }
[data-testid="stTextInput"] button { background: var(--pw-soft); color: var(--pw-ink); }
[data-testid="stTextInput"] svg { fill: currentColor; }
@media (max-width: 640px) {
  .st-key-topbar [data-testid="stHorizontalBlock"] { flex-direction: row!important; gap: 8px; }
  .st-key-topbar [data-testid="stColumn"] { min-width: 0!important; }
  .st-key-topbar [data-testid="stColumn"]:first-child { flex: 1.5 1 0!important; }
  .st-key-topbar [data-testid="stColumn"]:not(:first-child) { flex: 1 1 0!important; }
  .st-key-topbar button p { font-size: 11px!important; }
  .top-brand { font-size: 14px; gap: 6px; letter-spacing: .02em; }
  .brand-symbol { width: 26px; height: 28px; font-size: 17px; flex-shrink: 0; }
  .auth-hero { padding: 20px; }
  .auth-hero h1 { font-size: 29px; margin: 0; }
  .hero-eyebrow, .auth-hero .hero-copy { display: none; }
}
/* Keep deployment controls out of the end-user interface. */


/* ==========================================================================
   FreightQuote AI — Enterprise Maritime Logistics Dashboard
   Modern, Breathtaking SaaS Design System
   ========================================================================== */

:root {
  --bg: #f8fafc;
  --surface: #ffffff;
  --card: #ffffff;
  --card-hover: #fcfdfe;
  --line: #e2e8f0;
  --line-strong: #cbd5e1;
  --line-subtle: #f1f5f9;
  
  --tx: #0f172a;
  --tx2: #334155;
  --mut: #64748b;
  --mut-light: #94a3b8;
  
  --pri: #0284c7;
  --pri2: #0369a1;
  --pri-bg: #e0f2fe;
  --pri-gradient: linear-gradient(135deg, #0ea5e9 0%, #0284c7 100%);
  
  --teal: #0d9488;
  --teal-bg: #ccfbf1;
  --blue: #2563eb;
  --blue-bg: #dbeafe;
  --amber: #d97706;
  --amber-bg: #fef3c7;
  --red: #dc2626;
  --red-bg: #fee2e2;
  --green: #059669;
  --green-bg: #d1fae5;
  --violet: #7c3aed;
  --violet-bg: #ede9fe;
  
  --sidebar-bg: #0b132b;
  --sidebar-surface: #111b38;
  --sidebar-border: #1c274c;
  --sidebar-tx: #94a3b8;
  --sidebar-tx-active: #ffffff;
  --sidebar-active-bg: linear-gradient(135deg, rgba(14, 165, 233, 0.22) 0%, rgba(2, 132, 199, 0.15) 100%);
  --sidebar-active-border: #0ea5e9;

  --r: 12px;
  --r-sm: 8px;
  --r-lg: 16px;
  --sh: 0 1px 3px rgba(0,0,0,0.05), 0 6px 16px -2px rgba(15,23,42,0.05);
  --sh-lg: 0 10px 30px -4px rgba(15,23,42,0.1), 0 4px 12px -2px rgba(15,23,42,0.05);
  --sh-glow: 0 0 20px rgba(14, 165, 233, 0.25);
}

body.fq-dark {
  --bg: #070b14;
  --surface: #0f172a;
  --card: #111b33;
  --card-hover: #152240;
  --line: #1e293b;
  --line-strong: #334155;
  --line-subtle: #141e33;
  
  --tx: #f8fafc;
  --tx2: #cbd5e1;
  --mut: #94a3b8;
  --mut-light: #64748b;
  
  --pri: #38bdf8;
  --pri2: #0ea5e9;
  --pri-bg: rgba(56, 189, 248, 0.15);
  --pri-gradient: linear-gradient(135deg, #38bdf8 0%, #0284c7 100%);
  
  --teal: #2dd4bf;
  --teal-bg: rgba(45, 212, 191, 0.15);
  --blue: #60a5fa;
  --blue-bg: rgba(96, 165, 250, 0.15);
  --amber: #fbbf24;
  --amber-bg: rgba(251, 191, 36, 0.15);
  --red: #f87171;
  --red-bg: rgba(248, 113, 113, 0.15);
  --green: #34d399;
  --green-bg: rgba(52, 211, 153, 0.15);
  --violet: #a78bfa;
  --violet-bg: rgba(167, 139, 250, 0.15);
  
  --sidebar-bg: #050810;
  --sidebar-surface: #090e1c;
  --sidebar-border: #16203a;
  --sidebar-tx: #94a3b8;
  --sidebar-tx-active: #ffffff;
  --sidebar-active-bg: rgba(56, 189, 248, 0.18);
  --sidebar-active-border: #38bdf8;
  
  --sh: 0 4px 20px rgba(0,0,0,0.35);
  --sh-lg: 0 14px 40px rgba(0,0,0,0.5);
}

/* Base resets & layout */
* { box-sizing: border-box; }
html, body {
  margin: 0;
  height: 100%;
  background: var(--bg);
  color: var(--tx);
  font-family: 'Plus Jakarta Sans', -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;
  font-size: 14px;
  line-height: 1.5;
  -webkit-font-smoothing: antialiased;
  -moz-osx-font-smoothing: grayscale;
}

button, input, select, textarea {
  font-family: inherit;
  font-size: inherit;
  color: inherit;
}
a { color: var(--pri); text-decoration: none; transition: color .15s; }
a:hover { color: var(--pri2); text-decoration: none; }

#app {
  display: flex;
  height: 100vh;
  overflow: hidden;
  background: var(--bg);
}

/* ------------------------------------------------------------------ Sidebar */
.side {
  width: 256px;
  flex: none;
  background: var(--sidebar-bg);
  border-right: 1px solid var(--sidebar-border);
  display: flex;
  flex-direction: column;
  padding: 18px 14px;
  overflow-y: auto;
  gap: 4px;
  user-select: none;
}

.brand {
  display: flex;
  align-items: center;
  gap: 12px;
  padding: 4px 8px 16px;
  font-weight: 700;
  font-size: 15px;
  letter-spacing: -0.01em;
  color: #fff;
  border-bottom: 1px solid var(--sidebar-border);
  margin-bottom: 8px;
}
.brand i {
  width: 36px;
  height: 36px;
  border-radius: 10px;
  background: linear-gradient(135deg, #0ea5e9, #0284c7);
  display: grid;
  place-items: center;
  color: #fff;
  box-shadow: 0 4px 16px rgba(14, 165, 233, 0.4);
  flex-shrink: 0;
}
.brand i svg {
  width: 20px;
  height: 20px;
  stroke: #fff;
  fill: none;
  stroke-width: 2;
}
.brand small {
  display: block;
  color: #94a3b8;
  font-weight: 500;
  font-size: 11px;
  margin-top: 1px;
  max-width: 160px;
  overflow: hidden;
  text-overflow: ellipsis;
  white-space: nowrap;
}

.ksearch {
  display: flex;
  align-items: center;
  gap: 8px;
  padding: 8px 12px;
  border: 1px solid var(--sidebar-border);
  border-radius: var(--r-sm);
  color: #94a3b8;
  cursor: pointer;
  font-size: 12.5px;
  background: var(--sidebar-surface);
  margin: 6px 0 10px;
  transition: all .15s;
}
.ksearch:hover {
  border-color: #38bdf8;
  color: #f8fafc;
}
.ksearch kbd {
  margin-left: auto;
  font-family: 'JetBrains Mono', monospace;
  font-size: 10.5px;
  font-weight: 600;
  border: 1px solid var(--sidebar-border);
  border-radius: 4px;
  padding: 1px 6px;
  background: rgba(255,255,255,0.06);
  color: #94a3b8;
}

.ngrp {
  font-size: 10.5px;
  font-weight: 700;
  letter-spacing: 0.08em;
  text-transform: uppercase;
  color: #64748b;
  padding: 14px 10px 4px;
}

.nav {
  display: flex;
  align-items: center;
  gap: 11px;
  padding: 9px 12px;
  border-radius: var(--r-sm);
  color: var(--sidebar-tx);
  cursor: pointer;
  border: 1px solid transparent;
  font-weight: 500;
  font-size: 13px;
  transition: all .14s ease;
  text-decoration: none;
}
.nav:hover {
  background: rgba(255, 255, 255, 0.06);
  color: #f8fafc;
}
.nav.on {
  background: var(--sidebar-active-bg);
  color: var(--sidebar-tx-active);
  border-color: var(--sidebar-active-border);
  font-weight: 600;
  box-shadow: 0 2px 10px rgba(14, 165, 233, 0.15);
}
.nav svg {
  width: 17px;
  height: 17px;
  flex: none;
  stroke: currentColor;
  fill: none;
  stroke-width: 1.9;
  stroke-linecap: round;
  stroke-linejoin: round;
  opacity: 0.9;
}
.nav .bd {
  margin-left: auto;
  background: var(--red);
  color: #fff;
  border-radius: 99px;
  font-size: 10px;
  padding: 1px 7px;
  font-weight: 700;
}
.nav .bd.a {
  background: var(--amber);
  color: #fff;
}

.fq-theme-toggle {
  display: flex;
  align-items: center;
  justify-content: center;
  gap: 8px;
  padding: 8px 12px;
  border-radius: var(--r-sm);
  background: var(--sidebar-surface);
  border: 1px solid var(--sidebar-border);
  color: #94a3b8;
  font-size: 12px;
  font-weight: 600;
  cursor: pointer;
  margin-top: 10px;
  transition: all .15s;
}
.fq-theme-toggle:hover {
  background: rgba(255,255,255,0.08);
  color: #f8fafc;
  border-color: #38bdf8;
}

/* ------------------------------------------------------------------ Main View */
.main {
  flex: 1;
  min-width: 0;
  display: flex;
  flex-direction: column;
  overflow: hidden;
  background: var(--bg);
}

.top {
  display: flex;
  align-items: center;
  gap: 16px;
  padding: 14px 28px;
  border-bottom: 1px solid var(--line);
  background: var(--surface);
  backdrop-filter: blur(12px);
  flex-wrap: wrap;
  z-index: 10;
}
.top h1 {
  font-size: 20px;
  font-weight: 700;
  margin: 0;
  letter-spacing: -0.02em;
  color: var(--tx);
}
.top .sub {
  color: var(--mut);
  font-size: 12.5px;
  margin: 2px 0 0;
  font-weight: 500;
}

.chip {
  display: inline-flex;
  gap: 7px;
  align-items: center;
  padding: 5px 12px;
  border-radius: 99px;
  border: 1px solid var(--line);
  background: var(--surface);
  font-size: 12px;
  font-weight: 500;
  color: var(--tx2);
  box-shadow: 0 1px 2px rgba(0,0,0,0.04);
}
.live-chip {
  background: var(--teal-bg);
  border-color: var(--teal);
  color: var(--teal);
  font-weight: 600;
}
.live-dot {
  width: 8px;
  height: 8px;
  border-radius: 50%;
  background: var(--teal);
  box-shadow: 0 0 0 3px rgba(13, 148, 136, 0.2);
  animation: pulse-dot 1.8s infinite;
}
@keyframes pulse-dot {
  0%, 100% { opacity: 1; transform: scale(1); }
  50% { opacity: 0.4; transform: scale(0.9); }
}

.content {
  flex: 1;
  overflow-y: auto;
  padding: 24px 28px 80px;
}

/* ------------------------------------------------------------------ Tabs */
.tabs {
  display: flex;
  gap: 4px;
  border-bottom: 1px solid var(--line);
  margin-bottom: 20px;
  flex-wrap: wrap;
}
.tabs b {
  padding: 10px 16px;
  cursor: pointer;
  color: var(--mut);
  border-bottom: 2px solid transparent;
  font-weight: 600;
  font-size: 13.5px;
  margin-bottom: -1px;
  transition: all .15s;
}
.tabs b:hover {
  color: var(--tx);
}
.tabs b.on {
  color: var(--pri);
  border-color: var(--pri);
}

/* ------------------------------------------------------------------ Blocks & Cards */
.blocks > * { margin-bottom: 20px; }

.card {
  background: var(--card);
  border-radius: var(--r);
  border: 1px solid var(--line);
  box-shadow: var(--sh);
  padding: 20px 22px;
  transition: box-shadow .2s ease;
}
.card:hover {
  box-shadow: var(--sh-lg);
}
.card h3 {
  margin: 0 0 14px;
  font-size: 15px;
  font-weight: 700;
  letter-spacing: -0.01em;
  color: var(--tx);
}

.row {
  display: grid;
  gap: 20px;
  grid-template-columns: 1fr 1fr;
}
.col {
  display: flex;
  flex-direction: column;
  gap: 20px;
}

/* ------------------------------------------------------------------ KPI Cards */
.kpis {
  display: grid;
  grid-template-columns: repeat(auto-fit, minmax(180px, 1fr));
  gap: 16px;
}
.kpi {
  background: var(--card);
  border: 1px solid var(--line);
  border-radius: var(--r);
  padding: 16px 18px;
  box-shadow: var(--sh);
  position: relative;
  overflow: hidden;
  transition: transform .15s, box-shadow .15s;
}
.kpi:hover {
  transform: translateY(-2px);
  box-shadow: var(--sh-lg);
}
.kpi .l {
  font-size: 12px;
  font-weight: 600;
  color: var(--mut);
  text-transform: uppercase;
  letter-spacing: 0.04em;
  margin-bottom: 6px;
}
.kpi .v {
  font-size: 26px;
  font-weight: 800;
  color: var(--tx);
  letter-spacing: -0.02em;
}
.kpi .s {
  font-size: 11.5px;
  color: var(--mut);
  margin-top: 4px;
}
.kpi.Low { border-top: 3px solid var(--green); }
.kpi.Warning { border-top: 3px solid var(--amber); }
.kpi.Critical { border-top: 3px solid var(--red); }

/* ------------------------------------------------------------------ Tables */
.tbl {
  overflow: auto;
  border: 1px solid var(--line);
  border-radius: var(--r);
  background: var(--card);
}
.tbl table {
  width: 100%;
  border-collapse: collapse;
  text-align: left;
  font-size: 13px;
}
.tbl th {
  position: sticky;
  top: 0;
  background: var(--line-subtle);
  color: var(--tx2);
  padding: 11px 16px;
  font-weight: 700;
  font-size: 11.5px;
  letter-spacing: 0.04em;
  text-transform: uppercase;
  border-bottom: 1px solid var(--line);
  cursor: pointer;
  user-select: none;
  white-space: nowrap;
}
.tbl th:hover {
  background: var(--line);
}
.tbl td {
  padding: 11px 16px;
  border-bottom: 1px solid var(--line-subtle);
  color: var(--tx);
  white-space: nowrap;
}
.tbl tr:last-child td {
  border-bottom: none;
}
.tbl tr:hover td {
  background: rgba(14, 165, 233, 0.04);
}
.clickable-row {
  cursor: pointer;
}

.srch {
  width: min(320px, 100%);
  padding: 9px 14px;
  border: 1px solid var(--line);
  border-radius: var(--r-sm);
  background: var(--card);
  margin-bottom: 14px;
  font-size: 13px;
  color: var(--tx);
  transition: all .15s;
}
.srch:focus {
  border-color: var(--pri);
  outline: none;
  box-shadow: 0 0 0 3px rgba(14, 165, 233, 0.15);
}

/* ------------------------------------------------------------------ Status Pills */
.pill {
  display: inline-flex;
  align-items: center;
  padding: 3px 10px;
  border-radius: 99px;
  font-size: 11px;
  font-weight: 700;
  letter-spacing: 0.02em;
  border: 1px solid transparent;
}
.pill.Low, .pill.Approved, .pill.Active {
  background: var(--green-bg);
  color: var(--green);
  border-color: rgba(5, 150, 105, 0.25);
}
.pill.Warning, .pill.Pending, .pill.Medium {
  background: var(--amber-bg);
  color: var(--amber);
  border-color: rgba(217, 119, 6, 0.25);
}
.pill.Critical, .pill.On_Hold, .pill.Rejected, .pill.Locked {
  background: var(--red-bg);
  color: var(--red);
  border-color: rgba(220, 38, 38, 0.25);
}
.pill.In_Transit, .pill.Info {
  background: var(--blue-bg);
  color: var(--blue);
  border-color: rgba(37, 99, 235, 0.25);
}

/* ------------------------------------------------------------------ Buttons */
.btn {
  display: inline-flex;
  align-items: center;
  justify-content: center;
  gap: 7px;
  padding: 9px 18px;
  border-radius: var(--r-sm);
  font-weight: 600;
  font-size: 13px;
  cursor: pointer;
  border: 1px solid transparent;
  transition: all .14s ease;
  user-select: none;
}
.btn:active { transform: scale(0.98); }
.btn:disabled { opacity: 0.6; cursor: not-allowed; }

.btn.primary {
  background: var(--pri-gradient);
  color: #fff;
  box-shadow: 0 2px 8px rgba(2, 132, 199, 0.3);
}
.btn.primary:hover:not(:disabled) {
  box-shadow: 0 4px 14px rgba(2, 132, 199, 0.45);
  filter: brightness(1.06);
}

.btn.danger {
  background: linear-gradient(135deg, #ef4444 0%, #dc2626 100%);
  color: #fff;
  box-shadow: 0 2px 8px rgba(220, 38, 38, 0.3);
}
.btn.danger:hover:not(:disabled) {
  filter: brightness(1.08);
}

.btn.ok {
  background: linear-gradient(135deg, #10b981 0%, #059669 100%);
  color: #fff;
}
.btn.ghost {
  background: transparent;
  border: 1px solid var(--line);
  color: var(--tx2);
}
.btn.ghost:hover {
  background: var(--line-subtle);
  border-color: var(--line-strong);
}

.btn.sm {
  padding: 6px 12px;
  font-size: 12px;
  border-radius: 6px;
}

/* ------------------------------------------------------------------ Forms */
.f {
  display: grid;
  gap: 16px;
}
.f.inline {
  display: flex;
  align-items: flex-end;
  gap: 12px;
  flex-wrap: wrap;
}
.fld {
  display: flex;
  flex-direction: column;
  gap: 6px;
}
.fld label {
  font-size: 12.5px;
  font-weight: 600;
  color: var(--tx2);
}
.fld input, .fld select, .fld textarea, input.in, select.in {
  padding: 9px 13px;
  border: 1px solid var(--line);
  border-radius: var(--r-sm);
  background: var(--card);
  color: var(--tx);
  font-size: 13.5px;
  transition: all .15s;
}
.fld input:focus, .fld select:focus, .fld textarea:focus, input.in:focus, select.in:focus {
  outline: none;
  border-color: var(--pri);
  box-shadow: 0 0 0 3px rgba(14, 165, 233, 0.16);
}
.fld textarea { min-height: 80px; resize: vertical; }

.chk {
  display: flex;
  gap: 12px;
  flex-wrap: wrap;
}
.chk label {
  display: flex;
  align-items: center;
  gap: 6px;
  cursor: pointer;
  font-size: 13px;
}

/* ------------------------------------------------------------------ Callouts */
.callout {
  padding: 14px 18px;
  border-radius: var(--r);
  border-left: 4px solid;
  margin-bottom: 16px;
  font-size: 13.5px;
}
.callout.info {
  background: var(--pri-bg);
  border-color: var(--pri);
  color: var(--tx);
}
.callout.success {
  background: var(--green-bg);
  border-color: var(--green);
  color: var(--tx);
}
.callout.warn {
  background: var(--amber-bg);
  border-color: var(--amber);
  color: var(--tx);
}
.callout.error {
  background: var(--red-bg);
  border-color: var(--red);
  color: var(--tx);
}

/* ------------------------------------------------------------------ Detail Drawer & Backdrop */
.drawer-backdrop {
  position: fixed;
  inset: 0;
  background: rgba(11, 19, 43, 0.5);
  backdrop-filter: blur(4px);
  z-index: 1999;
  opacity: 0;
  pointer-events: none;
  transition: opacity 0.25s cubic-bezier(0.16, 1, 0.3, 1);
}
.drawer-backdrop.on {
  opacity: 1;
  pointer-events: auto;
}

#drawer {
  position: fixed;
  top: 0;
  right: 0;
  bottom: 0;
  width: min(720px, 94vw);
  background: var(--surface);
  border-left: 1px solid var(--line);
  box-shadow: -16px 0 50px rgba(0, 0, 0, 0.25);
  transform: translateX(105%);
  transition: transform 0.28s cubic-bezier(0.16, 1, 0.3, 1);
  z-index: 2000;
  display: flex;
  flex-direction: column;
}
#drawer.on {
  transform: translateX(0);
}
#drawer .dh {
  display: flex;
  align-items: center;
  justify-content: space-between;
  padding: 18px 24px;
  border-bottom: 1px solid var(--line);
  background: var(--surface);
}
.drawer-badge {
  background: var(--pri-bg);
  color: var(--pri);
  font-size: 11px;
  font-weight: 700;
  padding: 3px 9px;
  border-radius: 99px;
  text-transform: uppercase;
}
#drawer b#dt {
  font-size: 17px;
  font-weight: 700;
  color: var(--tx);
}
#db {
  padding: 24px;
  overflow-y: auto;
  flex: 1;
  background: var(--bg);
}
.x {
  background: transparent;
  border: none;
  color: var(--mut);
  cursor: pointer;
  font-size: 20px;
  width: 32px;
  height: 32px;
  display: grid;
  place-items: center;
  border-radius: 6px;
  transition: all .15s;
}
.x:hover {
  background: var(--line);
  color: var(--tx);
}

/* ------------------------------------------------------------------ Admin Modal */
.admin-modal-overlay {
  position: fixed;
  inset: 0;
  background: rgba(11, 19, 43, 0.55);
  backdrop-filter: blur(5px);
  display: flex;
  align-items: center;
  justify-content: center;
  z-index: 2500;
  padding: 16px;
  animation: modal-fade .2s ease;
}
@keyframes modal-fade {
  from { opacity: 0; } to { opacity: 1; }
}
.admin-modal-card {
  background: var(--card);
  border-radius: var(--r-lg);
  border: 1px solid var(--line);
  box-shadow: 0 25px 60px rgba(0,0,0,0.3);
  padding: 28px;
  width: min(500px, 94vw);
}
.admin-modal-title {
  margin: 0 0 10px;
  font-size: 18px;
  font-weight: 700;
  color: var(--tx);
}
.admin-modal-desc {
  margin: 0 0 22px;
  color: var(--mut);
  font-size: 13.5px;
  line-height: 1.55;
}
.admin-modal-acts {
  display: flex;
  gap: 12px;
  justify-content: flex-end;
}

/* ------------------------------------------------------------------ Admin Dashboard */
.admin-container {
  display: flex;
  flex-direction: column;
  gap: 24px;
}
.admin-kpi-grid {
  display: grid;
  grid-template-columns: repeat(auto-fit, minmax(220px, 1fr));
  gap: 16px;
}
.admin-kpi-card {
  background: var(--card);
  border: 1px solid var(--line);
  border-radius: var(--r);
  padding: 18px 20px;
  box-shadow: var(--sh);
}
.admin-kpi-title {
  font-size: 12px;
  font-weight: 600;
  color: var(--mut);
  text-transform: uppercase;
  letter-spacing: 0.04em;
  display: block;
}
.admin-kpi-val {
  font-size: 28px;
  font-weight: 800;
  color: var(--tx);
  margin: 4px 0 2px;
  display: block;
}
.admin-kpi-sub {
  font-size: 11.5px;
  color: var(--mut);
  display: block;
}

.admin-section {
  background: var(--card);
  border: 1px solid var(--line);
  border-radius: var(--r);
  box-shadow: var(--sh);
  padding: 22px 24px;
}
.admin-section-header {
  margin-bottom: 18px;
}
.admin-section-title {
  margin: 0 0 4px;
  font-size: 16px;
  font-weight: 700;
  color: var(--tx);
}
.admin-section-sub {
  color: var(--mut);
  font-size: 12.5px;
}

.admin-table-wrap {
  overflow-x: auto;
  border: 1px solid var(--line);
  border-radius: var(--r-sm);
}
.admin-table {
  width: 100%;
  border-collapse: collapse;
  font-size: 13px;
  text-align: left;
}
.admin-table th {
  background: var(--line-subtle);
  color: var(--tx2);
  padding: 11px 14px;
  font-size: 11.5px;
  font-weight: 700;
  text-transform: uppercase;
  letter-spacing: 0.04em;
  border-bottom: 1px solid var(--line);
  white-space: nowrap;
}
.admin-table td {
  padding: 12px 14px;
  border-bottom: 1px solid var(--line-subtle);
  color: var(--tx);
  white-space: nowrap;
}
.admin-table tr:last-child td { border-bottom: none; }
.admin-table tr:hover td { background: rgba(14, 165, 233, 0.03); }

.admin-select {
  padding: 6px 10px;
  border: 1px solid var(--line);
  border-radius: 6px;
  background: var(--card);
  color: var(--tx);
  font-size: 12px;
  font-weight: 500;
}
.admin-btn-action {
  padding: 6px 12px;
  border-radius: 6px;
  font-size: 12px;
  font-weight: 600;
  cursor: pointer;
  border: 1px solid transparent;
  transition: all .14s;
}
.admin-btn-lock {
  background: var(--amber-bg);
  color: var(--amber);
  border-color: rgba(217, 119, 6, 0.3);
}
.admin-btn-unlock {
  background: var(--green-bg);
  color: var(--green);
  border-color: rgba(5, 150, 105, 0.3);
}
.admin-btn-delete {
  background: var(--red-bg);
  color: var(--red);
  border-color: rgba(220, 38, 38, 0.3);
}
.admin-actions-cell {
  display: flex;
  align-items: center;
  gap: 8px;
}

/* ------------------------------------------------------------------ Profile Page */
.profile-container {
  display: flex;
  flex-direction: column;
  gap: 22px;
  max-width: 900px;
}
.profile-header-card {
  background: var(--card);
  border: 1px solid var(--line);
  border-radius: var(--r);
  box-shadow: var(--sh);
  padding: 24px 26px;
}
.profile-header-top {
  display: flex;
  align-items: center;
  gap: 20px;
  margin-bottom: 22px;
}
.profile-avatar {
  width: 64px;
  height: 64px;
  border-radius: 50%;
  display: grid;
  place-items: center;
  color: #fff;
  font-size: 26px;
  font-weight: 800;
  flex-shrink: 0;
}
.profile-info h2 {
  margin: 0 0 4px;
  font-size: 20px;
  font-weight: 700;
  color: var(--tx);
}
.profile-info p {
  margin: 0 0 8px;
  color: var(--mut);
  font-size: 13.5px;
}
.profile-role-badge {
  display: inline-block;
  padding: 3px 12px;
  border-radius: 99px;
  font-size: 11px;
  font-weight: 700;
  text-transform: uppercase;
  border: 1px solid;
}
.profile-meta-grid {
  display: grid;
  grid-template-columns: repeat(auto-fit, minmax(180px, 1fr));
  gap: 14px;
  border-top: 1px solid var(--line);
  padding-top: 18px;
}
.profile-meta-item {
  display: flex;
  flex-direction: column;
  gap: 2px;
}
.profile-meta-label {
  font-size: 11.5px;
  color: var(--mut);
  text-transform: uppercase;
  font-weight: 600;
}
.profile-meta-val {
  font-size: 14px;
  font-weight: 600;
  color: var(--tx);
}
.profile-input-wrap {
  position: relative;
  display: flex;
  align-items: center;
}
.profile-input-wrap input { width: 100%; padding-right: 60px; }
.fq-pw-toggle {
  position: absolute;
  right: 10px;
  background: transparent;
  border: none;
  color: var(--mut);
  cursor: pointer;
  font-size: 12px;
  font-weight: 600;
}

/* ------------------------------------------------------------------ Copilot Chat */
.chat {
  display: flex;
  flex-direction: column;
  gap: 14px;
  min-height: 480px;
}
.chat-suggestions {
  display: flex;
  gap: 8px;
  flex-wrap: wrap;
  padding-bottom: 12px;
  border-bottom: 1px solid var(--line);
}
.chat-chip {
  padding: 6px 14px;
  border-radius: 99px;
  background: var(--pri-bg);
  color: var(--pri);
  font-size: 12px;
  font-weight: 600;
  cursor: pointer;
  border: 1px solid rgba(14, 165, 233, 0.25);
  transition: all .15s;
}
.chat-chip:hover {
  background: var(--pri);
  color: #fff;
  transform: translateY(-1px);
}
.msgs {
  display: flex;
  flex-direction: column;
  gap: 16px;
  flex: 1;
  overflow-y: auto;
  padding: 10px 4px;
}
.m {
  max-width: 85%;
  padding: 14px 18px;
  border-radius: 16px;
  font-size: 13.5px;
  line-height: 1.55;
  box-shadow: var(--sh);
}
.m.u {
  align-self: flex-end;
  background: var(--pri-gradient);
  color: #fff;
  border-bottom-right-radius: 4px;
}
.m.a {
  align-self: flex-start;
  background: var(--card);
  color: var(--tx);
  border: 1px solid var(--line);
  border-bottom-left-radius: 4px;
}
.m.a table {
  width: 100%;
  border-collapse: collapse;
  margin: 10px 0;
  font-size: 12.5px;
}
.m.a th {
  background: var(--line-subtle);
  padding: 8px 12px;
  border-bottom: 1px solid var(--line);
  font-weight: 700;
}
.m.a td {
  padding: 8px 12px;
  border-bottom: 1px solid var(--line-subtle);
}
.cites {
  margin-top: 10px;
  padding: 8px 12px;
  border-radius: var(--r-sm);
  background: var(--line-subtle);
  border: 1px solid var(--line);
  font-size: 11.5px;
}
.cites summary {
  cursor: pointer;
  font-weight: 600;
  color: var(--pri);
}
.cites pre {
  margin: 8px 0 0;
  font-family: 'JetBrains Mono', monospace;
  font-size: 11px;
  color: var(--tx2);
  white-space: pre-wrap;
}

.cin {
  display: flex;
  gap: 10px;
  align-items: center;
  padding-top: 10px;
  border-top: 1px solid var(--line);
}
.cin input {
  flex: 1;
  padding: 12px 16px;
  border-radius: var(--r-sm);
  border: 1px solid var(--line);
  background: var(--card);
  color: var(--tx);
  font-size: 13.5px;
}
.cin input:focus {
  outline: none;
  border-color: var(--pri);
  box-shadow: 0 0 0 3px rgba(14, 165, 233, 0.16);
}

/* ------------------------------------------------------------------ Auth Screens */
.fq-auth-wrap {
  display: flex;
  min-height: 100vh;
  width: 100%;
  background: var(--bg);
}
.fq-auth-hero {
  flex: 1.1;
  background: #07131e;
  position: relative;
  display: flex;
  flex-direction: column;
  justify-content: flex-end;
  padding: 60px;
  overflow: hidden;
}
.fq-auth-hero-img {
  position: absolute;
  inset: 0;
  width: 100%;
  height: 100%;
  object-fit: cover;
  opacity: 0.65;
}
.fq-auth-hero-gradient {
  position: absolute;
  inset: 0;
  background: linear-gradient(180deg, rgba(7,19,30,0.2) 0%, rgba(7,19,30,0.92) 100%);
}
.fq-auth-hero-content {
  position: relative;
  z-index: 2;
  color: #fff;
  max-width: 480px;
}
.fq-auth-hero-title {
  font-size: 32px;
  font-weight: 800;
  margin: 0 0 12px;
  letter-spacing: -0.02em;
  line-height: 1.2;
}
.fq-auth-hero-desc {
  font-size: 15px;
  color: #cbd5e1;
  line-height: 1.6;
  margin: 0;
}

.fq-auth-card-wrap {
  flex: 1;
  display: flex;
  align-items: center;
  justify-content: center;
  padding: 40px 24px;
  overflow-y: auto;
}
.fq-auth-card {
  width: min(440px, 100%);
  background: var(--card);
  border-radius: var(--r-lg);
  border: 1px solid var(--line);
  box-shadow: var(--sh-lg);
  padding: 36px 32px;
}
.fq-auth-header {
  margin-bottom: 24px;
}
.fq-auth-title {
  font-size: 24px;
  font-weight: 800;
  margin: 0 0 6px;
  color: var(--tx);
  letter-spacing: -0.02em;
}
.fq-auth-sub {
  color: var(--mut);
  font-size: 13.5px;
  margin: 0;
}

.fq-alert {
  padding: 12px 16px;
  border-radius: var(--r-sm);
  font-size: 13px;
  margin-bottom: 16px;
}
.fq-alert-error {
  background: var(--red-bg);
  color: var(--red);
  border: 1px solid rgba(220, 38, 38, 0.25);
}
.fq-alert-success {
  background: var(--green-bg);
  color: var(--green);
  border: 1px solid rgba(5, 150, 105, 0.25);
}

.fq-btn-primary {
  width: 100%;
  padding: 11px 18px;
  border-radius: var(--r-sm);
  background: var(--pri-gradient);
  color: #fff;
  font-weight: 700;
  font-size: 14px;
  border: none;
  cursor: pointer;
  box-shadow: 0 2px 8px rgba(2, 132, 199, 0.3);
  transition: all .15s;
}
.fq-btn-primary:hover:not(:disabled) {
  filter: brightness(1.06);
  box-shadow: 0 4px 14px rgba(2, 132, 199, 0.45);
}
.fq-btn-outline {
  padding: 8px 16px;
  border-radius: var(--r-sm);
  background: transparent;
  border: 1px solid var(--line);
  color: var(--tx2);
  font-weight: 600;
  font-size: 13px;
  cursor: pointer;
  transition: all .15s;
}
.fq-btn-outline:hover {
  background: var(--line-subtle);
  border-color: var(--line-strong);
}

#toast {
  position: fixed;
  right: 20px;
  bottom: 20px;
  display: flex;
  flex-direction: column;
  gap: 10px;
  z-index: 3500;
}
#toast div {
  background: #0f172a;
  color: #fff;
  padding: 12px 20px;
  border-radius: var(--r-sm);
  max-width: 400px;
  font-size: 13px;
  font-weight: 500;
  box-shadow: 0 10px 30px rgba(0,0,0,0.25);
  animation: toast-in .25s ease;
}
#toast div.bad {
  background: #dc2626;
}
@keyframes toast-in {
  from { transform: translateY(12px); opacity: 0; }
  to { transform: translateY(0); opacity: 1; }
}

/* ------------------------------------------------------------------ Responsive */
@media (max-width: 860px) {
  #app { flex-direction: column; }
  .side {
    width: 100%;
    flex-direction: row;
    padding: 8px 12px;
    border-right: none;
    border-bottom: 1px solid var(--sidebar-border);
    overflow-x: auto;
  }
  .side .brand, .side .ksearch, .ngrp { display: none; }
  .row { grid-template-columns: 1fr; }
  .fq-auth-hero { display: none; }
  #drawer { width: 100vw; }
}

/* Freight map legends: visually separate data-source labels and markers. */
.fq-legend-layout-fix{}
.lg{display:flex;align-items:center;flex-wrap:wrap;gap:10px 18px;margin:10px 0 2px;padding:6px 0;color:var(--mut,#64748b);font-size:12px;line-height:1.5}
.lg span{display:inline-flex;align-items:center;gap:7px;white-space:normal}
.lg i{display:inline-block;flex:0 0 10px;width:10px;height:10px;border-radius:50%;border:1px solid rgba(0,0,0,.08)}

/* ===============================================================
   FreightQuote AI — Maritime Workspace finish
   Dashboard-only decorative styling. Public landing/auth unaffected.
   =============================================================== */
#app:has(.main) {
  --r: 16px;
  --r-sm: 10px;
  --r-lg: 20px;
  --sh: 0 2px 4px rgba(15,37,55,.025), 0 12px 32px rgba(20,70,86,.045);
  --sh-lg: 0 8px 18px rgba(15,37,55,.06), 0 20px 45px rgba(20,70,86,.085);
}
#app:has(.main) .main {
  background: var(--bg);
}
#app:has(.main) .top {
  position: relative;
  isolation: isolate;
  overflow: hidden;
  min-height: 92px;
  padding: 19px 30px;
  background: linear-gradient(105deg, var(--surface) 0%, var(--surface) 57%, rgba(14,165,175,.075) 100%);
  border-bottom: 1px solid var(--line);
  box-shadow: 0 6px 28px rgba(12,69,86,.03);
}
#app:has(.main) .top::after {
  content: '';
  position: absolute;
  right: -5px;
  top: 50%;
  width: min(47%, 620px);
  height: 175px;
  background: url("data:image/svg+xml,%3Csvg%20xmlns%3D%22http%3A%2F%2Fwww.w3.org%2F2000%2Fsvg%22%20width%3D%22700%22%20height%3D%22320%22%20viewBox%3D%220%200%20700%20320%22%20fill%3D%22none%22%3E%3Cg%20stroke%3D%22%230e9aab%22%20stroke-width%3D%221.1%22%20opacity%3D%22.26%22%3E%3Ccircle%20cx%3D%22525%22%20cy%3D%22152%22%20r%3D%2252%22%2F%3E%3Ccircle%20cx%3D%22525%22%20cy%3D%22152%22%20r%3D%2286%22%2F%3E%3Ccircle%20cx%3D%22525%22%20cy%3D%22152%22%20r%3D%22123%22%2F%3E%3Ccircle%20cx%3D%22525%22%20cy%3D%22152%22%20r%3D%22160%22%2F%3E%3Cpath%20d%3D%22M365%20152H687M525-8V312M392%2076Q525%20154%20661%2070M390%20235Q526%20150%20664%20235M416%2035Q535%20205%20631%20280M411%20272Q516%20102%20645%2023%22%2F%3E%3C%2Fg%3E%3Cg%20stroke%3D%22%2314b8a6%22%20stroke-width%3D%222%22%20stroke-linecap%3D%22round%22%20opacity%3D%22.43%22%3E%3Cpath%20d%3D%22M140%20233C190%20171%20229%20188%20270%20124S357%2095%20396%20153%22%20stroke-dasharray%3D%224%209%22%2F%3E%3Ccircle%20cx%3D%22140%22%20cy%3D%22233%22%20r%3D%225%22%20fill%3D%22%2314b8a6%22%2F%3E%3Ccircle%20cx%3D%22270%22%20cy%3D%22124%22%20r%3D%224%22%20fill%3D%22%2314b8a6%22%2F%3E%3Ccircle%20cx%3D%22396%22%20cy%3D%22153%22%20r%3D%225%22%20fill%3D%22%2314b8a6%22%2F%3E%3C%2Fg%3E%3Cg%20opacity%3D%22.36%22%20stroke%3D%22%230e7490%22%20stroke-linecap%3D%22round%22%20stroke-width%3D%221.5%22%3E%3Cpath%20d%3D%22M485%20177h80l-13%2016h-52z%22%2F%3E%3Cpath%20d%3D%22M518%20143v34m-14-24h26v24%22%2F%3E%3Cpath%20d%3D%22M468%20205c17-6%2030-6%2047%200s30%206%2047%200%22%2F%3E%3C%2Fg%3E%3C%2Fsvg%3E") center center / cover no-repeat;
  transform: translateY(-50%);
  opacity: .6;
  pointer-events: none;
  z-index: -1;
}
#app:has(.main) .top h1 {
  font-size: clamp(21px, 2vw, 27px);
  font-weight: 800;
  letter-spacing: -.045em;
  line-height: 1.22;
}
#app:has(.main) .top .sub {
  font-size: 13px;
  letter-spacing: .005em;
  margin-top: 4px;
}
#app:has(.main) .content {
  background-color: var(--bg);
  background-image:
    radial-gradient(ellipse at 95% 2%, rgba(14,165,175,.085), transparent 32%),
    radial-gradient(ellipse at 8% 43%, rgba(29,111,185,.055), transparent 37%),
    repeating-linear-gradient(90deg, transparent 0, transparent 79px, rgba(42,137,164,.019) 80px),
    repeating-linear-gradient(0deg, transparent 0, transparent 79px, rgba(42,137,164,.019) 80px);
  padding: 27px clamp(18px, 2.7vw, 42px) 90px;
  scroll-behavior: smooth;
}
#app:has(.main) .side {
  background:
    radial-gradient(ellipse at 20% -3%, rgba(22,193,181,.15), transparent 39%),
    linear-gradient(178deg, #0b263b 0%, #0a182e 55%, #081325 100%);
  border-right-color: rgba(146,196,211,.12);
  box-shadow: 8px 0 28px rgba(4,20,42,.09);
}
#app:has(.main) .side .brand {
  padding-bottom: 19px;
  margin-bottom: 12px;
}
#app:has(.main) .side .brand i {
  border-radius: 12px;
  background: linear-gradient(135deg,#10b5a9,#147da7);
  box-shadow: 0 6px 20px rgba(8,190,184,.3);
}
#app:has(.main) .side .nav {
  border-radius: 11px;
  margin-bottom: 2px;
  min-height: 40px;
  transition: background-color .18s ease, transform .18s ease, box-shadow .18s ease;
}
#app:has(.main) .side .nav:hover {
  background: rgba(99,224,214,.095);
  transform: translateX(2px);
}
#app:has(.main) .side .nav.on {
  background: linear-gradient(100deg,rgba(21,188,180,.25),rgba(30,133,186,.14));
  border: 1px solid rgba(86,218,211,.24);
  box-shadow: inset 3px 0 0 #2dd4bf, 0 5px 17px rgba(4,23,42,.22);
}
#app:has(.main) .card,
#app:has(.main) .admin-section,
#app:has(.main) .admin-table-wrap {
  background-color: var(--card);
  border: 1px solid var(--line);
  border-radius: 19px;
  box-shadow: var(--sh);
}
#app:has(.main) .card {
  padding: 23px 25px;
  transition: box-shadow .22s ease, transform .22s ease;
}
#app:has(.main) .card:hover {
  box-shadow: var(--sh-lg);
}
#app:has(.main) .card h3 {
  font-weight: 760;
  font-size: 15px;
  letter-spacing: -.025em;
}
#app:has(.main) .kpis,
#app:has(.main) .admin-kpi-grid { gap: 18px; }
#app:has(.main) .kpi,
#app:has(.main) .admin-kpi-card {
  position: relative;
  overflow: hidden;
  min-height: 115px;
  background:
    radial-gradient(circle at 98% 1%, rgba(14,165,175,.11), transparent 42%),
    var(--card);
  border: 1px solid var(--line);
  border-radius: 19px;
  box-shadow: var(--sh);
  padding: 19px 21px;
  transition: transform .2s ease, box-shadow .2s ease;
}
#app:has(.main) .kpi::before,
#app:has(.main) .admin-kpi-card::before {
  content: '';
  position: absolute;
  inset: 0 auto 0 0;
  width: 3px;
  background: linear-gradient(180deg,#0ea5a6,#0284c7);
  border-radius: 12px;
}
#app:has(.main) .kpi::after,
#app:has(.main) .admin-kpi-card::after {
  content: '';
  position: absolute;
  top: -45px;
  right: -45px;
  width: 125px;
  height: 125px;
  border: 1px solid rgba(22,159,172,.09);
  border-radius: 50%;
  box-shadow: 0 0 0 22px rgba(22,159,172,.027),0 0 0 47px rgba(22,159,172,.019);
  pointer-events: none;
}
#app:has(.main) .kpi:hover,
#app:has(.main) .admin-kpi-card:hover {
  transform: translateY(-3px);
  box-shadow: var(--sh-lg);
}
#app:has(.main) .kpi .l,
#app:has(.main) .admin-kpi-title {
  font-size: 10.5px;
  font-weight: 750;
  letter-spacing: .095em;
}
#app:has(.main) .kpi .v,
#app:has(.main) .admin-kpi-val {
  font-size: clamp(24px,2.2vw,31px);
  font-weight: 820;
  letter-spacing: -.055em;
  font-variant-numeric: tabular-nums;
}
#app:has(.main) .tbl,
#app:has(.main) .admin-table-wrap {
  border-radius: 16px;
  overflow-x: auto;
  border: 1px solid var(--line);
  box-shadow: 0 3px 15px rgba(18,66,91,.025);
}
#app:has(.main) .tbl th,
#app:has(.main) .admin-table th {
  background: linear-gradient(180deg,var(--line-subtle),var(--surface));
  letter-spacing: .065em;
  font-weight: 740;
}
#app:has(.main) .tbl td,
#app:has(.main) .admin-table td {
  border-bottom-color: var(--line-subtle);
}
#app:has(.main) .tabs {
  gap: 8px;
  padding-bottom: 5px;
}
#app:has(.main) .tabs b {
  border-radius: 9px 9px 0 0;
  padding: 11px 15px;
  transition: background .18s ease, color .18s ease;
}
#app:has(.main) .tabs b.on {
  background: var(--pri-bg);
}
#app:has(.main) button:not(.nav),
#app:has(.main) input,
#app:has(.main) select {
  transition: border-color .18s ease, box-shadow .18s ease, background .18s ease;
}
#app:has(.main) input:focus-visible,
#app:has(.main) select:focus-visible,
#app:has(.main) button:focus-visible {
  outline: 2px solid var(--pri);
  outline-offset: 2px;
}
body.fq-dark #app:has(.main) .content {
  background-image:
    radial-gradient(ellipse at 95% 2%, rgba(23,159,168,.10), transparent 38%),
    radial-gradient(ellipse at 3% 45%, rgba(38,101,172,.085), transparent 38%);
}
body.fq-dark #app:has(.main) .top {
  background: linear-gradient(105deg, var(--surface) 0%, var(--surface) 60%, rgba(14,165,175,.085) 100%);
}
body.fq-dark #app:has(.main) .kpi,
body.fq-dark #app:has(.main) .admin-kpi-card {
  background: radial-gradient(circle at 98% 1%,rgba(35,188,181,.10),transparent 50%),var(--card);
}
@media (max-width: 1100px) {
  #app:has(.main) .top::after { opacity:.31; width:36%; }
}
@media (max-width: 860px) {
  #app:has(.main) .top { min-height:72px; padding: 14px 19px; }
  #app:has(.main) .top::after { display:none; }
  #app:has(.main) .content { padding:18px 15px 65px; }
  #app:has(.main) .card { padding:18px; }
  #app:has(.main) .side .nav:hover { transform:none; }
}
@media (prefers-reduced-motion: reduce) {
  #app:has(.main) .kpi,
  #app:has(.main) .card,
  #app:has(.main) .admin-kpi-card,
  #app:has(.main) .side .nav { transition:none !important; transform:none !important; }
}


/* FreightQuote AI dashboard-only harbour photograph. All page/agent markup is unchanged. */
#app:has(.main) .content {
  background-color: var(--bg);
  background-image:
    linear-gradient(115deg, rgba(246,250,252,.89) 0%, rgba(246,250,252,.84) 48%, rgba(246,250,252,.86) 100%),
    radial-gradient(ellipse at 92% 8%, rgba(15,169,178,.10), transparent 42%),
    url('/static/dashboard-harbor.webp');
  background-position: center, center, center;
  background-size: cover, cover, cover;
  background-repeat: no-repeat;
}
body.fq-dark #app:has(.main) .content {
  background-color: var(--bg);
  background-image:
    linear-gradient(115deg, rgba(7,19,34,.86), rgba(10,23,40,.83)),
    radial-gradient(ellipse at 93% 6%, rgba(21,165,172,.12), transparent 40%),
    url('/static/dashboard-harbor.webp');
  background-position: center, center, center;
  background-size: cover, cover, cover;
  background-repeat: no-repeat;
}
@media (max-width: 860px) {
  #app:has(.main) .content, body.fq-dark #app:has(.main) .content {
    background-position: center, center, 61% center;
  }
}

/* Copilot voice controls – browser speech recognition and optional read-back. */
.fq-mic-button { white-space:nowrap; min-width:90px; }
.fq-mic-button[aria-pressed="true"] { outline:2px solid var(--pri); outline-offset:2px; }
.fq-voice-status { display:block; padding:3px 14px 12px; color:var(--mut); font-size:12px; }
.fq-voice-read { font-size:12px; padding:4px 9px; margin-top:8px; }
@media(max-width:600px) { .cin { flex-wrap:wrap; } .cin input { flex-basis:100%; } }


In [ ]:
%%writefile /content/app/static/app.js
"use strict";
const $ = (s, r = document) => r.querySelector(s);
const esc = s => String(s == null ? "" : s).replace(/[&<>"']/g, c => ({ "&": "&amp;", "<": "&lt;", ">": "&gt;", '"': "&quot;", "'": "&#39;" }[c]));

function h(tag, attrs, ...kids) {
  const e = document.createElement(tag);
  for (const k in (attrs || {})) {
    const v = attrs[k];
    if (v == null || v === false) continue;
    if (k === "class") e.className = v;
    else if (k === "html") e.innerHTML = v;
    else if (k.startsWith("on")) e.addEventListener(k.slice(2), v);
    else e.setAttribute(k, v === true ? "" : v);
  }
  for (const c of kids.flat()) {
    if (c != null && c !== false) e.append(c.nodeType ? c : document.createTextNode(c));
  }
  return e;
}

const TONE = {
  Critical: "#dc2626", High: "#dc2626", Fail: "#dc2626", rejected: "#dc2626", "On Hold": "#dc2626", overdue: "#dc2626",
  Warning: "#d97706", Medium: "#d97706", "Pending Approval": "#d97706", proposed: "#7c3aed",
  Info: "#2563eb", open: "#2563eb", Low: "#059669", Pass: "#059669", Approved: "#059669", approved: "#059669",
  resolved: "#059669", "auto-closed": "#64748b", live: "#059669", Healthy: "#059669", Watch: "#d97706", Rejected: "#64748b"
};
const tone = v => TONE[v] || (/crit|high|fail|bad|reject|hold/i.test(v) ? "#dc2626" : /warn|med|pend/i.test(v) ? "#d97706" : /ok|pass|low|good|appr|done|clear|live/i.test(v) ? "#059669" : "#64748b");

const ICON = { bot: "M12 8V4H8M4 12a2 2 0 0 1 2-2h12a2 2 0 0 1 2 2v6a2 2 0 0 1-2 2H6a2 2 0 0 1-2-2zM2 14h2M20 14h2M9 15v1M15 15v1", users: "M16 21v-2a4 4 0 0 0-4-4H6a4 4 0 0 0-4 4v2M9 11a4 4 0 1 0 0-8 4 4 0 0 0 0 8M22 21v-2a4 4 0 0 0-3-3.9M16 3.1a4 4 0 0 1 0 7.8", store: "M3 9l1-5h16l1 5M3 9v11h18V9M3 9a3 3 0 0 0 6 0 3 3 0 0 0 6 0 3 3 0 0 0 6 0M9 20v-6h6v6", boxes: "M21 8l-9-5-9 5 9 5zM3 8v8l9 5 9-5V8M12 13v8", megaphone: "M3 11v2a1 1 0 0 0 1 1h3l8 5V5L7 10H4a1 1 0 0 0-1 1zM19 8a5 5 0 0 1 0 8", messages: "M21 15a2 2 0 0 1-2 2H7l-4 4V5a2 2 0 0 1 2-2h14a2 2 0 0 1 2 2z", clipboard: "M9 4h6v3H9zM8 5H6a2 2 0 0 0-2 2v13a2 2 0 0 0 2 2h12a2 2 0 0 0 2-2V7a2 2 0 0 0-2-2h-2M9 13l2 2 4-4", bell: "M18 8a6 6 0 0 0-12 0c0 7-3 9-3 9h18s-3-2-3-9M13.7 21a2 2 0 0 1-3.4 0", network: "M12 2v6M12 16v6M5 6l5 4M19 6l-5 4M5 18l5-4M19 18l-5-4M12 8a4 4 0 1 0 0 8 4 4 0 0 0 0-8", flask: "M10 2v7L4 20a1 1 0 0 0 1 1h14a1 1 0 0 0 1-1l-6-11V2M8 2h8M7 15h10", upload: "M21 15v4a2 2 0 0 1-2 2H5a2 2 0 0 1-2-2v-4M17 8l-5-5-5 5M12 3v12", shield: "M12 22s8-4 8-10V5l-8-3-8 3v7c0 6 8 10 8 10", radar: "M12 12l7-7M12 2a10 10 0 1 0 10 10M12 6a6 6 0 1 0 6 6M12 10a2 2 0 1 0 2 2", route: "M6 19a3 3 0 1 0 0-6 3 3 0 0 0 0 6M18 11a3 3 0 1 0 0-6 3 3 0 0 0 0 6M9 16h6a3 3 0 0 0 0-6H9a3 3 0 0 1 0-6h3", dollar: "M12 2v20M17 5H9.5a3.5 3.5 0 0 0 0 7h5a3.5 3.5 0 0 1 0 7H6", truck: "M1 3h15v13H1zM16 8h4l3 3v5h-7M5.5 21a2.5 2.5 0 1 0 0-5 2.5 2.5 0 0 0 0 5M18.5 21a2.5 2.5 0 1 0 0-5 2.5 2.5 0 0 0 0 5", cloud: "M17.5 19a4.5 4.5 0 1 0-1.4-8.8A6 6 0 1 0 5 15a4 4 0 0 0 1 4z", trend: "M3 17l6-6 4 4 8-8M15 7h6v6", chart: "M3 3v18h18M7 15l4-4 3 3 5-6", check: "M22 11.1V12a10 10 0 1 1-5.9-9.1M22 4L12 14l-3-3", user: "M20 21v-2a4 4 0 0 0-4-4H8a4 4 0 0 0-4 4v2M12 11a4 4 0 1 0 0-8 4 4 0 0 0 0 8", file: "M14 2H6a2 2 0 0 0-2 2v16a2 2 0 0 0 2 2h12a2 2 0 0 0 2-2V8zM14 2v6h6M9 13h6M9 17h6" };

const icon = n => {
  const s = document.createElementNS("http://www.w3.org/2000/svg", "svg");
  s.setAttribute("viewBox", "0 0 24 24");
  const d = ICON[n] || ICON.bot;
  s.innerHTML = d.split("|").map(p => `<path d="${p}" />`).join("");
  return s;
};

async function api(url, opts = {}) {
  const token = localStorage.getItem("fq_token");
  const headers = { ...(opts.headers || {}) };
  if (token) headers["Authorization"] = "Bearer " + token;
  let body = opts.body;
  if (opts.json) {
    headers["Content-Type"] = "application/json";
    body = JSON.stringify(opts.json);
  }
  try {
    const res = await fetch(url, { ...opts, method: opts.method || (opts.json !== undefined ? 'POST' : 'GET'), headers, body });
    // Auth endpoints return meaningful errors (including account lockout).
    // Never overwrite them with the generic protected-session message.
    if (res.status === 401) {
      const data = await res.json().catch(() => ({}));
      const reason = data.msg || data.detail || "Session expired. Please sign in.";
      if (!url.startsWith("/api/auth/")) {
        localStorage.removeItem("fq_token");
        S.me = null;
        shellMounted = false;
        if (!location.hash.startsWith("#/login") && !location.hash.startsWith("#/signup") && location.hash !== "#/" && location.hash !== "") {
          window.location.hash = "#/login";
        }
      }
      return { ...data, ok: false, status: 401, msg: reason };
    }
    const ct = res.headers.get("content-type") || "";
    if (ct.includes("application/json")) {
      const data = await res.json();
      if (!res.ok && data && typeof data === "object") {
        data.ok = false;
        data.status = res.status;
        data.msg = data.msg || data.detail || `Request failed (HTTP ${res.status}).`;
      }
      return data;
    }
    const responseText = await res.text();
    return { ok: res.ok, status: res.status, msg: res.ok ? "Done" : `Server error (HTTP ${res.status}).`, text: responseText };
  } catch (err) {
    console.error("FreightQuote API connection failed:", url, err);
    return { ok: false, msg: "Cannot reach the server. Check that the Colab runtime and application are still running." };
  }
}

function toast(msg, bad = false) {
  if (!msg) return;
  const t = $("#toast");
  if (!t) return;
  const d = h("div", { class: bad ? "bad" : "" }, msg);
  t.append(d);
  setTimeout(() => {
    d.style.opacity = "0";
    d.style.transform = "translateY(10px)";
    d.style.transition = "all 0.25s";
    setTimeout(() => d.remove(), 260);
  }, 3500);
}

function md(text) {
  if (!text) return "";
  let s = esc(text);
  // Markdown tables
  s = s.replace(/((?:\|[^\n]+\|\r?\n)+)/g, (match) => {
    const lines = match.trim().split(/\r?\n/);
    if (lines.length >= 2 && lines[0].includes("|") && lines[1].includes("---")) {
      const headers = lines[0].split("|").slice(1, -1).map(c => `<th>${c.trim()}</th>`).join("");
      const bodyRows = lines.slice(2).map(row => {
        const cells = row.split("|").slice(1, -1).map(c => `<td>${c.trim()}</td>`).join("");
        return `<tr>${cells}</tr>`;
      }).join("");
      return `<div class="tbl" style="margin:10px 0;"><table style="width:100%;border-collapse:collapse;"><thead><tr>${headers}</tr></thead><tbody>${bodyRows}</tbody></table></div>`;
    }
    return match;
  });
  s = s.replace(/^### (.*$)/gim, '<h4 style="margin:8px 0 4px;font-size:14px;">$1</h4>');
  s = s.replace(/^## (.*$)/gim, '<h3 style="margin:10px 0 6px;font-size:15px;">$1</h3>');
  s = s.replace(/^# (.*$)/gim, '<h2 style="margin:12px 0 8px;font-size:16px;">$1</h2>');
  s = s.replace(/\*\*(.*?)\*\*/g, '<b>$1</b>');
  s = s.replace(/\*(.*?)\*/g, '<i>$1</i>');
  s = s.replace(/`([^`]+)`/g, '<code style="background:rgba(0,0,0,0.06);padding:2px 5px;border-radius:4px;font-size:12px;font-family:monospace;">$1</code>');
  s = s.replace(/\n/g, '<br>');
  return s;
}

const S = {
  me: null,
  page: null,
  tab: null,
  params: {},
  seq: 0,
  timer: null,
  live: 0,
  tree: null,
  es: null,
  pulse: null,
  lang: "English"
};

// ---------------------------------------------------------------- mountList & updList
function mountList(parent, blocks, key) { const hs = blocks.map((b, i) => mount(b, key + "/" + i)); hs.forEach(x => parent.append(x.el)); return hs }
function updList(parent, hs, blocks, key) { if (hs.length !== blocks.length || hs.some((x, i) => x.t !== blocks[i].t || x.k !== (blocks[i].key || blocks[i].id || ""))) { parent.replaceChildren(); return mountList(parent, blocks, key) } hs.forEach((x, i) => x.update(blocks[i])); return hs }

const R = {};

// Every server-driven page is assembled from typed blocks. Keep the adapter
// separate from individual renderers so all pages (including nested layouts)
// can render and update consistently.
function mount(block, key = "block") {
  const b = block && typeof block === "object" ? block : { t: "callout", tone: "warn", text: "This section could not be displayed." };
  const type = String(b.t || "");
  const id = String(b.key || b.id || "");
  const factory = R[type];
  if (typeof factory !== "function") {
    const el = h("div", { class: "callout warn", role: "status" }, "This section is temporarily unavailable.");
    console.warn("FreightQuote: unrecognized page block", type);
    return { t: type, k: id, el, update() {} };
  }
  try {
    const view = factory(b, id, key);
    if (!view || !view.el) throw new Error("Invalid block view");
    return {
      t: type, k: id, el: view.el,
      update(next) {
        try {
          if (typeof view.update === "function") view.update(next);
        } catch (err) {
          console.error("FreightQuote: update error in", type, err);
          view.el.replaceChildren(h("div", { class: "callout warn" }, "Unable to refresh this section. Please reopen the page."));
        }
      }
    };
  } catch (err) {
    console.error("FreightQuote: render error in", type, err);
    const el = h("div", { class: "callout warn", role: "alert" }, "Unable to display this section. Please try again.");
    return { t: type, k: id, el, update() {} };
  }
}

// ---------------------------------------------------------------- Blocks: Table
R.table = b => {
  const el = h("div");
  const title = h("h3", { style: "margin:0 0 6px;font-size:14px;font-weight:700;color:var(--tx);" });
  const sub = h("div", { class: "sub", style: "color:var(--mut);font-size:12.5px;margin-bottom:8px;" });
  const srch = h("input", { class: "srch", placeholder: "Search records…", oninput: () => { st.q = srch.value.toLowerCase(); draw(); } });
  const box = h("div", { class: "tbl" });
  const st = { sort: null, dir: 1, q: "" };
  let cur = b;
  el.append(title, sub, srch, box);

  const draw = () => {
    const b = cur;
    title.textContent = b.title || "";
    title.style.display = b.title ? "" : "none";
    sub.textContent = b.sub || "";
    sub.style.display = b.sub ? "" : "none";
    srch.style.display = b.search ? "" : "none";
    if (b.h) box.style.maxHeight = b.h + "px";

    let rows = b.rows || [];
    if (st.q) rows = rows.filter(r => Object.values(r).some(v => String(v).toLowerCase().includes(st.q)));
    if (st.sort) rows = [...rows].sort((x, y) => {
      const a = x[st.sort], c = y[st.sort];
      return (a == null) - (c == null) || (a > c ? 1 : a < c ? -1 : 0) * st.dir;
    });

    const th = h("tr", null, ...b.cols.map(c => h("th", {
      onclick: () => {
        st.dir = st.sort === c.k ? -st.dir : 1;
        st.sort = c.k;
        draw();
      }
    }, c.label + (st.sort === c.k ? (st.dir > 0 ? " ▲" : " ▼") : ""))));

    const tb = rows.slice(0, 400).map(r => {
      const tr = h("tr", null, ...b.cols.map(c => {
        const v = r[c.k];
        let n;
        if (c.type === "pill") n = h("span", { class: "pill " + String(v).replace(/\s+/g, "_") }, String(v));
        else if (c.type === "money") n = "$" + Number(v || 0).toLocaleString();
        else if (c.type === "num") n = Number(v || 0).toLocaleString();
        else if (c.type === "bar") {
          const pct = Math.min(100, Math.max(0, (Number(v || 0) / (c.max || 5)) * 100));
          n = h("div", { style: "display:flex;align-items:center;gap:8px;" },
            h("div", { style: `height:6px;width:70px;background:var(--line);border-radius:99px;overflow:hidden;` },
              h("div", { style: `height:100%;width:${pct}%;background:var(--pri);border-radius:99px;` })
            ),
            h("span", { style: "font-size:12px;font-weight:600;" }, String(v))
          );
        }
        else n = String(v == null ? "—" : v);
        return h("td", null, n);
      }));

      if (b.open) {
        tr.classList.add("clickable-row");
        tr.onclick = (e) => {
          if (e.target.closest("button, a, input, select, label, .no-open")) return;
          UI.openEntity(r[b.open.col]);
        };
      }
      return tr;
    });

    const sc = box.scrollTop;
    box.replaceChildren(h("table", null, h("thead", null, th), h("tbody", null, ...tb)));
    box.scrollTop = sc;
    if (!rows.length) box.append(h("div", { style: "padding:20px;text-align:center;color:var(--mut);font-size:13px;" }, "No matching records found."));
  };

  draw();
  return { el, update: nb => { cur = nb; draw(); } };
};

// ---------------------------------------------------------------- Blocks: KPIs, Hero, Cards
R.kpis = b => {
  const el = h("div", { class: "kpis" });
  const render = it => {
    // The server-side blocks.KPIs contract uses {label, value, sub, tone}.
    // Handle old payloads without ever showing literal "undefined".
    const v = it.value ?? it.val;
    return h("div", { class: "kpi " + (it.tone || it.sev || "") },
      h("div", { class: "l" }, it.label || "Metric"),
      h("div", { class: "v" }, v == null || v === "" ? "—" : String(v)),
      it.sub ? h("div", { class: "s" }, it.sub) : null
    );
  };
  const update = next => el.replaceChildren(...(next.items || []).map(render));
  update(b);
  return { el, update };
};

R.card = (b, id, pfx) => {
  const el = h("div", { class: "card" });
  if (b.title) el.append(h("h3", null, b.title));
  mountList(el, b.blocks || [], pfx);
  return { el, update() {} };
};

R.row = (b, id, pfx) => {
  const el = h("div", { class: "row" });
  if (b.ratio) el.style.gridTemplateColumns = b.ratio.map(r => r + "fr").join(" ");
  mountList(el, b.blocks || [], pfx);
  return { el, update() {} };
};

R.col = (b, id, pfx) => {
  const el = h("div", { class: "col" });
  mountList(el, b.blocks || [], pfx);
  return { el, update() {} };
};

R.callout = b => {
  const el = h("div", { class: "callout " + (b.tone || "info") });
  if (b.title) el.append(h("div", { style: "font-weight:700;margin-bottom:4px;" }, b.title));
  el.append(h("div", null, b.text));
  return { el, update() {} };
};

R.button = b => {
  const el = h("button", {
    class: "btn " + (b.tone || "primary"),
    onclick: async () => {
      if (b.payload && b.payload.download) { window.open(b.url, "_blank"); return; }
      if (b.confirm && !confirm(b.confirm)) return;
      el.disabled = true;
      const r = await api(b.url, { json: b.payload || {} });
      toast(r.msg || "Done", !r.ok);
      el.disabled = false;
      if (r.refresh) reloadPage();
    }
  }, b.label);
  return { el, update() {} };
};

R.hero = b => {
  const el = h("div", { class: "card", style: "display:flex;align-items:center;gap:20px;border-left:4px solid var(--pri);" });
  const up = b => {
    const c = b.risk ? tone(b.risk) : "var(--pri)";
    el.replaceChildren(
      h("div", { style: "flex:1" },
        h("div", { style: "display:flex;align-items:center;gap:10px;margin-bottom:6px;" },
          h("h2", { style: "margin:0;font-size:17px;font-weight:700;color:var(--tx);" }, b.title),
          b.risk ? h("span", { class: "pill " + b.risk }, b.risk) : null
        ),
        h("p", { style: "margin:0;color:var(--mut);font-size:13px;" }, b.blurb)
      ),
      b.can_run ? h("button", {
        class: "btn primary sm",
        onclick: async e => {
          e.target.disabled = true;
          const r = await api("/api/run_agent", { json: { agent: b.agent } });
          toast(r.msg, !r.ok);
          e.target.disabled = false;
          reloadPage();
        }
      }, "Run analysis") : null
    );
  };
  up(b);
  return { el, update: up };
};

// ---------------------------------------------------------------- Tickets block
R.tickets = b => {
  const el = h("div"), st = { f: "all", open: new Set() }; let cur = b; const body = h("div"), fl = h("div", { class: "filters" }); el.append(h("h3", { style: "margin:0 0 6px;font-size:13.5px" }, b.title || "Tickets"), h("div", { class: "sub" }, b.sub || ""), fl, body);
  const act = async (id, a, json) => { const r = await api(`/api/ticket/${id}/${a}`, { json: json || {} }); toast(r.msg, !r.ok); if (document.activeElement) document.activeElement.blur(); st.sig = null; reloadPage(true) };
  const draw = () => {
    const items = cur.items; st.sig = JSON.stringify(items) + st.f + [...st.open]; const cnt = k => k === "all" ? items.length : k === "proposed" ? items.filter(i => i.status === "proposed").length : k === "overdue" ? items.filter(i => i.overdue).length : items.filter(i => i.sev === k).length;
    fl.replaceChildren(...["all", "proposed", "Critical", "Warning", "Info", "overdue"].map(k => h("span", { class: "chip" + (st.f === k ? " on" : ""), onclick: () => { st.f = k; draw() } }, (k === "proposed" ? "awaiting approval" : k) + " · " + cnt(k))));
    const list = items.filter(i => st.f === "all" || (st.f === "proposed" ? i.status === "proposed" : st.f === "overdue" ? i.overdue : i.sev === st.f));
    body.replaceChildren(...list.slice(0, 80).map(i => {
      const c = tone(i.sev); const t = h("div", { class: "tk " + i.sev + " " + i.status },
        h("div", { class: "h" }, h("b", null, "#" + i.id + " " + i.type.replace(/_/g, " ")), h("span", { class: "pill", style: `color:${c};border-color:${c}66` }, i.sev), i.status === "proposed" ? h("span", { class: "pill", style: "color:#8e4ec6;border-color:#8e4ec666" }, "awaiting approval") : null, i.overdue ? h("span", { class: "pill", style: "color:#e5484d;border-color:#e5484d66" }, "overdue") : null, h("a", { href: "#", onclick: e => { e.preventDefault(); UI.openEntity(i.target) } }, i.target)),
        h("div", { class: "d" }, i.detail), h("div", { class: "m" }, h("span", null, "agent: " + i.agent), h("span", null, "owner: " + (i.owner || "–")), h("span", null, "assignee: " + (i.assignee || "unassigned")), h("span", null, "due " + (i.due || "–")), h("span", null, i.ts)),
        i.chain && i.chain.length ? h("div", { class: "m" }, h("span", null, "approval chain: "), ...i.chain.map(s => h("span", { class: "pill", style: `color:${s.status === "approved" ? "#1a9e6e" : s.status === "rejected" ? "#e5484d" : "#8e4ec6"};border-color:currentColor` }, "L" + s.step + " " + s.role + (s.status === "approved" ? " ✓ " + s.by : s.status === "rejected" ? " ✗" : " …")))) : null,
        i.status === "proposed" && i.turn && !i.can_approve ? h("div", { class: "m" }, "Waiting for a " + i.turn + " to decide") : null,
        i.note ? h("div", { class: "m" }, "note: " + i.note) : null,
        h("div", { class: "a" }, i.status === "proposed" && i.can_approve ? [h("button", { class: "btn ok sm", onclick: () => act(i.id, "approve") }, "Approve & execute"), h("button", { class: "btn danger sm", onclick: () => act(i.id, "reject") }, "Reject")] : null,
          i.quote && i.can_approve && i.status !== "proposed" ? [h("button", { class: "btn ok sm", onclick: () => act(i.id, "approve") }, "Approve quote"), h("button", { class: "btn danger sm", onclick: () => act(i.id, "reject") }, "Reject")] : null,
          i.can_resolve ? [h("button", { class: "btn sm", onclick: () => act(i.id, "resolve") }, "Resolve"), (() => { const a = h("input", { class: "in", style: "width:170px", placeholder: "assign to email", value: i.assignee || "" }); return h("span", { style: "display:flex;gap:4px" }, a, h("button", { class: "btn sm", onclick: () => a.value && act(i.id, "assign", { assignee: a.value }) }, "Assign")) })()] : null,
          h("button", { class: "btn sm", onclick: () => { st.open.has(i.id) ? st.open.delete(i.id) : st.open.add(i.id); draw() } }, "Comments (" + i.comments.length + ")")),
        st.open.has(i.id) ? h("div", { class: "cm" }, ...i.comments.map(c => h("div", null, h("b", null, c.by), ` ${c.ts}: ${c.body}`)), (() => { const t = h("input", { class: "in", placeholder: "Write a comment and press Enter", onkeydown: e => { if (e.key === "Enter" && t.value.trim()) act(i.id, "comment", { body: t.value }) } }); return t })()) : null); return t
    })); if (!list.length) body.append(h("div", { class: "sub" }, "Nothing here."))
  };
  draw(); return { el, update: nb => { if (el.contains(document.activeElement) && document.activeElement.tagName === "INPUT") { cur = nb; return } if (JSON.stringify(nb.items) + st.f + [...st.open] === st.sig) return; cur = nb; draw() } }
};

// ---------------------------------------------------------------- Charts block
const FN = { size3: v => Math.max(8, Math.min(34, Math.sqrt(Math.abs(v[3] || 10)) * 1.6)), scatterTip: p => `<b>${esc(p.value[2] || "")}</b><br>${esc(p.seriesName || "")}<br>${p.value[0]}, ${p.value[1]}`, nodeTip: p => p.dataType === "node" ? `<b>${esc(p.name)}</b><br>${esc(p.data.tip || "")}` : "", nodeLabel: p => p.data.showLabel ? p.name : "" };
function revive(o) { if (Array.isArray(o)) return o.map(revive); if (o && typeof o === "object") { if (o.__fn) return FN[o.__fn]; const r = {}; for (const k in o) r[k] = revive(o[k]); return r } return o }
if(typeof echarts!=="undefined"&&echarts&&echarts.registerTheme)echarts.registerTheme("lt", { color: ["#5b5bd6", "#12a594", "#f5a524", "#8e4ec6", "#e5484d", "#30a46c", "#d6409f", "#3e63dd", "#8b8d98"], backgroundColor: "transparent", textStyle: { fontFamily: "Inter, system-ui, sans-serif", color: "#454a58" }, legend: { textStyle: { color: "#454a58" } }, categoryAxis: { axisLine: { lineStyle: { color: "#d9dce8" } }, axisTick: { show: false }, axisLabel: { color: "#6b7080" }, splitLine: { lineStyle: { color: "#eceef6" } } }, valueAxis: { axisLine: { show: false }, axisTick: { show: false }, axisLabel: { color: "#6b7080" }, splitLine: { lineStyle: { color: "#eceef6" } } }, tooltip: { backgroundColor: "#fff", borderColor: "#e6e8f0", textStyle: { color: "#1c2024" }, extraCssText: "box-shadow:0 8px 24px rgba(20,24,40,.12);border-radius:10px" } });
const CH = []; function lazy(box, fn) { let d = 0; const t = () => { if (d) return; if (box.isConnected && box.clientWidth > 0) { d = 1; fn() } else setTimeout(t, 60) }; t() }
R.chart = b => {
  const el = h("div"), box = h("div", { style: `height:${b.h || 290}px` }); el.append(h("h3", { style: "margin:0 0 4px;font-size:13.5px;display:" + (b.title ? "block" : "none") }, b.title || ""), box); let ch = null, cur = b;
  const init = () => {
    if(typeof echarts==="undefined"||!echarts||!echarts.init){box.innerHTML="<div style=\"padding:16px;color:#64748b;font-size:13px;\">Chart library loading...</div>";return;}ch = echarts.init(box, "lt", { renderer: "canvas" }); CH.push(ch); ch.setOption(revive(cur.opt), true);
    ch.on("click", p => { const o = cur.open; if (!o) return; let id = null; if (o.bar) id = p.name; else if (o.scatter) id = p.value && p.value[2]; else if (o.graph) id = p.data && p.data.open; else if (p.data && p.data.open) id = p.data.open; if (id) UI.openEntity(id) });
    if (cur.open) box.style.cursor = "pointer"
  };
  lazy(box, init);
  return { el, update: nb => { cur = nb; if (ch) { const r = JSON.stringify(nb.opt); if (box._h !== r) { box._h = r; ch.setOption(revive(nb.opt), { replaceMerge: ["series", "xAxis", "yAxis"] }) } } }, };
};
window.addEventListener("resize", () => CH.forEach(c => { try { c.resize() } catch (e) { } }));

// ---------------------------------------------------------------- Maps block
function shiftGeo(g, sh) { const mv = c => typeof c[0] === "number" ? [c[0] + sh, c[1]] : c.map(mv); return { type: "FeatureCollection", features: g.features.map(f => ({ type: "Feature", properties: f.properties, geometry: { type: f.geometry.type, coordinates: mv(f.geometry.coordinates) } })) } }
let WORLD = null; async function world() { if (WORLD) return WORLD; try { const t = await (await fetch("/static/vendor/world.topo.json")).json(); WORLD = topojson.feature(t, t.objects.countries) } catch (e) { WORLD = { type: "FeatureCollection", features: [] } } return WORLD }
R.map = b => {
  const el = h("div"), box = h("div", { class: "map", style: `height:${b.h || 470}px` }), lg = h("div", { class: "lg" }); el.append(box, lg); let M = null, grp = null, cur = b;
  const draw = () => {
    grp.clearLayers(); const SH = [-360, 0, 360]; cur.layers.forEach(L => {
      if (L.type === "lines") L.lines.forEach(l => { if (l.pts.length < 2) return; SH.forEach(sh => { const p = L2.polyline(l.pts.map(q => [q[0], q[1] + sh]), { color: l.color || "#5b5bd6", weight: l.w || 2, dashArray: l.dash || null, opacity: .9, lineJoin: "round", renderer: M._r }); if (l.tip) p.bindTooltip(l.tip, { sticky: true }); grp.addLayer(p) }) });
      else if (L.type === "circles") L.items.forEach(c => SH.forEach(sh => { const k = L2.circle([c.lat, c.lon + sh], { radius: c.r_m, color: c.color || "#e5484d", weight: 1, fillOpacity: .18, renderer: M._r }); if (c.tip) k.bindTooltip(c.tip); grp.addLayer(k) }));
      else if (L.type === "points") [...L.pts].sort((a, c) => (a.z || 0) - (c.z || 0)).forEach(p => SH.forEach(sh => { const m = L2.circleMarker([p.lat, p.lon + sh], { radius: p.r || 5, color: "#ffffff", weight: 1.2, fillColor: p.color || "#5b5bd6", fillOpacity: .95, renderer: M._r }); if (p.tip) m.bindTooltip(p.tip, { sticky: true }); const o = p.open || (cur.open ? p.id : null) || p.id; if (o) m.on("click", () => UI.openEntity(o)); grp.addLayer(m) }))
    });
    lg.replaceChildren(...(cur.legend || []).map(([t, c]) => h("span", null, h("i", { style: "background:" + c }), t)))
  };
  const L2 = typeof L !== "undefined" ? L : null; if (!L2) { box.innerHTML = "<div style=\"padding:16px;color:#64748b;font-size:13px;\">Map library loading...</div>"; return; } const go = async () => {
    M = L2.map(box, { zoomControl: true, worldCopyJump: true, attributionControl: false, minZoom: 1, preferCanvas: true }); M._r = L2.canvas({ padding: .5 }); const w = await world();[-360, 0, 360].forEach(sh => L2.geoJSON(sh ? shiftGeo(w, sh) : w, { renderer: M._r, style: { color: "#b9b2a0", weight: .8, fillColor: "#f3efe6", fillOpacity: 1 }, interactive: false }).addTo(M)); grp = L2.layerGroup().addTo(M);
    const v = cur.view || {}; if (v.bounds) M.fitBounds(v.bounds); else M.setView(v.center || [20, 0], v.zoom || 2); draw(); setTimeout(() => M.invalidateSize(), 50)
  };
  lazy(box, go);
  return { el, update: nb => { cur = nb; if (M && grp) draw() } }
};

// ---------------------------------------------------------------- Forms block
R.form = b => {
  const el = h("div");
  const res = h("div", { class: "res" });
  const f = h("form", { class: "f" + (b.inline ? " inline" : "") });
  let ins = {};

  b.fields.forEach(fl => {
    const w = h("div", { class: "fld" });
    if (fl.type !== "hidden") w.append(h("label", null, fl.label || fl.name));
    let i;
    if (fl.type === "select") i = h("select", { name: fl.name }, ...fl.options.map(o => h("option", { value: o, selected: String(o) === String(fl.value) }, o)));
    else if (fl.type === "multi") { i = h("select", { name: fl.name, multiple: true, size: 5 }, ...fl.options.map(o => h("option", { value: o, selected: (fl.value || []).includes(o) }, o))); }
    else if (fl.type === "checks") { i = h("div", { class: "chk" }, ...fl.options.map(o => h("label", null, h("input", { type: "checkbox", value: o, checked: (fl.value || []).includes(o) }), o))); i.get = () => [...i.querySelectorAll("input:checked")].map(x => x.value); }
    else if (fl.type === "textarea") i = h("textarea", { name: fl.name, placeholder: fl.placeholder || "" }, fl.value || "");
    else if (fl.type === "range") { const v = h("span", { style: "color:var(--pri);font-weight:700;margin-left:8px;" }, fl.value); i = h("input", { type: "range", name: fl.name, min: fl.min, max: fl.max, step: fl.step || 1, value: fl.value, oninput: e => v.textContent = e.target.value }); w.firstChild.append(" ", v); }
    else if (fl.type === "file") i = h("input", { type: "file", name: fl.name, accept: fl.accept || null });
    else if (fl.type === "hidden") { i = h("input", { type: "hidden", name: fl.name, value: fl.value }); w.style.display = "none"; }
    else i = h("input", { type: fl.type === "number" ? "number" : fl.type === "date" ? "date" : fl.type === "password" ? "password" : "text", name: fl.name, value: fl.value == null ? "" : fl.value, min: fl.min, max: fl.max, step: fl.step || (fl.type === "number" ? "any" : null), placeholder: fl.placeholder || "" });
    i.fl = fl;
    ins[fl.name] = i;
    w.append(i);
    f.append(w);
  });

  const btn = h("button", { class: "btn " + (b.tone || "primary"), type: "submit" }, b.submit || "Submit");
  f.append(h("div", { class: "fld" }, btn));

  f.addEventListener("submit", async e => {
    e.preventDefault();
    const v = {};
    let fd = null;
    for (const n in ins) {
      const i = ins[n], fl = i.fl;
      if (fl.type === "file") { if (i.files[0]) { fd = fd || new FormData(); fd.append(n, i.files[0]); } }
      else if (fl.type === "checks") v[n] = i.get();
      else if (fl.type === "multi") v[n] = [...i.selectedOptions].map(o => o.value);
      else if (fl.type === "number" || fl.type === "range") v[n] = i.value === "" ? null : +i.value;
      else v[n] = i.value;
    }
    if (b.url.startsWith("get:")) {
      const q = new URLSearchParams();
      for (const n in v) if (v[n] !== "" && v[n] != null) q.set(n, v[n]);
      window.open(b.url.slice(4) + "?" + q, "_blank");
      return;
    }
    if (b.url.startsWith("page:")) {
      const parts = b.url.split(":");
      const pid = parts[1];
      const tab = parts[2] || S.tab;
      UI.go(pid, tab, v);
      return;
    }
    btn.disabled = true;
    const t0 = btn.textContent;
    btn.textContent = "Processing…";
    let r;
    if (fd) {
      for (const n in v) fd.append(n, Array.isArray(v[n]) ? v[n].join(",") : v[n]);
      r = await api(b.url, { method: "POST", body: fd });
    } else {
      r = await api(b.url, { json: v });
    }
    btn.disabled = false;
    btn.textContent = t0;
    toast(r.msg || (r.ok ? "Done" : "Failed"), !r.ok);
    res.replaceChildren();
    if (r.blocks) mountList(res, r.blocks, "res");
    if (r.refresh) reloadPage(true);
  });

  if (b.title || b.sub) {
    const c = h("div", { class: "card" }, b.title ? h("h3", null, b.title) : null, b.sub ? h("div", { class: "sub", style: "color:var(--mut);font-size:12.5px;margin-bottom:14px;" }, b.sub) : null, f, res);
    el.append(c);
  } else {
    el.append(f, res);
  }
  return { el, update() {} };
};

// ---------------------------------------------------------------- Blocks: AI Copilot Chat
async function sse(url, json, on) {
  const token = localStorage.getItem("fq_token");
  const headers = { "Content-Type": "application/json" };
  if (token) headers["Authorization"] = "Bearer " + token;
  const r = await fetch(url, { method: "POST", credentials: "same-origin", headers, body: JSON.stringify(json) });
  if (!r.ok) {
    on({ ev: "final", text: "Error " + r.status + " connecting to database." });
    return;
  }
  const rd = r.body.getReader(), dec = new TextDecoder();
  let buf = "";
  for (; ;) {
    const { value, done } = await rd.read();
    if (done) break;
    buf += dec.decode(value, { stream: true });
    let i;
    while ((i = buf.indexOf("\n\n")) >= 0) {
      const m = buf.slice(0, i);
      buf = buf.slice(i + 2);
      if (m.startsWith("data: ")) {
        try { on(JSON.parse(m.slice(6))); } catch (e) {}
      }
    }
  }
}

R.chat = b => {
  let cid = null, hist = [];
  const el = h("div", { class: "card chat" });
  const msgs = h("div", { class: "msgs" });
  const inp = h("input", {
    placeholder: "Ask about routes, quotes, carriers, weather or customs in plain English…",
    onkeydown: e => { if (e.key === "Enter") send(inp.value); }
  });
  const SpeechRecognitionClass = window.SpeechRecognition || window.webkitSpeechRecognition;
  const voiceStatus = h("small", { class: "fq-voice-status", role: "status", "aria-live": "polite" }, "You can type a question or use the microphone.");
  let recognition = null;
  let listening = false;
  const micBtn = h("button", {
    type: "button", class: "btn fq-mic-button",
    title: "Speak your Copilot question", "aria-label": "Speak your Copilot question",
    onclick: () => {
      if (!SpeechRecognitionClass) {
        voiceStatus.textContent = "Voice input is not supported here. Try Chrome or Edge, or type your question.";
        return;
      }
      if (!window.isSecureContext) {
        voiceStatus.textContent = "Voice input needs HTTPS or localhost.";
        return;
      }
      if (listening && recognition) { recognition.stop(); return; }
      try {
        recognition = new SpeechRecognitionClass();
        recognition.lang = navigator.language || "en-US";
        recognition.interimResults = true;
        recognition.continuous = false;
        recognition.maxAlternatives = 1;
        let finalText = "";
        recognition.onstart = () => {
          listening = true; micBtn.textContent = "Stop";
          micBtn.setAttribute("aria-pressed", "true");
          voiceStatus.textContent = "Listening… speak your freight question.";
        };
        recognition.onresult = event => {
          let interim = "";
          for (let i = event.resultIndex; i < event.results.length; i++) {
            const part = event.results[i][0].transcript;
            if (event.results[i].isFinal) finalText += part + " ";
            else interim += part;
          }
          inp.value = (finalText + interim).trim();
          if (finalText.trim()) voiceStatus.textContent = "Question captured. Review it, then press Ask Copilot.";
        };
        recognition.onerror = event => {
          const reason = event.error === "not-allowed" || event.error === "service-not-allowed"
            ? "Microphone access was denied. Allow microphone access in your browser."
            : event.error === "no-speech" ? "No speech detected. Try again."
            : "Voice recognition unavailable. You can type instead.";
          voiceStatus.textContent = reason;
        };
        recognition.onend = () => {
          listening = false; micBtn.textContent = "🎤 Speak";
          micBtn.setAttribute("aria-pressed", "false");
          if (voiceStatus.textContent.startsWith("Listening"))
            voiceStatus.textContent = inp.value.trim() ? "Question captured. Press Ask Copilot." : "Press Speak to try again.";
        };
        recognition.start();
      } catch (err) {
        listening = false; micBtn.textContent = "🎤 Speak";
        voiceStatus.textContent = "Microphone could not start. Check browser permissions or type your question.";
      }
    }
  }, "🎤 Speak");
  if (!SpeechRecognitionClass) micBtn.title = "Voice dictation requires a browser with SpeechRecognition support";
  const attachReadButton = (parent, content) => {
    if (!("speechSynthesis" in window)) return;
    parent.append(h("button", { class: "btn fq-voice-read", type: "button", "aria-label": "Read Copilot answer aloud", onclick: () => {
      window.speechSynthesis.cancel();
      const utterance = new SpeechSynthesisUtterance(String(content || "").slice(0, 2500));
      utterance.lang = navigator.language || "en-US";
      window.speechSynthesis.speak(utterance);
    } }, "🔊 Read answer"));
  };

  const examples = b.examples || [
    "Which carrier is cheapest to Rotterdam?",
    "How many shipments are on customs hold?",
    "Show quotes below 10% margin",
    "Which ports are most congested?",
    "Show storms near the Red Sea"
  ];
  const chips = h("div", { class: "chat-suggestions" },
    ...examples.map(ex => h("span", {
      class: "chat-chip",
      onclick: () => send(ex)
    }, ex))
  );

  const showMsg = m => {
    if (m.role === "user") {
      msgs.append(h("div", { class: "m u" }, m.text));
    } else {
      const o = h("div", { class: "md" });
      o.innerHTML = md(m.text);
      const a = h("div", { class: "m a" }, o);
      if (m.sql) {
        a.append(h("details", { class: "cites" },
          h("summary", null, `Show exact SQL query (${m.rows !== undefined ? m.rows : 0} rows)`),
          h("pre", null, m.sql)
        ));
      }
      attachReadButton(a, m.text);
      msgs.append(a);
    }
    msgs.scrollTop = msgs.scrollHeight;
  };

  const send = async q => {
    q = q.trim();
    if (!q) return;
    inp.value = "";
    hist.push({ role: "user", text: q });
    showMsg({ role: "user", text: q });

    const a = h("div", { class: "m a" });
    const loadingText = h("div", { style: "color:var(--mut);display:flex;align-items:center;gap:8px;" },
      h("div", { class: "spin", style: "width:16px;height:16px;border-width:2px;margin:0;" }),
      "Querying database…"
    );
    a.append(loadingText);
    msgs.append(a);
    msgs.scrollTop = msgs.scrollHeight;

    let received = false;
    await sse("/api/copilot", { q }, ev => {
      if (ev.ev === "final") {
        received = true;
        a.replaceChildren();
        const o = h("div", { class: "md" });
        o.innerHTML = md(ev.text);
        a.append(o);
        if (ev.sql) {
          a.append(h("details", { class: "cites" },
            h("summary", null, `Show exact SQL query (${ev.rows !== undefined ? ev.rows : 0} rows)`),
            h("pre", null, ev.sql)
          ));
        }
        attachReadButton(a, ev.text);
        hist.push({ role: "assistant", text: ev.text, sql: ev.sql, rows: ev.rows });
        msgs.scrollTop = msgs.scrollHeight;
      }
    });
    if (!received) {
      loadingText.textContent = "No response received.";
    }
  };

  const inputBar = h("div", { class: "cin" },
    inp,
    micBtn,
    h("button", { class: "btn primary", onclick: () => send(inp.value) }, "Ask Copilot")
  );

  el.append(chips, msgs, inputBar, voiceStatus);
  return { el, update() {} };
};

// ---------------------------------------------------------------- Admin Dashboard Block
R.admin_page = b => {
  const el = h("div", { class: "admin-container" });
  const metrics = b.metrics || { total_users: 0, locked_users: 0, roles: {} };
  const currentAdmin = b.current_admin || (S.me ? S.me.user.username : "");

  const rolesSummary = Object.entries(metrics.roles || {}).map(([r, c]) => `${r}: ${c}`).join(" · ");

  // 1. KPI Cards
  const kpiGrid = h("div", { class: "admin-kpi-grid" },
    h("div", { class: "admin-kpi-card" },
      h("span", { class: "admin-kpi-title" }, "Total Platform Accounts"),
      h("span", { class: "admin-kpi-val" }, String(metrics.total_users || 0)),
      h("span", { class: "admin-kpi-sub" }, rolesSummary || "All registered users")
    ),
    h("div", { class: "admin-kpi-card" },
      h("span", { class: "admin-kpi-title" }, "Active Accounts"),
      h("span", { class: "admin-kpi-val", style: "color:var(--green);" }, String((metrics.total_users || 0) - (metrics.locked_users || 0))),
      h("span", { class: "admin-kpi-sub" }, "Sign-in enabled")
    ),
    h("div", { class: "admin-kpi-card" },
      h("span", { class: "admin-kpi-title" }, "Locked Accounts"),
      h("span", { class: "admin-kpi-val", style: "color:" + ((metrics.locked_users || 0) > 0 ? "var(--red)" : "var(--mut)") + ";" }, String(metrics.locked_users || 0)),
      h("span", { class: "admin-kpi-sub" }, "Access suspended")
    )
  );

  // 2. Add User Form
  const addAlert = h("div", { class: "fq-alert", style: "display:none;" });
  const newU = h("input", { class: "in", placeholder: "e.g. operations_lead", required: true });
  const newE = h("input", { class: "in", placeholder: "e.g. user@freight.ai", type: "email", required: true });
  const newP = h("input", { class: "in", placeholder: "Temporary strong password", type: "password", required: true });
  let showPw = false;
  const pwToggle = h("button", {
    type: "button", class: "fq-pw-toggle", onclick: () => {
      showPw = !showPw;
      newP.type = showPw ? "text" : "password";
      pwToggle.textContent = showPw ? "Hide" : "Show";
    }
  }, "Show");

  const newR = h("select", { class: "in" },
    h("option", { value: "Customer" }, "Customer"),
    h("option", { value: "Freight Broker" }, "Freight Broker"),
    h("option", { value: "Operations Manager" }, "Operations Manager"),
    h("option", { value: "Auditor" }, "Auditor"),
    h("option", { value: "Admin" }, "Admin")
  );
  const newC = h("input", { class: "in", placeholder: "Optional (e.g. CUST-002)" });
  const addBtn = h("button", { class: "btn primary", type: "submit" }, "Create User Account");

  const addForm = h("form", {
    onsubmit: async e => {
      e.preventDefault();
      addAlert.style.display = "none";
      addBtn.disabled = true;
      addBtn.textContent = "Creating account…";

      const r = await api("/api/admin/users/create", {
        json: {
          username: newU.value.trim(),
          email: newE.value.trim(),
          password: newP.value,
          role: newR.value,
          customer_id: newC.value.trim() || null
        }
      });
      addBtn.disabled = false;
      addBtn.textContent = "Create User Account";

      if (r.ok) {
        toast(r.msg || "User created successfully.", false);
        loadPage(false);
      } else {
        addAlert.className = "fq-alert fq-alert-error";
        addAlert.textContent = r.msg || "Failed to create user.";
        addAlert.style.display = "block";
      }
    }
  },
    h("div", { style: "display:grid;grid-template-columns:repeat(auto-fit, minmax(200px, 1fr));gap:16px;margin-bottom:16px;" },
      h("div", { class: "fld" }, h("label", null, "Username"), newU),
      h("div", { class: "fld" }, h("label", null, "Email Address"), newE),
      h("div", { class: "fld" }, h("label", null, "RBAC Role"), newR),
      h("div", { class: "fld" }, h("label", null, "Customer ID (Optional)"), newC),
      h("div", { class: "fld", style: "grid-column: 1 / -1;" },
        h("label", null, "Temporary Password"),
        h("div", { class: "profile-input-wrap" }, newP, pwToggle),
        h("small", { style: "color:var(--mut);font-size:11.5px;margin-top:4px;" }, "Requires 8+ characters, uppercase, lowercase, digit, and special symbol.")
      )
    ),
    addAlert,
    addBtn
  );

  const addSection = h("div", { class: "admin-section" },
    h("div", { class: "admin-section-header" },
      h("h3", { class: "admin-section-title" }, "Provision New User Account"),
      h("div", { class: "admin-section-sub" }, "Create platform credentials and assign authorization role.")
    ),
    addForm
  );

  // 2b. Link Customer Account (map a public signup to an existing freight customer profile)
  const linkUsers = (b.users || []).filter(u => u.role === "Customer" && !u.customer_id);
  const linkCusts = b.customers || [];
  const linkAlert = h("div", { class: "fq-alert", style: "display:none;" });
  const linkU = h("select", { class: "in" },
    ...linkUsers.map(u => h("option", { value: String(u.id) }, `${u.username} - ${u.email}`)));
  const linkC = h("select", { class: "in" },
    ...linkCusts.map(c => h("option", { value: c.customer_id }, `${c.customer_id} - ${c.name || ""}`)));
  const linkBtn = h("button", { class: "btn primary", type: "submit" }, "Link Account");
  const linkForm = h("form", {
    onsubmit: async e => {
      e.preventDefault();
      linkAlert.style.display = "none";
      linkBtn.disabled = true;
      linkBtn.textContent = "Linking account…";
      const r = await api("/api/admin/link_customer", { json: { user_id: linkU.value, customer_id: linkC.value } });
      linkBtn.disabled = false;
      linkBtn.textContent = "Link Account";
      if (r.ok) {
        toast(r.msg || "Account linked.", false);
        loadPage(false);
      } else {
        linkAlert.className = "fq-alert fq-alert-error";
        linkAlert.textContent = r.msg || "Failed to link account.";
        linkAlert.style.display = "block";
      }
    }
  },
    h("div", { style: "display:grid;grid-template-columns:repeat(auto-fit, minmax(200px, 1fr));gap:16px;margin-bottom:16px;" },
      h("div", { class: "fld" }, h("label", null, "Pending Customer Account"), linkU),
      h("div", { class: "fld" }, h("label", null, "Corporate Freight Profile"), linkC)
    ),
    linkAlert,
    linkBtn
  );
  const linkSection = h("div", { class: "admin-section" },
    h("div", { class: "admin-section-header" },
      h("h3", { class: "admin-section-title" }, "Link Customer Account"),
      h("div", { class: "admin-section-sub" }, "Connect a signed-up user to their freight data.")
    ),
    (linkUsers.length && linkCusts.length)
      ? linkForm
      : h("div", { style: "color:var(--mut);font-size:13px;" }, "No unlinked customer accounts are waiting to be linked.")
  );

  // 3. User Table Section
  const modalContainer = h("div", null);
  const usersRows = (b.users || []).map(u => {
    const isSelf = u.username === currentAdmin;

    const roleSelect = h("select", {
      class: "admin-select",
      onchange: async e => {
        const nextRole = e.target.value;
        const res = await api("/api/admin/users/role", { json: { target_id: u.id, role: nextRole } });
        if (res.ok) {
          toast(res.msg || "Role updated.", false);
          loadPage(false);
        } else {
          toast(res.msg || "Failed to update role.", true);
          e.target.value = u.role;
        }
      }
    },
      h("option", { value: "Customer", selected: u.role === "Customer" }, "Customer"),
      h("option", { value: "Freight Broker", selected: u.role === "Freight Broker" }, "Freight Broker"),
      h("option", { value: "Operations Manager", selected: u.role === "Operations Manager" }, "Operations Manager"),
      h("option", { value: "Auditor", selected: u.role === "Auditor" }, "Auditor"),
      h("option", { value: "Admin", selected: u.role === "Admin" }, "Admin")
    );

    const lockBtn = h("button", {
      class: "admin-btn-action " + (u.locked ? "admin-btn-unlock" : "admin-btn-lock"),
      disabled: isSelf,
      onclick: async () => {
        const actionUrl = u.locked ? "/api/admin/users/unlock" : "/api/admin/users/lock";
        const res = await api(actionUrl, { json: { target_id: u.id } });
        if (res.ok) {
          toast(res.msg || "Status updated.", false);
          loadPage(false);
        } else {
          toast(res.msg || "Failed to change lock status.", true);
        }
      }
    }, u.locked ? "Unlock" : "Lock");

    const deleteBtn = h("button", {
      class: "admin-btn-action admin-btn-delete",
      disabled: isSelf,
      onclick: () => {
        const modal = h("div", { class: "admin-modal-overlay", onclick: e => { if (e.target === modal) modal.remove(); } },
          h("div", { class: "admin-modal-card" },
            h("h3", { class: "admin-modal-title" }, "Confirm Permanent Deletion"),
            h("p", { class: "admin-modal-desc" }, `Permanently delete account ${u.username} (${u.email})? This action cannot be undone.`),
            h("div", { class: "admin-modal-acts" },
              h("button", { class: "btn ghost sm", onclick: () => modal.remove() }, "Cancel"),
              h("button", {
                class: "btn danger sm",
                onclick: async () => {
                  modal.remove();
                  const res = await api("/api/admin/users/delete", { json: { target_id: u.id } });
                  if (res.ok) {
                    toast(res.msg || "User deleted.", false);
                    loadPage(false);
                  } else {
                    toast(res.msg || "Failed to delete user.", true);
                  }
                }
              }, "Confirm Delete")
            )
          )
        );
        modalContainer.replaceChildren(modal);
      }
    }, "Delete");

    const rolePill = h("span", { class: "pill " + (u.role === "Admin" ? "Info" : u.role === "Operations Manager" ? "Low" : u.role === "Freight Broker" ? "Approved" : u.role === "Auditor" ? "proposed" : "Warning") }, u.role);
    const statusPill = h("span", { class: "pill " + (u.locked ? "Critical" : "Approved") }, u.locked ? "Locked" : "Active");

    return h("tr", null,
      h("td", { style: "font-weight:700;" }, u.username, isSelf ? h("span", { style: "margin-left:6px;font-size:11px;color:var(--mut);" }, "(You)") : null),
      h("td", null, u.email),
      h("td", null, rolePill),
      h("td", null, statusPill),
      h("td", { style: "color:var(--mut);" }, u.customer_id || "—"),
      h("td", { style: "color:var(--mut);font-size:12px;" }, u.created_at ? u.created_at.slice(0, 16) : "—"),
      h("td", { style: "color:var(--mut);font-size:12px;" }, u.last_login ? u.last_login.slice(0, 16) : "Never"),
      h("td", null, h("div", { class: "admin-actions-cell" }, roleSelect, lockBtn, deleteBtn))
    );
  });

  const usersTable = h("div", { class: "admin-table-wrap" },
    h("table", { class: "admin-table" },
      h("thead", null,
        h("tr", null,
          h("th", null, "Username"),
          h("th", null, "Email"),
          h("th", null, "Role"),
          h("th", null, "Status"),
          h("th", null, "Customer ID"),
          h("th", null, "Joined Date"),
          h("th", null, "Last Login"),
          h("th", null, "Actions")
        )
      ),
      h("tbody", null, ...usersRows)
    )
  );

  const usersSection = h("div", { class: "admin-section" },
    h("div", { class: "admin-section-header" },
      h("h3", { class: "admin-section-title" }, "Platform Accounts & Access Control"),
      h("div", { class: "admin-section-sub" }, "Manage roles, lock accounts, or remove users. Self-protection and last-admin rules enforced.")
    ),
    usersTable
  );

  // 4. Activity Log Table
  const actRows = (b.activity || []).map(a => h("tr", null,
    h("td", { style: "font-family:monospace;font-size:12px;color:var(--mut);" }, a.timestamp ? a.timestamp.slice(0, 19).replace("T", " ") : "—"),
    h("td", { style: "font-weight:600;" }, a.actor),
    h("td", null, h("span", { class: "pill Info", style: "font-family:monospace;font-size:11px;" }, a.action)),
    h("td", null, a.details)
  ));

  const activitySection = h("div", { class: "admin-section" },
    h("div", { class: "admin-section-header" },
      h("h3", { class: "admin-section-title" }, "Admin Activity Audit Log (Latest 50 Events)"),
      h("div", { class: "admin-section-sub" }, "Immutable record of user creations, promotions, demotions, locks, and removals.")
    ),
    h("div", { class: "admin-table-wrap" },
      h("table", { class: "admin-table" },
        h("thead", null,
          h("tr", null,
            h("th", null, "Timestamp (UTC)"),
            h("th", null, "Admin Actor"),
            h("th", null, "Action"),
            h("th", null, "Details")
          )
        ),
        h("tbody", null, ...actRows)
      )
    )
  );

  // 5. RBAC Permission Matrix Reference
  const permMatrixSection = h("div", { class: "admin-section" },
    h("div", { class: "admin-section-header" },
      h("h3", { class: "admin-section-title" }, "Role-Based Access Control (RBAC) Privilege Matrix"),
      h("div", { class: "admin-section-sub" }, "Module authorization mapping across platform roles.")
    ),
    h("div", { class: "admin-table-wrap" },
      h("table", { class: "admin-table", style: "text-align:center;" },
        h("thead", null,
          h("tr", null,
            h("th", { style: "text-align:left;" }, "Role"),
            h("th", null, "Copilot"),
            h("th", null, "Route AI"),
            h("th", null, "Spot Quotes"),
            h("th", null, "Carriers"),
            h("th", null, "Weather"),
            h("th", null, "Margin"),
            h("th", null, "Customs"),
            h("th", null, "Run Agents"),
            h("th", null, "Manage Users")
          )
        ),
        h("tbody", null,
          h("tr", null, h("td", { style: "text-align:left;font-weight:700;" }, "Admin"), h("td", null, "✓"), h("td", null, "✓"), h("td", null, "✓"), h("td", null, "✓"), h("td", null, "✓"), h("td", null, "✓"), h("td", null, "✓"), h("td", null, "✓"), h("td", null, "✓")),
          h("tr", null, h("td", { style: "text-align:left;font-weight:700;" }, "Operations Manager"), h("td", null, "✓"), h("td", null, "✓"), h("td", { style: "color:var(--mut);" }, "✗"), h("td", null, "✓"), h("td", null, "✓"), h("td", { style: "color:var(--mut);" }, "✗"), h("td", null, "✓"), h("td", null, "✓"), h("td", { style: "color:var(--mut);" }, "✗")),
          h("tr", null, h("td", { style: "text-align:left;font-weight:700;" }, "Freight Broker"), h("td", null, "✓"), h("td", { style: "color:var(--mut);" }, "✗"), h("td", null, "✓"), h("td", null, "✓"), h("td", { style: "color:var(--mut);" }, "✗"), h("td", null, "✓"), h("td", { style: "color:var(--mut);" }, "✗"), h("td", { style: "color:var(--mut);" }, "✗"), h("td", { style: "color:var(--mut);" }, "✗")),
          h("tr", null, h("td", { style: "text-align:left;font-weight:700;" }, "Auditor (Read-Only)"), h("td", null, "✓"), h("td", null, "✓"), h("td", null, "✓"), h("td", null, "✓"), h("td", null, "✓"), h("td", null, "✓"), h("td", null, "✓"), h("td", { style: "color:var(--mut);" }, "✗"), h("td", { style: "color:var(--mut);" }, "✗")),
          h("tr", null, h("td", { style: "text-align:left;font-weight:700;" }, "Customer (Scoped)"), h("td", null, "✓"), h("td", { style: "color:var(--mut);" }, "✗"), h("td", null, "✓ (Own)"), h("td", { style: "color:var(--mut);" }, "✗"), h("td", { style: "color:var(--mut);" }, "✗"), h("td", { style: "color:var(--mut);" }, "✗"), h("td", { style: "color:var(--mut);" }, "✗"), h("td", { style: "color:var(--mut);" }, "✗"), h("td", { style: "color:var(--mut);" }, "✗"))
        )
      )
    )
  );

  el.append(kpiGrid, addSection, linkSection, usersSection, activitySection, permMatrixSection, modalContainer);
  return { el, update() {} };
};

// ---------------------------------------------------------------- Profile Page Block
R.profile_page = b => {
  const u = b.user || (S.me ? S.me.user : {});
  const el = h("div", { class: "profile-container" });

  const initial = (u.username || "U")[0].toUpperCase();
  const roleColor = u.role === "Admin" ? "var(--blue)" : u.role === "Operations Manager" ? "var(--teal)" : u.role === "Freight Broker" ? "var(--green)" : u.role === "Auditor" ? "var(--violet)" : "var(--amber)";

  const avatarEl = h("div", { class: "profile-avatar", style: "background:linear-gradient(135deg, " + roleColor + ", var(--pri));box-shadow:0 6px 18px rgba(2,132,199,0.3);" }, initial);
  const titleEl = h("h2", null, u.username);
  const emailEl = h("p", null, u.email);
  const badgeEl = h("span", { class: "profile-role-badge", style: "background:rgba(2,132,199,0.12);color:" + roleColor + ";border-color:currentColor;" }, u.role);

  const headerCard = h("div", { class: "profile-header-card" },
    h("div", { class: "profile-header-top" },
      avatarEl,
      h("div", { class: "profile-info" }, titleEl, emailEl, badgeEl)
    ),
    h("div", { class: "profile-meta-grid" },
      h("div", { class: "profile-meta-item" }, h("span", { class: "profile-meta-label" }, "Account Role"), h("span", { class: "profile-meta-val" }, u.role)),
      h("div", { class: "profile-meta-item" }, h("span", { class: "profile-meta-label" }, "Customer Identifier"), h("span", { class: "profile-meta-val" }, u.customer_id || "None (Internal User)")),
      h("div", { class: "profile-meta-item" }, h("span", { class: "profile-meta-label" }, "Joined Date"), h("span", { class: "profile-meta-val" }, u.created_at ? u.created_at.slice(0, 10) : "—")),
      h("div", { class: "profile-meta-item" }, h("span", { class: "profile-meta-label" }, "Last Active Session"), h("span", { class: "profile-meta-val" }, u.last_login ? u.last_login.slice(0, 16).replace("T", " ") : "Active now"))
    )
  );

  // Form 1: Username Update
  const unAlert = h("div", { class: "fq-alert", style: "display:none;" });
  const newUsernameInput = h("input", { class: "in", type: "text", value: u.username, required: true });
  const unPwInput = h("input", { class: "in", type: "password", placeholder: "Enter current password to verify", required: true });
  let showUnPw = false;
  const unPwToggle = h("button", {
    type: "button", class: "fq-pw-toggle", onclick: () => {
      showUnPw = !showUnPw;
      unPwInput.type = showUnPw ? "text" : "password";
      unPwToggle.textContent = showUnPw ? "Hide" : "Show";
    }
  }, "Show");
  const unBtn = h("button", { class: "btn primary", type: "submit" }, "Update Username");

  const unForm = h("form", {
    onsubmit: async e => {
      e.preventDefault();
      unAlert.style.display = "none";
      unBtn.disabled = true;
      unBtn.textContent = "Updating…";

      const r = await api("/api/profile/username", {
        json: {
          new_username: newUsernameInput.value.trim(),
          current_password: unPwInput.value
        }
      });
      unBtn.disabled = false;
      unBtn.textContent = "Update Username";

      if (r.ok && r.token) {
        localStorage.setItem("fq_token", r.token);
        S.me.user.username = r.user.username;
        titleEl.textContent = r.user.username;
        avatarEl.textContent = r.user.username[0].toUpperCase();
        renderNav();
        toast("Username updated successfully!", false);
        unAlert.className = "fq-alert fq-alert-success";
        unAlert.textContent = r.msg || "Username updated successfully.";
        unAlert.style.display = "block";
        unPwInput.value = "";
      } else {
        unAlert.className = "fq-alert fq-alert-error";
        unAlert.textContent = r.msg || "Could not update username.";
        unAlert.style.display = "block";
      }
    }
  },
    h("div", { class: "card" },
      h("h3", null, "Change Username"),
      h("div", { style: "display:grid;gap:14px;max-width:440px;" },
        h("div", { class: "fld" }, h("label", null, "New Username"), newUsernameInput),
        h("div", { class: "fld" }, h("label", null, "Current Password"), h("div", { class: "profile-input-wrap" }, unPwInput, unPwToggle)),
        unAlert,
        unBtn
      )
    )
  );

  // Form 2: Password Update
  const pwAlert = h("div", { class: "fq-alert", style: "display:none;" });
  const curPwInput = h("input", { class: "in", type: "password", placeholder: "Current password", required: true });
  const newPwInput = h("input", { class: "in", type: "password", placeholder: "New strong password", required: true });
  const confPwInput = h("input", { class: "in", type: "password", placeholder: "Confirm new password", required: true });
  let showCur = false, showNew = false;
  const curToggle = h("button", { type: "button", class: "fq-pw-toggle", onclick: () => { showCur = !showCur; curPwInput.type = showCur ? "text" : "password"; curToggle.textContent = showCur ? "Hide" : "Show"; } }, "Show");
  const newToggle = h("button", { type: "button", class: "fq-pw-toggle", onclick: () => { showNew = !showNew; newPwInput.type = showNew ? "text" : "password"; newToggle.textContent = showNew ? "Hide" : "Show"; } }, "Show");
  const pwBtn = h("button", { class: "btn primary", type: "submit" }, "Update Password");

  const pwForm = h("form", {
    onsubmit: async e => {
      e.preventDefault();
      pwAlert.style.display = "none";
      pwBtn.disabled = true;
      pwBtn.textContent = "Updating password…";

      const r = await api("/api/profile/password", {
        json: {
          current_password: curPwInput.value,
          new_password: newPwInput.value,
          confirm_password: confPwInput.value
        }
      });
      pwBtn.disabled = false;
      pwBtn.textContent = "Update Password";

      if (r.ok) {
        toast("Password changed successfully!", false);
        pwAlert.className = "fq-alert fq-alert-success";
        pwAlert.textContent = r.msg || "Password updated successfully.";
        pwAlert.style.display = "block";
        curPwInput.value = "";
        newPwInput.value = "";
        confPwInput.value = "";
      } else {
        pwAlert.className = "fq-alert fq-alert-error";
        pwAlert.textContent = r.msg || "Failed to update password.";
        pwAlert.style.display = "block";
      }
    }
  },
    h("div", { class: "card" },
      h("h3", null, "Security Credentials"),
      h("div", { style: "display:grid;gap:14px;max-width:440px;" },
        h("div", { class: "fld" }, h("label", null, "Current Password"), h("div", { class: "profile-input-wrap" }, curPwInput, curToggle)),
        h("div", { class: "fld" }, h("label", null, "New Password"), h("div", { class: "profile-input-wrap" }, newPwInput, newToggle)),
        h("div", { class: "fld" }, h("label", null, "Confirm New Password"), confPwInput),
        h("small", { style: "color:var(--mut);font-size:11.5px;" }, "Requires 8+ characters, uppercase, lowercase, digit, and special symbol."),
        pwAlert,
        pwBtn
      )
    )
  );

  el.append(headerCard, unForm, pwForm);
  return { el, update() {} };
};

// ---------------------------------------------------------------- Navigation & Command Shell
const GROUPS = [
  ["Assistant", ["copilot"]],
  ["Operations", ["route", "quotes", "carriers", "weather", "margin", "customs"]],
  ["Account", ["profile", "admin"]]
];

function renderNav() {
  const nav = $("#nav");
  if (!nav || !S.me) return;
  const items = [];
  const used = new Set();
  const mk = m => h("div", {
    class: "nav" + (m.id === S.page ? " on" : ""),
    "data-id": m.id,
    onclick: () => UI.go(m.id)
  },
    icon(m.icon || "bot"),
    h("span", null, m.label.replace(/^Agent \d+ · /, "")),
    h("em", { class: "bd", style: "display:none" })
  );

  GROUPS.forEach(([g, ids]) => {
    const ms = (S.me.menu || []).filter(m => ids.includes(m.id));
    if (!ms.length) return;
    items.push(h("div", { class: "ngrp" }, g));
    ms.forEach(m => { used.add(m.id); items.push(mk(m)); });
  });
  (S.me.menu || []).filter(m => !used.has(m.id)).forEach(m => items.push(mk(m)));
  nav.replaceChildren(...items);
  updBadges();
}

function updBadges() {
  const p = S.pulse;
  if (!p) return;
  const hl = $("#hl");
  if (hl) {
    const state = p.worker_alive ? (p.health == null ? "Checking health" : `Health ${Math.round(p.health)}%`) : "Starting";
    hl.innerHTML = `<span class="live-dot"></span>Fleet engine · ${state}`;
  }
}

function openStream() {
  if (S.es) S.es.close();
  let active = true;
  const update = async () => {
    if (!active || !S.me) return;
    try {
      const p = await api("/api/pulse");
      if (!active || !p || p.ok === false || p.stage === undefined) return;
      const prev = S.pulse;
      S.pulse = p;
      updBadges();
      if (prev && p.dec !== prev.dec && S.live) loadPage(true);
    } catch (e) {}
  };
  const timer = setInterval(update, 8000);
  S.es = { close() { active = false; clearInterval(timer); } };
  update();
}

// ---------------------------------------------------------------- UI Global Methods
const UI = {};

UI.openEntity = async id => {
  if (!id) return;
  $("#dt").textContent = id;
  $("#db").replaceChildren(h("div", { class: "boot", style: "min-height:200px;display:flex;align-items:center;justify-content:center;gap:10px;" }, h("div", { class: "spin" }), "Loading details…"));
  $("#drawer").classList.add("on");
  const b = $("#drawer-backdrop");
  if (b) b.classList.add("on");

  const d = await api("/api/detail/" + encodeURIComponent(id));
  if (d.title === undefined) {
    $("#db").replaceChildren(h("div", { class: "callout warn" }, d.msg || "Details not accessible for this account role."));
    return;
  }
  $("#dt").textContent = d.title || id;
  $("#db").replaceChildren();
  mountList($("#db"), d.blocks || [], "d");
};

UI.closeDrawer = () => {
  const d = $("#drawer");
  const b = $("#drawer-backdrop");
  if (d) d.classList.remove("on");
  if (b) b.classList.remove("on");
};

UI.go = (id, tab, params) => {
  S.page = id;
  S.tab = tab || null;
  S.params = params || {};
  window.location.hash = "#/app/" + id + (tab ? ":" + tab : "");
  UI.closeDrawer();
  renderNav();
  loadPage(false);
};

UI.palette = () => {
  if ($(".kpal")) return;
  let sel = 0, rows = [];
  const inp = h("input", { placeholder: "Jump to module, entity (e.g. QTE-001, SHP-…), or question…" });
  const lst = h("div", { class: "lst" });
  const ov = h("div", { class: "kpal", onclick: e => { if (e.target === ov) close(); } }, h("div", null, inp, lst));

  const close = () => { ov.remove(); document.removeEventListener("keydown", kd, true); };
  const build = () => {
    const q = inp.value.trim(), ql = q.toLowerCase();
    rows = [];
    (S.me ? S.me.menu : []).filter(m => !ql || m.label.toLowerCase().includes(ql) || m.id.includes(ql)).forEach(m => {
      rows.push({ g: "Modules", t: m.label.replace(/^Agent \d+ · /, ""), s: "navigate", f: () => UI.go(m.id) });
    });
    if (/^[A-Za-z]{2,5}-?[A-Za-z]?\d+$/.test(q)) {
      rows.unshift({ g: "Entity", t: q.toUpperCase(), s: "inspect", f: () => UI.openEntity(q.toUpperCase()) });
    }
    if (q.length > 2) {
      rows.push({
        g: "Copilot", t: "Ask Copilot: " + q, s: "query", f: () => {
          UI.go("copilot");
          setTimeout(() => {
            const i = document.querySelector(".cin input");
            if (i) { i.value = q; i.dispatchEvent(new KeyboardEvent("keydown", { key: "Enter" })); }
          }, 500);
        }
      });
    }
    if (sel >= rows.length) sel = Math.max(0, rows.length - 1);
    let last = "";
    lst.replaceChildren(...rows.flatMap((r, i) => {
      const e = [];
      if (r.g !== last) { last = r.g; e.push(h("div", { class: "gl" }, r.g)); }
      e.push(h("div", { class: "it" + (i === sel ? " on" : ""), onclick: () => { close(); r.f(); } }, r.t, h("small", null, r.s)));
      return e;
    }));
  };

  const kd = e => {
    if (e.key === "Escape") close();
    else if (e.key === "ArrowDown") { sel = Math.min(rows.length - 1, sel + 1); build(); e.preventDefault(); }
    else if (e.key === "ArrowUp") { sel = Math.max(0, sel - 1); build(); e.preventDefault(); }
    else if (e.key === "Enter" && document.activeElement === inp) { const r = rows[sel]; if (r) { close(); r.f(); } }
  };

  document.addEventListener("keydown", kd, true);
  inp.addEventListener("input", () => { sel = 0; build(); });
  document.body.append(ov);
  build();
  inp.focus();
};

document.addEventListener("keydown", e => {
  if (e.key === "Escape") {
    UI.closeDrawer();
    const m = $(".admin-modal-overlay");
    if (m) m.remove();
  }
  if ((e.ctrlKey || e.metaKey) && e.key.toLowerCase() === "k" && S.me) {
    e.preventDefault();
    UI.palette();
  }
});

// ---------------------------------------------------------------- Page Loader
async function loadPage(silent) {
  const seq = ++S.seq;
  clearTimeout(S.timer);
  const q = new URLSearchParams();
  if (S.tab) q.set("tab", S.tab);
  if (Object.keys(S.params || {}).length) q.set("p", JSON.stringify(S.params));

  if (!silent) {
    S.tree = null;
    $("#blocks").replaceChildren(
      h("div", { class: "boot", style: "min-height:240px;display:flex;align-items:center;justify-content:center;gap:12px;" },
        h("div", { class: "spin" }),
        h("span", { style: "color:var(--mut);font-weight:600;" }, "Loading module data…")
      )
    );
  }

  const d = await api(`/api/page/${S.page}?` + q);
  if (seq !== S.seq) return;

  if (d.blocks === undefined) {
    $("#blocks").replaceChildren(h("div", { class: "callout error" }, d.msg || d.detail || "Could not load module data."));
    return;
  }

  $("#ttl").textContent = d.title || "";
  $("#sub").textContent = d.sub || "";
  document.title = (d.title || "") + " · " + (S.me ? S.me.name : "FreightQuote AI");

  // Tabs
  const tabs = $("#tabs");
  if (d.tabs && d.tabs.length) {
    tabs.style.display = "flex";
    tabs.replaceChildren(
      ...d.tabs.map(x => h("b", {
        class: (S.tab || d.tabs[0].id) === x.id ? "on" : "",
        onclick: () => {
          S.params = {};
          UI.go(S.page, x.id);
        }
      }, x.label))
    );
  } else {
    tabs.style.display = "none";
    tabs.replaceChildren();
  }

  // Actions
  $("#acts").replaceChildren(
    ...(d.actions || []).map(a => h("button", {
      class: "btn " + (a.tone || "primary"),
      onclick: async e => {
        e.target.disabled = true;
        const r = await api(a.url, { json: {} });
        toast(r.msg, !r.ok);
        e.target.disabled = false;
        reloadPage(true);
      }
    }, a.label))
  );

  const box = $("#blocks");
  try {
    const blocks = Array.isArray(d.blocks) ? d.blocks : [];
    if (!S.tree || silent === false || !S.tree.ok || S.tree.id !== S.page || S.tree.tab !== S.tab) {
      box.replaceChildren();
      S.tree = { hs: mountList(box, blocks, "p"), ok: 1, id: S.page, tab: S.tab };
    } else {
      S.tree.hs = updList(box, S.tree.hs, blocks, "p");
    }
    if (!blocks.length) {
      box.replaceChildren(h("div", { class: "card fq-empty-module" },
        h("h3", null, "Nothing to show yet"),
        h("p", null, "This section will appear when there is data to display.")));
    }
  } catch (err) {
    console.error("FreightQuote: page render error", err);
    S.tree = null;
    box.replaceChildren(h("div", { class: "card fq-empty-module", role: "alert" },
      h("h3", null, "We couldn't open this page"),
      h("p", null, "Please try again. If the issue continues, check the application log."),
      h("button", { class: "btn primary", onclick: () => loadPage(false) }, "Try again")));
  }

  S.live = d.live || 0;
  sched();
}

function sched() {
  clearTimeout(S.timer);
  if (S.live && !document.hidden) {
    S.timer = setTimeout(() => loadPage(true), S.live * 1000);
  }
}

function reloadPage(soft) {
  loadPage(!!soft);
}

// ---------------------------------------------------------------- Application Shell Mount
let shellMounted = false;

function mountAppShell() {
  const app = $("#app");
  const signOutBtn = h("button", {
    class: "btn sm danger",
    style: "margin-top:12px;width:100%;",
    onclick: () => {
      localStorage.removeItem("fq_token");
      S.me = null;
      shellMounted = false;
      toast("Signed out successfully.", false);
      window.location.hash = "#/";
    }
  }, "Sign out");

  app.replaceChildren(
    h("div", { class: "side" },
      h("div", { class: "brand" },
        h("i", null, icon("radar")),
        h("div", null,
          "FreightQuote AI",
          h("small", null, S.me.user.role + " · " + S.me.user.username)
        )
      ),
      h("div", { class: "ksearch", onclick: () => UI.palette() }, "Quick search…", h("kbd", null, "Ctrl K")),
      h("div", { id: "nav" }),
      h("div", { style: "flex:1;" }),
      h("button", {
        class: "fq-theme-toggle",
        type: "button",
        onclick: (e) => {
          const enabled = !document.body.classList.contains("fq-dark");
          document.body.classList.toggle("fq-dark", enabled);
          localStorage.setItem("fq_dark", enabled ? "1" : "0");
          e.currentTarget.textContent = enabled ? "☀  Light appearance" : "☾  Dark appearance";
        }
      }, localStorage.getItem("fq_dark") === "1" ? "☀  Light appearance" : "☾  Dark appearance"),
      signOutBtn
    ),
    h("div", { class: "main" },
      h("div", { class: "top" },
        h("div", null, h("h1", { id: "ttl" }), h("div", { class: "sub", id: "sub" })),
        h("div", { style: "flex:1;" }),
        h("div", { id: "acts", style: "display:flex;gap:8px;" }),
        h("span", { class: "chip live-chip", id: "hl" },
          h("span", { class: "live-dot" }),
          "Live Fleet Engine"
        )
      ),
      h("div", { class: "content" },
        h("div", { class: "tabs", id: "tabs" }),
        h("div", { id: "pg" }, h("div", { class: "blocks", id: "blocks" }))
      )
    )
  );
  openStream();
}

async function renderApp(pageId, tabId) {
  const app = $("#app");
  if (!app) return;

  const token = localStorage.getItem("fq_token");
  if (!token) {
    shellMounted = false;
    window.location.hash = "#/login";
    return;
  }

  if (!shellMounted || !S.me) {
    const m = await api("/api/me");
    if (!m || !m.user) {
      localStorage.removeItem("fq_token");
      shellMounted = false;
      window.location.hash = "#/login";
      return;
    }
    S.me = m;
    mountAppShell();
    shellMounted = true;
  }

  document.body.classList.toggle("fq-dark", localStorage.getItem("fq_dark") === "1");

  const allowedMenu = S.me.menu || [];
  const validPage = allowedMenu.find(item => item.id === pageId);
  if (!validPage) {
    S.page = null;
    $("#ttl").textContent = "Access Denied";
    $("#sub").textContent = "403 Forbidden - Role Permission Required";
    $("#tabs").style.display = "none";
    $("#acts").replaceChildren();
    $("#blocks").replaceChildren(
      h("div", { class: "callout error", style: "margin: 20px 0; padding: 24px;" },
        h("h3", { style: "margin-top:0; color:#ef4444;" }, "Access Denied (403)"),
        h("p", null, `Your role (${S.me.user.role}) does not have permission to access the '${pageId}' module. Please choose an authorized module from the sidebar.`)
      )
    );
    renderNav();
    return;
  }

  S.page = pageId;
  S.tab = tabId || null;
  UI.closeDrawer();
  renderNav();
  loadPage(false);
}

// ---------------------------------------------------------------- Landing Page View
let cachedLandingHTML = null;

async function renderLanding() {
  const app = $("#app");
  shellMounted = false;
  if (S.es) { S.es.close(); S.es = null; }
  document.body.classList.remove("fq-dark");

  if (cachedLandingHTML) {
    app.innerHTML = cachedLandingHTML;
    bindLandingEvents(app);
    return;
  }

  try {
    const response = await fetch("/static/landing_exact.html");
    if (!response.ok) throw new Error("Landing page unavailable");
    cachedLandingHTML = await response.text();
    app.innerHTML = cachedLandingHTML;
    bindLandingEvents(app);
  } catch (err) {
    app.replaceChildren(h("div", { class: "boot", style: "padding:60px;text-align:center;" },
      h("h2", null, "FreightQuote AI"),
      h("p", { style: "color:var(--mut);" }, "Unable to load landing page."),
      h("button", { class: "btn primary", onclick: renderLanding }, "Retry")
    ));
  }
}

function bindLandingEvents(app) {
  const root = app.querySelector(".fq-exact-page") || app;
  root.querySelectorAll(".fq-access-login").forEach(el => el.addEventListener("click", () => { location.hash = "#/login"; }));
  root.querySelectorAll(".fq-access-create").forEach(el => el.addEventListener("click", () => { location.hash = "#/signup"; }));
  root.querySelectorAll(".fq-nav-links a").forEach(el => el.addEventListener("click", event => {
    event.preventDefault();
    const id = el.getAttribute("href").replace(/^#\/?/, "");
    history.replaceState(null, "", "#/" + id);
    root.querySelector("#" + id)?.scrollIntoView({ behavior: "smooth", block: "start" });
  }));
  const section = location.hash.replace(/^#\/?/, "");
  if (["platform", "how-it-works", "security", "faq"].includes(section)) {
    requestAnimationFrame(() => root.querySelector("#" + section)?.scrollIntoView({ behavior: "instant", block: "start" }));
  } else {
    scrollTo(0, 0);
  }
}

// ---------------------------------------------------------------- Auth Views
function authHero(title, desc) {
  return h("div", { class: "fq-auth-hero" },
    h("img", { class: "fq-auth-hero-img", src: "/static/login-hero.webp", onerror: (e) => { e.target.src = "/static/ship.jpg"; }, alt: "Maritime cargo vessel" }),
    h("div", { class: "fq-auth-hero-gradient" }),
    h("div", { class: "fq-auth-hero-content" },
      h("h1", { class: "fq-auth-hero-title" }, title),
      h("p", { class: "fq-auth-hero-desc" }, desc)
    )
  );
}

function renderLogin() {
  const app = $("#app");
  shellMounted = false;
  if (S.es) { S.es.close(); S.es = null; }
  document.body.classList.remove("fq-dark");

  const alertBox = h("div", { class: "fq-alert", style: "display:none;" });
  const unInput = h("input", { class: "in", type: "text", placeholder: "Username or email address", required: true });
  const pwInput = h("input", { class: "in", type: "password", placeholder: "Enter password", required: true });
  // The role is assigned by an administrator, not granted by this selector.
  const roleInput = h("select", { class: "in", required: true, "aria-label": "Assigned account role" },
    h("option", { value: "", disabled: true, selected: true }, "Select your assigned role"),
    ...["Admin", "Operations Manager", "Freight Broker", "Auditor", "Customer"].map(role => h("option", { value: role }, role))
  );

  let showPw = false;
  const pwToggle = h("button", {
    type: "button", class: "fq-pw-toggle", onclick: () => {
      showPw = !showPw;
      pwInput.type = showPw ? "text" : "password";
      pwToggle.textContent = showPw ? "Hide" : "Show";
    }
  }, "Show");

  const submitBtn = h("button", { class: "fq-btn-primary", type: "submit" }, "Sign in to Workspace");

  const form = h("form", {
    onsubmit: async e => {
      e.preventDefault();
      alertBox.style.display = "none";
      submitBtn.disabled = true;
      submitBtn.textContent = "Authenticating…";

      const res = await api("/api/auth/login", {
        method: "POST",
        json: { identity: unInput.value.trim(), password: pwInput.value, role: roleInput.value }
      });

      submitBtn.disabled = false;
      submitBtn.textContent = "Sign in to Workspace";

      if (res.ok && res.token) {
        localStorage.setItem("fq_token", res.token);
        toast("Sign in successful!", false);
        window.location.hash = "#/app/route";
      } else {
        alertBox.className = "fq-alert fq-alert-error";
        alertBox.textContent = res.msg || "Invalid username/email or password.";
        alertBox.style.display = "block";
      }
    }
  },
    h("div", { style: "display:grid;gap:16px;" },
      h("div", { class: "fld" }, h("label", null, "Account Identifier"), unInput),
      h("div", { class: "fld" }, h("label", null, "Assigned Role"), roleInput,
        h("small", { style: "color:var(--mut);" }, "Use the role assigned to your account by an administrator.")),
      h("div", { class: "fld" },
        h("div", { style: "display:flex;justify-content:space-between;align-items:center;" },
          h("label", null, "Password"),
          h("a", { href: "#/forgot", style: "font-size:12px;" }, "Forgot password?")
        ),
        h("div", { class: "profile-input-wrap" }, pwInput, pwToggle)
      ),
      alertBox,
      submitBtn,
      h("div", { style: "text-align:center;font-size:13px;color:var(--mut);margin-top:10px;" },
        h("a", { href: "#/verify-email", style: "font-weight:600;" }, "Verify your email"),
        h("span", null, " · "),
        "Don't have an account? ",
        h("a", { href: "#/signup", style: "font-weight:600;" }, "Create customer account")
      )
    )
  );

  const card = h("div", { class: "fq-auth-card" },
    h("div", { style: "display:flex;align-items:center;justify-content:space-between;margin-bottom:20px;" },
      h("div", { style: "display:flex;align-items:center;gap:10px;cursor:pointer;", onclick: () => { window.location.hash = "#/"; } },
        h("div", { class: "brand", style: "padding:0;border:none;margin:0;" }, h("i", null, icon("radar"))),
        h("b", { style: "font-size:16px;" }, "FreightQuote AI")
      ),
      h("a", { href: "#/", class: "fq-btn-outline", style: "padding:6px 12px;font-size:12px;" }, "Home")
    ),
    h("div", { class: "fq-auth-header" },
      h("h2", { class: "fq-auth-title" }, "Sign in"),
      h("p", { class: "fq-auth-sub" }, "Access your freight logistics workspace.")
    ),
    form
  );

  app.replaceChildren(h("div", { class: "fq-auth-wrap" },
    authHero("Ocean Freight Intelligence", "Automate spot quotes, track carrier reliability, and analyze voyage congestion with verified data."),
    h("div", { class: "fq-auth-card-wrap" }, card)
  ));
}

function renderSignup() {
  const app = $("#app");
  shellMounted = false;
  if (S.es) { S.es.close(); S.es = null; }
  document.body.classList.remove("fq-dark");

  const alertBox = h("div", { class: "fq-alert", style: "display:none;" });
  const unInput = h("input", { class: "in", type: "text", placeholder: "Choose username", required: true });
  const emailInput = h("input", { class: "in", type: "email", placeholder: "e.g. shipper@company.com", required: true });
  const pwInput = h("input", { class: "in", type: "password", placeholder: "Choose strong password", required: true });
  const confPwInput = h("input", { class: "in", type: "password", placeholder: "Confirm password", required: true });

  const questionSelect = h("select", { class: "in" },
    h("option", { value: "What was the name of your first school?" }, "What was the name of your first school?"),
    h("option", { value: "In which city were you born?" }, "In which city were you born?"),
    h("option", { value: "What was the name of your first pet?" }, "What was the name of your first pet?"),
    h("option", { value: "What is the middle name of your oldest sibling?" }, "What is the middle name of your oldest sibling?")
  );
  const ansInput = h("input", { class: "in", type: "text", placeholder: "Your security answer", required: true });

  const submitBtn = h("button", { class: "fq-btn-primary", type: "submit" }, "Create Customer Account");

  const form = h("form", {
    onsubmit: async e => {
      e.preventDefault();
      alertBox.style.display = "none";
      submitBtn.disabled = true;
      submitBtn.textContent = "Creating account…";

      const res = await api("/api/auth/signup", {
        method: "POST",
        json: {
          username: unInput.value.trim(),
          email: emailInput.value.trim(),
          password: pwInput.value,
          confirm_password: confPwInput.value,
          question: questionSelect.value,
          answer: ansInput.value.trim(),
          role: "Customer"
        }
      });

      submitBtn.disabled = false;
      submitBtn.textContent = "Create Customer Account";

      if (res.ok) {
        toast("Account created. Verify your email to continue.", false);
        sessionStorage.setItem("fq_verify_email", emailInput.value.trim());
        window.location.hash = "#/verify-email";
      } else {
        alertBox.className = "fq-alert fq-alert-error";
        alertBox.textContent = res.msg || "Registration failed.";
        alertBox.style.display = "block";
      }
    }
  },
    h("div", { style: "display:grid;gap:14px;" },
      h("div", { class: "fld" }, h("label", null, "Username"), unInput),
      h("div", { class: "fld" }, h("label", null, "Email Address"), emailInput),
      h("div", { class: "fld" }, h("label", null, "Password"), pwInput),
      h("div", { class: "fld" }, h("label", null, "Confirm Password"), confPwInput),
      h("small", { style: "color:var(--mut);font-size:11.5px;" }, "Password needs 8+ chars, uppercase, lowercase, digit, and symbol."),
      h("div", { class: "fld" }, h("label", null, "Security Question"), questionSelect),
      h("div", { class: "fld" }, h("label", null, "Security Answer"), ansInput),
      alertBox,
      submitBtn,
      h("div", { style: "text-align:center;font-size:13px;color:var(--mut);margin-top:10px;" },
        "Already have an account? ",
        h("a", { href: "#/login", style: "font-weight:600;" }, "Sign in")
      )
    )
  );

  const card = h("div", { class: "fq-auth-card" },
    h("div", { style: "display:flex;align-items:center;justify-content:space-between;margin-bottom:20px;" },
      h("div", { style: "display:flex;align-items:center;gap:10px;cursor:pointer;", onclick: () => { window.location.hash = "#/"; } },
        h("div", { class: "brand", style: "padding:0;border:none;margin:0;" }, h("i", null, icon("radar"))),
        h("b", { style: "font-size:16px;" }, "FreightQuote AI")
      ),
      h("a", { href: "#/", class: "fq-btn-outline", style: "padding:6px 12px;font-size:12px;" }, "Home")
    ),
    h("div", { class: "fq-auth-header" },
      h("h2", { class: "fq-auth-title" }, "Register"),
      h("p", { class: "fq-auth-sub" }, "Create a verified customer shipper account.")
    ),
    form
  );

  app.replaceChildren(h("div", { class: "fq-auth-wrap" },
    authHero("Maritime Logistics Platform", "Access instant rate calculations, customs documentation checks, and live voyage monitoring."),
    h("div", { class: "fq-auth-card-wrap" }, card)
  ));
}

function renderVerifyEmail() {
  const app = $("#app");
  shellMounted = false;
  if (S.es) { S.es.close(); S.es = null; }
  document.body.classList.remove("fq-dark");
  const email = h("input", { class: "in", type: "email", placeholder: "Registered email address", required: true, value: sessionStorage.getItem("fq_verify_email") || "" });
  const code = h("input", { class: "in", type: "text", inputmode: "numeric", maxlength: "6", pattern: "[0-9]{6}", placeholder: "Six-digit verification code", required: true });
  const alert = h("div", { class: "fq-alert", style: "display:none" });
  const sendBtn = h("button", { class: "fq-btn-outline", type: "button" }, "Send verification code");
  const verifyBtn = h("button", { class: "fq-btn-primary", type: "submit" }, "Verify email");
  const show = (msg, ok=false) => { alert.style.display="block"; alert.className="fq-alert " + (ok?"fq-alert-success":"fq-alert-error"); alert.textContent=msg; };
  let cooldown = 0;
  sendBtn.onclick = async () => {
    sendBtn.disabled=true;
    const r = await api("/api/auth/verify-email/request", {method:"POST",json:{email:email.value.trim()}});
    show(r.msg || (r.ok?"Code sent":"Unable to send code"),!!r.ok);
    if(r.ok) sessionStorage.setItem("fq_verify_email",email.value.trim());
    cooldown = Number(r.retry_after_seconds||0);
    if(cooldown>0){sendBtn.textContent=`Resend in ${cooldown}s`;const tick=setInterval(()=>{if(!sendBtn.isConnected){clearInterval(tick);return;}cooldown--;if(cooldown<=0){clearInterval(tick);sendBtn.disabled=false;sendBtn.textContent="Resend code";}else sendBtn.textContent=`Resend in ${cooldown}s`;},1000);} else sendBtn.disabled=false;
  };
  const form=h("form",{onsubmit:async e=>{e.preventDefault();verifyBtn.disabled=true;const r=await api("/api/auth/verify-email/confirm",{method:"POST",json:{email:email.value.trim(),code:code.value.trim()}});verifyBtn.disabled=false;if(r.ok){sessionStorage.removeItem("fq_verify_email");toast("Email verified. Sign in to continue.",false);location.hash="#/login";}else show(r.msg||"Verification failed");}},
    h("div",{style:"display:grid;gap:15px"},h("div",{class:"fld"},h("label",null,"Email address"),email),h("div",{class:"fld"},h("label",null,"Six-digit code"),code),sendBtn,alert,verifyBtn,h("a",{href:"#/login",style:"text-align:center"},"Back to Sign In")));
  app.replaceChildren(h("div",{class:"fq-auth-wrap"},authHero("Verify Your Email","Your account is almost ready. Check your inbox for the verification code."),h("div",{class:"fq-auth-card-wrap"},h("div",{class:"fq-auth-card"},h("div",{class:"fq-auth-header"},h("h2",{class:"fq-auth-title"},"Email Verification"),h("p",{class:"fq-auth-sub"},"Verify your address before signing in.")),form))));
}

function renderForgot() {
  const app=$("#app"); shellMounted=false;
  if(S.es){S.es.close();S.es=null;}
  document.body.classList.remove("fq-dark");
  let method="otp", grant=null, verifiedIdentity="", currentIdentity="", cooldown=0;
  const card=h("div",{class:"fq-auth-card"});
  const draw=()=>{
    const alert=h("div",{class:"fq-alert",style:"display:none"});
    const show=(msg,ok=false)=>{alert.style.display="block";alert.className="fq-alert "+(ok?"fq-alert-success":"fq-alert-error");alert.textContent=msg;};
    if(grant){
      const pw=h("input",{class:"in",type:"password",required:true,placeholder:"New password"});
      const confirm=h("input",{class:"in",type:"password",required:true,placeholder:"Confirm new password"});
      const btn=h("button",{type:"submit",class:"fq-btn-primary"},"Reset Password");
      const form=h("form",{onsubmit:async e=>{e.preventDefault();btn.disabled=true;const r=await api("/api/auth/reset",{method:"POST",json:{identity:verifiedIdentity,grant,new_password:pw.value,confirm_password:confirm.value}});btn.disabled=false;if(r.ok){toast("Password updated. Sign in with your new password.",false);location.hash="#/login";}else show(r.msg||"Password reset failed.");}},
        h("div",{style:"display:grid;gap:15px"},h("div",{class:"fld"},h("label",null,"New password"),pw),h("div",{class:"fld"},h("label",null,"Confirm new password"),confirm),h("small",{style:"color:var(--mut)"},"8+ characters, uppercase, lowercase, digit, symbol, no spaces."),alert,btn));
      card.replaceChildren(h("div",{class:"fq-auth-header"},h("h2",{class:"fq-auth-title"},"Set New Password"),h("p",{class:"fq-auth-sub"},"Verification successful. Choose a new password.")),form);return;
    }
    const otpChoice=h("input",{type:"radio",name:"fq-recovery-method",value:"otp",checked:method==="otp"});
    const questionChoice=h("input",{type:"radio",name:"fq-recovery-method",value:"question",checked:method==="question"});
    for(const [button,kind] of [[otpChoice,"otp"],[questionChoice,"question"]]) button.addEventListener("change",()=>{method=kind;currentIdentity="";draw();});
    const id=h("input",{class:"in",type:method==="otp"?"email":"text",required:true,placeholder:method==="otp"?"Registered email address":"Username or email",value:currentIdentity});
    id.addEventListener("input",()=>{currentIdentity=id.value.trim();});
    const code=h("input",{class:"in",type:"text",inputmode:"numeric",maxlength:"6",placeholder:"Six-digit code",required:true});
    const answer=h("input",{class:"in",type:"password",placeholder:"Your security answer",required:true});
    const question=h("div",{style:"display:none;font-weight:600;padding:12px;background:var(--line-subtle);border-radius:9px"});
    let questionLoaded = false;
    let questionLocked = false;
    const loadQuestion = async () => {
      const identity = id.value.trim();
      if (!identity) { show("Enter your username or email first."); return; }
      question.textContent = "Finding your security question…";
      question.style.display = "block";
      const r = await api("/api/auth/forgot/question", { method: "POST", json: { identity } });
      if (identity !== id.value.trim() || !question.isConnected) return;
      questionLoaded = !!(r.ok && r.question);
      questionLocked = !!r.locked;
      question.textContent = questionLoaded ? r.question : "Security question unavailable.";
      answer.disabled = !questionLoaded || questionLocked;
      submit.disabled = !questionLoaded || questionLocked;
      if (questionLocked) show(r.msg || "Account locked. Contact the administrator to unlock it.");
      else if (!questionLoaded) show(r.msg || "Unable to find security question.");
      else alert.style.display = "none";
    };
    if (method === "question") {
      answer.disabled = true;
      id.addEventListener("blur", loadQuestion);
      id.addEventListener("input", () => {
        questionLoaded = false; questionLocked = false;
        question.style.display = "none";
        answer.disabled = true;
        submit.disabled = true;
        alert.style.display = "none";
      });
    }
    const showQuestion = h("button", { type: "button", class: "fq-btn-outline", onclick: loadQuestion }, "Show security question");
    const send=h("button",{type:"button",class:"fq-btn-outline"},"Send email code");
    send.addEventListener("click",async()=>{send.disabled=true;const r=await api("/api/auth/forgot/request_otp",{method:"POST",json:{email:id.value.trim()}});show(r.msg||"Unable to send code",!!r.ok);cooldown=Number(r.retry_after_seconds||0);if(cooldown>0){send.textContent=`Resend in ${cooldown}s`;const tick=setInterval(()=>{if(!send.isConnected){clearInterval(tick);return;}cooldown--;if(cooldown<=0){clearInterval(tick);send.disabled=false;send.textContent="Resend code";}else send.textContent=`Resend in ${cooldown}s`;},1000);}else send.disabled=false;});
    const submit=h("button",{class:"fq-btn-primary",type:"submit",disabled:method==="question"},method==="otp"?"Verify email code":"Verify security answer");
    const form=h("form",{onsubmit:async e=>{e.preventDefault();submit.disabled=true;const identity=id.value.trim();const path=method==="otp"?"/api/auth/forgot/verify_otp":"/api/auth/forgot/verify_answer";const json=method==="otp"?{email:identity,code:code.value.trim()}:{identity,answer:answer.value};const r=await api(path,{method:"POST",json});submit.disabled=false;if(r.ok&&r.grant){grant=r.grant;verifiedIdentity=identity;draw();}else show(r.msg||"Verification failed.");}},
      h("div",{style:"display:grid;gap:15px"},
        h("div",{style:"display:flex;gap:18px;flex-wrap:wrap;align-items:center"},h("label",{style:"display:flex;gap:8px;align-items:center"},otpChoice,"Email OTP"),h("label",{style:"display:flex;gap:8px;align-items:center"},questionChoice,"Security question")),
        h("div",{class:"fld"},h("label",null,method==="otp"?"Registered email":"Username or email"),id),
        ...(method==="otp"?[send,h("div",{class:"fld"},h("label",null,"Six-digit verification code"),code)]:[showQuestion,question,h("div",{class:"fld"},h("label",null,"Security answer"),answer)]),
        alert,submit,h("a",{href:"#/login",style:"text-align:center"},"Back to Sign In")));
    card.replaceChildren(h("div",{class:"fq-auth-header"},h("h2",{class:"fq-auth-title"},"Password Recovery"),h("p",{class:"fq-auth-sub"},"Recover your account using email verification or a security question.")),form);
  };
  draw();app.replaceChildren(h("div",{class:"fq-auth-wrap"},authHero("Account Recovery","Recover access to your FreightQuote AI workspace."),h("div",{class:"fq-auth-card-wrap"},card)));
}

// ---------------------------------------------------------------- Hash Router
async function handleRouting() {
  const hash = window.location.hash || "#/";

  if (hash === "#/" || hash === "#" || hash === "") {
    renderLanding();
  } else if (hash.startsWith("#/platform") || hash.startsWith("#/how-it-works") || hash.startsWith("#/security") || hash.startsWith("#/faq")) {
    renderLanding();
  } else if (hash === "#/login") {
    renderLogin();
  } else if (hash === "#/signup") {
    renderSignup();
  } else if (hash === "#/forgot") {
    renderForgot();
  } else if (hash === "#/verify-email") {
    renderVerifyEmail();
  } else if (hash.startsWith("#/app/")) {
    const raw = hash.slice(6);
    const parts = raw.split(":");
    const pid = parts[0];
    const tab = parts[1] || null;
    await renderApp(pid, tab);
  } else {
    renderLanding();
  }
}

window.addEventListener("hashchange", handleRouting);
window.UI = UI;

// Bootstrapping
handleRouting();

In [ ]:
from pathlib import Path
import base64
Path('/content/app/static/freight_port.webp').write_bytes(base64.b64decode('...'))

# Original cargo ship used on the landing page; kept independent of the login artwork.
Path('/content/app/static/ship.jpg').write_bytes(base64.b64decode('...'))

# Dedicated cinematic ship artwork for the login panel.
Path('/content/app/static/login-hero.webp').write_bytes(base64.b64decode('...'))

# Dashboard-only background image (included offline for every Colab run).
Path('/content/app/static/dashboard-harbor.webp').write_bytes(base64.b64decode(
    '...'
))


In [ ]:
import os, sys, subprocess
echarts_file = '/content/app/static/vendor/echarts.min.js'
if (not os.path.exists(echarts_file) or os.path.getsize(echarts_file) == 0) and os.path.exists('/content/app/_vendor_bundle.py'):
    subprocess.run([sys.executable, '/content/app/_vendor_bundle.py'], check=True)
    print('Vendor assets unpacked successfully.')
else:
    print('Vendor assets already verified.')


In [ ]:
%%writefile /content/app/README.md
# FreightQuote AI — Milestone 2

A multi-user freight workspace built with FastAPI, SQLite, JWT, five-role RBAC, the six freight agents and a SQL-based Copilot. The application is intended to run in Google Colab for the internship demonstration.

## Run in Google Colab

1. Upload `FreightQuote_AI_Milestone2_Final_Deployment.ipynb` to Colab. Use a CPU runtime.
2. In **Secrets**, add `JWT_SECRET` (at least 32 characters; stable across restarts) and a strong `ADMIN_PASSWORD` (12–72 UTF-8 bytes). For email OTP, add `EMAIL_ADDRESS` and `EMAIL_PASSWORD` (a supported SMTP app password). Optionally add `ADMIN_EMAIL` and `NGROK_AUTHTOKEN`.
3. Select **Runtime → Run all** and open the link printed by the final cell.
4. Sign in with the configured administrator username (`admin` by default) and `ADMIN_PASSWORD`. Change or securely rotate initial credentials before sharing access.
5. Use Administration to create internal-role accounts. Public signup creates only Customer accounts. To show account-specific quotes, link the Customer to the correct `customer_id` as an administrator.

## Latest targeted fixes

- **Copilot voice:** Press **🎤 Speak**, allow microphone permission in Chrome/Edge on HTTPS or localhost, review the recognized text, then press Ask Copilot. The optional **🔊 Read answer** button reads a result aloud. If the browser does not support speech recognition, the typed question continues to work. Speech recognition may use the browser/provider's speech service; do not dictate confidential content without checking your organization's privacy policy.
- **Locked-account sign-in:** The exact server error, **Account locked. Contact the administrator to unlock it**, is displayed instead of the generic session-expired message.
- **Security-question recovery:** A locked user can still view their configured question and a clear lock notice, but cannot verify an answer or reset the password until an administrator unlocks the account.
- **Sign-in role:** The login form provides a role selector; it must match the role already assigned by an administrator. Choosing a role cannot grant or change permissions. Public signup remains Customer-only.
- **Security prerequisites:** There are no published default administrator passwords; JWT_SECRET and ADMIN_PASSWORD must be configured before launching.

## Permission matrix

| Role | Copilot | Route | Quotes | Carriers | Weather | Margin | Customs | Run agents | Manage users |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| Admin | ✓ | ✓ | ✓ | ✓ | ✓ | ✓ | ✓ | ✓ | ✓ |
| Operations Manager | ✓ | ✓ | — | ✓ | ✓ | — | ✓ | ✓ | — |
| Freight Broker | ✓ | — | ✓ | ✓ | — | ✓ | — | — | — |
| Auditor | ✓ | ✓ | ✓ | ✓ | ✓ | ✓ | ✓ | — | — |
| Customer | ✓ | — | Own only | — | — | — | — | — | — |

Permissions are enforced on the server, not merely by hiding navigation. Customer-facing quote data and relevant Copilot queries must be scoped to the linked customer account.

## Copilot examples to verify against the seeded dataset

- Which carrier is cheapest to Rotterdam?
- Which shipments have a high delay risk?
- Which shipments are on customs hold?
- Which ports are most congested?
- Show quotes with low margins.
- Show recent agent runs.
- Which carriers have the lowest on-time percentage?

Results must be obtained from the actual SQLite database. The engine includes 17 SQL query definitions and role-specific access checks. Record the live response, SQL statement, and row count during submission testing; sample records can change, so this README does not invent answers.

## Release validation checklist

Run `python -m pytest -q tests` from the release folder after installing dependencies. Live microphone capture, email delivery and Colab tunnel access must additionally be tested in your own browser and configured runtime; they cannot be fully validated by offline tests.

## Submission validation checklist

- Login, signup, forgot password with OTP and security question, Profile username and password updates
- Five test accounts, one per role; permitted and forbidden page/action requests (403)
- Admin create, role change, lock/unlock, delete, last-Admin protection, activity log
- Linked Customer sees only their own quotes
- Copilot query displays SQL and number of rows and enforces role scoping
- Run a quote through the seven-step pipeline, weather page after an injected storm, and verify background cycles
- Capture genuine screenshots of signup, login, Profile, Admin, allowed/denied cases, Copilot, quotes and weather
- Confirm no keys, tokens or databases are committed

## Deployment considerations

Colab sessions and `/content` storage are temporary; for continuous production hosting, migrate to a durable database and storage volume, deploy behind HTTPS, configure trusted proxy handling, secure SMTP, backups, logging, health checks, and an independent full end-to-end test. Seed records are demonstration data and must not be used as verified commercial quotations or authoritative customs guidance.

## Design

The landing-page sections and ship image are restored from the approved teammate design. A separate ocean-freight image is used for the responsive login and registration experience. The workspace uses a consistent maritime palette and offers light/dark appearance controls.


In [ ]:
# Preflight: validate generated application files before starting the web server.
from pathlib import Path
import py_compile
root = Path('/content/app')
required = [root/'server.py', root/'auth_service.py', root/'rbac.py',
            root/'static/app.js', root/'static/style.css', root/'static/index.html',
            root/'static/landing_exact.html', root/'static/ship.jpg', root/'static/login-hero.webp']
missing = [str(p) for p in required if not p.is_file() or p.stat().st_size == 0]
if missing:
    raise RuntimeError('Run the source and image cells first. Missing files: ' + ', '.join(missing))
# Prepare the map and routing data in case a previous Colab run only unpacked
# some of the vendor assets. The coastline file is required by Route AI.
import importlib.util
vendor = root/'static/vendor'
coastline = vendor/'world.topo.json'
if not coastline.is_file() or coastline.stat().st_size == 0:
    module_spec = importlib.util.spec_from_file_location('_fq_vendor_preflight', root/'_vendor_bundle.py')
    module = importlib.util.module_from_spec(module_spec)
    module_spec.loader.exec_module(module)
if not coastline.is_file() or coastline.stat().st_size == 0:
    raise RuntimeError('The bundled coastline file is missing; rerun the vendor-assets cell before launching.')
for source in root.glob('*.py'):
    py_compile.compile(str(source), doraise=True)
print('Preflight passed: application modules, interface and both images are present.')

In [ ]:
import os, sys, subprocess, time, urllib.request, html, secrets
from pathlib import Path
from IPython.display import display, HTML

_FQ_BUNDLE = (
    '...'
)
# Recover application sources if the runtime was reset or the setup cells were skipped.
# The archive contains the exact source and image files already present in this notebook.
from pathlib import Path
import base64 as _fq_b64, io as _fq_io, zipfile as _fq_zip, subprocess as _fq_sub, sys as _fq_sys
_fq_root = Path('/content/app')
_fq_files = ['server.py', 'auth_service.py', 'rbac.py', 'config.py', 'static/app.js', 'static/style.css', 'static/index.html', 'static/ship.jpg', 'static/login-hero.webp', 'static/dashboard-harbor.webp']
_fq_missing = [name for name in _fq_files if not (_fq_root / name).is_file()]
if _fq_missing:
    print('Restoring application files from the notebook after a runtime reset...')
    _fq_root.mkdir(parents=True, exist_ok=True)
    with _fq_zip.ZipFile(_fq_io.BytesIO(_fq_b64.b64decode(_FQ_BUNDLE))) as _fq_archive:
        _fq_archive.extractall(_fq_root)
if not (_fq_root/'static/vendor/world.topo.json').is_file():
    _fq_sub.run([_fq_sys.executable, str(_fq_root/'_vendor_bundle.py')], cwd=str(_fq_root), check=True)
if not (_fq_root/'server.py').is_file():
    raise RuntimeError('App sources could not be initialized; run all cells from the top.')
# Recovery also works when only this final cell is rerun after a Colab reset.
import sys, subprocess, importlib

# Use the Python environment Colab already provides (no virtualenv/ensurepip).
FQ_PYTHON = sys.executable
_FQ_DEPENDENCIES = (
    ('fastapi', 'fastapi>=0.110,<1'),
    ('uvicorn', 'uvicorn>=0.30,<1'),
    ('multipart', 'python-multipart>=0.0.9'),
    ('pandas', 'pandas>=2.1,<3'),
    ('numpy', 'numpy>=1.26,<3'),
    ('sklearn', 'scikit-learn>=1.4,<2'),
    ('jwt', 'PyJWT>=2.8,<3'),
    ('bcrypt', 'bcrypt>=4,<6'),
    ('pyngrok', 'pyngrok>=7,<8'),
    ('requests', 'requests>=2.31,<3'),
    ('scipy', 'scipy>=1.11,<2'),
    ('matplotlib', 'matplotlib>=3.8,<4'),
    ('reportlab', 'reportlab>=4,<5'),
)

def _fq_ensure_dependencies():
    missing = []
    for import_name, requirement in _FQ_DEPENDENCIES:
        try:
            importlib.import_module(import_name)
        except ImportError:
            missing.append(requirement)
    if missing:
        print('Installing missing packages into the existing Colab environment:', ', '.join(missing))
        installation = subprocess.run(
            [FQ_PYTHON, '-m', 'pip', 'install', '-q', '--disable-pip-version-check', *missing],
            stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True,
        )
        if installation.returncode:
            raise RuntimeError(
                'Package installation failed in the Colab environment. '
                'Check internet access and retry the cell. Pip details:\n' +
                installation.stderr[-3000:]
            )
        importlib.invalidate_caches()
        for import_name, _ in _FQ_DEPENDENCIES:
            importlib.import_module(import_name)
    return FQ_PYTHON

_fq_ensure_dependencies()
# Existing launch logic continues unchanged below.

FQ_PYTHON = sys.executable  # Colab's installed Python (never the failed virtualenv)

import socket
FQ_PORT = 8000
while FQ_PORT < 8100:
    with socket.socket() as probe:
        try:
            probe.bind(('127.0.0.1', FQ_PORT))
            break
        except OSError:
            FQ_PORT += 1
if FQ_PORT == 8100:
    raise RuntimeError('No available application port.')

def redacted(text):
    for key, value in os.environ.items():
        if any(word in key.upper() for word in ('SECRET', 'TOKEN', 'PASSWORD')) and value:
            text = text.replace(value, '[REDACTED]')
    return text

os.environ['APP_KIND'] = 'Freight'
os.environ['RATE_LIMIT_OFF'] = '0'
os.environ.setdefault('PERSIST_DIR', '/content/freightquote_data')

try:
    from google.colab import userdata
    for key in ['JWT_SECRET', 'ADMIN_PASSWORD', 'ADMIN_EMAIL', 'EMAIL_ADDRESS', 'EMAIL_PASSWORD', 'NGROK_AUTHTOKEN']:
        try:
            value = userdata.get(key)
            if value:
                os.environ[key] = value
        except Exception:
            pass
except ImportError:
    pass

# A stable secret is needed to preserve authenticated sessions after restarts.
if len(os.environ.get('JWT_SECRET', '')) < 32:
    raise RuntimeError('Configure JWT_SECRET in Colab Secrets (32+ characters) before starting the application.')

if not (12 <= len(os.environ.get('ADMIN_PASSWORD', '').encode()) <= 72):
    raise RuntimeError('Configure ADMIN_PASSWORD in Colab Secrets (12–72 UTF-8 bytes, with upper/lowercase letters, number and symbol).')

if not os.environ.get('ADMIN_EMAIL'):
    os.environ['ADMIN_EMAIL'] = "admin@infosys.com"

if globals().get('FQ_PROCESS') and FQ_PROCESS.poll() is None:
    FQ_PROCESS.terminate()
    FQ_PROCESS.wait(timeout=20)

check = subprocess.run([
    FQ_PYTHON, '-c',
    'import server, config, auth_service as a; assert not a.password_errors(config.ADMIN_PASSWORD), "ADMIN_PASSWORD needs uppercase, lowercase, digit and symbol"'
], cwd='/content/app', capture_output=True, text=True)

if check.returncode:
    print(redacted(check.stderr[-6000:]))
    raise RuntimeError('Application preflight failed; see the redacted error above.')

log_path = Path('/content/freightquote_server.log')
with log_path.open('w') as log:
    FQ_PROCESS = subprocess.Popen([
        FQ_PYTHON, '-m', 'uvicorn', 'server:app', '--host', '0.0.0.0', '--port', str(FQ_PORT)
    ], cwd='/content/app', stdout=log, stderr=log)

ready = False
for _ in range(90):
    if FQ_PROCESS.poll() is not None:
        break
    try:
        with urllib.request.urlopen(f'http://127.0.0.1:{FQ_PORT}/api/public', timeout=2) as response:
            ready = response.status == 200
        if ready:
            break
    except Exception:
        time.sleep(1)

if not ready:
    print(redacted(log_path.read_text()[-6000:]))
    raise RuntimeError('Server startup failed; see the redacted error above.')

# Verify the actual authenticated API before presenting a demo link.
# These are read-only requests; no sample accounts or quote data are changed.
import json as _fq_json
from urllib.error import HTTPError as _FQHTTPError

def _fq_request(path, token=None, payload=None):
    raw = _fq_json.dumps(payload).encode('utf-8') if payload is not None else None
    headers = {'Content-Type': 'application/json'} if raw is not None else {}
    if token:
        headers['Authorization'] = 'Bearer ' + token
    req = urllib.request.Request(f'http://127.0.0.1:{FQ_PORT}{path}', data=raw, headers=headers, method='POST' if raw is not None else 'GET')
    try:
        with urllib.request.urlopen(req, timeout=90) as res:
            return res.status, _fq_json.loads(res.read().decode('utf-8'))
    except _FQHTTPError as exc:
        try:
            return exc.code, _fq_json.loads(exc.read().decode('utf-8'))
        except Exception:
            return exc.code, {'msg': 'Non-JSON HTTP error'}

_login_status, _login_body = _fq_request('/api/auth/login', payload={
    'identity': os.environ.get('ADMIN_EMAIL', 'admin@infosys.com'),
    'password': os.environ['ADMIN_PASSWORD'], 'role': 'Admin'
})
if _login_status != 200 or not _login_body.get('token'):
    raise RuntimeError('Server started but administrator login failed. Check the ADMIN_PASSWORD secret and existing database account. Do not publish the demo link yet.')
_api_token = _login_body['token']
_failed = []
for _pid in ('admin','profile','copilot','route','quotes','carriers','weather','margin','customs'):
    _status, _body = _fq_request('/api/page/' + _pid, _api_token)
    if _status != 200 or not isinstance(_body.get('blocks'), list):
        _failed.append((_pid, _status, _body.get('request_id','none')))
if _failed:
    print('Pages failing preflight:', _failed)
    # The exception middleware saves tracebacks to the persistent data folder,
    # not to uvicorn's stdout log. Read that diagnostic when a page fails.
    error_log = Path(os.environ['PERSIST_DIR']) / 'errors.log'
    if error_log.exists():
        error_text = error_log.read_text(errors='replace')
        print('Server exception details (redacted):')
        print(redacted(error_text[-10000:]))
    else:
        print('No errors.log found in', os.environ['PERSIST_DIR'])
        print(redacted(log_path.read_text(errors='replace')[-5000:]))
    raise RuntimeError('Authenticated page checks failed; see the server exception above.')
print('Demo API preflight passed: login + all nine page endpoints.')

admin_email = os.environ.get('ADMIN_EMAIL', 'admin@infosys.com')
admin_pw = os.environ.get('ADMIN_PASSWORD', '')

try:
    from google.colab import output
    url = output.eval_js(f"google.colab.kernel.proxyPort({FQ_PORT}, {{'cache': false}})")
    display(HTML(f'''
    <div style="font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif; padding: 22px; background: linear-gradient(135deg, #0f172a 0%, #1e293b 100%); border: 1px solid #38bdf8; border-radius: 12px; margin: 16px 0; box-shadow: 0 10px 25px -5px rgba(0,0,0,0.3); color: #f8fafc;">
        <div style="display: flex; align-items: center; justify-content: space-between; margin-bottom: 14px;">
            <h3 style="margin: 0; color: #38bdf8; font-size: 18px; display: flex; align-items: center; gap: 10px;">
                <span style="display:inline-block; width:12px; height:12px; border-radius:50%; background:#10b981; box-shadow: 0 0 10px #10b981;"></span>
                FreightQuote AI · Server Online
            </h3>
            <span style="font-size: 12px; color: #94a3b8; background: #334155; padding: 3px 10px; border-radius: 9999px;">Milestone 2</span>
        </div>
        <p style="margin: 8px 0 16px;">
            <a href="{html.escape(url, quote=True)}" target="_blank" style="display: inline-block; padding: 11px 24px; background: linear-gradient(135deg, #0284c7 0%, #0369a1 100%); color: #ffffff; text-decoration: none; border-radius: 8px; font-weight: 600; font-size: 15px; box-shadow: 0 4px 14px rgba(2,132,199,0.4);">Launch Web Application &rarr;</a>
        </p>
        <div style="font-size: 13px; color: #cbd5e1; line-height: 1.7; background: rgba(15, 23, 42, 0.6); padding: 12px 16px; border-radius: 8px; border: 1px solid #334155;">
            <div><b>Default Administrator:</b> <code style="color: #38bdf8; background: #1e293b; padding: 2px 6px; border-radius: 4px;">{admin_email}</code></div>
            <div><b>Admin Password:</b> Set in Colab Secrets (not displayed)</div>
        </div>
    </div>
    '''))
except Exception:
    print(f'Server ready on port {FQ_PORT}.')
    print(f'Administrator: {admin_email} (password stored in Colab Secrets)')

os.environ['FQ_PORT'] = str(FQ_PORT)
if os.environ.get('NGROK_AUTHTOKEN'):
    code = '''import os, json
from pyngrok import ngrok, conf
conf.get_default().auth_token = os.environ['NGROK_AUTHTOKEN']
tunnels = ngrok.get_tunnels()
existing = next((t for t in tunnels if str(t.config.get('addr', '')).endswith(':' + os.environ['FQ_PORT'])), None)
print(existing.public_url if existing else ngrok.connect(int(os.environ['FQ_PORT'])).public_url)
'''
    tunnel = subprocess.run([FQ_PYTHON, '-c', code], capture_output=True, text=True)
    if tunnel.returncode == 0:
        url = tunnel.stdout.strip().splitlines()[-1]
        display(HTML(f'<p style="margin-top:8px;"><a href="{html.escape(url, quote=True)}" target="_blank" style="font-weight:600; color:#38bdf8;">Open public ngrok link &rarr;</a></p>'))
    else:
        print('Public tunnel unavailable. Local server is running.')